In [2]:
import sys
from pathlib import Path

# Start from the notebook's current working directory
p = Path.cwd()

src_path = None
for parent in [p] + list(p.parents):
    candidate = parent / "src"
    if candidate.exists():
        src_path = candidate
        break

if src_path is None:
    raise RuntimeError("Could not find 'src/' directory above current notebook path")

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print("Using src path:", src_path)


Using src path: C:\Users\phemm\orb_lab\src


In [2]:
import sys
print(sys.path[0])


C:\Users\phemm\orb_lab\src


In [1]:
import pandas as pd
import numpy as np

from stop_engine import StopState
from trade_stop_manager import TradeStopManager

# --- Synthetic intraday dataframe ---
idx = pd.date_range(
    "2025-01-07 09:30",
    periods=40,
    freq="1min",
    tz="US/Eastern",
)

np.random.seed(42)

df = pd.DataFrame(
    {
        "open": 100 + np.random.randn(len(idx)) * 0.2,
        "high": 100.5 + np.random.randn(len(idx)) * 0.2,
        "low": 99.5 + np.random.randn(len(idx)) * 0.2,
        "close": 100 + np.random.randn(len(idx)) * 0.2,
        "vwap": 100 + np.random.randn(len(idx)) * 0.1,
    },
    index=idx,
)

# --- Simulated entry context ---
entry_price = 100.00
atr = 1.00
direction = "LONG"
rr_desired = 2.0
entry_bar_idx = 10
vwap = df["vwap"].iloc[entry_bar_idx]

# --- Initialize manager ---
mgr = TradeStopManager(stop_state=StopState())

mgr.enter_trade(
    direction=direction,
    entry_price=entry_price,
    atr=atr,
    rr_desired=rr_desired,
    day_df=df,
    bar_idx=entry_bar_idx,
    vwap=vwap,
)

print("Stop type selected:", mgr.stop_type)
print("Achievable RR:", round(mgr.achievable_rr, 2))
print("Initial stop:", mgr.stop_state.current_stop)

# --- Walk bars forward ---
for i in range(entry_bar_idx + 1, len(df)):
    bar = df.iloc[i]

    should_exit, reason, px = mgr.process_bar(
        bar_high=bar["high"],
        bar_low=bar["low"],
        break_even_rr=0.5,
    )

    if should_exit:
        print(f"EXIT @ bar {i} ({df.index[i]})")
        print("  Reason:", reason)
        print("  Exit price:", px)
        break
else:
    print("Trade still open at end of data")


ModuleNotFoundError: No module named 'stop_engine'

In [3]:
import pandas as pd
import numpy as np

from stop_engine import StopState
from trade_stop_manager import TradeStopManager

# --- Synthetic intraday dataframe ---
idx = pd.date_range(
    "2025-01-07 09:30",
    periods=40,
    freq="1min",
    tz="US/Eastern",
)

np.random.seed(42)

df = pd.DataFrame(
    {
        "open": 100 + np.random.randn(len(idx)) * 0.2,
        "high": 100.5 + np.random.randn.randn(len(idx)) * 0.2,
        "low": 99.5 + np.random.randn(len(idx)) * 0.2,
        "close": 100 + np.random.randn(len(idx)) * 0.2,
        "vwap": 100 + np.random.randn(len(idx)) * 0.1,
    },
    index=idx,
)

# --- Simulated entry context ---
entry_price = 100.00
atr = 1.00
direction = "LONG"
rr_desired = 2.0
entry_bar_idx = 10
vwap = df["vwap"].iloc[entry_bar_idx]

# --- Initialize manager ---
mgr = TradeStopManager(stop_state=StopState())

mgr.enter_trade(
    direction=direction,
    entry_price=entry_price,
    atr=atr,
    rr_desired=rr_desired,
    day_df=df,
    bar_idx=entry_bar_idx,
    vwap=vwap,
)

print("Stop type selected:", mgr.stop_type)
print("Achievable RR:", round(mgr.achievable_rr, 2))
print("Initial stop:", mgr.stop_state.current_stop)

# --- Walk bars forward ---
for i in range(entry_bar_idx + 1, len(df)):
    bar = df.iloc[i]

    should_exit, reason, px = mgr.process_bar(
        bar_high=bar["high"],
        bar_low=bar["low"],
        break_even_rr=0.5,
    )

    if should_exit:
        print(f"EXIT @ bar {i} ({df.index[i]})")
        print("  Reason:", reason)
        print("  Exit price:", px)
        break
else:
    print("Trade still open at end of data")


ModuleNotFoundError: No module named 'stop_engine'

In [4]:
import sys
from pathlib import Path

print("CWD:", Path.cwd())
print("sys.path:")
for p in sys.path[:5]:
    print(" ", p)


CWD: C:\Users\phemm\orb_lab\notebooks
sys.path:
  C:\Users\phemm\orb_lab\src
  C:\Users\phemm\anaconda3\python313.zip
  C:\Users\phemm\anaconda3\DLLs
  C:\Users\phemm\anaconda3\Lib
  C:\Users\phemm\anaconda3


In [5]:
import os

print(os.listdir(r"C:\Users\phemm\orb_lab\src"))


['confluence_indicators.py', 'data_collector.py', 'orb_backtester.py', 'orb_core.py', 'orb_optimizer.py', 'single_day_tracer_v3.py', 'single_day_tracer_v4.py', 'src_trade_stop_manager.py', '__pycache__']


In [6]:
import os

print("Files in src/:")
for f in sorted(os.listdir(r"C:\Users\phemm\orb_lab\src")):
    if f.endswith(".py"):
        print(" ", f)


Files in src/:
  confluence_indicators.py
  data_collector.py
  orb_backtester.py
  orb_core.py
  orb_optimizer.py
  single_day_tracer_v3.py
  single_day_tracer_v4.py
  stop_engine.py
  stop_selector.py
  trade_stop_manager.py


In [7]:
# src/single_trade_runner.py
"""
Single Trade Runner
-------------------

Purpose:
    Execute ONE trade from entry to exit, bar-by-bar, using
    Pine-parity stop logic.

Scope:
    - Assumes entry signal already exists
    - Uses TradeStopManager to manage stops
    - Walks forward until exit

This module intentionally does NOT:
    - search for entries
    - manage multiple trades
    - calculate P&L
    - loop over symbols or days

This is a truth-validation layer, not a backtester.
"""

from __future__ import annotations

from typing import Dict, Optional
import pandas as pd

from stop_engine import StopState
from trade_stop_manager import TradeStopManager


def run_single_trade(
    *,
    df: pd.DataFrame,
    entry_bar_idx: int,
    direction: str,
    atr: float,
    rr_desired: float,
    break_even_rr: float = 0.5,
) -> Dict[str, Optional[object]]:
    """
    Run a single trade forward until exit.

    Args:
        df: intraday DataFrame (indexed by timestamp)
        entry_bar_idx: integer index of entry bar
        direction: 'LONG' or 'SHORT'
        atr: ATR value at entry (Pine-equivalent)
        rr_desired: desired R:R at entry
        break_even_rr: RR multiple to trigger break-even

    Returns:
        dict with entry/exit details
    """
    entry_ts = df.index[entry_bar_idx]
    entry_price = float(df["close"].iloc[entry_bar_idx])
    vwap = float(df["vwap"].iloc[entry_bar_idx])

    manager = TradeStopManager(stop_state=StopState())

    # Initialize trade
    manager.enter_trade(
        direction=direction,
        entry_price=entry_price,
        atr=atr,
        rr_desired=rr_desired,
        day_df=df,
        bar_idx=entry_bar_idx,
        vwap=vwap,
    )

    # Walk forward bar-by-bar
    for i in range(entry_bar_idx + 1, len(df)):
        bar = df.iloc[i]

        should_exit, reason, exit_price = manager.process_bar(
            bar_high=float(bar["high"]),
            bar_low=float(bar["low"]),
            break_even_rr=break_even_rr,
        )

        if should_exit:
            return {
                "entry_time": entry_ts,
                "entry_price": entry_price,
                "exit_time": df.index[i],
                "exit_price": exit_price,
                "exit_reason": reason,
                "stop_type": manager.stop_type,
                "achievable_rr": manager.achievable_rr,
            }

    # Trade still open at end of data
    return {
        "entry_time": entry_ts,
        "entry_price": entry_price,
        "exit_time": None,
        "exit_price": None,
        "exit_reason": None,
        "stop_type": manager.stop_type,
        "achievable_rr": manager.achievable_rr,
    }


In [2]:
import sys
from pathlib import Path

# Ensure src/ is on path (safe even if already present)
p = Path.cwd()
for parent in [p] + list(p.parents):
    candidate = parent / "src"
    if candidate.exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break

import pandas as pd
import numpy as np

from single_trade_runner import run_single_trade

# --- Synthetic intraday data ---
idx = pd.date_range(
    "2025-01-07 09:30",
    periods=50,
    freq="1min",
    tz="US/Eastern",
)

np.random.seed(123)

df = pd.DataFrame(
    {
        "open": 100 + np.random.randn(len(idx)) * 0.2,
        "high": 100.6 + np.random.randn(len(idx)) * 0.2,
        "low": 99.4 + np.random.randn(len(idx)) * 0.2,
        "close": 100 + np.random.randn(len(idx)) * 0.2,
        "vwap": 100 + np.random.randn(len(idx)) * 0.1,
    },
    index=idx,
)

# --- Simulated entry ---
entry_bar_idx = 10
atr = 1.0
rr_desired = 2.0
direction = "LONG"

result = run_single_trade(
    df=df,
    entry_bar_idx=entry_bar_idx,
    direction=direction,
    atr=atr,
    rr_desired=rr_desired,
    break_even_rr=0.5,
)

print("Single trade result:")
for k, v in result.items():
    print(f"  {k}: {v}")


SyntaxError: '{' was never closed (single_trade_runner.py, line 83)

In [2]:
import sys
from pathlib import Path

# Ensure src/ is on path (safe even if already present)
p = Path.cwd()
for parent in [p] + list(p.parents):
    candidate = parent / "src"
    if candidate.exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break

import pandas as pd
import numpy as np

from single_trade_runner import run_single_trade

# --- Synthetic intraday data ---
idx = pd.date_range(
    "2025-01-07 09:30",
    periods=50,
    freq="1min",
    tz="US/Eastern",
)

np.random.seed(123)

df = pd.DataFrame(
    {
        "open": 100 + np.random.randn(len(idx)) * 0.2,
        "high": 100.6 + np.random.randn(len(idx)) * 0.2,
        "low": 99.4 + np.random.randn(len(idx)) * 0.2,
        "close": 100 + np.random.randn(len(idx)) * 0.2,
        "vwap": 100 + np.random.randn(len(idx)) * 0.1,
    },
    index=idx,
)

# --- Simulated entry ---
entry_bar_idx = 10
atr = 1.0
rr_desired = 2.0
direction = "LONG"

result = run_single_trade(
    df=df,
    entry_bar_idx=entry_bar_idx,
    direction=direction,
    atr=atr,
    rr_desired=rr_desired,
    break_even_rr=0.5,
)

print("Single trade result:")
for k, v in result.items():
    print(f"  {k}: {v}")

Single trade result:
  entry_time: 2025-01-07 09:40:00-05:00
  entry_price: 100.00406319964692
  exit_time: 2025-01-07 09:41:00-05:00
  exit_price: 100.00406319964692
  exit_reason: BREAK-EVEN
  stop_type: SWING
  achievable_rr: 2.0


In [3]:
from massive_loader import load_one_day

try:
    load_one_day(symbol="AAPL", trade_date="2025-12-19")
except NotImplementedError as e:
    print("Expected:", e)


Expected: Massive API not wired yet — next step will add this safely.


In [4]:
import sys
from pathlib import Path

# Ensure src/ is on path
p = Path.cwd()
for parent in [p] + list(p.parents):
    candidate = parent / "src"
    if candidate.exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break

from massive_loader import load_one_day

df = load_one_day(symbol="AAPL", trade_date="2025-12-19")

print("Rows:", len(df))
print("TZ:", df.index.tz)
print("First bar:", df.index[0])
print("Last bar:", df.index[-1])
print("Columns:", df.columns.tolist())

print("\nRTH check:")
print(df.between_time("09:30", "09:35").head())


NotImplementedError: Massive API not wired yet — next step will add this safely.

In [1]:
from massive_loader import load_one_day

df = load_one_day(symbol="AAPL", trade_date="2025-12-19")

print(len(df))
print(df.index[0], df.index[-1])
print(df.columns)


ModuleNotFoundError: No module named 'massive_loader'

In [1]:
//@version=5
indicator("ATR Stop Debug", overlay=true)

atrPeriod = input.int(14, "ATR Period")
atrMultiplier = input.float(2.0, "ATR Stop Multiplier")

// RTH ticker for ATR
rthTicker = ticker.new(syminfo.prefix, syminfo.ticker, session.regular)
rthATR = request.security(rthTicker, timeframe.period, ta.rma(ta.tr(true), atrPeriod), barmerge.gaps_off, barmerge.lookahead_off)

// Entry at current close, test both directions
entryPrice = close

// ATR Stop calc - Pine line 1136
atrStopLong = entryPrice - (rthATR * atrMultiplier)
atrStopShort = entryPrice + (rthATR * atrMultiplier)

// Debug label
var label debugLabel = na
if barstate.islast
    txt = "═══ ATR STOP DEBUG ═══\n"
    txt += "Time: " + str.tostring(hour) + ":" + str.tostring(minute) + "\n"
    txt += "═══ INPUTS ═══\n"
    txt += "Entry: " + str.tostring(entryPrice, "#.##") + "\n"
    txt += "rthATR: " + str.tostring(rthATR, "#.####") + "\n"
    txt += "Multiplier: " + str.tostring(atrMultiplier, "#.##") + "\n"
    txt += "═══ ATR STOP ═══\n"
    txt += "LONG stop: " + str.tostring(atrStopLong, "#.##") + "\n"
    txt += "SHORT stop: " + str.tostring(atrStopShort, "#.##") + "\n"
    label.delete(debugLabel)
    debugLabel := label.new(bar_index, high, txt, style=label.style_label_lower_left, size=size.normal, textcolor=color.white, color=color.new(color.green, 20))

SyntaxError: invalid syntax (3521843183.py, line 1)

In [2]:
"""
ATR Stop Validation - Compare Python to Pine
Target: AAPL Dec 19 @ 10:19
"""
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
from orb_backtester import ORBBacktester
from orb_core import calc_atr_rth

# Load data
bt = ORBBacktester(symbol='AAPL', start_date='2025-12-19', end_date='2025-12-19', verbose=False)
bt.load_data()
df = bt.df

# Get 10:19 bar
ts = pd.Timestamp('2025-12-19 10:19', tz='US/Eastern')
bar = df.loc[ts]

# Entry = close
entry_price = bar['close']

# ATR from our calc
atr_series = calc_atr_rth(df, period=14)
rth_atr = atr_series.loc[ts]

# ATR stop calc - Pine line 1136
atr_multiplier = 2.0
atr_stop_long = entry_price - (rth_atr * atr_multiplier)
atr_stop_short = entry_price + (rth_atr * atr_multiplier)

print("═══ ATR STOP DEBUG (Python) ═══")
print(f"Time: 10:19")
print("═══ INPUTS ═══")
print(f"Entry: {entry_price:.2f}")
print(f"rthATR: {rth_atr:.4f}")
print(f"Multiplier: {atr_multiplier:.0f}")
print("═══ ATR STOP ═══")
print(f"LONG stop: {atr_stop_long:.2f}")
print(f"SHORT stop: {atr_stop_short:.2f}")
print()
print("═══ PINE VALUES ═══")
print("Entry: 271.59")
print("rthATR: 0.3155")
print("LONG stop: 270.96")
print("SHORT stop: 272.22")
print()
print("═══ MATCH? ═══")
print(f"Entry: {'✓' if abs(entry_price - 271.59) < 0.01 else '✗'}")
print(f"ATR: {'✓' if abs(rth_atr - 0.3155) < 0.001 else '✗'} (diff: {abs(rth_atr - 0.3155):.4f})")
print(f"LONG stop: {'✓' if abs(atr_stop_long - 270.96) < 0.01 else '✗'}")
print(f"SHORT stop: {'✓' if abs(atr_stop_short - 272.22) < 0.01 else '✗'}")


[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AAPL (1-min bars, 180 days, extended)...
  ✓ Loading from cache (6.7 hours old)
═══ ATR STOP DEBUG (Python) ═══
Time: 10:19
═══ INPUTS ═══
Entry: 271.59
rthATR: 0.3157
Multiplier: 2
═══ ATR STOP ═══
LONG stop: 270.96
SHORT stop: 272.22

═══ PINE VALUES ═══
Entry: 271.59
rthATR: 0.3155
LONG stop: 270.96
SHORT stop: 272.22

═══ MATCH? ═══
Entry: ✓
ATR: ✓ (diff: 0.0002)
LONG stop: ✓
SHORT stop: ✓


In [1]:
%run C:\Users\phemm\orb_lab\src\test_vwap_stop.py

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AAPL (1-min bars, 180 days, extended)...
  ⚠ Cache stale (0 hours), refreshing...
  ✓ Fetched 89528 bars, cached for next time
═══ VWAP STOP DEBUG (Python) ═══
Time: 10:19
═══ INPUTS ═══
Entry: 271.59
rthATR: 0.3157
VWAP: 272.13
Distance mult: 0.3
═══ VWAP STOP ═══
LONG stop: 272.04
SHORT stop: 272.23

═══ PINE VALUES ═══
Entry: 271.59
rthATR: 0.3377
VWAP: 272.16
LONG stop: 272.06
SHORT stop: 272.26

═══ MATCH? ═══
Entry: ✓
ATR: ✗ (Python: 0.3157, Pine: 0.3377, diff: 0.0220)
VWAP: ✗ (Python: 272.13)
LONG stop: ✗ (Python: 272.04)
SHORT stop: ✗ (Python: 272.23)


In [2]:
%run C:\Users\phemm\orb_lab\src\test_vwap_variance.py

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AAPL (1-min bars, 180 days, extended)...
  ✓ Loading from cache (0.2 hours old)
═══ VWAP VARIANCE CHECK ═══
AAPL Dec 19, 2025
Time        Polygon    Session       Diff     Diff %
09:35        272.31     272.31       0.00     0.000%
09:45        272.29     272.29       0.00     0.000%
10:00        272.21     272.21      -0.00    -0.000%
10:15        272.14     272.14       0.00     0.000%
10:19        272.13     272.13       0.00     0.000%
10:30        272.12     272.12       0.00     0.000%
11:00        272.03     272.03       0.00     0.000%
12:00        271.92     271.92      -0.00    -0.000%
13:00        271.77     271.77       0.00     0.000%
14:00        271.69     271.69       0.00     0.000%
15:00        271.66     271.66       0.00     0.000%
15:30        271.61     271.61       0.00     0.000%
Max absolute diff: 0.0000
Avg absolute diff: 0.0000
Min absolute diff: 0.0000


In [3]:
ts = pd.Timestamp('2025-12-19 10:19', tz='US/Eastern')
print(f"ATR at 10:19: {atr_series.loc[ts]:.4f}")
print(f"ATR at 10:18: {atr_series.shift(1).loc[ts]:.4f}")
print(f"ATR at 10:17: {atr_series.shift(2).loc[ts]:.4f}")

ATR at 10:19: 0.3157
ATR at 10:18: 0.3125
ATR at 10:17: 0.3066


In [4]:
ts = pd.Timestamp('2025-12-19 10:19', tz='US/Eastern')
bar = df.loc[ts]
print(f"10:19 - O:{bar['open']:.2f} H:{bar['high']:.2f} L:{bar['low']:.2f} C:{bar['close']:.2f}")

10:19 - O:271.32 H:271.62 L:271.26 C:271.59


In [5]:
"""
Swing Stop Validation - Compare Python to Pine
Target: AAPL Dec 19 @ 10:19
"""
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
from orb_backtester import ORBBacktester
from orb_core import calc_atr_rth

# Load data
bt = ORBBacktester(symbol='AAPL', start_date='2025-12-19', end_date='2025-12-19', verbose=False)
bt.load_data()
df = bt.df

# Get Dec 19 data
dec19 = df[df.index.date == pd.Timestamp('2025-12-19').date()].copy()

# Get 10:19 bar
ts = pd.Timestamp('2025-12-19 10:19', tz='US/Eastern')
bar_idx = dec19.index.get_loc(ts)
bar = dec19.iloc[bar_idx]

# Entry = close
entry_price = bar['close']

# ATR
atr_series = calc_atr_rth(df, period=14)
rth_atr = atr_series.loc[ts]

# Swing params
swing_lookback = 5
swing_buffer = 0.02

# Find Swing Low/High from RTH bars only - matching Pine logic
def find_swing_low_rth(day_df, bar_idx, lookback):
    """Pine lines 1104-1117: findSwingLowRegularHours"""
    swing_low = None
    valid_bars = 0
    
    for i in range(1, lookback * 3 + 1):
        if bar_idx - i < 0:
            break
        check_ts = day_df.index[bar_idx - i]
        # Check if RTH (09:30-16:00)
        mins = check_ts.hour * 60 + check_ts.minute
        if 9*60+30 <= mins < 16*60:
            bar_low = day_df['low'].iloc[bar_idx - i]
            if swing_low is None or bar_low < swing_low:
                swing_low = bar_low
            valid_bars += 1
            if valid_bars >= lookback:
                break
    
    if swing_low is None:
        swing_low = day_df['low'].iloc[bar_idx - 1]
    
    return swing_low

def find_swing_high_rth(day_df, bar_idx, lookback):
    """Pine lines 1119-1132: findSwingHighRegularHours"""
    swing_high = None
    valid_bars = 0
    
    for i in range(1, lookback * 3 + 1):
        if bar_idx - i < 0:
            break
        check_ts = day_df.index[bar_idx - i]
        # Check if RTH (09:30-16:00)
        mins = check_ts.hour * 60 + check_ts.minute
        if 9*60+30 <= mins < 16*60:
            bar_high = day_df['high'].iloc[bar_idx - i]
            if swing_high is None or bar_high > swing_high:
                swing_high = bar_high
            valid_bars += 1
            if valid_bars >= lookback:
                break
    
    if swing_high is None:
        swing_high = day_df['high'].iloc[bar_idx - 1]
    
    return swing_high

swing_low = find_swing_low_rth(dec19, bar_idx, swing_lookback)
swing_high = find_swing_high_rth(dec19, bar_idx, swing_lookback)

# Swing Stop calc - Pine line 1140
swing_stop_long = swing_low - (rth_atr * swing_buffer)
swing_stop_short = swing_high + (rth_atr * swing_buffer)

print("═══ SWING STOP DEBUG (Python) ═══")
print(f"Time: 10:19")
print("═══ INPUTS ═══")
print(f"Entry: {entry_price:.2f}")
print(f"rthATR: {rth_atr:.4f}")
print(f"Lookback: {swing_lookback}")
print(f"Buffer mult: {swing_buffer}")
print("═══ SWING VALUES ═══")
print(f"SwingLow: {swing_low:.2f}")
print(f"SwingHigh: {swing_high:.2f}")
print("═══ SWING STOP ═══")
print(f"LONG stop: {swing_stop_long:.2f}")
print(f"SHORT stop: {swing_stop_short:.2f}")
print()
print("═══ PINE VALUES ═══")
print("Entry: 271.32")
print("rthATR: 0.2926")
print("SwingLow: 271.28")
print("SwingHigh: 271.99")
print("LONG stop: 271.27")
print("SHORT stop: 272.00")
print()
print("═══ MATCH? ═══")
print(f"SwingLow: {'✓' if abs(swing_low - 271.28) < 0.01 else '✗'} (Python: {swing_low:.2f})")
print(f"SwingHigh: {'✓' if abs(swing_high - 271.99) < 0.01 else '✗'} (Python: {swing_high:.2f})")
print(f"LONG stop: {'✓' if abs(swing_stop_long - 271.27) < 0.01 else '✗'} (Python: {swing_stop_long:.2f})")
print(f"SHORT stop: {'✓' if abs(swing_stop_short - 272.00) < 0.01 else '✗'} (Python: {swing_stop_short:.2f})")

# Show the RTH bars used
print()
print("═══ RTH BARS USED ═══")
valid_bars = 0
for i in range(1, swing_lookback * 3 + 1):
    if bar_idx - i < 0:
        break
    check_ts = dec19.index[bar_idx - i]
    mins = check_ts.hour * 60 + check_ts.minute
    if 9*60+30 <= mins < 16*60:
        bar_data = dec19.iloc[bar_idx - i]
        valid_bars += 1
        marker = " <-- used" if valid_bars <= swing_lookback else ""
        print(f"  {check_ts.strftime('%H:%M')} H={bar_data['high']:.2f} L={bar_data['low']:.2f}{marker}")
        if valid_bars >= swing_lookback:
            break

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AAPL (1-min bars, 180 days, extended)...
  ✓ Loading from cache (0.6 hours old)
═══ SWING STOP DEBUG (Python) ═══
Time: 10:19
═══ INPUTS ═══
Entry: 271.59
rthATR: 0.3157
Lookback: 5
Buffer mult: 0.02
═══ SWING VALUES ═══
SwingLow: 271.28
SwingHigh: 271.99
═══ SWING STOP ═══
LONG stop: 271.27
SHORT stop: 272.00

═══ PINE VALUES ═══
Entry: 271.32
rthATR: 0.2926
SwingLow: 271.28
SwingHigh: 271.99
LONG stop: 271.27
SHORT stop: 272.00

═══ MATCH? ═══
SwingLow: ✓ (Python: 271.28)
SwingHigh: ✓ (Python: 271.99)
LONG stop: ✓ (Python: 271.27)
SHORT stop: ✓ (Python: 272.00)

═══ RTH BARS USED ═══
  10:18 H=271.66 L=271.28 <-- used
  10:17 H=271.86 L=271.51 <-- used
  10:16 H=271.99 L=271.77 <-- used
  10:15 H=271.92 L=271.75 <-- used
  10:14 H=271.83 L=271.34 <-- used


In [1]:
"""
Auto-Select Stop Validation - Compare Python to Pine
Target: AAPL Dec 19 @ 10:18 SHORT
"""
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
from orb_backtester import ORBBacktester
from orb_core import (calc_atr_rth, calc_atr_stop, calc_vwap_stop, calc_swing_stop,
                      calc_hybrid_stop, check_stop_validity, calc_achievable_rr,
                      select_best_stop)

# Load data
bt = ORBBacktester(symbol='AAPL', start_date='2025-12-19', end_date='2025-12-19', verbose=False)
bt.load_data()
df = bt.df

# Get Dec 19 data
dec19 = df[df.index.date == pd.Timestamp('2025-12-19').date()].copy()

# Get 10:18 bar (the actual entry bar)
ts = pd.Timestamp('2025-12-19 10:18', tz='US/Eastern')
bar_idx = dec19.index.get_loc(ts)
bar = dec19.iloc[bar_idx]

# Entry = close, SHORT trade
entry_price = bar['close']
is_long = False

# ATR and VWAP
atr_series = calc_atr_rth(df, period=14)
atr = atr_series.loc[ts]
vwap = bar['vwap']

print("═══ AUTO-SELECT DEBUG (Python) ═══")
print(f"Time: 10:18")
print(f"Entry: {entry_price:.2f}")
print(f"Direction: SHORT")
print(f"ATR: {atr:.4f}")
print(f"VWAP: {vwap:.2f}")

# Calculate all stops
atr_stop = calc_atr_stop(entry_price, atr, is_long)
swing_stop = calc_swing_stop(dec19, bar_idx, atr, is_long)
vwap_stop = calc_vwap_stop(vwap, atr, is_long)
hybrid_stop = calc_hybrid_stop(atr_stop, swing_stop, vwap_stop, is_long)

print("═══ STOPS ═══")
print(f"ATR: {atr_stop:.2f}")
print(f"Swing: {swing_stop:.2f}")
print(f"VWAP: {vwap_stop:.2f}")
print(f"Hybrid: {hybrid_stop:.2f}")

# Risk
print("═══ RISK (stop - entry) ═══")
print(f"ATR risk: {abs(atr_stop - entry_price):.2f}")
print(f"Swing risk: {abs(swing_stop - entry_price):.2f}")
print(f"VWAP risk: {abs(vwap_stop - entry_price):.2f}")

# Validity
atr_valid = check_stop_validity(atr_stop, entry_price, atr, is_long)
swing_valid = check_stop_validity(swing_stop, entry_price, atr, is_long)
vwap_valid = check_stop_validity(vwap_stop, entry_price, atr, is_long, vwap=vwap)
hybrid_valid = check_stop_validity(hybrid_stop, entry_price, atr, is_long)

# R:R
atr_rr = calc_achievable_rr(entry_price, atr_stop, atr) if atr_valid else -1
swing_rr = calc_achievable_rr(entry_price, swing_stop, atr) if swing_valid else -1
vwap_rr = calc_achievable_rr(entry_price, vwap_stop, atr) if vwap_valid else -1
hybrid_rr = calc_achievable_rr(entry_price, hybrid_stop, atr) if hybrid_valid else -1

print("═══ ACHIEVABLE R:R ═══")
print(f"ATR: {atr_rr:.2f}" + ("" if atr_valid else " (invalid)"))
print(f"Swing: {swing_rr:.2f}" + ("" if swing_valid else " (invalid)"))
print(f"VWAP: {vwap_rr:.2f}" + ("" if vwap_valid else " (invalid)"))
print(f"Hybrid: {hybrid_rr:.2f}" + ("" if hybrid_valid else " (invalid)"))

# Auto-select
best_stop, best_type, best_rr = select_best_stop(
    entry_price, atr, vwap, is_long, dec19, bar_idx
)

print("═══ WINNER ═══")
print(f"{best_type} @ R:R {best_rr:.2f}")
print(f"Stop price: {best_stop:.2f}")

print()
print("═══ PINE VALUES ═══")
print("ATR Stop: 271.98")
print("ATR R:R: 1.5")
print("Winner: ATR @ R:R 1.5")

print()
print("═══ MATCH? ═══")
print(f"Stop: {'✓' if abs(best_stop - 271.98) < 0.01 else '✗'} (Python: {best_stop:.2f}, Pine: 271.98)")
print(f"Type: {'✓' if best_type == 'ATR' else '✗'} (Python: {best_type}, Pine: ATR)")
print(f"R:R: {'✓' if abs(best_rr - 1.5) < 0.01 else '✗'} (Python: {best_rr:.2f}, Pine: 1.5)")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AAPL (1-min bars, 180 days, extended)...
  ✓ Loading from cache (6.1 hours old)
═══ AUTO-SELECT DEBUG (Python) ═══
Time: 10:18
Entry: 271.35
Direction: SHORT
ATR: 0.3125
VWAP: 272.14
═══ STOPS ═══
ATR: 271.98
Swing: 272.00
VWAP: 272.23
Hybrid: 271.98
═══ RISK (stop - entry) ═══
ATR risk: 0.63
Swing risk: 0.65
VWAP risk: 0.88
═══ ACHIEVABLE R:R ═══
ATR: 1.50
Swing: 1.45
VWAP: 1.07
Hybrid: 1.50
═══ WINNER ═══
ATR @ R:R 1.50
Stop price: 271.98

═══ PINE VALUES ═══
ATR Stop: 271.98
ATR R:R: 1.5
Winner: ATR @ R:R 1.5

═══ MATCH? ═══
Stop: ✓ (Python: 271.98, Pine: 271.98)
Type: ✓ (Python: ATR, Pine: ATR)
R:R: ✓ (Python: 1.50, Pine: 1.5)


In [2]:
"""
Full December Validation - Compare Python auto-select to Pine trades
AAPL December 2025
"""
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
from orb_backtester import ORBBacktester
from orb_core import (calc_atr_rth, select_best_stop, build_orb, 
                      check_long_breakout, check_short_breakout)

# Load data
bt = ORBBacktester(symbol='AAPL', start_date='2025-12-01', end_date='2025-12-31', verbose=False)
bt.load_data()
df = bt.df

# Get ATR series
atr_series = calc_atr_rth(df, period=14)

print("═══ DECEMBER 2025 AAPL - AUTO-SELECT VALIDATION ═══")
print(f"{'Date':<12} {'Time':<6} {'Dir':<6} {'Entry':>8} {'Stop':>8} {'Type':<8} {'R:R':>6}")
print("=" * 65)

# Process each trading day
for date in pd.date_range('2025-12-01', '2025-12-31', freq='B'):  # Business days
    day_df = df[df.index.date == date.date()]
    if len(day_df) == 0:
        continue
    
    # Build ORB
    orb = build_orb(day_df)
    if not orb.orb_complete:
        continue
    
    # Scan for breakouts after ORB (09:35 - 10:30)
    for ts, bar in day_df.iterrows():
        if ts.hour == 9 and ts.minute < 35:
            continue  # Skip ORB window
        if ts.hour > 10 or (ts.hour == 10 and ts.minute > 30):
            break  # Stop after 10:30
        
        atr = atr_series.loc[ts]
        if pd.isna(atr):
            continue
            
        # Check for breakout
        is_long_breakout = check_long_breakout(
            bar['open'], bar['high'], bar['low'], bar['close'],
            orb.session_high, orb.session_low, atr, orb.orb_complete
        )
        is_short_breakout = check_short_breakout(
            bar['open'], bar['high'], bar['low'], bar['close'],
            orb.session_high, orb.session_low, atr, orb.orb_complete
        )
        
        if is_long_breakout or is_short_breakout:
            is_long = is_long_breakout
            direction = "LONG" if is_long else "SHORT"
            entry_price = bar['close']
            vwap = bar['vwap']
            bar_idx = day_df.index.get_loc(ts)
            
            # Get Python's auto-select result
            best_stop, best_type, best_rr = select_best_stop(
                entry_price, atr, vwap, is_long, day_df, bar_idx
            )
            
            print(f"{date.strftime('%Y-%m-%d'):<12} {ts.strftime('%H:%M'):<6} {direction:<6} {entry_price:>8.2f} {best_stop:>8.2f} {best_type:<8} {best_rr:>6.2f}")
            
            # Only first breakout per day
            break

print("=" * 65)
print("Compare these against Pine's actual trades in TradingView")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AAPL (1-min bars, 180 days, extended)...
  ✓ Loading from cache (6.2 hours old)
═══ DECEMBER 2025 AAPL - AUTO-SELECT VALIDATION ═══
Date         Time   Dir       Entry     Stop Type        R:R
2025-12-01   07:04  SHORT    277.11   277.41 Swing      2.00
2025-12-02   07:08  SHORT    282.25   282.79 ATR        1.50
2025-12-03   08:30  SHORT    285.75   285.90 Swing      2.00
2025-12-04   09:37  SHORT    283.05   283.93 ATR        1.50
2025-12-05   04:05  LONG     281.19   280.97 Swing      2.00
2025-12-08   09:39  SHORT    277.68   278.53 ATR        1.50
2025-12-09   09:37  LONG     279.39   278.74 ATR        1.50
2025-12-10   09:35  SHORT    276.79   277.53 ATR        1.50
2025-12-11   06:06  LONG     279.65   279.26 ATR        1.50
2025-12-12   09:46  LONG     279.10   278.33 ATR        1.50
2025-12-15   08:01  SHORT    277.58   278.16 ATR        1.50
2025-12-16   08:30  LONG     273.99   273.38 ATR        1.50
2

In [3]:
"""
Full December Validation - Compare Python auto-select to Pine trades
AAPL December 2025 - RTH ONLY
"""
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
from orb_backtester import ORBBacktester
from orb_core import (calc_atr_rth, select_best_stop, build_orb, 
                      check_long_breakout, check_short_breakout)

# Load data
bt = ORBBacktester(symbol='AAPL', start_date='2025-12-01', end_date='2025-12-31', verbose=False)
bt.load_data()
df = bt.df

# Get ATR series
atr_series = calc_atr_rth(df, period=14)

print("═══ DECEMBER 2025 AAPL - AUTO-SELECT VALIDATION ═══")
print(f"{'Date':<12} {'Time':<6} {'Dir':<6} {'Entry':>8} {'Stop':>8} {'Type':<8} {'R:R':>6}")
print("=" * 65)

# Process each trading day
for date in pd.date_range('2025-12-01', '2025-12-31', freq='B'):  # Business days
    day_df = df[df.index.date == date.date()]
    if len(day_df) == 0:
        continue
    
    # Build ORB
    orb = build_orb(day_df)
    if not orb.orb_complete:
        continue
    
    # Scan for breakouts after ORB (09:35 - 10:30) - RTH ONLY
    for ts, bar in day_df.iterrows():
        # RTH filter: must be between 09:35 and 10:30
        mins = ts.hour * 60 + ts.minute
        if mins < 9*60+35:  # Before 09:35
            continue
        if mins > 10*60+30:  # After 10:30
            break
        
        atr = atr_series.loc[ts]
        if pd.isna(atr):
            continue
            
        # Check for breakout
        is_long_breakout = check_long_breakout(
            bar['open'], bar['high'], bar['low'], bar['close'],
            orb.session_high, orb.session_low, atr, orb.orb_complete
        )
        is_short_breakout = check_short_breakout(
            bar['open'], bar['high'], bar['low'], bar['close'],
            orb.session_high, orb.session_low, atr, orb.orb_complete
        )
        
        if is_long_breakout or is_short_breakout:
            is_long = is_long_breakout
            direction = "LONG" if is_long else "SHORT"
            entry_price = bar['close']
            vwap = bar['vwap']
            bar_idx = day_df.index.get_loc(ts)
            
            # Get Python's auto-select result
            best_stop, best_type, best_rr = select_best_stop(
                entry_price, atr, vwap, is_long, day_df, bar_idx
            )
            
            print(f"{date.strftime('%Y-%m-%d'):<12} {ts.strftime('%H:%M'):<6} {direction:<6} {entry_price:>8.2f} {best_stop:>8.2f} {best_type:<8} {best_rr:>6.2f}")
            
            # Only first breakout per day
            break

print("=" * 65)
print("Compare these against Pine's actual trades in TradingView")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AAPL (1-min bars, 180 days, extended)...
  ✓ Loading from cache (6.2 hours old)
═══ DECEMBER 2025 AAPL - AUTO-SELECT VALIDATION ═══
Date         Time   Dir       Entry     Stop Type        R:R
2025-12-01   09:41  SHORT    277.23   277.93 ATR        1.50
2025-12-02   09:36  LONG     284.36   283.41 VWAP       1.57
2025-12-04   09:37  SHORT    283.05   283.93 ATR        1.50
2025-12-05   09:35  SHORT    279.35   280.07 ATR        1.50
2025-12-08   09:39  SHORT    277.68   278.53 ATR        1.50
2025-12-09   09:37  LONG     279.39   278.74 ATR        1.50
2025-12-10   09:35  SHORT    276.79   277.53 ATR        1.50
2025-12-11   09:36  SHORT    277.91   278.61 ATR        1.50
2025-12-12   09:46  LONG     279.10   278.33 ATR        1.50
2025-12-18   09:35  SHORT    269.31   270.54 ATR        1.50
2025-12-19   10:18  SHORT    271.35   271.98 ATR        1.50
2025-12-22   09:45  SHORT    272.47   273.24 Swing      1.97
2

In [4]:
"""
Full December Validation with Exit Management
AAPL December 2025 - Shows entry, stop type, exit reason
"""
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
from orb_backtester import ORBBacktester
from orb_core import (calc_atr_rth, select_best_stop, build_orb, 
                      check_long_breakout, check_short_breakout,
                      PositionState, manage_position_exit)

# Load data
bt = ORBBacktester(symbol='AAPL', start_date='2025-12-01', end_date='2025-12-31', verbose=False)
bt.load_data()
df = bt.df

# Get ATR series
atr_series = calc_atr_rth(df, period=14)

print("═══ DECEMBER 2025 AAPL - FULL TRADE SIMULATION ═══")
print(f"{'Date':<12} {'Dir':<6} {'Entry':>8} {'Stop':>8} {'Type':<8} {'Exit':>8} {'Reason':<15} {'P&L':>8}")
print("=" * 85)

# Process each trading day
for date in pd.date_range('2025-12-01', '2025-12-31', freq='B'):
    day_df = df[df.index.date == date.date()]
    if len(day_df) == 0:
        continue
    
    # Build ORB
    orb = build_orb(day_df)
    if not orb.orb_complete:
        continue
    
    position = None
    
    # Scan for breakouts and manage position
    for ts, bar in day_df.iterrows():
        mins = ts.hour * 60 + ts.minute
        
        # Skip pre-RTH
        if mins < 9*60+30:
            continue
        
        atr = atr_series.loc[ts]
        if pd.isna(atr):
            continue
        
        # If no position, look for entry (09:35 - 10:30)
        if position is None and mins >= 9*60+35 and mins <= 10*60+30:
            is_long_breakout = check_long_breakout(
                bar['open'], bar['high'], bar['low'], bar['close'],
                orb.session_high, orb.session_low, atr, orb.orb_complete
            )
            is_short_breakout = check_short_breakout(
                bar['open'], bar['high'], bar['low'], bar['close'],
                orb.session_high, orb.session_low, atr, orb.orb_complete
            )
            
            if is_long_breakout or is_short_breakout:
                is_long = is_long_breakout
                entry_price = bar['close']
                vwap = bar['vwap']
                bar_idx = day_df.index.get_loc(ts)
                
                best_stop, stop_type, best_rr = select_best_stop(
                    entry_price, atr, vwap, is_long, day_df, bar_idx
                )
                
                position = PositionState(
                    entry_price=entry_price,
                    initial_stop=best_stop,
                    stop_type=stop_type,
                    is_long=is_long,
                    risk=abs(entry_price - best_stop),
                    target_rr=best_rr
                )
                entry_time = ts
                direction = "LONG" if is_long else "SHORT"
        
        # If in position, manage exit
        elif position is not None:
            is_closed, exit_price, exit_reason = manage_position_exit(
                position, bar, atr,
                break_even_rr=0.5, profit_target_rr=2.0,
                trailing_stop_distance=1.2,
                use_break_even=True, use_trailing=True
            )
            
            if is_closed:
                if position.is_long:
                    pnl = exit_price - position.entry_price
                else:
                    pnl = position.entry_price - exit_price
                
                print(f"{date.strftime('%Y-%m-%d'):<12} {direction:<6} {position.entry_price:>8.2f} {position.initial_stop:>8.2f} {position.stop_type:<8} {exit_price:>8.2f} {exit_reason:<15} {pnl:>+8.2f}")
                position = None
                break
        
        # EOD close
        if mins >= 15*60+55 and position is not None:
            exit_price = bar['close']
            if position.is_long:
                pnl = exit_price - position.entry_price
            else:
                pnl = position.entry_price - exit_price
            print(f"{date.strftime('%Y-%m-%d'):<12} {direction:<6} {position.entry_price:>8.2f} {position.initial_stop:>8.2f} {position.stop_type:<8} {exit_price:>8.2f} {'EOD Close':<15} {pnl:>+8.2f}")
            position = None
            break

print("=" * 85)

ImportError: cannot import name 'PositionState' from 'orb_core' (C:\Users\phemm\orb_lab\src\orb_core.py)

In [ ]:
"""
Full December Validation with Exit Management
AAPL December 2025 - Shows entry, stop type, exit reason
"""
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
from orb_backtester import ORBBacktester
from orb_core import (calc_atr_rth, select_best_stop, build_orb, 
                      check_long_breakout, check_short_breakout,
                      PositionState, manage_position_exit)

# Load data
bt = ORBBacktester(symbol='AAPL', start_date='2025-12-01', end_date='2025-12-31', verbose=False)
bt.load_data()
df = bt.df

# Get ATR series
atr_series = calc_atr_rth(df, period=14)

print("═══ DECEMBER 2025 AAPL - FULL TRADE SIMULATION ═══")
print(f"{'Date':<12} {'Dir':<6} {'Entry':>8} {'Stop':>8} {'Type':<8} {'Exit':>8} {'Reason':<15} {'P&L':>8}")
print("=" * 85)

# Process each trading day
for date in pd.date_range('2025-12-01', '2025-12-31', freq='B'):
    day_df = df[df.index.date == date.date()]
    if len(day_df) == 0:
        continue
    
    # Build ORB
    orb = build_orb(day_df)
    if not orb.orb_complete:
        continue
    
    position = None
    
    # Scan for breakouts and manage position
    for ts, bar in day_df.iterrows():
        mins = ts.hour * 60 + ts.minute
        
        # Skip pre-RTH
        if mins < 9*60+30:
            continue
        
        atr = atr_series.loc[ts]
        if pd.isna(atr):
            continue
        
        # If no position, look for entry (09:35 - 10:30)
        if position is None and mins >= 9*60+35 and mins <= 10*60+30:
            is_long_breakout = check_long_breakout(
                bar['open'], bar['high'], bar['low'], bar['close'],
                orb.session_high, orb.session_low, atr, orb.orb_complete
            )
            is_short_breakout = check_short_breakout(
                bar['open'], bar['high'], bar['low'], bar['close'],
                orb.session_high, orb.session_low, atr, orb.orb_complete
            )
            
            if is_long_breakout or is_short_breakout:
                is_long = is_long_breakout
                entry_price = bar['close']
                vwap = bar['vwap']
                bar_idx = day_df.index.get_loc(ts)
                
                best_stop, stop_type, best_rr = select_best_stop(
                    entry_price, atr, vwap, is_long, day_df, bar_idx
                )
                
                position = PositionState(
                    entry_price=entry_price,
                    initial_stop=best_stop,
                    stop_type=stop_type,
                    is_long=is_long,
                    risk=abs(entry_price - best_stop),
                    target_rr=best_rr
                )
                entry_time = ts
                direction = "LONG" if is_long else "SHORT"
        
        # If in position, manage exit
        elif position is not None:
            is_closed, exit_price, exit_reason = manage_position_exit(
                position, bar, atr,
                break_even_rr=0.5, profit_target_rr=2.0,
                trailing_stop_distance=1.2,
                use_break_even=True, use_trailing=True
            )
            
            if is_closed:
                if position.is_long:
                    pnl = exit_price - position.entry_price
                else:
                    pnl = position.entry_price - exit_price
                
                print(f"{date.strftime('%Y-%m-%d'):<12} {direction:<6} {position.entry_price:>8.2f} {position.initial_stop:>8.2f} {position.stop_type:<8} {exit_price:>8.2f} {exit_reason:<15} {pnl:>+8.2f}")
                position = None
                break
        
        # EOD close
        if mins >= 15*60+55 and position is not None:
            exit_price = bar['close']
            if position.is_long:
                pnl = exit_price - position.entry_price
            else:
                pnl = position.entry_price - exit_price
            print(f"{date.strftime('%Y-%m-%d'):<12} {direction:<6} {position.entry_price:>8.2f} {position.initial_stop:>8.2f} {position.stop_type:<8} {exit_price:>8.2f} {'EOD Close':<15} {pnl:>+8.2f}")
            position = None
            break

print("=" * 85)

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AAPL (1-min bars, 180 days, extended)...
  ⚠ Cache stale (23 hours), refreshing...
  ✓ Fetched 89585 bars, cached for next time


In [7]:
with open(r'C:\Users\phemm\orb_lab\src\orb_core.py', 'r') as f:
    content = f.read()
    print(f"File size: {len(content)} chars")
    print(f"Contains PositionState: {'class PositionState' in content}")

UnicodeDecodeError: 'charmap' codec can't decode byte 0x90 in position 248: character maps to <undefined>

In [1]:
import os
files = os.listdir(r'C:\Users\phemm\orb_lab\src')
print(files)

['confluence_indicators.py', 'data_collector.py', 'massive_loader.py', 'orb_backtester.py', 'orb_core.py', 'orb_optimizer.py', 'single_day_tracer_v3.py', 'single_day_tracer_v4.py', 'single_trade_runner.py', 'stop_engine.py', 'stop_selector.py', 'test_atr_stop.py', 'test_vwap_stop.py', 'test_vwap_variance.py', 'trade_stop_manager.py', '__pycache__']


In [2]:
import shutil
cache_path = r'C:\Users\phemm\orb_lab\src\__pycache__'
if os.path.exists(cache_path):
    shutil.rmtree(cache_path)
    print("Deleted cache")

Deleted cache


In [1]:
from orb_core import PositionState, manage_position_exit
print("Import OK")

ModuleNotFoundError: No module named 'orb_core'

In [2]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')

from orb_core import PositionState, manage_position_exit
print("Import OK")

Import OK


In [3]:
"""
Full December Validation with Exit Management
AAPL December 2025 - Shows entry, stop type, exit reason
"""
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
from orb_backtester import ORBBacktester
from orb_core import (calc_atr_rth, select_best_stop, build_orb, 
                      check_long_breakout, check_short_breakout,
                      PositionState, manage_position_exit)

# Load data
bt = ORBBacktester(symbol='AAPL', start_date='2025-12-01', end_date='2025-12-31', verbose=False)
bt.load_data()
df = bt.df

# Get ATR series
atr_series = calc_atr_rth(df, period=14)

print("═══ DECEMBER 2025 AAPL - FULL TRADE SIMULATION ═══")
print(f"{'Date':<12} {'Dir':<6} {'Entry':>8} {'Stop':>8} {'Type':<8} {'Exit':>8} {'Reason':<15} {'P&L':>8}")
print("=" * 85)

# Process each trading day
for date in pd.date_range('2025-12-01', '2025-12-31', freq='B'):
    day_df = df[df.index.date == date.date()]
    if len(day_df) == 0:
        continue
    
    # Build ORB
    orb = build_orb(day_df)
    if not orb.orb_complete:
        continue
    
    position = None
    
    # Scan for breakouts and manage position
    for ts, bar in day_df.iterrows():
        mins = ts.hour * 60 + ts.minute
        
        # Skip pre-RTH
        if mins < 9*60+30:
            continue
        
        atr = atr_series.loc[ts]
        if pd.isna(atr):
            continue
        
        # If no position, look for entry (09:35 - 10:30)
        if position is None and mins >= 9*60+35 and mins <= 10*60+30:
            is_long_breakout = check_long_breakout(
                bar['open'], bar['high'], bar['low'], bar['close'],
                orb.session_high, orb.session_low, atr, orb.orb_complete
            )
            is_short_breakout = check_short_breakout(
                bar['open'], bar['high'], bar['low'], bar['close'],
                orb.session_high, orb.session_low, atr, orb.orb_complete
            )
            
            if is_long_breakout or is_short_breakout:
                is_long = is_long_breakout
                entry_price = bar['close']
                vwap = bar['vwap']
                bar_idx = day_df.index.get_loc(ts)
                
                best_stop, stop_type, best_rr = select_best_stop(
                    entry_price, atr, vwap, is_long, day_df, bar_idx
                )
                
                position = PositionState(
                    entry_price=entry_price,
                    initial_stop=best_stop,
                    stop_type=stop_type,
                    is_long=is_long,
                    risk=abs(entry_price - best_stop),
                    target_rr=best_rr
                )
                entry_time = ts
                direction = "LONG" if is_long else "SHORT"
        
        # If in position, manage exit
        elif position is not None:
            is_closed, exit_price, exit_reason = manage_position_exit(
                position, bar, atr,
                break_even_rr=0.5, profit_target_rr=2.0,
                trailing_stop_distance=1.2,
                use_break_even=True, use_trailing=True
            )
            
            if is_closed:
                if position.is_long:
                    pnl = exit_price - position.entry_price
                else:
                    pnl = position.entry_price - exit_price
                
                print(f"{date.strftime('%Y-%m-%d'):<12} {direction:<6} {position.entry_price:>8.2f} {position.initial_stop:>8.2f} {position.stop_type:<8} {exit_price:>8.2f} {exit_reason:<15} {pnl:>+8.2f}")
                position = None
                break
        
        # EOD close
        if mins >= 15*60+55 and position is not None:
            exit_price = bar['close']
            if position.is_long:
                pnl = exit_price - position.entry_price
            else:
                pnl = position.entry_price - exit_price
            print(f"{date.strftime('%Y-%m-%d'):<12} {direction:<6} {position.entry_price:>8.2f} {position.initial_stop:>8.2f} {position.stop_type:<8} {exit_price:>8.2f} {'EOD Close':<15} {pnl:>+8.2f}")
            position = None
            break

print("=" * 85)

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AAPL (1-min bars, 180 days, extended)...
  ✓ Loading from cache (0.0 hours old)
═══ DECEMBER 2025 AAPL - FULL TRADE SIMULATION ═══
Date         Dir       Entry     Stop Type         Exit Reason               P&L
2025-12-01   SHORT    277.23   277.93 ATR        277.23 Break Even         +0.00
2025-12-02   LONG     284.36   283.41 VWAP       284.36 Break Even         +0.00
2025-12-04   SHORT    283.05   283.93 ATR        283.05 Break Even         +0.00
2025-12-05   SHORT    279.35   280.07 ATR        280.07 Initial Stop       -0.72
2025-12-08   SHORT    277.68   278.53 ATR        277.68 Break Even         +0.00
2025-12-09   LONG     279.39   278.74 ATR        279.39 Break Even         +0.00
2025-12-10   SHORT    276.79   277.53 ATR        277.53 Initial Stop       -0.73
2025-12-11   SHORT    277.91   278.61 ATR        276.95 Trailing Stop      +0.96
2025-12-12   LONG     279.10   278.33 ATR        278.33 Initial St

In [4]:
"""
Full December Validation with Exit Management + EMA Exit
AAPL December 2025
"""
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
from orb_backtester import ORBBacktester
from orb_core import (calc_atr_rth, select_best_stop, build_orb, 
                      check_long_breakout, check_short_breakout,
                      PositionState, manage_position_exit)

# Load data
bt = ORBBacktester(symbol='AAPL', start_date='2025-12-01', end_date='2025-12-31', verbose=False)
bt.load_data()
df = bt.df

# Get ATR series
atr_series = calc_atr_rth(df, period=14)

# Calculate EMA9
df['ema9'] = df['close'].ewm(span=9, adjust=False).mean()

print("═══ DECEMBER 2025 AAPL - FULL TRADE SIMULATION ═══")
print(f"{'Date':<12} {'Dir':<6} {'Entry':>8} {'Stop':>8} {'Type':<8} {'Exit':>8} {'Reason':<15} {'P&L':>8}")
print("=" * 85)

# Process each trading day
for date in pd.date_range('2025-12-01', '2025-12-31', freq='B'):
    day_df = df[df.index.date == date.date()]
    if len(day_df) == 0:
        continue
    
    # Build ORB
    orb = build_orb(day_df)
    if not orb.orb_complete:
        continue
    
    position = None
    
    # Scan for breakouts and manage position
    for ts, bar in day_df.iterrows():
        mins = ts.hour * 60 + ts.minute
        
        # Skip pre-RTH
        if mins < 9*60+30:
            continue
        
        atr = atr_series.loc[ts]
        if pd.isna(atr):
            continue
        
        ema = bar['ema9']
        
        # If no position, look for entry (09:35 - 10:30)
        if position is None and mins >= 9*60+35 and mins <= 10*60+30:
            is_long_breakout = check_long_breakout(
                bar['open'], bar['high'], bar['low'], bar['close'],
                orb.session_high, orb.session_low, atr, orb.orb_complete
            )
            is_short_breakout = check_short_breakout(
                bar['open'], bar['high'], bar['low'], bar['close'],
                orb.session_high, orb.session_low, atr, orb.orb_complete
            )
            
            if is_long_breakout or is_short_breakout:
                is_long = is_long_breakout
                entry_price = bar['close']
                vwap = bar['vwap']
                bar_idx = day_df.index.get_loc(ts)
                
                best_stop, stop_type, best_rr = select_best_stop(
                    entry_price, atr, vwap, is_long, day_df, bar_idx
                )
                
                position = PositionState(
                    entry_price=entry_price,
                    initial_stop=best_stop,
                    stop_type=stop_type,
                    is_long=is_long,
                    risk=abs(entry_price - best_stop),
                    target_rr=best_rr
                )
                entry_time = ts
                direction = "LONG" if is_long else "SHORT"
        
        # If in position, manage exit
        elif position is not None:
            is_closed, exit_price, exit_reason = manage_position_exit(
                position, bar, atr,
                break_even_rr=0.5, profit_target_rr=2.0,
                trailing_stop_distance=1.2,
                use_break_even=True, use_trailing=True,
                use_ema_exit=True, ema=ema, ema_confirmation_bars=1
            )
            
            if is_closed:
                if position.is_long:
                    pnl = exit_price - position.entry_price
                else:
                    pnl = position.entry_price - exit_price
                
                print(f"{date.strftime('%Y-%m-%d'):<12} {direction:<6} {position.entry_price:>8.2f} {position.initial_stop:>8.2f} {position.stop_type:<8} {exit_price:>8.2f} {exit_reason:<15} {pnl:>+8.2f}")
                position = None
                break
        
        # EOD close
        if mins >= 15*60+55 and position is not None:
            exit_price = bar['close']
            if position.is_long:
                pnl = exit_price - position.entry_price
            else:
                pnl = position.entry_price - exit_price
            print(f"{date.strftime('%Y-%m-%d'):<12} {direction:<6} {position.entry_price:>8.2f} {position.initial_stop:>8.2f} {position.stop_type:<8} {exit_price:>8.2f} {'EOD Close':<15} {pnl:>+8.2f}")
            position = None
            break

print("=" * 85)

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AAPL (1-min bars, 180 days, extended)...
  ✓ Loading from cache (0.2 hours old)
═══ DECEMBER 2025 AAPL - FULL TRADE SIMULATION ═══
Date         Dir       Entry     Stop Type         Exit Reason               P&L


TypeError: manage_position_exit() got an unexpected keyword argument 'use_ema_exit'

In [1]:
"""
Full December Validation with Exit Management + EMA Exit
AAPL December 2025
"""
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
from orb_backtester import ORBBacktester
from orb_core import (calc_atr_rth, select_best_stop, build_orb, 
                      check_long_breakout, check_short_breakout,
                      PositionState, manage_position_exit)

# Load data
bt = ORBBacktester(symbol='AAPL', start_date='2025-12-01', end_date='2025-12-31', verbose=False)
bt.load_data()
df = bt.df

# Get ATR series
atr_series = calc_atr_rth(df, period=14)

# Calculate EMA9
df['ema9'] = df['close'].ewm(span=9, adjust=False).mean()

print("═══ DECEMBER 2025 AAPL - FULL TRADE SIMULATION ═══")
print(f"{'Date':<12} {'Dir':<6} {'Entry':>8} {'Stop':>8} {'Type':<8} {'Exit':>8} {'Reason':<15} {'P&L':>8}")
print("=" * 85)

# Process each trading day
for date in pd.date_range('2025-12-01', '2025-12-31', freq='B'):
    day_df = df[df.index.date == date.date()]
    if len(day_df) == 0:
        continue
    
    # Build ORB
    orb = build_orb(day_df)
    if not orb.orb_complete:
        continue
    
    position = None
    
    # Scan for breakouts and manage position
    for ts, bar in day_df.iterrows():
        mins = ts.hour * 60 + ts.minute
        
        # Skip pre-RTH
        if mins < 9*60+30:
            continue
        
        atr = atr_series.loc[ts]
        if pd.isna(atr):
            continue
        
        ema = bar['ema9']
        
        # If no position, look for entry (09:35 - 10:30)
        if position is None and mins >= 9*60+35 and mins <= 10*60+30:
            is_long_breakout = check_long_breakout(
                bar['open'], bar['high'], bar['low'], bar['close'],
                orb.session_high, orb.session_low, atr, orb.orb_complete
            )
            is_short_breakout = check_short_breakout(
                bar['open'], bar['high'], bar['low'], bar['close'],
                orb.session_high, orb.session_low, atr, orb.orb_complete
            )
            
            if is_long_breakout or is_short_breakout:
                is_long = is_long_breakout
                entry_price = bar['close']
                vwap = bar['vwap']
                bar_idx = day_df.index.get_loc(ts)
                
                best_stop, stop_type, best_rr = select_best_stop(
                    entry_price, atr, vwap, is_long, day_df, bar_idx
                )
                
                position = PositionState(
                    entry_price=entry_price,
                    initial_stop=best_stop,
                    stop_type=stop_type,
                    is_long=is_long,
                    risk=abs(entry_price - best_stop),
                    target_rr=best_rr
                )
                entry_time = ts
                direction = "LONG" if is_long else "SHORT"
        
        # If in position, manage exit
        elif position is not None:
            is_closed, exit_price, exit_reason = manage_position_exit(
                position, bar, atr,
                break_even_rr=0.5, profit_target_rr=2.0,
                trailing_stop_distance=1.2,
                use_break_even=True, use_trailing=True,
                use_ema_exit=True, ema=ema, ema_confirmation_bars=1
            )
            
            if is_closed:
                if position.is_long:
                    pnl = exit_price - position.entry_price
                else:
                    pnl = position.entry_price - exit_price
                
                print(f"{date.strftime('%Y-%m-%d'):<12} {direction:<6} {position.entry_price:>8.2f} {position.initial_stop:>8.2f} {position.stop_type:<8} {exit_price:>8.2f} {exit_reason:<15} {pnl:>+8.2f}")
                position = None
                break
        
        # EOD close
        if mins >= 15*60+55 and position is not None:
            exit_price = bar['close']
            if position.is_long:
                pnl = exit_price - position.entry_price
            else:
                pnl = position.entry_price - exit_price
            print(f"{date.strftime('%Y-%m-%d'):<12} {direction:<6} {position.entry_price:>8.2f} {position.initial_stop:>8.2f} {position.stop_type:<8} {exit_price:>8.2f} {'EOD Close':<15} {pnl:>+8.2f}")
            position = None
            break

print("=" * 85)

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AAPL (1-min bars, 180 days, extended)...
  ✓ Loading from cache (0.2 hours old)
═══ DECEMBER 2025 AAPL - FULL TRADE SIMULATION ═══
Date         Dir       Entry     Stop Type         Exit Reason               P&L
2025-12-01   SHORT    277.23   277.93 ATR        277.05 EMA Exit           +0.18
2025-12-02   LONG     284.36   283.41 VWAP       284.36 Break Even         +0.00
2025-12-04   SHORT    283.05   283.93 ATR        283.05 Break Even         +0.00
2025-12-05   SHORT    279.35   280.07 ATR        280.07 Initial Stop       -0.72
2025-12-08   SHORT    277.68   278.53 ATR        277.37 EMA Exit           +0.31
2025-12-09   LONG     279.39   278.74 ATR        279.39 Break Even         +0.00
2025-12-10   SHORT    276.79   277.53 ATR        277.53 Initial Stop       -0.73
2025-12-11   SHORT    277.91   278.61 ATR        276.95 Trailing Stop      +0.96
2025-12-12   LONG     279.10   278.33 ATR        278.33 Initial St

In [2]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
from orb_backtester import ORBBacktester
from orb_core import (calc_atr_rth, calc_atr_stop, calc_vwap_stop, calc_swing_stop,
                      calc_hybrid_stop, check_stop_validity, calc_achievable_rr,
                      build_orb, check_long_breakout)

bt = ORBBacktester(symbol='AAPL', start_date='2025-12-02', end_date='2025-12-02', verbose=False)
bt.load_data()
df = bt.df

day_df = df[df.index.date == pd.Timestamp('2025-12-02').date()]
atr_series = calc_atr_rth(df, period=14)

# Find the entry bar (09:36 based on our earlier analysis)
ts = pd.Timestamp('2025-12-02 09:36', tz='US/Eastern')
bar = day_df.loc[ts]
bar_idx = day_df.index.get_loc(ts)

entry_price = bar['close']
atr = atr_series.loc[ts]
vwap = bar['vwap']
is_long = True

print(f"Entry: {entry_price:.2f}")
print(f"ATR: {atr:.4f}")
print(f"VWAP: {vwap:.2f}")

# Calculate all stops
atr_stop = calc_atr_stop(entry_price, atr, is_long)
swing_stop = calc_swing_stop(day_df, bar_idx, atr, is_long)
vwap_stop = calc_vwap_stop(vwap, atr, is_long)
hybrid_stop = calc_hybrid_stop(atr_stop, swing_stop, vwap_stop, is_long)

print(f"\nATR Stop: {atr_stop:.2f}")
print(f"Swing Stop: {swing_stop:.2f}")
print(f"VWAP Stop: {vwap_stop:.2f}")
print(f"Hybrid Stop: {hybrid_stop:.2f}")

# Check validity
atr_valid = check_stop_validity(atr_stop, entry_price, atr, is_long)
swing_valid = check_stop_validity(swing_stop, entry_price, atr, is_long)
vwap_valid = check_stop_validity(vwap_stop, entry_price, atr, is_long, vwap=vwap)
hybrid_valid = check_stop_validity(hybrid_stop, entry_price, atr, is_long)

print(f"\nATR valid: {atr_valid}")
print(f"Swing valid: {swing_valid}")
print(f"VWAP valid: {vwap_valid}")
print(f"Hybrid valid: {hybrid_valid}")

# R:R
atr_rr = calc_achievable_rr(entry_price, atr_stop, atr) if atr_valid else -1
swing_rr = calc_achievable_rr(entry_price, swing_stop, atr) if swing_valid else -1
vwap_rr = calc_achievable_rr(entry_price, vwap_stop, atr) if vwap_valid else -1
hybrid_rr = calc_achievable_rr(entry_price, hybrid_stop, atr) if hybrid_valid else -1

print(f"\nATR R:R: {atr_rr:.2f}")
print(f"Swing R:R: {swing_rr:.2f}")
print(f"VWAP R:R: {vwap_rr:.2f}")
print(f"Hybrid R:R: {hybrid_rr:.2f}")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AAPL (1-min bars, 180 days, extended)...
  ✓ Loading from cache (0.3 hours old)
Entry: 284.36
ATR: 0.4970
VWAP: 283.56

ATR Stop: 283.37
Swing Stop: 282.62
VWAP Stop: 283.41
Hybrid Stop: 283.41

ATR valid: True
Swing valid: True
VWAP valid: True
Hybrid valid: True

ATR R:R: 1.50
Swing R:R: 0.86
VWAP R:R: 1.57
Hybrid R:R: 1.57


In [1]:
"""
Full December Validation with Exit Management + EMA Exit
AAPL December 2025
"""
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
from orb_backtester import ORBBacktester
from orb_core import (calc_atr_rth, select_best_stop, build_orb, 
                      check_long_breakout, check_short_breakout,
                      PositionState, manage_position_exit)

# Load data
bt = ORBBacktester(symbol='AAPL', start_date='2025-12-01', end_date='2025-12-31', verbose=False)
bt.load_data()
df = bt.df

# Get ATR series
atr_series = calc_atr_rth(df, period=14)

# Calculate EMA9
df['ema9'] = df['close'].ewm(span=9, adjust=False).mean()

print("═══ DECEMBER 2025 AAPL - FULL TRADE SIMULATION ═══")
print(f"{'Date':<12} {'Dir':<6} {'Entry':>8} {'Stop':>8} {'Type':<8} {'Exit':>8} {'Reason':<15} {'P&L':>8}")
print("=" * 85)

# Process each trading day
for date in pd.date_range('2025-12-01', '2025-12-31', freq='B'):
    day_df = df[df.index.date == date.date()]
    if len(day_df) == 0:
        continue
    
    # Build ORB
    orb = build_orb(day_df)
    if not orb.orb_complete:
        continue
    
    position = None
    
    # Scan for breakouts and manage position
    for ts, bar in day_df.iterrows():
        mins = ts.hour * 60 + ts.minute
        
        # Skip pre-RTH
        if mins < 9*60+30:
            continue
        
        atr = atr_series.loc[ts]
        if pd.isna(atr):
            continue
        
        ema = bar['ema9']
        
        # If no position, look for entry (09:35 - 10:30)
        if position is None and mins >= 9*60+35 and mins <= 10*60+30:
            is_long_breakout = check_long_breakout(
                bar['open'], bar['high'], bar['low'], bar['close'],
                orb.session_high, orb.session_low, atr, orb.orb_complete
            )
            is_short_breakout = check_short_breakout(
                bar['open'], bar['high'], bar['low'], bar['close'],
                orb.session_high, orb.session_low, atr, orb.orb_complete
            )
            
            if is_long_breakout or is_short_breakout:
                is_long = is_long_breakout
                entry_price = bar['close']
                vwap = bar['vwap']
                bar_idx = day_df.index.get_loc(ts)
                
                best_stop, stop_type, best_rr = select_best_stop(
                    entry_price, atr, vwap, is_long, day_df, bar_idx
                )
                
                position = PositionState(
                    entry_price=entry_price,
                    initial_stop=best_stop,
                    stop_type=stop_type,
                    is_long=is_long,
                    risk=abs(entry_price - best_stop),
                    target_rr=best_rr
                )
                entry_time = ts
                direction = "LONG" if is_long else "SHORT"
        
        # If in position, manage exit
        elif position is not None:
            is_closed, exit_price, exit_reason = manage_position_exit(
                position, bar, atr,
                break_even_rr=0.5, profit_target_rr=2.0,
                trailing_stop_distance=1.2,
                use_break_even=True, use_trailing=True,
                use_ema_exit=True, ema=ema, ema_confirmation_bars=1,
                use_aggressive_trailing=True, ema_tighten_zone=0.3, tightened_trail_distance=0.3
            )
            
            if is_closed:
                if position.is_long:
                    pnl = exit_price - position.entry_price
                else:
                    pnl = position.entry_price - exit_price
                
                print(f"{date.strftime('%Y-%m-%d'):<12} {direction:<6} {position.entry_price:>8.2f} {position.initial_stop:>8.2f} {position.stop_type:<8} {exit_price:>8.2f} {exit_reason:<15} {pnl:>+8.2f}")
                position = None
                break
        
        # EOD close
        if mins >= 15*60+55 and position is not None:
            exit_price = bar['close']
            if position.is_long:
                pnl = exit_price - position.entry_price
            else:
                pnl = position.entry_price - exit_price
            print(f"{date.strftime('%Y-%m-%d'):<12} {direction:<6} {position.entry_price:>8.2f} {position.initial_stop:>8.2f} {position.stop_type:<8} {exit_price:>8.2f} {'EOD Close':<15} {pnl:>+8.2f}")
            position = None
            break

print("=" * 85)

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AAPL (1-min bars, 180 days, extended)...
  ✓ Loading from cache (0.9 hours old)
═══ DECEMBER 2025 AAPL - FULL TRADE SIMULATION ═══
Date         Dir       Entry     Stop Type         Exit Reason               P&L
2025-12-01   SHORT    277.23   277.93 ATR        277.05 EMA Exit           +0.18
2025-12-02   LONG     284.36   283.41 VWAP       284.36 Break Even         +0.00
2025-12-04   SHORT    283.05   283.93 ATR        283.05 Break Even         +0.00
2025-12-05   SHORT    279.35   280.07 ATR        280.07 Initial Stop       -0.72
2025-12-08   SHORT    277.68   278.53 ATR        277.37 EMA Exit           +0.31
2025-12-09   LONG     279.39   278.74 ATR        279.39 Break Even         +0.00
2025-12-10   SHORT    276.79   277.53 ATR        277.53 Initial Stop       -0.73
2025-12-11   SHORT    277.91   278.61 ATR        274.56 Trailing Stop      +3.35
2025-12-12   LONG     279.10   278.33 ATR        278.33 Initial St

In [1]:
"""
Full December Validation with Exit Management + EMA Exit
AAPL December 2025
"""
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
from orb_backtester import ORBBacktester
from orb_core import (calc_atr_rth, select_best_stop, build_orb, 
                      check_long_breakout, check_short_breakout,
                      PositionState, manage_position_exit)

# Load data
bt = ORBBacktester(symbol='AAPL', start_date='2025-12-01', end_date='2025-12-31', verbose=False)
bt.load_data()
df = bt.df

# Get ATR series
atr_series = calc_atr_rth(df, period=14)

# Calculate EMA9
df['ema9'] = df['close'].ewm(span=9, adjust=False).mean()

print("═══ DECEMBER 2025 AAPL - FULL TRADE SIMULATION ═══")
print(f"{'Date':<12} {'Dir':<6} {'Entry':>8} {'Stop':>8} {'Type':<8} {'Exit':>8} {'Reason':<15} {'P&L':>8}")
print("=" * 85)

# Process each trading day
for date in pd.date_range('2025-12-01', '2025-12-31', freq='B'):
    day_df = df[df.index.date == date.date()]
    if len(day_df) == 0:
        continue
    
    # Build ORB
    orb = build_orb(day_df)
    if not orb.orb_complete:
        continue
    
    position = None
    
    # Scan for breakouts and manage position
    for ts, bar in day_df.iterrows():
        mins = ts.hour * 60 + ts.minute
        
        # Skip pre-RTH
        if mins < 9*60+30:
            continue
        
        atr = atr_series.loc[ts]
        if pd.isna(atr):
            continue
        
        ema = bar['ema9']
        
        # If no position, look for entry (09:35 - 10:30)
        if position is None and mins >= 9*60+35 and mins <= 10*60+30:
            is_long_breakout = check_long_breakout(
                bar['open'], bar['high'], bar['low'], bar['close'],
                orb.session_high, orb.session_low, atr, orb.orb_complete
            )
            is_short_breakout = check_short_breakout(
                bar['open'], bar['high'], bar['low'], bar['close'],
                orb.session_high, orb.session_low, atr, orb.orb_complete
            )
            
            if is_long_breakout or is_short_breakout:
                is_long = is_long_breakout
                entry_price = bar['close']
                vwap = bar['vwap']
                bar_idx = day_df.index.get_loc(ts)
                
                best_stop, stop_type, best_rr = select_best_stop(
                    entry_price, atr, vwap, is_long, day_df, bar_idx
                )
                
                position = PositionState(
                    entry_price=entry_price,
                    initial_stop=best_stop,
                    stop_type=stop_type,
                    is_long=is_long,
                    risk=abs(entry_price - best_stop),
                    target_rr=best_rr
                )
                entry_time = ts
                direction = "LONG" if is_long else "SHORT"
        
        # If in position, manage exit
        elif position is not None:
            is_closed, exit_price, exit_reason = manage_position_exit(
                position, bar, atr,
                break_even_rr=0.5, profit_target_rr=2.0,
                trailing_stop_distance=1.2,
                use_break_even=True, use_trailing=True,
                use_ema_exit=True, ema=ema, ema_confirmation_bars=1,
                use_aggressive_trailing=True, ema_tighten_zone=0.3, tightened_trail_distance=0.3
            )
            
            if is_closed:
                if position.is_long:
                    pnl = exit_price - position.entry_price
                else:
                    pnl = position.entry_price - exit_price
                
                print(f"{date.strftime('%Y-%m-%d'):<12} {direction:<6} {position.entry_price:>8.2f} {position.initial_stop:>8.2f} {position.stop_type:<8} {exit_price:>8.2f} {exit_reason:<15} {pnl:>+8.2f}")
                position = None
                break
        
        # EOD close
        if mins >= 15*60+55 and position is not None:
            exit_price = bar['close']
            if position.is_long:
                pnl = exit_price - position.entry_price
            else:
                pnl = position.entry_price - exit_price
            print(f"{date.strftime('%Y-%m-%d'):<12} {direction:<6} {position.entry_price:>8.2f} {position.initial_stop:>8.2f} {position.stop_type:<8} {exit_price:>8.2f} {'EOD Close':<15} {pnl:>+8.2f}")
            position = None
            break

print("=" * 85)

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AAPL (1-min bars, 180 days, extended)...
  ✓ Loading from cache (1.2 hours old)
═══ DECEMBER 2025 AAPL - FULL TRADE SIMULATION ═══
Date         Dir       Entry     Stop Type         Exit Reason               P&L
2025-12-01   SHORT    277.23   277.93 ATR        277.05 EMA Exit           +0.18
2025-12-02   LONG     284.36   283.41 VWAP       284.36 Break Even         +0.00
2025-12-04   SHORT    283.05   283.93 ATR        283.05 Break Even         +0.00
2025-12-05   SHORT    279.35   280.07 ATR        280.07 Initial Stop       -0.72
2025-12-08   SHORT    277.68   278.53 ATR        277.37 EMA Exit           +0.31
2025-12-09   LONG     279.39   278.74 ATR        279.39 Break Even         +0.00
2025-12-10   SHORT    276.79   277.53 ATR        277.53 Initial Stop       -0.73
2025-12-11   SHORT    277.91   278.61 ATR        274.56 Trailing Stop      +3.35
2025-12-12   LONG     279.10   278.33 ATR        278.33 Initial St

In [2]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import inspect
from orb_core import check_stop_hit
print(inspect.getsource(check_stop_hit))

def check_stop_hit(current_bar: pd.Series, position: PositionState) -> bool:
    """
    Check if stop was hit on this bar.
    
    For LONG: stop hit if low <= current_stop
    For SHORT: stop hit if high >= current_stop
    
    CRITICAL: BE stop must NOT fire if position is still in profit.
    This allows EMA exit to take priority over BE.
    (From Surgeon's analysis of Pine behavior)
    """
    if position.is_long:
        if current_bar['low'] <= position.current_stop:
            # Block BE exit if still in profit
            if (position.stop_moved_to_be 
                and not position.trailing_activated
                and current_bar['close'] > position.entry_price):
                return False
            return True
    else:
        if current_bar['high'] >= position.current_stop:
            # Block BE exit if still in profit
            if (position.stop_moved_to_be 
                and not position.trailing_activated
                and current_bar['close'] < posi

In [3]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
from orb_backtester import ORBBacktester
from orb_core import (calc_atr_rth, select_best_stop, build_orb, 
                      check_long_breakout, check_short_breakout,
                      PositionState, check_stop_hit, check_break_even_hit)

bt = ORBBacktester(symbol='AAPL', start_date='2025-12-04', end_date='2025-12-04', verbose=False)
bt.load_data()
df = bt.df

day_df = df[df.index.date == pd.Timestamp('2025-12-04').date()]
atr_series = calc_atr_rth(df, period=14)

# Simulate the trade
entry_price = 283.05
initial_stop = 283.93
risk = initial_stop - entry_price  # 0.88

position = PositionState(
    entry_price=entry_price,
    initial_stop=initial_stop,
    stop_type="ATR",
    is_long=False,
    risk=risk,
    target_rr=1.5
)

print(f"Entry: SHORT @ {entry_price}, Stop: {initial_stop}, Risk: {risk:.2f}")
print(f"BE target (0.5R): {entry_price - risk * 0.5:.2f}")
print()

# Walk through bars after entry
for ts, bar in day_df.iterrows():
    if ts.hour < 9 or (ts.hour == 9 and ts.minute < 38):
        continue
    if ts.hour > 10 or (ts.hour == 10 and ts.minute > 10):
        break
    
    atr = atr_series.loc[ts]
    
    # Check BE hit
    be_hit = check_break_even_hit(bar['close'], entry_price, risk, False, 0.5)
    
    # Check stop hit  
    stop_hit = check_stop_hit(bar, position)
    
    print(f"{ts.strftime('%H:%M')} C={bar['close']:.2f} H={bar['high']:.2f} | "
          f"BE_armed={position.stop_moved_to_be} stop={position.current_stop:.2f} | "
          f"be_hit={be_hit} stop_hit={stop_hit}")
    
    # Simulate BE arming
    if be_hit and not position.stop_moved_to_be:
        position.stop_moved_to_be = True
        position.current_stop = entry_price
        print(f"  >>> BE ARMED, stop moved to {entry_price}")
    
    if stop_hit:
        print(f"  >>> STOP HIT - would exit here")
        break

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AAPL (1-min bars, 180 days, extended)...
  ✓ Loading from cache (1.3 hours old)
Entry: SHORT @ 283.05, Stop: 283.93, Risk: 0.88
BE target (0.5R): 282.61

09:38 C=282.88 H=283.02 | BE_armed=False stop=283.93 | be_hit=False stop_hit=False
09:39 C=283.06 H=283.13 | BE_armed=False stop=283.93 | be_hit=False stop_hit=False
09:40 C=282.58 H=283.30 | BE_armed=False stop=283.93 | be_hit=True stop_hit=False
  >>> BE ARMED, stop moved to 283.05
09:41 C=282.57 H=282.73 | BE_armed=True stop=283.05 | be_hit=True stop_hit=False
09:42 C=282.72 H=282.76 | BE_armed=True stop=283.05 | be_hit=False stop_hit=False
09:43 C=283.22 H=283.24 | BE_armed=True stop=283.05 | be_hit=False stop_hit=True
  >>> STOP HIT - would exit here


In [4]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
from orb_backtester import ORBBacktester

bt = ORBBacktester(symbol='AAPL', start_date='2025-12-04', end_date='2025-12-04', verbose=False)
bt.load_data()
df = bt.df

day_df = df[df.index.date == pd.Timestamp('2025-12-04').date()]

# Show bars around 09:43
for ts, bar in day_df.iterrows():
    if ts.hour == 9 and 40 <= ts.minute <= 45:
        print(f"{ts.strftime('%H:%M')} O={bar['open']:.2f} H={bar['high']:.2f} L={bar['low']:.2f} C={bar['close']:.2f}")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AAPL (1-min bars, 180 days, extended)...
  ✓ Loading from cache (1.7 hours old)
09:40 O=283.09 H=283.30 L=282.57 C=282.58
09:41 O=282.58 H=282.73 L=282.36 C=282.57
09:42 O=282.55 H=282.76 L=282.32 C=282.72
09:43 O=282.75 H=283.24 L=282.69 C=283.22
09:44 O=283.22 H=283.22 L=282.77 C=282.77
09:45 O=282.77 H=282.77 L=282.42 C=282.44


In [5]:
"""
Full December Validation with Exit Management + EMA Exit
AAPL December 2025 - Multiple trades per day allowed
"""
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
from orb_backtester import ORBBacktester
from orb_core import (calc_atr_rth, select_best_stop, build_orb, 
                      check_long_breakout, check_short_breakout,
                      PositionState, manage_position_exit)

# Load data
bt = ORBBacktester(symbol='AAPL', start_date='2025-12-01', end_date='2025-12-31', verbose=False)
bt.load_data()
df = bt.df

# Get ATR series
atr_series = calc_atr_rth(df, period=14)

# Calculate EMA9
df['ema9'] = df['close'].ewm(span=9, adjust=False).mean()

print("═══ DECEMBER 2025 AAPL - MULTIPLE TRADES PER DAY ═══")
print(f"{'Date':<12} {'Dir':<6} {'Entry':>8} {'Stop':>8} {'Type':<8} {'Exit':>8} {'Reason':<15} {'P&L':>8}")
print("=" * 85)

total_trades = 0

# Process each trading day
for date in pd.date_range('2025-12-01', '2025-12-31', freq='B'):
    day_df = df[df.index.date == date.date()]
    if len(day_df) == 0:
        continue
    
    # Build ORB
    orb = build_orb(day_df)
    if not orb.orb_complete:
        continue
    
    position = None
    last_exit_bar = -10  # Track last exit to prevent immediate re-entry
    
    # Scan for breakouts and manage position
    for bar_num, (ts, bar) in enumerate(day_df.iterrows()):
        mins = ts.hour * 60 + ts.minute
        
        # Skip pre-RTH
        if mins < 9*60+30:
            continue
        
        atr = atr_series.loc[ts]
        if pd.isna(atr):
            continue
        
        ema = bar['ema9']
        
        # If no position, look for entry (09:35 - 10:30)
        if position is None and mins >= 9*60+35 and mins <= 10*60+30:
            # Cooldown after exit (at least 5 bars)
            if bar_num - last_exit_bar < 5:
                continue
                
            is_long_breakout = check_long_breakout(
                bar['open'], bar['high'], bar['low'], bar['close'],
                orb.session_high, orb.session_low, atr, orb.orb_complete
            )
            is_short_breakout = check_short_breakout(
                bar['open'], bar['high'], bar['low'], bar['close'],
                orb.session_high, orb.session_low, atr, orb.orb_complete
            )
            
            if is_long_breakout or is_short_breakout:
                is_long = is_long_breakout
                entry_price = bar['close']
                vwap = bar['vwap']
                bar_idx = day_df.index.get_loc(ts)
                
                best_stop, stop_type, best_rr = select_best_stop(
                    entry_price, atr, vwap, is_long, day_df, bar_idx
                )
                
                position = PositionState(
                    entry_price=entry_price,
                    initial_stop=best_stop,
                    stop_type=stop_type,
                    is_long=is_long,
                    risk=abs(entry_price - best_stop),
                    target_rr=best_rr
                )
                direction = "LONG" if is_long else "SHORT"
        
        # If in position, manage exit
        elif position is not None:
            is_closed, exit_price, exit_reason = manage_position_exit(
                position, bar, atr,
                break_even_rr=0.5, profit_target_rr=2.0,
                trailing_stop_distance=1.2,
                use_break_even=True, use_trailing=True,
                use_ema_exit=True, ema=ema, ema_confirmation_bars=1,
                use_aggressive_trailing=True, ema_tighten_zone=0.3, tightened_trail_distance=0.3
            )
            
            if is_closed:
                if position.is_long:
                    pnl = exit_price - position.entry_price
                else:
                    pnl = position.entry_price - exit_price
                
                print(f"{date.strftime('%Y-%m-%d'):<12} {direction:<6} {position.entry_price:>8.2f} {position.initial_stop:>8.2f} {position.stop_type:<8} {exit_price:>8.2f} {exit_reason:<15} {pnl:>+8.2f}")
                position = None
                last_exit_bar = bar_num
                total_trades += 1
                # NO break - continue scanning for re-entry
        
        # EOD close
        if mins >= 15*60+55 and position is not None:
            exit_price = bar['close']
            if position.is_long:
                pnl = exit_price - position.entry_price
            else:
                pnl = position.entry_price - exit_price
            print(f"{date.strftime('%Y-%m-%d'):<12} {direction:<6} {position.entry_price:>8.2f} {position.initial_stop:>8.2f} {position.stop_type:<8} {exit_price:>8.2f} {'EOD Close':<15} {pnl:>+8.2f}")
            position = None
            total_trades += 1

print("=" * 85)
print(f"Total trades: {total_trades}")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AAPL (1-min bars, 180 days, extended)...
  ✓ Loading from cache (6.2 hours old)
═══ DECEMBER 2025 AAPL - MULTIPLE TRADES PER DAY ═══
Date         Dir       Entry     Stop Type         Exit Reason               P&L
2025-12-01   SHORT    277.23   277.93 ATR        277.05 EMA Exit           +0.18
2025-12-01   SHORT    277.25   277.62 VWAP       277.22 EMA Exit           +0.03
2025-12-02   LONG     284.36   283.41 VWAP       284.36 Break Even         +0.00
2025-12-04   SHORT    283.05   283.93 ATR        283.05 Break Even         +0.00
2025-12-05   SHORT    279.35   280.07 ATR        280.07 Initial Stop       -0.72
2025-12-05   SHORT    279.39   280.15 ATR        280.15 Initial Stop       -0.76
2025-12-08   SHORT    277.68   278.53 ATR        277.37 EMA Exit           +0.31
2025-12-09   LONG     279.39   278.74 ATR        279.39 Break Even         +0.00
2025-12-09   SHORT    277.77   278.45 ATR        277.58 EMA Exit

In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
from orb_backtester import ORBBacktester
from orb_core import (calc_atr_rth, select_best_stop, build_orb, 
                      check_short_breakout, PositionState, 
                      check_break_even_hit, check_stop_hit)

bt = ORBBacktester(symbol='AAPL', start_date='2025-12-22', end_date='2025-12-22', verbose=False)
bt.load_data()
df = bt.df

day_df = df[df.index.date == pd.Timestamp('2025-12-22').date()]
atr_series = calc_atr_rth(df, period=14)
df['ema9'] = df['close'].ewm(span=9, adjust=False).mean()

# From output: SHORT @ 272.47, Swing Stop @ 273.24
entry_price = 272.47
initial_stop = 273.24
risk = initial_stop - entry_price  # 0.77

position = PositionState(
    entry_price=entry_price,
    initial_stop=initial_stop,
    stop_type="Swing",
    is_long=False,
    risk=risk,
    target_rr=1.5
)

be_target = entry_price - (risk * 0.5)
print(f"Entry: SHORT @ {entry_price}, Stop: {initial_stop}, Risk: {risk:.2f}")
print(f"BE target (0.5R): {be_target:.2f}")
print()

# Find entry bar and trace forward
entry_found = False
for ts, bar in day_df.iterrows():
    if ts.hour < 9 or (ts.hour == 9 and ts.minute < 35):
        continue
    if ts.hour > 11:
        break
    
    atr = atr_series.loc[ts]
    ema = df.loc[ts, 'ema9']
    
    # Check if this is around entry
    if abs(bar['close'] - entry_price) < 0.05 and not entry_found:
        entry_found = True
        print(f">>> ENTRY BAR: {ts.strftime('%H:%M')}")
        continue
    
    if not entry_found:
        continue
    
    # R-Multiple
    r_mult = (entry_price - bar['close']) / risk
    
    # BE check
    be_hit = bar['close'] <= be_target
    
    # Stop hit check
    stop_hit = bar['high'] >= position.current_stop
    
    # EMA cross (for SHORT: close > ema means crossed)
    ema_cross = bar['close'] > ema
    
    print(f"{ts.strftime('%H:%M')} C={bar['close']:.2f} H={bar['high']:.2f} EMA={ema:.2f} | "
          f"R={r_mult:.2f} BE_armed={position.stop_moved_to_be} stop={position.current_stop:.2f} | "
          f"be_hit={be_hit} stop_hit={stop_hit} ema_cross={ema_cross}")
    
    # Simulate BE arming
    if be_hit and not position.stop_moved_to_be:
        position.stop_moved_to_be = True
        position.current_stop = entry_price
        print(f"  >>> BE ARMED, stop moved to {entry_price}")
    
    # Check for exits
    if stop_hit and not (position.stop_moved_to_be and bar['close'] < entry_price):
        print(f"  >>> STOP HIT @ {position.current_stop:.2f}")
        break
    
    if position.stop_moved_to_be and ema_cross and bar['close'] < entry_price:
        position.bars_beyond_ema += 1
        if position.bars_beyond_ema >= 1:
            print(f"  >>> EMA EXIT @ {bar['close']:.2f}")
            break
    else:
        position.bars_beyond_ema = 0

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AAPL (1-min bars, 180 days, extended)...
  ✓ Loading from cache (6.4 hours old)
Entry: SHORT @ 272.47, Stop: 273.24, Risk: 0.77
BE target (0.5R): 272.09

>>> ENTRY BAR: 09:45
09:46 C=272.62 H=272.63 EMA=272.78 | R=-0.20 BE_armed=False stop=273.24 | be_hit=False stop_hit=False ema_cross=False
09:47 C=272.60 H=272.74 EMA=272.74 | R=-0.16 BE_armed=False stop=273.24 | be_hit=False stop_hit=False ema_cross=False
09:48 C=272.40 H=272.64 EMA=272.67 | R=0.09 BE_armed=False stop=273.24 | be_hit=False stop_hit=False ema_cross=False
09:49 C=272.19 H=272.50 EMA=272.58 | R=0.36 BE_armed=False stop=273.24 | be_hit=False stop_hit=False ema_cross=False
09:50 C=272.56 H=272.58 EMA=272.57 | R=-0.12 BE_armed=False stop=273.24 | be_hit=False stop_hit=False ema_cross=False
09:51 C=272.88 H=272.88 EMA=272.63 | R=-0.53 BE_armed=False stop=273.24 | be_hit=False stop_hit=False ema_cross=True
09:52 C=272.83 H=273.02 EMA=272.67 | R=-0.47 B

In [1]:
"""
Full December Validation with Exit Management + EMA Exit
AAPL December 2025 - Multiple trades per day allowed
"""
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
from orb_backtester import ORBBacktester
from orb_core import (calc_atr_rth, select_best_stop, build_orb, 
                      check_long_breakout, check_short_breakout,
                      PositionState, manage_position_exit)

# Load data
bt = ORBBacktester(symbol='AAPL', start_date='2025-12-01', end_date='2025-12-31', verbose=False)
bt.load_data()
df = bt.df

# Get ATR series
atr_series = calc_atr_rth(df, period=14)

# Calculate EMA9
df['ema9'] = df['close'].ewm(span=9, adjust=False).mean()

print("═══ DECEMBER 2025 AAPL - MULTIPLE TRADES PER DAY ═══")
print(f"{'Date':<12} {'Dir':<6} {'Entry':>8} {'Stop':>8} {'Type':<8} {'Exit':>8} {'Reason':<15} {'P&L':>8}")
print("=" * 85)

total_trades = 0

# Process each trading day
for date in pd.date_range('2025-12-01', '2025-12-31', freq='B'):
    day_df = df[df.index.date == date.date()]
    if len(day_df) == 0:
        continue
    
    # Build ORB
    orb = build_orb(day_df)
    if not orb.orb_complete:
        continue
    
    position = None
    last_exit_bar = -10  # Track last exit to prevent immediate re-entry
    
    # Scan for breakouts and manage position
    for bar_num, (ts, bar) in enumerate(day_df.iterrows()):
        mins = ts.hour * 60 + ts.minute
        
        # Skip pre-RTH
        if mins < 9*60+30:
            continue
        
        atr = atr_series.loc[ts]
        if pd.isna(atr):
            continue
        
        ema = bar['ema9']
        
        # If no position, look for entry (09:35 - 10:30)
        if position is None and mins >= 9*60+35 and mins <= 10*60+30:
            # Cooldown after exit (at least 5 bars)
            if bar_num - last_exit_bar < 5:
                continue
                
            is_long_breakout = check_long_breakout(
                bar['open'], bar['high'], bar['low'], bar['close'],
                orb.session_high, orb.session_low, atr, orb.orb_complete
            )
            is_short_breakout = check_short_breakout(
                bar['open'], bar['high'], bar['low'], bar['close'],
                orb.session_high, orb.session_low, atr, orb.orb_complete
            )
            
            if is_long_breakout or is_short_breakout:
                is_long = is_long_breakout
                entry_price = bar['close']
                vwap = bar['vwap']
                bar_idx = day_df.index.get_loc(ts)
                
                best_stop, stop_type, best_rr = select_best_stop(
                    entry_price, atr, vwap, is_long, day_df, bar_idx
                )
                
                position = PositionState(
                    entry_price=entry_price,
                    initial_stop=best_stop,
                    stop_type=stop_type,
                    is_long=is_long,
                    risk=abs(entry_price - best_stop),
                    target_rr=best_rr
                )
                direction = "LONG" if is_long else "SHORT"
        
        # If in position, manage exit
        elif position is not

SyntaxError: invalid syntax (1498724173.py, line 94)

In [1]:
"""
Full December Validation with Exit Management + EMA Exit
AAPL December 2025 - Multiple trades per day allowed
"""
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
from orb_backtester import ORBBacktester
from orb_core import (calc_atr_rth, select_best_stop, build_orb, 
                      check_long_breakout, check_short_breakout,
                      PositionState, manage_position_exit)

bt = ORBBacktester(symbol='AAPL', start_date='2025-12-01', end_date='2025-12-31', verbose=False)
bt.load_data()
df = bt.df
atr_series = calc_atr_rth(df, period=14)
df['ema9'] = df['close'].ewm(span=9, adjust=False).mean()

print("═══ DECEMBER 2025 AAPL - MULTIPLE TRADES PER DAY ═══")
print(f"{'Date':<12} {'Dir':<6} {'Entry':>8} {'Stop':>8} {'Type':<8} {'Exit':>8} {'Reason':<15} {'P&L':>8}")
print("=" * 85)

total_trades = 0

for date in pd.date_range('2025-12-01', '2025-12-31', freq='B'):
    day_df = df[df.index.date == date.date()]
    if len(day_df) == 0:
        continue
    orb = build_orb(day_df)
    if not orb.orb_complete:
        continue
    
    position = None
    last_exit_bar = -10
    
    for bar_num, (ts, bar) in enumerate(day_df.iterrows()):
        mins = ts.hour * 60 + ts.minute
        if mins < 9*60+30:
            continue
        atr = atr_series.loc[ts]
        if pd.isna(atr):
            continue
        ema = bar['ema9']
        
        if position is None and mins >= 9*60+35 and mins <= 10*60+30:
            if bar_num - last_exit_bar < 5:
                continue
            is_long_breakout = check_long_breakout(bar['open'], bar['high'], bar['low'], bar['close'],
                orb.session_high, orb.session_low, atr, orb.orb_complete)
            is_short_breakout = check_short_breakout(bar['open'], bar['high'], bar['low'], bar['close'],
                orb.session_high, orb.session_low, atr, orb.orb_complete)
            
            if is_long_breakout or is_short_breakout:
                is_long = is_long_breakout
                entry_price = bar['close']
                vwap = bar['vwap']
                bar_idx = day_df.index.get_loc(ts)
                best_stop, stop_type, best_rr = select_best_stop(entry_price, atr, vwap, is_long, day_df, bar_idx)
                position = PositionState(entry_price=entry_price, initial_stop=best_stop, stop_type=stop_type,
                    is_long=is_long, risk=abs(entry_price - best_stop), target_rr=best_rr)
                direction = "LONG" if is_long else "SHORT"
        
        elif position is not None:
            is_closed, exit_price, exit_reason = manage_position_exit(position, bar, atr,
                break_even_rr=0.5, profit_target_rr=2.0, trailing_stop_distance=1.2,
                use_break_even=True, use_trailing=True, use_ema_exit=True, ema=ema, ema_confirmation_bars=1,
                use_aggressive_trailing=True, ema_tighten_zone=0.3, tightened_trail_distance=0.3)
            
            if is_closed:
                pnl = (exit_price - position.entry_price) if position.is_long else (position.entry_price - exit_price)
                print(f"{date.strftime('%Y-%m-%d'):<12} {direction:<6} {position.entry_price:>8.2f} {position.initial_stop:>8.2f} {position.stop_type:<8} {exit_price:>8.2f} {exit_reason:<15} {pnl:>+8.2f}")
                position = None
                last_exit_bar = bar_num
                total_trades += 1
        
        if mins >= 15*60+55 and position is not None:
            exit_price = bar['close']
            pnl = (exit_price - position.entry_price) if position.is_long else (position.entry_price - exit_price)
            print(f"{date.strftime('%Y-%m-%d'):<12} {direction:<6} {position.entry_price:>8.2f} {position.initial_stop:>8.2f} {position.stop_type:<8} {exit_price:>8.2f} {'EOD Close':<15} {pnl:>+8.2f}")
            position = None
            total_trades += 1

print("=" * 85)
print(f"Total trades: {total_trades}")Claude is AI and can make mistakes. Please double-check responses. Opus 4.5Claude is AI and can make mistakes. Please double-check 

SyntaxError: invalid decimal literal (3174011338.py, line 84)

In [2]:
"""
Full December Validation with Exit Management + EMA Exit
AAPL December 2025 - Multiple trades per day allowed
"""
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
from orb_backtester import ORBBacktester
from orb_core import (calc_atr_rth, select_best_stop, build_orb, 
                      check_long_breakout, check_short_breakout,
                      PositionState, manage_position_exit)

bt = ORBBacktester(symbol='AAPL', start_date='2025-12-01', end_date='2025-12-31', verbose=False)
bt.load_data()
df = bt.df
atr_series = calc_atr_rth(df, period=14)
df['ema9'] = df['close'].ewm(span=9, adjust=False).mean()

print("═══ DECEMBER 2025 AAPL - MULTIPLE TRADES PER DAY ═══")
print(f"{'Date':<12} {'Dir':<6} {'Entry':>8} {'Stop':>8} {'Type':<8} {'Exit':>8} {'Reason':<15} {'P&L':>8}")
print("=" * 85)

total_trades = 0

for date in pd.date_range('2025-12-01', '2025-12-31', freq='B'):
    day_df = df[df.index.date == date.date()]
    if len(day_df) == 0:
        continue
    orb = build_orb(day_df)
    if not orb.orb_complete:
        continue
    
    position = None
    last_exit_bar = -10
    
    for bar_num, (ts, bar) in enumerate(day_df.iterrows()):
        mins = ts.hour * 60 + ts.minute
        if mins < 9*60+30:
            continue
        atr = atr_series.loc[ts]
        if pd.isna(atr):
            continue
        ema = bar['ema9']
        
        if position is None and mins >= 9*60+35 and mins <= 10*60+30:
            if bar_num - last_exit_bar < 5:
                continue
            is_long_breakout = check_long_breakout(bar['open'], bar['high'], bar['low'], bar['close'],
                orb.session_high, orb.session_low, atr, orb.orb_complete)
            is_short_breakout = check_short_breakout(bar['open'], bar['high'], bar['low'], bar['close'],
                orb.session_high, orb.session_low, atr, orb.orb_complete)
            
            if is_long_breakout or is_short_breakout:
                is_long = is_long_breakout
                entry_price = bar['close']
                vwap = bar['vwap']
                bar_idx = day_df.index.get_loc(ts)
                best_stop, stop_type, best_rr = select_best_stop(entry_price, atr, vwap, is_long, day_df, bar_idx)
                position = PositionState(entry_price=entry_price, initial_stop=best_stop, stop_type=stop_type,
                    is_long=is_long, risk=abs(entry_price - best_stop), target_rr=best_rr)
                direction = "LONG" if is_long else "SHORT"
        
        elif position is not None:
            is_closed, exit_price, exit_reason = manage_position_exit(position, bar, atr,
                break_even_rr=0.5, profit_target_rr=2.0, trailing_stop_distance=1.2,
                use_break_even=True, use_trailing=True, use_ema_exit=True, ema=ema, ema_confirmation_bars=1,
                use_aggressive_trailing=True, ema_tighten_zone=0.3, tightened_trail_distance=0.3)
            
            if is_closed:
                pnl = (exit_price - position.entry_price) if position.is_long else (position.entry_price - exit_price)
                print(f"{date.strftime('%Y-%m-%d'):<12} {direction:<6} {position.entry_price:>8.2f} {position.initial_stop:>8.2f} {position.stop_type:<8} {exit_price:>8.2f} {exit_reason:<15} {pnl:>+8.2f}")
                position = None
                last_exit_bar = bar_num
                total_trades += 1
        
        if mins >= 15*60+55 and position is not None:
            exit_price = bar['close']
            pnl = (exit_price - position.entry_price) if position.is_long else (position.entry_price - exit_price)
            print(f"{date.strftime('%Y-%m-%d'):<12} {direction:<6} {position.entry_price:>8.2f} {position.initial_stop:>8.2f} {position.stop_type:<8} {exit_price:>8.2f} {'EOD Close':<15} {pnl:>+8.2f}")
            position = None
            total_trades += 1

print("=" * 85)
print(f"Total trades: {total_trades}")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AAPL (1-min bars, 180 days, extended)...
  ✓ Loading from cache (6.5 hours old)
═══ DECEMBER 2025 AAPL - MULTIPLE TRADES PER DAY ═══
Date         Dir       Entry     Stop Type         Exit Reason               P&L
2025-12-01   SHORT    277.23   277.93 ATR        277.05 EMA Exit           +0.18
2025-12-01   SHORT    277.25   277.62 VWAP       277.22 EMA Exit           +0.03
2025-12-02   LONG     284.36   283.41 VWAP       284.36 Break Even         +0.00
2025-12-04   SHORT    283.05   283.93 ATR        283.05 Break Even         +0.00
2025-12-05   SHORT    279.35   280.07 ATR        280.07 Initial Stop       -0.72
2025-12-05   SHORT    279.39   280.15 ATR        280.15 Initial Stop       -0.76
2025-12-08   SHORT    277.68   278.53 ATR        277.37 EMA Exit           +0.31
2025-12-09   LONG     279.39   278.74 ATR        279.39 Break Even         +0.00
2025-12-09   SHORT    277.77   278.45 ATR        277.58 EMA Exit

In [3]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
from orb_backtester import ORBBacktester
from orb_core import (calc_atr_rth, build_orb, check_short_breakout)

bt = ORBBacktester(symbol='AAPL', start_date='2025-12-30', end_date='2025-12-30', verbose=False)
bt.load_data()
df = bt.df

day_df = df[df.index.date == pd.Timestamp('2025-12-30').date()]
atr_series = calc_atr_rth(df, period=14)
orb = build_orb(day_df)

print(f"ORB High: {orb.session_high:.2f}, Low: {orb.session_low:.2f}")
print()

# Scan for breakouts
for ts, bar in day_df.iterrows():
    mins = ts.hour * 60 + ts.minute
    if mins < 9*60+35 or mins > 10*60+30:
        continue
    
    atr = atr_series.loc[ts]
    if pd.isna(atr):
        continue
    
    is_short = check_short_breakout(bar['open'], bar['high'], bar['low'], bar['close'],
        orb.session_high, orb.session_low, atr, orb.orb_complete)
    
    if is_short:
        print(f"{ts.strftime('%H:%M')} SHORT BREAKOUT: O={bar['open']:.2f} H={bar['high']:.2f} L={bar['low']:.2f} C={bar['close']:.2f}")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AAPL (1-min bars, 180 days, extended)...
  ✓ Loading from cache (6.6 hours old)
ORB High: 274.08, Low: 272.72

09:35 SHORT BREAKOUT: O=272.81 H=272.84 L=272.54 C=272.54
10:24 SHORT BREAKOUT: O=272.79 H=272.79 L=272.64 C=272.64


In [1]:
"""
December Validation with Skip Logic
AAPL December 2025 - Shows SKIPs when R:R < min_acceptable_rr
"""
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
from orb_backtester import ORBBacktester
from orb_core import (calc_atr_rth, select_best_stop, build_orb, 
                      check_long_breakout, check_short_breakout,
                      PositionState, manage_position_exit, check_trade_feasibility)

bt = ORBBacktester(symbol='AAPL', start_date='2025-12-01', end_date='2025-12-31', verbose=False)
bt.load_data()
df = bt.df
atr_series = calc_atr_rth(df, period=14)
df['ema9'] = df['close'].ewm(span=9, adjust=False).mean()

MIN_ACCEPTABLE_RR = 1.5  # THE GATE

print("═══ DECEMBER 2025 AAPL - WITH SKIP LOGIC ═══")
print(f"Min Acceptable R:R: {MIN_ACCEPTABLE_RR}")
print(f"{'Date':<12} {'Dir':<6} {'Entry':>8} {'Stop':>8} {'Type':<8} {'R:R':>6} {'Exit':>8} {'Reason':<15} {'P&L':>8}")
print("=" * 95)

total_trades = 0
total_skips = 0

for date in pd.date_range('2025-12-01', '2025-12-31', freq='B'):
    day_df = df[df.index.date == date.date()]
    if len(day_df) == 0:
        continue
    orb = build_orb(day_df)
    if not orb.orb_complete:
        continue
    
    position = None
    last_exit_bar = -10
    
    for bar_num, (ts, bar) in enumerate(day_df.iterrows()):
        mins = ts.hour * 60 + ts.minute
        if mins < 9*60+30:
            continue
        atr = atr_series.loc[ts]
        if pd.isna(atr):
            continue
        ema = bar['ema9']
        
        if position is None and mins >= 9*60+35 and mins <= 10*60+30:
            if bar_num - last_exit_bar < 5:
                continue
            is_long_breakout = check_long_breakout(bar['open'], bar['high'], bar['low'], bar['close'],
                orb.session_high, orb.session_low, atr, orb.orb_complete)
            is_short_breakout = check_short_breakout(bar['open'], bar['high'], bar['low'], bar['close'],
                orb.session_high, orb.session_low, atr, orb.orb_complete)
            
            if is_long_breakout or is_short_breakout:
                is_long = is_long_breakout
                entry_price = bar['close']
                vwap = bar['vwap']
                bar_idx = day_df.index.get_loc(ts)
                best_stop, stop_type, achievable_rr = select_best_stop(entry_price, atr, vwap, is_long, day_df, bar_idx)
                
                # THE GATE - Check if trade is feasible
                is_feasible, should_skip = check_trade_feasibility(achievable_rr, MIN_ACCEPTABLE_RR)
                
                if should_skip:
                    direction = "LONG" if is_long else "SHORT"
                    print(f"{date.strftime('%Y-%m-%d'):<12} {direction:<6} {entry_price:>8.2f} {best_stop:>8.2f} {stop_type:<8} {achievable_rr:>6.2f} {'---':>8} {'SKIP':^15} {'---':>8}")
                    total_skips += 1
                    last_exit_bar = bar_num  # Consume the breakout
                    continue
                
                position = PositionState(entry_price=entry_price, initial_stop=best_stop, stop_type=stop_type,
                    is_long=is_long, risk=abs(entry_price - best_stop), target_rr=achievable_rr)
                direction = "LONG" if is_long else "SHORT"
        
        elif position is not None:
            is_closed, exit_price, exit_reason = manage_position_exit(position, bar, atr,
                break_even_rr=0.5, profit_target_rr=2.0, trailing_stop_distance=1.2,
                use_break_even=True, use_trailing=True, use_ema_exit=True, ema=ema, ema_confirmation_bars=1,
                use_aggressive_trailing=True, ema_tighten_zone=0.3, tightened_trail_distance=0.3)
            
            if is_closed:
                pnl = (exit_price - position.entry_price) if position.is_long else (position.entry_price - exit_price)
                print(f"{date.strftime('%Y-%m-%d'):<12} {direction:<6} {position.entry_price:>8.2f} {position.initial_stop:>8.2f} {position.stop_type:<8} {position.target_rr:>6.2f} {exit_price:>8.2f} {exit_reason:<15} {pnl:>+8.2f}")
                position = None
                last_exit_bar = bar_num
                total_trades += 1
        
        if mins >= 15*60+55 and position is not None:
            exit_price = bar['close']
            pnl = (exit_price - position.entry_price) if position.is_long else (position.entry_price - exit_price)
            print(f"{date.strftime('%Y-%m-%d'):<12} {direction:<6} {position.entry_price:>8.2f} {position.initial_stop:>8.2f} {position.stop_type:<8} {position.target_rr:>6.2f} {exit_price:>8.2f} {'EOD Close':<15} {pnl:>+8.2f}")
            position = None
            total_trades += 1

print("=" * 95)
print(f"Total trades: {total_trades}, Total skips: {total_skips}")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AAPL (1-min bars, 180 days, extended)...
  ⚠ Cache stale (14 hours), refreshing...
  ✓ Fetched 89842 bars, cached for next time
═══ DECEMBER 2025 AAPL - WITH SKIP LOGIC ═══
Min Acceptable R:R: 1.5
Date         Dir       Entry     Stop Type        R:R     Exit Reason               P&L
2025-12-01   SHORT    277.23   277.93 ATR        1.50   277.05 EMA Exit           +0.18
2025-12-01   SHORT    277.25   277.62 VWAP       2.00   277.22 EMA Exit           +0.03
2025-12-02   LONG     284.36   283.41 VWAP       1.57   284.36 Break Even         +0.00
2025-12-04   SHORT    283.05   283.93 ATR        1.50   283.05 Break Even         +0.00
2025-12-05   SHORT    279.35   280.07 ATR        1.50   280.07 Initial Stop       -0.72
2025-12-05   SHORT    279.39   280.15 ATR        1.50   280.15 Initial Stop       -0.76
2025-12-08   SHORT    277.68   278.53 ATR        1.50   277.37 EMA Exit           +0.31
2025-12-09   LONG     279

In [2]:
"""
January 2025 AMD Validation with Skip Logic
"""
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
from orb_backtester import ORBBacktester
from orb_core import (calc_atr_rth, select_best_stop, build_orb, 
                      check_long_breakout, check_short_breakout,
                      PositionState, manage_position_exit, check_trade_feasibility)

bt = ORBBacktester(symbol='AMD', start_date='2025-01-01', end_date='2025-01-16', verbose=False)
bt.load_data()
df = bt.df
atr_series = calc_atr_rth(df, period=14)
df['ema9'] = df['close'].ewm(span=9, adjust=False).mean()

MIN_ACCEPTABLE_RR = 1.5  # THE GATE

print("═══ JANUARY 2025 AMD - WITH SKIP LOGIC ═══")
print(f"Min Acceptable R:R: {MIN_ACCEPTABLE_RR}")
print(f"{'Date':<12} {'Dir':<6} {'Entry':>8} {'Stop':>8} {'Type':<8} {'R:R':>6} {'Exit':>8} {'Reason':<15} {'P&L':>8}")
print("=" * 95)

total_trades = 0
total_skips = 0

for date in pd.date_range('2025-01-01', '2025-01-16', freq='B'):
    day_df = df[df.index.date == date.date()]
    if len(day_df) == 0:
        continue
    orb = build_orb(day_df)
    if not orb.orb_complete:
        continue
    
    position = None
    last_exit_bar = -10
    
    for bar_num, (ts, bar) in enumerate(day_df.iterrows()):
        mins = ts.hour * 60 + ts.minute
        if mins < 9*60+30:
            continue
        atr = atr_series.loc[ts]
        if pd.isna(atr):
            continue
        ema = bar['ema9']
        
        if position is None and mins >= 9*60+35 and mins <= 10*60+30:
            if bar_num - last_exit_bar < 5:
                continue
            is_long_breakout = check_long_breakout(bar['open'], bar['high'], bar['low'], bar['close'],
                orb.session_high, orb.session_low, atr, orb.orb_complete)
            is_short_breakout = check_short_breakout(bar['open'], bar['high'], bar['low'], bar['close'],
                orb.session_high, orb.session_low, atr, orb.orb_complete)
            
            if is_long_breakout or is_short_breakout:
                is_long = is_long_breakout
                entry_price = bar['close']
                vwap = bar['vwap']
                bar_idx = day_df.index.get_loc(ts)
                best_stop, stop_type, achievable_rr = select_best_stop(entry_price, atr, vwap, is_long, day_df, bar_idx)
                
                # THE GATE - Check if trade is feasible
                is_feasible, should_skip = check_trade_feasibility(achievable_rr, MIN_ACCEPTABLE_RR)
                
                if should_skip:
                    direction = "LONG" if is_long else "SHORT"
                    print(f"{date.strftime('%Y-%m-%d'):<12} {direction:<6} {entry_price:>8.2f} {best_stop:>8.2f} {stop_type:<8} {achievable_rr:>6.2f} {'---':>8} {'SKIP':^15} {'---':>8}")
                    total_skips += 1
                    last_exit_bar = bar_num  # Consume the breakout
                    continue
                
                position = PositionState(entry_price=entry_price, initial_stop=best_stop, stop_type=stop_type,
                    is_long=is_long, risk=abs(entry_price - best_stop), target_rr=achievable_rr)
                direction = "LONG" if is_long else "SHORT"
        
        elif position is not None:
            is_closed, exit_price, exit_reason = manage_position_exit(position, bar, atr,
                break_even_rr=0.5, profit_target_rr=2.0, trailing_stop_distance=1.2,
                use_break_even=True, use_trailing=True, use_ema_exit=True, ema=ema, ema_confirmation_bars=1,
                use_aggressive_trailing=True, ema_tighten_zone=0.3, tightened_trail_distance=0.3)
            
            if is_closed:
                pnl = (exit_price - position.entry_price) if position.is_long else (position.entry_price - exit_price)
                print(f"{date.strftime('%Y-%m-%d'):<12} {direction:<6} {position.entry_price:>8.2f} {position.initial_stop:>8.2f} {position.stop_type:<8} {position.target_rr:>6.2f} {exit_price:>8.2f} {exit_reason:<15} {pnl:>+8.2f}")
                position = None
                last_exit_bar = bar_num
                total_trades += 1
        
        if mins >= 15*60+55 and position is not None:
            exit_price = bar['close']
            pnl = (exit_price - position.entry_price) if position.is_long else (position.entry_price - exit_price)
            print(f"{date.strftime('%Y-%m-%d'):<12} {direction:<6} {position.entry_price:>8.2f} {position.initial_stop:>8.2f} {position.stop_type:<8} {position.target_rr:>6.2f} {exit_price:>8.2f} {'EOD Close':<15} {pnl:>+8.2f}")
            position = None
            total_trades += 1

print("=" * 95)
print(f"Total trades: {total_trades}, Total skips: {total_skips}")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ⚠ Cache stale (14 hours), refreshing...
  ✓ Fetched 103124 bars, cached for next time
═══ JANUARY 2025 AMD - WITH SKIP LOGIC ═══
Min Acceptable R:R: 1.5
Date         Dir       Entry     Stop Type        R:R     Exit Reason               P&L
Total trades: 0, Total skips: 0


In [3]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
from orb_backtester import ORBBacktester
from orb_core import calc_atr_rth, build_orb, check_long_breakout, check_short_breakout

bt = ORBBacktester(symbol='AMD', start_date='2025-01-01', end_date='2025-01-16', verbose=False)
bt.load_data()
df = bt.df

print(f"Total rows: {len(df)}")
print(f"Date range: {df.index.min()} to {df.index.max()}")
print()

atr_series = calc_atr_rth(df, period=14)

for date in pd.date_range('2025-01-01', '2025-01-16', freq='B'):
    day_df = df[df.index.date == date.date()]
    if len(day_df) == 0:
        print(f"{date.strftime('%Y-%m-%d')}: No data")
        continue
    
    orb = build_orb(day_df)
    print(f"{date.strftime('%Y-%m-%d')}: {len(day_df)} bars, ORB complete={orb.orb_complete}, High={orb.session_high:.2f if orb.session_high else 'N/A'}, Low={orb.session_low:.2f if orb.session_low else 'N/A'}")
    
    # Check for breakouts
    breakouts = []
    for ts, bar in day_df.iterrows():
        mins = ts.hour * 60 + ts.minute
        if mins < 9*60+35 or mins > 10*60+30:
            continue
        atr = atr_series.loc[ts]
        if pd.isna(atr):
            continue
        
        is_long = check_long_breakout(bar['open'], bar['high'], bar['low'], bar['close'],
            orb.session_high, orb.session_low, atr, orb.orb_complete)
        is_short = check_short_breakout(bar['open'], bar['high'], bar['low'], bar['close'],
            orb.session_high, orb.session_low, atr, orb.orb_complete)
        
        if is_long:
            breakouts.append(f"LONG @ {ts.strftime('%H:%M')} C={bar['close']:.2f}")
        if is_short:
            breakouts.append(f"SHORT @ {ts.strftime('%H:%M')} C={bar['close']:.2f}")
    
    if breakouts:
        print(f"  Breakouts: {breakouts[:3]}")  # Show first 3

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (0.0 hours old)
Total rows: 103124
Date range: 2025-07-21 04:00:00-04:00 to 2026-01-16 10:47:00-05:00

2025-01-01: No data
2025-01-02: No data
2025-01-03: No data
2025-01-06: No data
2025-01-07: No data
2025-01-08: No data
2025-01-09: No data
2025-01-10: No data
2025-01-13: No data
2025-01-14: No data
2025-01-15: No data
2025-01-16: No data


In [5]:
"""
January 2026 AMD Validation with Skip Logic
"""
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
from orb_backtester import ORBBacktester
from orb_core import (calc_atr_rth, select_best_stop, build_orb, 
                      check_long_breakout, check_short_breakout,
                      PositionState, manage_position_exit, check_trade_feasibility)

bt = ORBBacktester(symbol='AMD', start_date='2026-01-01', end_date='2026-01-16', verbose=False)
bt.load_data()
df = bt.df
atr_series = calc_atr_rth(df, period=14)
df['ema9'] = df['close'].ewm(span=9, adjust=False).mean()

MIN_ACCEPTABLE_RR = 1.5  # THE GATE

print("═══ JANUARY 2026 AMD - WITH SKIP LOGIC ═══")
print(f"Min Acceptable R:R: {MIN_ACCEPTABLE_RR}")
print(f"{'Date':<12} {'Dir':<6} {'Entry':>8} {'Stop':>8} {'Type':<8} {'R:R':>6} {'Exit':>8} {'Reason':<15} {'P&L':>8}")
print("=" * 95)

total_trades = 0
total_skips = 0

for date in pd.date_range('2026-01-01', '2026-01-16', freq='B'):
    day_df = df[df.index.date == date.date()]
    if len(day_df) == 0:
        continue
    orb = build_orb(day_df)
    if not orb.orb_complete:
        continue
    
    position = None
    last_exit_bar = -10
    
    for bar_num, (ts, bar) in enumerate(day_df.iterrows()):
        mins = ts.hour * 60 + ts.minute
        if mins < 9*60+30:
            continue
        atr = atr_series.loc[ts]
        if pd.isna(atr):
            continue
        ema = bar['ema9']
        
        if position is None and mins >= 9*60+35 and mins <= 10*60+30:
            if bar_num - last_exit_bar < 5:
                continue
            is_long_breakout = check_long_breakout(bar['open'], bar['high'], bar['low'], bar['close'],
                orb.session_high, orb.session_low, atr, orb.orb_complete)
            is_short_breakout = check_short_breakout(bar['open'], bar['high'], bar['low'], bar['close'],
                orb.session_high, orb.session_low, atr, orb.orb_complete)
            
            if is_long_breakout or is_short_breakout:
                is_long = is_long_breakout
                entry_price = bar['close']
                vwap = bar['vwap']
                bar_idx = day_df.index.get_loc(ts)
                best_stop, stop_type, achievable_rr = select_best_stop(entry_price, atr, vwap, is_long, day_df, bar_idx)
                
                # THE GATE - Check if trade is feasible
                is_feasible, should_skip = check_trade_feasibility(achievable_rr, MIN_ACCEPTABLE_RR)
                
                if should_skip:
                    direction = "LONG" if is_long else "SHORT"
                    print(f"{date.strftime('%Y-%m-%d'):<12} {direction:<6} {entry_price:>8.2f} {best_stop:>8.2f} {stop_type:<8} {achievable_rr:>6.2f} {'---':>8} {'SKIP':^15} {'---':>8}")
                    total_skips += 1
                    last_exit_bar = bar_num  # Consume the breakout
                    continue
                
                position = PositionState(entry_price=entry_price, initial_stop=best_stop, stop_type=stop_type,
                    is_long=is_long, risk=abs(entry_price - best_stop), target_rr=achievable_rr)
                direction = "LONG" if is_long else "SHORT"
        
        elif position is not None:
            is_closed, exit_price, exit_reason = manage_position_exit(position, bar, atr,
                break_even_rr=0.5, profit_target_rr=2.0, trailing_stop_distance=1.2,
                use_break_even=True, use_trailing=True, use_ema_exit=True, ema=ema, ema_confirmation_bars=1,
                use_aggressive_trailing=True, ema_tighten_zone=0.3, tightened_trail_distance=0.3)
            
            if is_closed:
                pnl = (exit_price - position.entry_price) if position.is_long else (position.entry_price - exit_price)
                print(f"{date.strftime('%Y-%m-%d'):<12} {direction:<6} {position.entry_price:>8.2f} {position.initial_stop:>8.2f} {position.stop_type:<8} {position.target_rr:>6.2f} {exit_price:>8.2f} {exit_reason:<15} {pnl:>+8.2f}")
                position = None
                last_exit_bar = bar_num
                total_trades += 1
        
        if mins >= 15*60+55 and position is not None:
            exit_price = bar['close']
            pnl = (exit_price - position.entry_price) if position.is_long else (position.entry_price - exit_price)
            print(f"{date.strftime('%Y-%m-%d'):<12} {direction:<6} {position.entry_price:>8.2f} {position.initial_stop:>8.2f} {position.stop_type:<8} {position.target_rr:>6.2f} {exit_price:>8.2f} {'EOD Close':<15} {pnl:>+8.2f}")
            position = None
            total_trades += 1

print("=" * 95)
print(f"Total trades: {total_trades}, Total skips: {total_skips}")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (0.1 hours old)
═══ JANUARY 2026 AMD - WITH SKIP LOGIC ═══
Min Acceptable R:R: 1.5
Date         Dir       Entry     Stop Type        R:R     Exit Reason               P&L
2026-01-02   LONG     225.62   223.93 Swing      1.85   226.47 EMA Exit           +0.85
2026-01-05   SHORT    229.60   232.00 ATR        1.50   225.20 Trailing Stop      +4.40
2026-01-06   SHORT    219.16   220.67 ATR        1.50   217.60 EMA Exit           +1.56
2026-01-08   SHORT    208.10   209.16 Swing      1.63   207.82 EMA Exit           +0.28
2026-01-09   SHORT    204.44   205.53 ATR        1.50   204.44 Break Even         +0.00
2026-01-09   SHORT    204.71   204.88 Swing      2.00   204.88 Initial Stop       -0.17
2026-01-12   LONG     204.29   203.02 ATR        1.50   206.41 Trailing Stop      +2.12
2026-01-13   LONG     221.42   219.66 Swing      2.00   219.66 Initial Stop 

In [6]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
from orb_backtester import ORBBacktester
from orb_core import calc_atr_rth, build_orb, calc_stops, calc_achievable_rr

bt = ORBBacktester(symbol='AMD', start_date='2026-01-12', end_date='2026-01-12', verbose=False)
bt.load_data()
df = bt.df
atr_series = calc_atr_rth(df, period=14)

day_df = df[df.index.date == pd.Timestamp('2026-01-12').date()]
orb = build_orb(day_df)

# Find bar at 09:45
for ts, bar in day_df.iterrows():
    if ts.hour == 9 and ts.minute == 45:
        entry = bar['close']
        vwap = bar['vwap']
        atr = atr_series.loc[ts]
        bar_idx = day_df.index.get_loc(ts)
        
        print(f"Time: {ts.strftime('%H:%M')}")
        print(f"Entry: {entry:.2f}")
        print(f"VWAP: {vwap:.2f}")
        print(f"ATR: {atr:.4f}")
        print()
        
        # Calculate stops
        stops = calc_stops(entry, atr, vwap, True, day_df, bar_idx)
        print(f"ATR Stop: {stops['atr_stop']:.2f}")
        print(f"Swing Stop: {stops['swing_stop']:.2f} (valid={stops['swing_valid']})")
        print(f"VWAP Stop: {stops['vwap_stop']:.2f} (va

SyntaxError: unterminated f-string literal (detected at line 33) (1788311967.py, line 33)

In [7]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
from orb_backtester import ORBBacktester
from orb_core import calc_atr_rth, build_orb, calc_stops, calc_achievable_rr

bt = ORBBacktester(symbol='AMD', start_date='2026-01-12', end_date='2026-01-12', verbose=False)
bt.load_data()
df = bt.df
atr_series = calc_atr_rth(df, period=14)

day_df = df[df.index.date == pd.Timestamp('2026-01-12').date()]
orb = build_orb(day_df)

for ts, bar in day_df.iterrows():
    if ts.hour == 9 and ts.minute == 45:
        entry = bar['close']
        vwap = bar['vwap']
        atr = atr_series.loc[ts]
        bar_idx = day_df.index.get_loc(ts)
        
        print(f"Time: {ts.strftime('%H:%M')}")
        print(f"Entry: {entry:.2f}")
        print(f"VWAP: {vwap:.2f}")
        print(f"ATR: {atr:.4f}")
        print()
        
        stops = calc_stops(entry, atr, vwap, True, day_df, bar_idx)
        print(f"ATR Stop: {stops['atr_stop']:.2f}")
        print(f"Swing Stop: {stops['swing_stop']:.2f} (valid={stops['swing_valid']})")
        print(f"VWAP Stop: {stops['vwap_stop']:.2f} (valid={stops['vwap_valid']})")
        print()
        
        rr_desired = 2.0
        max_target_atr = 3.0
        
        for stop_type in ['atr', 'swing', 'vwap']:
            stop_price = stops[f'{stop_type}_stop']
            risk = abs(entry - stop_price)
            target_dist = risk * rr_desired
            target_atrs = target_dist / atr
            rr = calc_achievable_rr(entry, stop_price, atr, rr_desired, max_target_atr)
            print(f"{stop_type.upper()}: stop={stop_price:.2f}, risk={risk:.2f}, target_atrs={target_atrs:.2f}, RR={rr:.2f}")
        
        break

ImportError: cannot import name 'calc_stops' from 'orb_core' (C:\Users\phemm\orb_lab\src\orb_core.py)

In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
from orb_backtester import ORBBacktester
from orb_core import calc_atr_rth, build_orb, calc_stops, calc_achievable_rr

bt = ORBBacktester(symbol='AMD', start_date='2026-01-12', end_date='2026-01-12', verbose=False)
bt.load_data()
df = bt.df
atr_series = calc_atr_rth(df, period=14)

day_df = df[df.index.date == pd.Timestamp('2026-01-12').date()]
orb = build_orb(day_df)

for ts, bar in day_df.iterrows():
    if ts.hour == 9 and ts.minute == 45:
        entry = bar['close']
        vwap = bar['vwap']
        atr = atr_series.loc[ts]
        bar_idx = day_df.index.get_loc(ts)
        
        print(f"Time: {ts.strftime('%H:%M')}")
        print(f"Entry: {entry:.2f}")
        print(f"VWAP: {vwap:.2f}")
        print(f"ATR: {atr:.4f}")
        print()
        
        stops = calc_stops(entry, atr, vwap, True, day_df, bar_idx)
        print(f"ATR Stop: {stops['atr_stop']:.2f}")
        print(f"Swing Stop: {stops['swing_stop']:.2f} (valid={stops['swing_valid']})")
        print(f"VWAP Stop: {stops['vwap_stop']:.2f} (valid={stops['vwap_valid']})")
        print()
        
        rr_desired = 2.0
        max_target_atr = 3.0
        
        for stop_type in ['atr', 'swing', 'vwap']:
            stop_price = stops[f'{stop_type}_stop']
            risk = abs(entry - stop_price)
            target_dist = risk * rr_desired
            target_atrs = target_dist / atr
            rr = calc_achievable_rr(entry, stop_price, atr, rr_desired, max_target_atr)
            print(f"{stop_type.upper()}: stop={stop_price:.2f}, risk={risk:.2f}, target_atrs={target_atrs:.2f}, RR={rr:.2f}")
        
        break

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (0.2 hours old)
Time: 09:45
Entry: 205.05
VWAP: 203.48
ATR: 0.6316

ATR Stop: 203.79
Swing Stop: 204.80 (valid=True)
VWAP Stop: 203.29 (valid=True)

ATR: stop=203.79, risk=1.26, target_atrs=4.00, RR=1.50
SWING: stop=204.80, risk=0.25, target_atrs=0.80, RR=2.00
VWAP: stop=203.29, risk=1.76, target_atrs=5.56, RR=1.08


In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
from orb_backtester import ORBBacktester
from orb_core import (calc_atr_rth, calc_atr_stop, calc_swing_stop, 
                      calc_vwap_stop, calc_hybrid_stop, build_orb)

bt = ORBBacktester(symbol='AMD', start_date='2026-01-12', end_date='2026-01-12', verbose=False)
bt.load_data()
df = bt.df
atr_series = calc_atr_rth(df, period=14)

day_df = df[df.index.date == pd.Timestamp('2026-01-12').date()]

for ts, bar in day_df.iterrows():
    if ts.hour == 9 and ts.minute == 36:
        entry = 205.05
        atr = atr_series.loc[ts]
        vwap = bar['vwap']
        bar_idx = day_df.index.get_loc(ts)
        is_long = True
        
        atr_stop = calc_atr_stop(entry, atr, is_long)
        swing_stop = calc_swing_stop(day_df, bar_idx, atr, is_long)
        vwap_stop = calc_vwap_stop(vwap, atr, is_long)
        hybrid_stop = calc_hybrid_stop(atr_stop, swing_stop, vwap_stop, is_long)
        
        print(f"Time: 09:36")
        print(f"Entry: {entry}")
        print(f"ATR: {atr:.4f}")
        print(f"VWAP: {vwap:.2f}")
        print(f"═══ ALL STOPS ═══")
        print(f"ATR Stop: {atr_stop:.2f}")
        print(f"Swing Stop: {swing_stop:.2f}")
        print(f"VWAP Stop: {vwap_stop:.2f}")
        print(f"═══ HYBRID ═══")
        print(f"Hybrid Stop: {hybrid_stop:.2f}")
        
        # Which won?
        if hybrid_stop == atr_stop:
            print("Winner: ATR")
        elif hybrid_stop == swing_stop:
            print("Winner: Swing")
        else:
            print("Winner: VWAP")
        
        break

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (0.8 hours old)
Time: 09:36
Entry: 205.05
ATR: 0.6334
VWAP: 202.11
═══ ALL STOPS ═══
ATR Stop: 203.78
Swing Stop: 200.84
VWAP Stop: 201.92
═══ HYBRID ═══
Hybrid Stop: 203.78
Winner: ATR


In [1]:
"""
January 2026 AMD Validation with Skip Logic
"""
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
from orb_backtester import ORBBacktester
from orb_core import (calc_atr_rth, select_best_stop, build_orb, 
                      check_long_breakout, check_short_breakout,
                      PositionState, manage_position_exit, check_trade_feasibility)

bt = ORBBacktester(symbol='AMD', start_date='2026-01-01', end_date='2026-01-16', verbose=False)
bt.load_data()
df = bt.df
atr_series = calc_atr_rth(df, period=14)
df['ema9'] = df['close'].ewm(span=9, adjust=False).mean()

MIN_ACCEPTABLE_RR = 1.5

print("═══ JANUARY 2026 AMD - WITH SKIP LOGIC ═══")
print(f"Min Acceptable R:R: {MIN_ACCEPTABLE_RR}")
print(f"{'Date':<12} {'Dir':<6} {'Entry':>8} {'Stop':>8} {'Type':<8} {'R:R':>6} {'Exit':>8} {'Reason':<15} {'P&L':>8}")
print("=" * 95)

total_trades = 0
total_skips = 0

for date in pd.date_range('2026-01-01', '2026-01-16', freq='B'):
    day_df = df[df.index.date == date.date()]
    if len(day_df) == 0:
        continue
    orb = build_orb(day_df)
    if not orb.orb_complete:
        continue
    
    position = None
    last_exit_bar = -10
    
    for bar_num, (ts, bar) in enumerate(day_df.iterrows()):
        mins = ts.hour * 60 + ts.minute
        if mins < 9*60+30:
            continue
        atr = atr_series.loc[ts]
        if pd.isna(atr):
            continue
        ema = bar['ema9']
        
        if position is None and mins >= 9*60+35 and mins <= 10*60+30:
            if bar_num - last_exit_bar < 5:
                continue
            is_long_breakout = check_long_breakout(bar['open'], bar['high'], bar['low'], bar['close'],
                orb.session_high, orb.session_low, atr, orb.orb_complete)
            is_short_breakout = check_short_breakout(bar['open'], bar['high'], bar['low'], bar['close'],
                orb.session_high, orb.session_low, atr, orb.orb_complete)
            
            if is_long_breakout or is_short_breakout:
                is_long = is_long_breakout
                entry_price = bar['close']
                vwap = bar['vwap']
                bar_idx = day_df.index.get_loc(ts)
                best_stop, stop_type, achievable_rr = select_best_stop(entry_price, atr, vwap, is_long, day_df, bar_idx)
                
                is_feasible, should_skip = check_trade_feasibility(achievable_rr, MIN_ACCEPTABLE_RR)
                
                if should_skip:
                    direction = "LONG" if is_long else "SHORT"
                    print(f"{date.strftime('%Y-%m-%d'):<12} {direction:<6} {entry_price:>8.2f} {best_stop:>8.2f} {stop_type:<8} {achievable_rr:>6.2f} {'---':>8} {'SKIP':^15} {'---':>8}")
                    total_skips += 1
                    last_exit_bar = bar_num
                    continue
                
                position = PositionState(entry_price=entry_price, initial_stop=best_stop, stop_type=stop_type,
                    is_long=is_long, risk=abs(entry_price - best_stop), target_rr=achievable_rr)
                direction = "LONG" if is_long else "SHORT"
        
        elif position is not None:
            is_closed, exit_price, exit_reason = manage_position_exit(position, bar, atr,
                break_even_rr=0.5, profit_target_rr=2.0, trailing_stop_distance=1.2,
                use_break_even=True, use_trailing=True, use_ema_exit=True, ema=ema, ema_confirmation_bars=1,
                use_aggressive_trailing=True, ema_tighten_zone=0.3, tightened_trail_distance=0.3)
            
            if is_closed:
                pnl = (exit_price - position.entry_price) if position.is_long else (position.entry_price - exit_price)
                print(f"{date.strftime('%Y-%m-%d'):<12} {direction:<6} {position.entry_price:>8.2f} {position.initial_stop:>8.2f} {position.stop_type:<8} {position.target_rr:>6.2f} {exit_price:>8.2f} {exit_reason:<15} {pnl:>+8.2f}")
                position = None
                last_exit_bar = bar_num
                total_trades += 1
        
        if mins >= 15*60+55 and position is not None:
            exit_price = bar['close']
            pnl = (exit_price - position.entry_price) if position.is_long else (position.entry_price - exit_price)
            print(f"{date.strftime('%Y-%m-%d'):<12} {direction:<6} {position.entry_price:>8.2f} {position.initial_stop:>8.2f} {position.stop_type:<8} {position.target_rr:>6.2f} {exit_price:>8.2f} {'EOD Close':<15} {pnl:>+8.2f}")
            position = None
            total_trades += 1

print("=" * 95)
print(f"Total trades: {total_trades}, Total skips: {total_skips}")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (0.9 hours old)
═══ JANUARY 2026 AMD - WITH SKIP LOGIC ═══
Min Acceptable R:R: 1.5
Date         Dir       Entry     Stop Type        R:R     Exit Reason               P&L
2026-01-02   LONG     225.62   223.93 Swing      1.85   226.47 EMA Exit           +0.85
2026-01-05   SHORT    229.60   232.00 ATR        1.50   225.20 Trailing Stop      +4.40
2026-01-06   SHORT    219.16   220.67 ATR        1.50   217.60 EMA Exit           +1.56
2026-01-08   SHORT    208.10   209.16 Swing      1.63   207.82 EMA Exit           +0.28
2026-01-09   SHORT    204.44   205.53 ATR        1.50   204.44 Break Even         +0.00
2026-01-09   SHORT    204.71   204.88 Swing      2.00   204.88 Initial Stop       -0.17
2026-01-12   LONG     204.29   203.02 ATR        1.50   206.41 Trailing Stop      +2.12
2026-01-13   LONG     221.42   219.66 Swing      2.00   219.66 Initial Stop 

In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
from orb_backtester import ORBBacktester
from orb_core import (calc_atr_rth, calc_atr_stop, calc_swing_stop, 
                      calc_vwap_stop, calc_hybrid_stop, calc_achievable_rr)

bt = ORBBacktester(symbol='AMD', start_date='2026-01-12', end_date='2026-01-12', verbose=False)
bt.load_data()
df = bt.df
atr_series = calc_atr_rth(df, period=14)

day_df = df[df.index.date == pd.Timestamp('2026-01-12').date()]

entry = 205.05
is_long = True
rr_desired = 2.0
max_target_atr = 3.0
min_acceptable_rr = 1.5
min_stop_atr = 0.15

for ts, bar in day_df.iterrows():
    if ts.hour == 9 and 33 <= ts.minute <= 40:
        atr = atr_series.loc[ts]
        vwap = bar['vwap']
        bar_idx = day_df.index.get_loc(ts)
        
        # Calculate all stops
        atr_stop = calc_atr_stop(entry, atr, is_long)
        swing_stop = calc_swing_stop(day_df, bar_idx, atr, is_long)
        vwap_stop = calc_vwap_stop(vwap, atr, is_long)
        hybrid_stop = calc_hybrid_stop(atr_stop, swing_stop, vwap_stop, is_long)
        
        # Validity checks
        min_stop_dist = atr * min_stop_atr
        atr_valid = atr_stop < entry - min_stop_dist
        swing_valid = swing_stop < entry - min_stop_dist
        vwap_valid = (vwap < entry) and (vwap_stop < entry - min_stop_dist)
        hybrid_valid = hybrid_stop < entry - min_stop_dist
        
        # Risk for each
        atr_risk = abs(entry - atr_stop)
        swing_risk = abs(entry - swing_stop)
        vwap_risk = abs(entry - vwap_stop)
        hybrid_risk = abs(entry - hybrid_stop)
        
        # R:R for each
        atr_rr = calc_achievable_rr(entry, atr_stop, atr, rr_desired, max_target_atr) if atr_valid else -1
        swing_rr = calc_achievable_rr(entry, swing_stop, atr, rr_desired, max_target_atr) if swing_valid else -1
        vwap_rr = calc_achievable_rr(entry, vwap_stop, atr, rr_desired, max_target_atr) if vwap_valid else -1
        hybrid_rr = calc_achievable_rr(entry, hybrid_stop, atr, rr_desired, max_target_atr) if hybrid_valid else -1
        
        best_rr = max(atr_rr, swing_rr, vwap_rr, hybrid_rr)
        should_skip = best_rr < min_acceptable_rr
        
        print(f"═══ {ts.strftime('%H:%M')} ═══")
        print(f"ATR: {atr:.4f} VWAP: {vwap:.2f}")
        print(f"ATR: {atr_stop:.2f} R={atr_risk:.2f} RR={atr_rr:.2f} {'' if atr_valid else 'X'}")
        print(f"Swing: {swing_stop:.2f} R={swing_risk:.2f} RR={swing_rr:.2f} {'' if swing_valid else 'X'}")
        print(f"VWAP: {vwap_stop:.2f} R={vwap_risk:.2f} RR={vwap_rr:.2f} {'' if vwap_valid else 'X'}")
        print(f"Hybrid: {hybrid_stop:.2f} R={hybrid_risk:.2f} RR={hybrid_rr:.2f} {'' if hybrid_valid else 'X'}")
        print(f"Best RR: {best_rr:.2f} → {'SKIP' if should_skip else 'TAKE'}")
        print()

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (1.1 hours old)
═══ 09:33 ═══
ATR: 0.5576 VWAP: 201.37
ATR: 203.93 R=1.12 RR=1.50 
Swing: 199.79 R=5.26 RR=0.32 
VWAP: 201.20 R=3.85 RR=0.43 
Hybrid: 203.93 R=1.12 RR=1.50 
Best RR: 1.50 → SKIP

═══ 09:34 ═══
ATR: 0.6072 VWAP: 201.57
ATR: 203.84 R=1.21 RR=1.50 
Swing: 199.79 R=5.26 RR=0.35 
VWAP: 201.39 R=3.66 RR=0.50 
Hybrid: 203.84 R=1.21 RR=1.50 
Best RR: 1.50 → TAKE

═══ 09:35 ═══
ATR: 0.6352 VWAP: 201.86
ATR: 203.78 R=1.27 RR=1.50 
Swing: 199.79 R=5.26 RR=0.36 
VWAP: 201.67 R=3.38 RR=0.56 
Hybrid: 203.78 R=1.27 RR=1.50 
Best RR: 1.50 → TAKE

═══ 09:36 ═══
ATR: 0.6334 VWAP: 202.11
ATR: 203.78 R=1.27 RR=1.50 
Swing: 200.84 R=4.21 RR=0.45 
VWAP: 201.92 R=3.13 RR=0.61 
Hybrid: 203.78 R=1.27 RR=1.50 
Best RR: 1.50 → SKIP

═══ 09:37 ═══
ATR: 0.6317 VWAP: 202.40
ATR: 203.79 R=1.26 RR=1.50 
Swing: 201.90 R=3.15 RR=0.60 
VWAP: 202.21 R=2.84 RR=0.67 
Hybri

In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
from orb_backtester import ORBBacktester
from orb_core import select_best_stop, check_trade_feasibility, calc_atr_rth

bt = ORBBacktester(symbol='AMD', start_date='2026-01-12', end_date='2026-01-12', verbose=False)
bt.load_data()
df = bt.df
atr_series = calc_atr_rth(df, period=14)

day_df = df[df.index.date == pd.Timestamp('2026-01-12').date()]

for ts, bar in day_df.iterrows():
    if ts.hour == 9 and ts.minute == 36:
        entry = 205.05
        atr = atr_series.loc[ts]
        vwap = bar['vwap']
        bar_idx = day_df.index.get_loc(ts)
        
        # Test with EXTREME vol_state
        stop, stop_type, rr = select_best_stop(entry, atr, vwap, True, day_df, bar_idx, vol_state="EXTREME")
        is_feasible, should_skip = check_trade_feasibility(rr, 1.5)
        
        print(f"Entry: {entry}, Stop: {stop:.2f} ({stop_type}), R:R: {rr:.2f}")
        print(f"Feasible: {is_feasible}, Skip: {should_skip}")
        break

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (1.5 hours old)
Entry: 205.05, Stop: 203.78 (ATR), R:R: 1.20
Feasible: False, Skip: True


In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
from orb_backtester import ORBBacktester
from orb_core import (VolState, calc_vol_state, prepare_vol_columns, 
                      select_best_stop, check_trade_feasibility, calc_atr_rth)

bt = ORBBacktester(symbol='AMD', start_date='2026-01-12', end_date='2026-01-12', verbose=False)
bt.load_data()
df = bt.df

# Prepare vol columns
df = prepare_vol_columns(df)

day_df = df[df.index.date == pd.Timestamp('2026-01-12').date()]

# Track vol state
vol = VolState()

for ts, bar in day_df.iterrows():
    if ts.hour == 9 and ts.minute >= 30:
        # Calculate vol state for each bar
        vol_state = calc_vol_state(bar, ts, vol)
        
        if ts.minute == 36:
            entry = 205.05
            atr = bar['atr_rth']
            vwap = bar['vwap']
            bar_idx = day_df.index.get_loc(ts)
            
            print(f"Time: 09:36")
            print(f"VolFactor: {vol.vol_factor:.2f}")
            print(f"VolState: {vol.vol_state}")
            
            stop, stop_type, rr = select_best_stop(entry, atr, vwap, True, day_df, bar_idx, vol_state=vol_state)
            is_feasible, should_skip = check_trade_feasibility(rr, 1.5)
            
            print(f"Entry: {entry}, Stop: {stop:.2f} ({stop_type}), R:R: {rr:.2f}")
            print(f"Skip: {should_skip}")
            break

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (1.7 hours old)
Time: 09:36
VolFactor: 0.91
VolState: NORMAL
Entry: 205.05, Stop: 203.78 (ATR), R:R: 1.50
Skip: False


In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
from orb_backtester import ORBBacktester
from orb_core import (VolState, calc_vol_state, select_best_stop, 
                      check_trade_feasibility)

bt = ORBBacktester(symbol='AMD', start_date='2026-01-12', end_date='2026-01-12', verbose=False)
bt.load_data()
df = bt.df  # Already has vol columns from backtester's load_data()

day_df = df[df.index.date == pd.Timestamp('2026-01-12').date()]

vol = VolState()

for ts, bar in day_df.iterrows():
    if ts.hour == 9 and ts.minute >= 30:
        vol_state = calc_vol_state(bar, ts, vol)
        
        if ts.minute == 36:
            entry = 205.05
            atr = bar['atr_rth']
            vwap = bar['vwap']
            bar_idx = day_df.index.get_loc(ts)
            
            print(f"═══ AMD Jan 12 @ 09:36 ═══")
            print(f"atr_fast_rth: {bar['atr_fast_rth']:.4f}")
            print(f"orb_session_baseline: {vol.orb_session_baseline:.4f}")
            print(f"session_vf: {vol.session_vf:.4f}")
            print(f"daily_atr: {bar['daily_atr']:.4f}")
            print(f"daily_atr_slow: {bar['daily_atr_slow']:.4f}")
            print(f"htf_vf: {vol.htf_vf:.4f}")
            print(f"vol_factor: {vol.vol_factor:.4f}")
            print(f"vol_state: {vol.vol_state}")
            print()
            
            stop, stop_type, rr = select_best_stop(entry, atr, vwap, True, day_df, bar_idx, vol_state=vol_state)
            is_feasible, should_skip = check_trade_feasibility(rr, 1.5)
            
            print(f"Entry: {entry}, Stop: {stop:.2f} ({stop_type}), R:R: {rr:.2f}")
            print(f"Feasible: {is_feasible}, Skip: {should_skip}")
            break

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (1.9 hours old)
═══ AMD Jan 12 @ 09:36 ═══
atr_fast_rth: 0.8753
orb_session_baseline: 0.4756
session_vf: 1.8405
daily_atr: 8.6141
daily_atr_slow: 9.5130
htf_vf: 0.9055
vol_factor: 1.6666
vol_state: HIGH

Entry: 205.05, Stop: 203.96 (ATR), R:R: 1.50
Feasible: True, Skip: False


In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
from orb_backtester import ORBBacktester
from orb_core import VolState, calc_vol_state

bt = ORBBacktester(symbol='AMD', start_date='2026-01-12', end_date='2026-01-12', verbose=False)
bt.load_data()
df = bt.df

day_df = df[df.index.date == pd.Timestamp('2026-01-12').date()]
vol = VolState()

print("═══ AMD Jan 12 - Baseline Build (09:30-09:36) ═══")
for ts, bar in day_df.iterrows():
    if ts.hour == 9 and 30 <= ts.minute <= 36:
        vol_state = calc_vol_state(bar, ts, vol)
        print(f"{ts.strftime('%H:%M')} | atr_fast={bar['atr_fast_rth']:.4f} | atr_fast_sma={bar['atr_fast_sma_rth']:.4f} | baseline={vol.orb_session_baseline:.4f} | sessionVF={vol.session_vf:.2f}")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (2.0 hours old)
═══ AMD Jan 12 - Baseline Build (09:30-09:36) ═══
09:30 | atr_fast=0.2206 | atr_fast_sma=0.2206 | baseline=0.2206 | sessionVF=1.00
09:31 | atr_fast=0.5035 | atr_fast_sma=0.5035 | baseline=0.2489 | sessionVF=2.02
09:32 | atr_fast=0.7108 | atr_fast_sma=0.7108 | baseline=0.2950 | sessionVF=2.41
09:33 | atr_fast=0.8006 | atr_fast_sma=0.8006 | baseline=0.3456 | sessionVF=2.32
09:34 | atr_fast=0.7425 | atr_fast_sma=0.7425 | baseline=0.3853 | sessionVF=1.93
09:35 | atr_fast=0.8442 | atr_fast_sma=0.8442 | baseline=0.4312 | sessionVF=1.96
09:36 | atr_fast=0.8753 | atr_fast_sma=0.8753 | baseline=0.4756 | sessionVF=1.84


In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
from orb_backtester import ORBBacktester
from orb_core import (VolState, calc_vol_state, prepare_vol_columns,
                      select_best_stop, check_trade_feasibility)

bt = ORBBacktester(symbol='AMD', start_date='2026-01-12', end_date='2026-01-12', verbose=False)
bt.load_data()
df = bt.df

# Use our prepare_vol_columns to get the corrected atr_fast_sma
df = prepare_vol_columns(df)

day_df = df[df.index.date == pd.Timestamp('2026-01-12').date()]
vol = VolState()

print("═══ AMD Jan 12 - Baseline Build (09:30-09:36) ═══")
for ts, bar in day_df.iterrows():
    if ts.hour == 9 and 30 <= ts.minute <= 36:
        vol_state = calc_vol_state(bar, ts, vol)
        print(f"{ts.strftime('%H:%M')} | atr_fast={bar['atr_fast_rth']:.4f} | atr_fast_sma={bar['atr_fast_sma_rth']:.4f} | baseline={vol.orb_session_baseline:.4f} | sessionVF={vol.session_vf:.2f}")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (2.1 hours old)
═══ AMD Jan 12 - Baseline Build (09:30-09:36) ═══
09:30 | atr_fast=0.8584 | atr_fast_sma=0.2353 | baseline=0.2353 | sessionVF=3.65
09:31 | atr_fast=0.9947 | atr_fast_sma=0.2757 | baseline=0.2393 | sessionVF=4.16
09:32 | atr_fast=1.0277 | atr_fast_sma=0.3180 | baseline=0.2472 | sessionVF=4.16
09:33 | atr_fast=0.9242 | atr_fast_sma=0.3559 | baseline=0.2580 | sessionVF=3.58
09:34 | atr_fast=0.9895 | atr_fast_sma=0.3976 | baseline=0.2720 | sessionVF=3.64
09:35 | atr_fast=0.9916 | atr_fast_sma=0.4396 | baseline=0.2888 | sessionVF=3.43
09:36 | atr_fast=0.9153 | atr_fast_sma=0.4772 | baseline=0.3076 | sessionVF=2.98


In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
from orb_backtester import ORBBacktester
from orb_core import (VolState, calc_vol_state, prepare_vol_columns,
                      select_best_stop, check_trade_feasibility)

bt = ORBBacktester(symbol='AMD', start_date='2026-01-12', end_date='2026-01-12', verbose=False)
bt.load_data()
df = bt.df
df = prepare_vol_columns(df)

day_df = df[df.index.date == pd.Timestamp('2026-01-12').date()]
vol = VolState()

for ts, bar in day_df.iterrows():
    if ts.hour == 9 and ts.minute >= 30:
        vol_state = calc_vol_state(bar, ts, vol)
        
        if ts.minute == 36:
            entry = 205.05
            atr = bar['atr_rth']
            vwap = bar['vwap']
            bar_idx = day_df.index.get_loc(ts)
            
            print(f"═══ AMD Jan 12 @ 09:36 ═══")
            print(f"sessionVF: {vol.session_vf:.2f}")
            print(f"htfVF: {vol.htf_vf:.2f}")
            print(f"volFactor: {vol.vol_factor:.2f}")
            print(f"volState: {vol.vol_state}")
            print()
            
            stop, stop_type, rr = select_best_stop(entry, atr, vwap, True, day_df, bar_idx, vol_state=vol_state)
            is_feasible, should_skip = check_trade_feasibility(rr, 1.5)
            
            print(f"Entry: {entry}, Stop: {stop:.2f} ({stop_type}), R:R: {rr:.2f}")
            print(f"Skip: {should_skip}")
            break

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (2.1 hours old)
═══ AMD Jan 12 @ 09:36 ═══
sessionVF: 2.98
htfVF: 0.91
volFactor: 2.69
volState: EXTREME

Entry: 205.05, Stop: 203.78 (ATR), R:R: 1.20
Skip: True


In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
from orb_backtester import ORBBacktester
from orb_core import (VolState, calc_vol_state, prepare_vol_columns,
                      select_best_stop, check_trade_feasibility, build_orb,
                      check_long_breakout, check_short_breakout, 
                      PositionState, manage_position_exit)

bt = ORBBacktester(symbol='AMD', start_date='2026-01-01', end_date='2026-01-16', verbose=False)
bt.load_data()
df = bt.df
df = prepare_vol_columns(df)
df['ema9'] = df['close'].ewm(span=9, adjust=False).mean()

MIN_ACCEPTABLE_RR = 1.5

print("═══ JANUARY 2026 AMD - WITH SKIP LOGIC ═══")
print(f"Min Acceptable R:R: {MIN_ACCEPTABLE_RR}")
print(f"{'Date':<12} {'Dir':<6} {'Entry':>8} {'Stop':>8} {'Type':<8} {'R:R':>6} {'Exit':>8} {'Reason':<15} {'P&L':>8}")
print("=" * 95)

total_trades = 0
total_skips = 0

for date in pd.date_range('2026-01-01', '2026-01-16', freq='B'):
    day_df = df[df.index.date == date.date()]
    if len(day_df) == 0:
        continue
    orb = build_orb(day_df)
    if not orb.orb_complete:
        continue
    
    vol = VolState()
    position = None
    last_exit_bar = -10
    
    for bar_num, (ts, bar) in enumerate(day_df.iterrows()):
        mins = ts.hour * 60 + ts.minute
        if mins < 9*60+30:
            continue
        
        # Calculate vol_state for every RTH bar
        vol_state = calc_vol_state(bar, ts, vol)
        
        atr = bar['atr_rth']
        if pd.isna(atr):
            continue
        ema = bar['ema9']
        
        if position is None and mins >= 9*60+35 and mins <= 10*60+30:
            if bar_num - last_exit_bar < 5:
                continue
            is_long_breakout = check_long_breakout(bar['open'], bar['high'], bar['low'], bar['close'],
                orb.session_high, orb.session_low, atr, orb.orb_complete)
            is_short_breakout = check_short_breakout(bar['open'], bar['high'], bar['low'], bar['close'],
                orb.session_high, orb.session_low, atr, orb.orb_complete)
            
            if is_long_breakout or is_short_breakout:
                is_long = is_long_breakout
                entry_price = bar['close']
                vwap = bar['vwap']
                bar_idx = day_df.index.get_loc(ts)
                best_stop, stop_type, achievable_rr = select_best_stop(
                    entry_price, atr, vwap, is_long, day_df, bar_idx, vol_state=vol_state)
                
                is_feasible, should_skip = check_trade_feasibility(achievable_rr, MIN_ACCEPTABLE_RR)
                
                if should_skip:
                    direction = "LONG" if is_long else "SHORT"
                    print(f"{date.strftime('%Y-%m-%d'):<12} {direction:<6} {entry_price:>8.2f} {best_stop:>8.2f} {stop_type:<8} {achievable_rr:>6.2f} {'---':>8} {'SKIP':^15} {'---':>8}")
                    total_skips += 1
                    last_exit_bar = bar_num
                    continue
                
                position = PositionState(entry_price=entry_price, initial_stop=best_stop, stop_type=stop_type,
                    is_long=is_long, risk=abs(entry_price - best_stop), target_rr=achievable_rr)
                direction = "LONG" if is_long else "SHORT"
        
        elif position is not None:
            is_closed, exit_price, exit_reason = manage_position_exit(position, bar, atr,
                break_even_rr=0.5, profit_target_rr=2.0, trailing_stop_distance=1.2,
                use_break_even=True, use_trailing=True, use_ema_exit=True, ema=ema, ema_confirmation_bars=1,
                use_aggressive_trailing=True, ema_tighten_zone=0.3, tightened_trail_distance=0.3)
            
            if is_closed:
                pnl = (exit_price - position.entry_price) if position.is_long else (position.entry_price - exit_price)
                print(f"{date.strftime('%Y-%m-%d'):<12} {direction:<6} {position.entry_price:>8.2f} {position.initial_stop:>8.2f} {position.stop_type:<8} {position.target_rr:>6.2f} {exit_price:>8.2f} {exit_reason:<15} {pnl:>+8.2f}")
                position = None
                last_exit_bar = bar_num
                total_trades += 1
        
        if mins >= 15*60+55 and position is not None:
            exit_price = bar['close']
            pnl = (exit_price - position.entry_price) if position.is_long else (position.entry_price - exit_price)
            print(f"{date.strftime('%Y-%m-%d'):<12} {direction:<6} {position.entry_price:>8.2f} {position.initial_stop:>8.2f} {position.stop_type:<8} {position.target_rr:>6.2f} {exit_price:>8.2f} {'EOD Close':<15} {pnl:>+8.2f}")
            position

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (2.3 hours old)
═══ JANUARY 2026 AMD - WITH SKIP LOGIC ═══
Min Acceptable R:R: 1.5
Date         Dir       Entry     Stop Type        R:R     Exit Reason               P&L
2026-01-02   LONG     225.62   223.93 SWING      1.83   226.47 EMA Exit           +0.85
2026-01-05   SHORT    229.60   232.00 ATR        1.20      ---      SKIP            ---
2026-01-06   SHORT    219.16   220.55 ATR        1.20      ---      SKIP            ---
2026-01-08   SHORT    208.10   209.18 ATR        1.20      ---      SKIP            ---
2026-01-09   SHORT    204.44   205.49 ATR        1.50   204.44 Break Even         +0.00
2026-01-09   SHORT    204.71   204.88 SWING      2.50   204.88 Initial Stop       -0.17
2026-01-12   LONG     204.29   203.08 ATR        1.20      ---      SKIP            ---
2026-01-13   LONG     221.42   219.66 SWING      2.00   219.66 Initial Stop 

In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
from orb_backtester import ORBBacktester
from orb_core import (VolState, calc_vol_state, prepare_vol_columns,
                      select_best_stop, check_trade_feasibility, build_orb,
                      check_long_breakout, check_short_breakout, 
                      PositionState, manage_position_exit)

bt = ORBBacktester(symbol='AMD', start_date='2025-12-01', end_date='2025-12-31', verbose=False)
bt.load_data()
df = bt.df
df = prepare_vol_columns(df)
df['ema9'] = df['close'].ewm(span=9, adjust=False).mean()

MIN_ACCEPTABLE_RR = 1.5

print("═══ DECEMBER 2025 AMD - WITH SKIP LOGIC ═══")
print(f"Min Acceptable R:R: {MIN_ACCEPTABLE_RR}")
print(f"{'Date':<12} {'Dir':<6} {'Entry':>8} {'Stop':>8} {'Type':<8} {'R:R':>6} {'Exit':>8} {'Reason':<15} {'P&L':>8}")
print("=" * 95)

total_trades = 0
total_skips = 0

for date in pd.date_range('2025-12-01', '2025-12-31', freq='B'):
    day_df = df[df.index.date == date.date()]
    if len(day_df) == 0:
        continue
    orb = build_orb(day_df)
    if not orb.orb_complete:
        continue
    
    vol = VolState()
    position = None
    last_exit_bar = -10
    
    for bar_num, (ts, bar) in enumerate(day_df.iterrows()):
        mins = ts.hour * 60 + ts.minute
        if mins < 9*60+30:
            continue
        
        # Calculate vol_state for every RTH bar
        vol_state = calc_vol_state(bar, ts, vol)
        
        atr = bar['atr_rth']
        if pd.isna(atr):
            continue
        ema = bar['ema9']
        
        if position is None and mins >= 9*60+35 and mins <= 10*60+30:
            if bar_num - last_exit_bar < 5:
                continue
            is_long_breakout = check_long_breakout(bar['open'], bar['high'], bar['low'], bar['close'],
                orb.session_high, orb.session_low, atr, orb.orb_complete)
            is_short_breakout = check_short_breakout(bar['open'], bar['high'], bar['low'], bar['close'],
                orb.session_high, orb.session_low, atr, orb.orb_complete)
            
            if is_long_breakout or is_short_breakout:
                is_long = is_long_breakout
                entry_price = bar['close']
                vwap = bar['vwap']
                bar_idx = day_df.index.get_loc(ts)
                best_stop, stop_type, achievable_rr = select_best_stop(
                    entry_price, atr, vwap, is_long, day_df, bar_idx, vol_state=vol_state)
                
                is_feasible, should_skip = check_trade_feasibility(achievable_rr, MIN_ACCEPTABLE_RR)
                
                if should_skip:
                    direction = "LONG" if is_long else "SHORT"
                    print(f"{date.strftime('%Y-%m-%d'):<12} {direction:<6} {entry_price:>8.2f} {best_stop:>8.2f} {stop_type:<8} {achievable_rr:>6.2f} {'---':>8} {'SKIP':^15} {'---':>8}")
                    total_skips += 1
                    last_exit_bar = bar_num
                    continue
                
                position = PositionState(entry_price=entry_price, initial_stop=best_stop, stop_type=stop_type,
                    is_long=is_long, risk=abs(entry_price - best_stop), target_rr=achievable_rr)
                direction = "LONG" if is_long else "SHORT"
        
        elif position is not None:
            is_closed, exit_price, exit_reason = manage_position_exit(position, bar, atr,
                break_even_rr=0.5, profit_target_rr=2.0, trailing_stop_distance=1.2,
                use_break_even=True, use_trailing=True, use_ema_exit=True, ema=ema, ema_confirmation_bars=1,
                use_aggressive_trailing=True, ema_tighten_zone=0.3, tightened_trail_distance=0.3)
            
            if is_closed:
                pnl = (exit_price - position.entry_price) if position.is_long else (position.entry_price - exit_price)
                print(f"{date.strftime('%Y-%m-%d'):<12} {direction:<6} {position.entry_price:>8.2f} {position.initial_stop:>8.2f} {position.stop_type:<8} {position.target_rr:>6.2f} {exit_price:>8.2f} {exit_reason:<15} {pnl:>+8.2f}")
                position = None
                last_exit_bar = bar_num
                total_trades += 1
        
        if mins >= 15*60+55 and position is not None:
            exit_price = bar['close']
            pnl = (exit_price - position.entry_price) if position.is_long else (position.entry_price - exit_price)
            print(f"{date.strftime('%Y-%m-%d'):<12} {direction:<6} {position.entry_price:>8.2f} {position.initial_stop:>8.2f} {position.stop_type:<8} {position.target_rr:>6.2f} {exit_price:>8.2f} {'EOD Close':<15} {pnl:>+8.2f}")
            position = None
            total_trades += 1

print("=" * 95)
print(f"Total trades: {total_trades}, Total skips: {total_skips}")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (2.4 hours old)
═══ DECEMBER 2025 AMD - WITH SKIP LOGIC ═══
Min Acceptable R:R: 1.5
Date         Dir       Entry     Stop Type        R:R     Exit Reason               P&L
2025-12-01   LONG     216.22   214.90 ATR        1.20      ---      SKIP            ---
2025-12-01   LONG     216.55   215.14 ATR        1.50   218.52 Trailing Stop      +1.97
2025-12-02   LONG     224.33   222.89 ATR        1.20      ---      SKIP            ---
2025-12-03   SHORT    213.28   214.68 ATR        1.20      ---      SKIP            ---
2025-12-03   LONG     216.95   215.90 SWING      1.78   215.90 Initial Stop       -1.05
2025-12-04   SHORT    215.28   216.71 ATR        1.50   216.71 Initial Stop       -1.43
2025-12-05   SHORT    216.74   218.25 ATR        1.20      ---      SKIP            ---
2025-12-05   LONG     222.21   221.64 SWING      2.50   221.64 Initial Stop

In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
from orb_backtester import ORBBacktester
from orb_core import (VolState, calc_vol_state, prepare_vol_columns,
                      select_best_stop, check_trade_feasibility)

bt = ORBBacktester(symbol='AMD', start_date='2025-12-02', end_date='2025-12-02', verbose=False)
bt.load_data()
df = bt.df
df = prepare_vol_columns(df)

day_df = df[df.index.date == pd.Timestamp('2025-12-02').date()]
vol = VolState()

for ts, bar in day_df.iterrows():
    if ts.hour == 9 and ts.minute >= 30:
        vol_state = calc_vol_state(bar, ts, vol)
        
        if ts.minute == 37:
            entry = 224.33
            atr = bar['atr_rth']
            vwap = bar['vwap']
            bar_idx = day_df.index.get_loc(ts)
            
            print(f"═══ AMD Dec 2 @ 09:37 ═══")
            print(f"atr_fast: {bar['atr_fast_rth']:.4f}")
            print(f"atr_fast_sma: {bar['atr_fast_sma_rth']:.4f}")
            print(f"baseline: {vol.orb_session_baseline:.4f}")
            print(f"sessionVF: {vol.session_vf:.2f}")
            print(f"htfVF: {vol.htf_vf:.2f}")
            print(f"volFactor: {vol.vol_factor:.2f}")
            print(f"volState: {vol.vol_state}")
            print()
            
            stop, stop_type, rr = select_best_stop(entry, atr, vwap, True, day_df, bar_idx, vol_state=vol_state)
            is_feasible, should_skip = check_trade_feasibility(rr, 1.5)
            
            print(f"Entry: {entry}, Stop: {stop:.2f} ({stop_type}), R:R: {rr:.2f}")
            print(f"Skip: {should_skip}")
            break

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (2.9 hours old)
═══ AMD Dec 2 @ 09:37 ═══
atr_fast: 0.8885
atr_fast_sma: 0.9522
baseline: 0.6037
sessionVF: 1.47
htfVF: 1.00
volFactor: 1.47
volState: HIGH

Entry: 224.33, Stop: 222.88 (ATR), R:R: 1.50
Skip: False


In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
from orb_backtester import ORBBacktester
from orb_core import (VolState, calc_vol_state, prepare_vol_columns,
                      select_best_stop, check_trade_feasibility, build_orb,
                      check_long_breakout, check_short_breakout, 
                      PositionState, manage_position_exit)

bt = ORBBacktester(symbol='AMD', start_date='2025-12-01', end_date='2025-12-31', verbose=False)
bt.load_data()
df = bt.df
df = prepare_vol_columns(df)
df['ema9'] = df['close'].ewm(span=9, adjust=False).mean()

MIN_ACCEPTABLE_RR = 1.5

print("═══ DECEMBER 2025 AMD - WITH SKIP LOGIC ═══")
print(f"Min Acceptable R:R: {MIN_ACCEPTABLE_RR}")
print(f"{'Date':<12} {'Dir':<6} {'Entry':>8} {'Stop':>8} {'Type':<8} {'R:R':>6} {'Exit':>8} {'Reason':<15} {'P&L':>8}")
print("=" * 95)

total_trades = 0
total_skips = 0

for date in pd.date_range('2025-12-01', '2025-12-31', freq='B'):
    day_df = df[df.index.date == date.date()]
    if len(day_df) == 0:
        continue
    orb = build_orb(day_df)
    if not orb.orb_complete:
        continue
    
    vol = VolState()
    position = None
    last_exit_bar = -10
    
    for bar_num, (ts, bar) in enumerate(day_df.iterrows()):
        mins = ts.hour * 60 + ts.minute
        if mins < 9*60+30:
            continue
        
        vol_state = calc_vol_state(bar, ts, vol)
        
        atr = bar['atr_rth']
        if pd.isna(atr):
            continue
        ema = bar['ema9']
        
        if position is None and mins >= 9*60+35 and mins <= 10*60+30:
            if bar_num - last_exit_bar < 5:
                continue
            is_long_breakout = check_long_breakout(bar['open'], bar['high'], bar['low'], bar['close'],
                orb.session_high, orb.session_low, atr, orb.orb_complete)
            is_short_breakout = check_short_breakout(bar['open'], bar['high'], bar['low'], bar['close'],
                orb.session_high, orb.session_low, atr, orb.orb_complete)
            
            if is_long_breakout or is_short_breakout:
                is_long = is_long_breakout
                entry_price = bar['close']
                vwap = bar['vwap']
                bar_idx = day_df.index.get_loc(ts)
                best_stop, stop_type, achievable_rr = select_best_stop(
                    entry_price, atr, vwap, is_long, day_df, bar_idx, vol_state=vol_state)
                
                is_feasible, should_skip = check_trade_feasibility(achievable_rr, MIN_ACCEPTABLE_RR)
                
                if should_skip:
                    direction = "LONG" if is_long else "SHORT"
                    print(f"{date.strftime('%Y-%m-%d'):<12} {direction:<6} {entry_price:>8.2f} {best_stop:>8.2f} {stop_type:<8} {achievable_rr:>6.2f} {'---':>8} {'SKIP':^15} {'---':>8}")
                    total_skips += 1
                    last_exit_bar = bar_num
                    continue
                
                position = PositionState(entry_price=entry_price, initial_stop=best_stop, stop_type=stop_type,
                    is_long=is_long, risk=abs(entry_price - best_stop), target_rr=achievable_rr)
                direction = "LONG" if is_long else "SHORT"
        
        elif position is not None:
            is_closed, exit_price, exit_reason = manage_position_exit(position, bar, atr,
                break_even_rr=0.5, profit_target_rr=2.0, trailing_stop_distance=1.2,
                use_break_even=True, use_trailing=True, use_ema_exit=True, ema=ema, ema_confirmation_bars=1,
                use_aggressive_trailing=True, ema_tighten_zone=0.3, tightened_trail_distance=0.3)
            
            if is_closed:
                pnl = (exit_price - position.entry_price) if position.is_long else (position.entry_price - exit_price)
                print(f"{date.strftime('%Y-%m-%d'):<12} {direction:<6} {position.entry_price:>8.2f} {position.initial_stop:>8.2f} {position.stop_type:<8} {position.target_rr:>6.2f} {exit_price:>8.2f} {exit_reason:<15} {pnl:>+8.2f}")
                position = None
                last_exit_bar = bar_num
                total_trades += 1
        
        if mins >= 15*60+55 and position is not None:
            exit_price = bar['close']
            pnl = (exit_price - position.entry_price) if position.is_long else (position.entry_price - exit_price)
            print(f"{date.strftime('%Y-%m-%d'):<12} {direction:<6} {position.entry_price:>8.2f} {position.initial_stop:>8.2f} {position.stop_type:<8} {position.target_rr:>6.2f} {exit_price:>8.2f} {'EOD Close':<15} {pnl:>+8.2f}")
            position = None
            total_trades += 1

print("=" * 95)
print(f"Total trades: {total_trades}, Total skips: {total_skips}")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (2.9 hours old)
═══ DECEMBER 2025 AMD - WITH SKIP LOGIC ═══
Min Acceptable R:R: 1.5
Date         Dir       Entry     Stop Type        R:R     Exit Reason               P&L
2025-12-01   LONG     216.22   214.90 ATR        1.50   214.90 Initial Stop       -1.33
2025-12-02   LONG     224.33   222.89 ATR        1.50   224.71 EMA Exit           +0.38
2025-12-03   SHORT    213.28   214.68 ATR        1.20      ---      SKIP            ---
2025-12-03   LONG     216.95   215.90 SWING      1.78   215.90 Initial Stop       -1.05
2025-12-04   SHORT    215.28   216.71 ATR        1.50   216.71 Initial Stop       -1.43
2025-12-05   SHORT    216.74   218.25 ATR        1.50   218.25 Initial Stop       -1.51
2025-12-05   LONG     222.21   221.64 SWING      2.50   221.64 Initial Stop       -0.57
2025-12-08   LONG     221.94   220.82 ATR        1.50   221.94 Break Even  

In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
from orb_backtester import ORBBacktester
from orb_core import (VolState, calc_vol_state, prepare_vol_columns,
                      select_best_stop, check_trade_feasibility)

bt = ORBBacktester(symbol='AMD', start_date='2025-12-03', end_date='2025-12-03', verbose=False)
bt.load_data()
df = bt.df
df = prepare_vol_columns(df)

day_df = df[df.index.date == pd.Timestamp('2025-12-03').date()]
vol = VolState()

print("═══ AMD Dec 3 - Baseline Build ═══")
for ts, bar in day_df.iterrows():
    if ts.hour == 9 and 30 <= ts.minute <= 40:
        vol_state = calc_vol_state(bar, ts, vol)
        print(f"{ts.strftime('%H:%M')} | atr_fast={bar['atr_fast_rth']:.4f} | baseline={vol.orb_session_baseline:.4f} | sessionVF={vol.session_vf:.2f} | htfVF={vol.htf_vf:.2f} | volFactor={vol.vol_factor:.2f} | {vol.vol_state}")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (3.0 hours old)
═══ AMD Dec 3 - Baseline Build ═══
09:30 | atr_fast=0.8262 | baseline=0.3212 | sessionVF=2.57 | htfVF=0.96 | volFactor=2.46 | EXTREME
09:31 | atr_fast=1.0070 | baseline=0.3294 | sessionVF=3.06 | htfVF=0.96 | volFactor=2.92 | EXTREME
09:32 | atr_fast=1.1436 | baseline=0.3464 | sessionVF=3.30 | htfVF=0.96 | volFactor=3.16 | EXTREME
09:33 | atr_fast=1.0949 | baseline=0.3708 | sessionVF=2.95 | htfVF=0.96 | volFactor=2.82 | EXTREME
09:34 | atr_fast=1.1479 | baseline=0.4016 | sessionVF=2.86 | htfVF=0.96 | volFactor=2.73 | EXTREME
09:35 | atr_fast=1.3273 | baseline=0.4395 | sessionVF=3.02 | htfVF=0.96 | volFactor=2.89 | EXTREME
09:36 | atr_fast=1.3379 | baseline=0.4835 | sessionVF=2.77 | htfVF=0.96 | volFactor=2.65 | EXTREME
09:37 | atr_fast=1.2703 | baseline=0.5327 | sessionVF=2.38 | htfVF=0.96 | volFactor=2.28 | EXTREME
09:38 | atr_fast=1.2

In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
from orb_backtester import ORBBacktester
from orb_core import prepare_vol_columns

bt = ORBBacktester(symbol='AMD', start_date='2025-12-03', end_date='2025-12-03', verbose=False)
bt.load_data()
df = bt.df
df = prepare_vol_columns(df)

day_df = df[df.index.date == pd.Timestamp('2025-12-03').date()]

print("═══ AMD Dec 3 - ATR values around open ═══")
for ts, bar in day_df.iterrows():
    if ts.hour == 9 and 28 <= ts.minute <= 35:
        print(f"{ts.strftime('%H:%M')} | atr_fast={bar['atr_fast_rth']:.4f} | atr_fast_sma={bar['atr_fast_sma_rth']:.4f}")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (3.1 hours old)
═══ AMD Dec 3 - ATR values around open ═══
09:28 | atr_fast=0.2778 | atr_fast_sma=0.2569
09:29 | atr_fast=0.2778 | atr_fast_sma=0.2569
09:30 | atr_fast=0.8262 | atr_fast_sma=0.3212
09:31 | atr_fast=1.0070 | atr_fast_sma=0.4032
09:32 | atr_fast=1.1436 | atr_fast_sma=0.4994
09:33 | atr_fast=1.0949 | atr_fast_sma=0.5910
09:34 | atr_fast=1.1479 | atr_fast_sma=0.6786
09:35 | atr_fast=1.3273 | atr_fast_sma=0.7802


In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
from orb_backtester import ORBBacktester
from orb_core import prepare_vol_columns, find_swing_low_rth, calc_swing_stop

bt = ORBBacktester(symbol='AMD', start_date='2026-01-12', end_date='2026-01-12', verbose=False)
bt.load_data()
df = bt.df
df = prepare_vol_columns(df)

day_df = df[df.index.date == pd.Timestamp('2026-01-12').date()]

# Find 09:36 bar
for bar_idx, (ts, bar) in enumerate(day_df.iterrows()):
    if ts.hour == 9 and ts.minute == 36:
        atr = bar['atr_rth']
        
        # Show last 5 RTH bars' lows
        print("═══ Last 5 RTH bars before 09:36 ═══")
        swing_low = find_swing_low_rth(day_df, bar_idx, 5)
        
        for i in range(1, 10):
            if bar_idx - i < 0:
                break
            check_ts = day_df.index[bar_idx - i]
            mins = check_ts.hour * 60 + check_ts.minute
            is_rth = 9*60+30 <= mins < 16*60
            print(f"{check_ts.strftime('%H:%M')} | Low: {day_df['low'].iloc[bar_idx - i]:.2f} | RTH: {is_rth}")
        
        print(f"\nSwing Low: {swing_low:.2f}")
        print(f"ATR: {atr:.4f}")
        swing_stop = calc_swing_stop(day_df, bar_idx, atr, True)
        print(f"Swing Stop (LONG): {swing_stop:.2f}")
        break

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (3.4 hours old)
═══ Last 5 RTH bars before 09:36 ═══
09:35 | Low: 203.29 | RTH: True
09:34 | Low: 202.20 | RTH: True
09:33 | Low: 202.07 | RTH: True
09:32 | Low: 201.91 | RTH: True
09:31 | Low: 200.85 | RTH: True
09:30 | Low: 199.80 | RTH: True
09:29 | Low: 201.19 | RTH: False
09:28 | Low: 201.05 | RTH: False
09:27 | Low: 201.20 | RTH: False

Swing Low: 200.85
ATR: 0.6352
Swing Stop (LONG): 200.84


In [2]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
from orb_backtester import ORBBacktester
from orb_core import (VolState, calc_vol_state, prepare_vol_columns,
                      select_best_stop, check_trade_feasibility, build_orb,
                      check_long_breakout, check_short_breakout, 
                      PositionState, manage_position_exit)

bt = ORBBacktester(symbol='AMD', start_date='2025-12-01', end_date='2025-12-31', verbose=False)
bt.load_data()
df = bt.df
df = prepare_vol_columns(df)
df['ema9'] = df['close'].ewm(span=9, adjust=False).mean()

MIN_ACCEPTABLE_RR = 1.5

print("═══ DECEMBER 2025 AMD - WITH SKIP LOGIC ═══")
print(f"Min Acceptable R:R: {MIN_ACCEPTABLE_RR}")
print(f"{'Date':<12} {'Dir':<6} {'Entry':>8} {'Stop':>8} {'Type':<8} {'R:R':>6} {'Exit':>8} {'Reason':<15} {'P&L':>8}")
print("=" * 95)

total_trades = 0
total_skips = 0
total_pnl = 0

for date in pd.date_range('2025-12-01', '2025-12-31', freq='B'):
    day_df = df[df.index.date == date.date()]
    if len(day_df) == 0:
        continue
    orb = build_orb(day_df)
    if not orb.orb_complete:
        continue
    
    vol = VolState()
    position = None
    last_exit_bar = -10
    
    for bar_num, (ts, bar) in enumerate(day_df.iterrows()):
        mins = ts.hour * 60 + ts.minute
        if mins < 9*60+30:
            continue
        
        vol_state = calc_vol_state(bar, ts, vol)
        
        atr = bar['atr_rth']
        if pd.isna(atr):
            continue
        ema = bar['ema9']
        
        if position is None and mins >= 9*60+35 and mins <= 10*60+30:
            if bar_num - last_exit_bar < 5:
                continue
            is_long_breakout = check_long_breakout(bar['open'], bar['high'], bar['low'], bar['close'],
                orb.session_high, orb.session_low, atr, orb.orb_complete)
            is_short_breakout = check_short_breakout(bar['open'], bar['high'], bar['low'], bar['close'],
                orb.session_high, orb.session_low, atr, orb.orb_complete)
            
            if is_long_breakout or is_short_breakout:
                is_long = is_long_breakout
                entry_price = bar['close']
                vwap = bar['vwap']
                bar_idx = day_df.index.get_loc(ts)
                best_stop, stop_type, achievable_rr = select_best_stop(
                    entry_price, atr, vwap, is_long, day_df, bar_idx, vol_state=vol_state)
                
                is_feasible, should_skip = check_trade_feasibility(achievable_rr, MIN_ACCEPTABLE_RR)
                
                if should_skip:
                    direction = "LONG" if is_long else "SHORT"
                    print(f"{date.strftime('%Y-%m-%d'):<12} {direction:<6} {entry_price:>8.2f} {best_stop:>8.2f} {stop_type:<8} {achievable_rr:>6.2f} {'---':>8} {'SKIP':^15} {'---':>8}")
                    total_skips += 1
                    last_exit_bar = bar_num
                    continue
                
                position = PositionState(entry_price=entry_price, initial_stop=best_stop, stop_type=stop_type,
                    is_long=is_long, risk=abs(entry_price - best_stop), target_rr=achievable_rr)
                direction = "LONG" if is_long else "SHORT"
        
        elif position is not None:
            is_closed, exit_price, exit_reason = manage_position_exit(position, bar, atr,
                break_even_rr=0.5, profit_target_rr=2.0, trailing_stop_distance=1.2,
                use_break_even=True, use_trailing=True, use_ema_exit=True, ema=ema, ema_confirmation_bars=1,
                use_aggressive_trailing=True, ema_tighten_zone=0.3, tightened_trail_distance=0.3)
            
            if is_closed:
                pnl = (exit_price - position.entry_price) if position.is_long else (position.entry_price - exit_price)
                total_pnl += pnl
                print(f"{date.strftime('%Y-%m-%d'):<12} {direction:<6} {position.entry_price:>8.2f} {position.initial_stop:>8.2f} {position.stop_type:<8} {position.target_rr:>6.2f} {exit_price:>8.2f} {exit_reason:<15} {pnl:>+8.2f}")
                position = None
                last_exit_bar = bar_num
                total_trades += 1
        
        if mins >= 15*60+55 and position is not None:
            exit_price = bar['close']
            pnl = (exit_price - position.entry_price) if position.is_long else (position.entry_price - exit_price)
            total_pnl += pnl
            print(f"{date.strftime('%Y-%m-%d'):<12} {direction:<6} {position.entry_price:>8.2f} {position.initial_stop:>8.2f} {position.stop_type:<8} {position.target_rr:>6.2f} {exit_price:>8.2f} {'EOD Close':<15} {pnl:>+8.2f}")
            position = None
            total_trades += 1

print("=" * 95)
print(f"Total trades: {total_trades}, Total skips: {total_skips}, Total P&L: ${total_pnl:+.2f}")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (3.5 hours old)
═══ DECEMBER 2025 AMD - WITH SKIP LOGIC ═══
Min Acceptable R:R: 1.5
Date         Dir       Entry     Stop Type        R:R     Exit Reason               P&L
2025-12-01   LONG     216.22   214.90 ATR        1.50   214.90 Initial Stop       -1.33
2025-12-02   LONG     224.33   222.89 ATR        1.50   224.71 EMA Exit           +0.38
2025-12-03   SHORT    213.28   214.68 ATR        1.20      ---      SKIP            ---
2025-12-03   LONG     216.95   215.90 SWING      1.78   215.90 Initial Stop       -1.05
2025-12-04   SHORT    215.28   216.71 ATR        1.50   216.71 Initial Stop       -1.43
2025-12-05   SHORT    216.74   218.25 ATR        1.50   218.25 Initial Stop       -1.51
2025-12-05   LONG     222.21   221.64 SWING      2.50   221.64 Initial Stop       -0.57
2025-12-08   LONG     221.94   220.82 ATR        1.50   221.94 Break Even  

In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
from orb_backtester import ORBBacktester
from orb_core import build_orb, is_orb_window, ORBState

bt = ORBBacktester(symbol='AMD', start_date='2025-12-05', end_date='2025-12-05', verbose=False)
bt.load_data()
df = bt.df

day_df = df[df.index.date == pd.Timestamp('2025-12-05').date()]

print("═══ DEC 5 ORB DIAGNOSTIC ═══\n")

# Method 1: build_orb function
orb1 = build_orb(day_df)
print(f"build_orb() result:")
print(f"  ORB High: {orb1.session_high:.2f}")
print(f"  ORB Low:  {orb1.session_low:.2f}")
print(f"  Complete: {orb1.orb_complete}")

# Method 2: Manual check of 09:30-09:34 bars
print(f"\n═══ ORB Window Bars (09:30-09:34) ═══")
for ts, bar in day_df.iterrows():
    if ts.hour == 9 and 30 <= ts.minute < 35:
        in_window = is_orb_window(ts, 5)
        print(f"{ts.strftime('%H:%M')} | O:{bar['open']:.2f} H:{bar['high']:.2f} L:{bar['low']:.2f} C:{bar['close']:.2f} | in_window={in_window}")

print(f"\n═══ Pine Values (from chart) ═══")
print(f"  ORB High: 221.94")
print(f"  ORB Low:  217.00")

# Check all bars from 09:35 to 10:00 for potential breakouts
print(f"\n═══ Post-ORB Bars Analysis ═══")
for ts, bar in day_df.iterrows():
    if ts.hour == 9 and 35 <= ts.minute <= 59:
        open_in_range = (bar['open'] >= orb1.session_low) and (bar['open'] <= orb1.session_high)
        close_above_high = bar['close'] > orb1.session_high
        close_below_low = bar['close'] < orb1.session_low
        
        potential = ""
        if close_above_high and open_in_range:
            potential = "LONG BREAKOUT?"
        elif close_below_low and open_in_range:
            potential = "SHORT BREAKOUT?"
        elif close_above_high:
            potential = "(close above but open not in range)"
        elif close_below_low:
            potential = "(close below but open not in range)"
            
        print(f"{ts.strftime('%H:%M')} | O:{bar['open']:.2f} H:{bar['high']:.2f} L:{bar['low']:.2f} C:{bar['close']:.2f} | open_in_range={open_in_range} {potential}")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ⚠ Cache stale (8 hours), refreshing...
  ✓ Fetched 103566 bars, cached for next time
═══ DEC 5 ORB DIAGNOSTIC ═══

build_orb() result:
  ORB High: 221.94
  ORB Low:  217.00
  Complete: True

═══ ORB Window Bars (09:30-09:34) ═══
09:30 | O:217.16 H:219.26 L:217.00 C:218.80 | in_window=True
09:31 | O:218.89 H:220.20 L:218.80 C:220.16 | in_window=True
09:32 | O:220.16 H:221.05 L:220.10 C:220.86 | in_window=True
09:33 | O:220.83 H:221.48 L:220.75 C:221.31 | in_window=True
09:34 | O:221.32 H:221.94 L:221.22 C:221.56 | in_window=True

═══ Pine Values (from chart) ═══
  ORB High: 221.94
  ORB Low:  217.00

═══ Post-ORB Bars Analysis ═══
09:35 | O:221.56 H:221.77 L:220.01 C:220.08 | open_in_range=True 
09:36 | O:220.09 H:220.16 L:219.62 C:219.85 | open_in_range=True 
09:37 | O:219.87 H:220.07 L:219.09 C:219.26 | open_in_range=True 
09:38 | O:219.26 H:219.70 L:218.85 C:219.00 | op

In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
from orb_backtester import ORBBacktester
from orb_core import build_orb

bt = ORBBacktester(symbol='AMD', start_date='2025-12-05', end_date='2025-12-05', verbose=False)
bt.load_data()
df = bt.df

day_df = df[df.index.date == pd.Timestamp('2025-12-05').date()]
orb = build_orb(day_df)

print(f"ORB High: {orb.session_high:.2f}, ORB Low: {orb.session_low:.2f}")
print(f"\n═══ Post-ORB Bars 09:35-10:30 ═══")

for ts, bar in day_df.iterrows():
    mins = ts.hour * 60 + ts.minute
    if 9*60+35 <= mins <= 10*60+30:
        open_in_range = (bar['open'] >= orb.session_low) and (bar['open'] <= orb.session_high)
        close_above = bar['close'] > orb.session_high
        close_below = bar['close'] < orb.session_low
        
        flag = ""
        if close_above and open_in_range:
            flag = "<<< LONG BREAKOUT"
        elif close_below and open_in_range:
            flag = "<<< SHORT BREAKOUT"
        elif close_above:
            flag = "(close above, open NOT in range)"
        elif close_below:
            flag = "(close below, open NOT in range)"
            
        print(f"{ts.strftime('%H:%M')} | O:{bar['open']:.2f} H:{bar['high']:.2f} L:{bar['low']:.2f} C:{bar['close']:.2f} | open_in_range={open_in_range} {flag}")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (0.0 hours old)
ORB High: 221.94, ORB Low: 217.00

═══ Post-ORB Bars 09:35-10:30 ═══
09:35 | O:221.56 H:221.77 L:220.01 C:220.08 | open_in_range=True 
09:36 | O:220.09 H:220.16 L:219.62 C:219.85 | open_in_range=True 
09:37 | O:219.87 H:220.07 L:219.09 C:219.26 | open_in_range=True 
09:38 | O:219.26 H:219.70 L:218.85 C:219.00 | open_in_range=True 
09:39 | O:219.05 H:219.18 L:218.00 C:218.07 | open_in_range=True 
09:40 | O:218.04 H:218.87 L:217.67 C:217.87 | open_in_range=True 
09:41 | O:217.81 H:217.99 L:216.38 C:216.74 | open_in_range=True <<< SHORT BREAKOUT
09:42 | O:216.67 H:217.54 L:216.50 C:217.41 | open_in_range=False 
09:43 | O:217.40 H:218.23 L:217.37 C:217.74 | open_in_range=True 
09:44 | O:217.74 H:218.12 L:217.57 C:217.95 | open_in_range=True 
09:45 | O:218.05 H:218.75 L:218.04 C:218.73 | open_in_range=True 
09:46 | O:218.69 H:219.08 L:218.5

In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
from orb_backtester import ORBBacktester
from orb_core import build_orb

bt = ORBBacktester(symbol='AMD', start_date='2025-12-05', end_date='2025-12-05', verbose=False)
bt.load_data()
df = bt.df

day_df = df[df.index.date == pd.Timestamp('2025-12-05').date()]
orb = build_orb(day_df)

print(f"ORB High: {orb.session_high:.2f}, ORB Low: {orb.session_low:.2f}")

for ts, bar in day_df.iterrows():
    if ts.hour == 10 and ts.minute in [5, 6, 7, 8]:
        atr = bar['atr_rth']
        threshold = atr * 0.1
        
        o, h, l, c = bar['open'], bar['high'], bar['low'], bar['close']
        
        candle_body = abs(c - o)
        candle_range = h - l
        body_ratio = candle_body / candle_range if candle_range > 0 else 0
        strong_body = body_ratio >= 0.5
        
        open_in_range = (o >= orb.session_low) and (o <= orb.session_high)
        sufficient_high = c > orb.session_high + threshold
        
        print(f"\n═══ {ts.strftime('%H:%M')} ═══")
        print(f"O:{o:.2f} H:{h:.2f} L:{l:.2f} C:{c:.2f}")
        print(f"ATR: {atr:.4f}, Threshold: {threshold:.4f}")
        print(f"Body ratio: {body_ratio:.2f} (need >= 0.5) -> strong={strong_body}")
        print(f"Open in range: {open_in_range} ({orb.session_low:.2f} <= {o:.2f} <= {orb.session_high:.2f})")
        print(f"Sufficient high: {sufficient_high} ({c:.2f} > {orb.session_high + threshold:.2f})")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (0.1 hours old)
ORB High: 221.94, ORB Low: 217.00

═══ 10:05 ═══
O:221.53 H:222.24 L:221.53 C:221.95
ATR: 0.2320, Threshold: 0.0232
Body ratio: 0.59 (need >= 0.5) -> strong=True
Open in range: True (217.00 <= 221.53 <= 221.94)
Sufficient high: False (221.95 > 221.96)

═══ 10:06 ═══
O:221.96 H:222.52 L:221.94 C:222.25
ATR: 0.2320, Threshold: 0.0232
Body ratio: 0.49 (need >= 0.5) -> strong=False
Open in range: False (217.00 <= 221.96 <= 221.94)
Sufficient high: True (222.25 > 221.96)

═══ 10:07 ═══
O:222.22 H:222.37 L:221.95 C:222.28
ATR: 0.2320, Threshold: 0.0232
Body ratio: 0.15 (need >= 0.5) -> strong=False
Open in range: False (217.00 <= 222.22 <= 221.94)
Sufficient high: True (222.28 > 221.96)

═══ 10:08 ═══
O:222.26 H:222.60 L:222.09 C:222.53
ATR: 0.2320, Threshold: 0.0232
Body ratio: 0.53 (need >= 0.5) -> strong=True
Open in range: False (217.00 

In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
from orb_backtester import ORBBacktester
from orb_core import build_orb

bt = ORBBacktester(symbol='AMD', start_date='2025-12-05', end_date='2025-12-05', verbose=False)
bt.load_data()
df = bt.df

day_df = df[df.index.date == pd.Timestamp('2025-12-05').date()]
orb = build_orb(day_df)

print(f"ORB High: {orb.session_high:.2f}, ORB Low: {orb.session_low:.2f}")

for ts, bar in day_df.iterrows():
    if ts.hour == 10 and ts.minute == 25:
        atr = bar['atr_rth']
        threshold = atr * 0.1
        
        o, h, l, c = bar['open'], bar['high'], bar['low'], bar['close']
        
        candle_body = abs(c - o)
        candle_range = h - l
        body_ratio = candle_body / candle_range if candle_range > 0 else 0
        strong_body = body_ratio >= 0.5
        
        open_in_range = (o >= orb.session_low) and (o <= orb.session_high)
        sufficient_high = c > orb.session_high + threshold
        
        print(f"\n═══ {ts.strftime('%H:%M')} ═══")
        print(f"O:{o:.2f} H:{h:.2f} L:{l:.2f} C:{c:.2f}")
        print(f"ATR: {atr:.4f}, Threshold: {threshold:.4f}")
        print(f"Body ratio: {body_ratio:.2f} (need >= 0.5) -> strong={strong_body}")
        print(f"Open in range: {open_in_range} ({orb.session_low:.2f} <= {o:.2f} <= {orb.session_high:.2f})")
        print(f"Sufficient high: {sufficient_high} ({c:.2f} > {orb.session_high + threshold:.2f})")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (0.2 hours old)
ORB High: 221.94, ORB Low: 217.00

═══ 10:25 ═══
O:221.84 H:222.21 L:221.75 C:222.21
ATR: 0.2320, Threshold: 0.0232
Body ratio: 0.81 (need >= 0.5) -> strong=True
Open in range: True (217.00 <= 221.84 <= 221.94)
Sufficient high: True (222.21 > 221.96)


In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
from orb_backtester import ORBBacktester
from orb_core import (VolState, calc_vol_state, prepare_vol_columns,
                      select_best_stop, check_trade_feasibility, build_orb,
                      check_long_breakout, check_short_breakout, 
                      PositionState, manage_position_exit, ORBState)

bt = ORBBacktester(symbol='AMD', start_date='2025-12-01', end_date='2025-12-31', verbose=False)
bt.load_data()
df = bt.df
df = prepare_vol_columns(df)
df['ema9'] = df['close'].ewm(span=9, adjust=False).mean()

MIN_ACCEPTABLE_RR = 1.5

print("═══ DECEMBER 2025 AMD - WITH BREAKOUT TRACKING ═══")
print(f"Min Acceptable R:R: {MIN_ACCEPTABLE_RR}")
print(f"{'Date':<12} {'Dir':<6} {'Entry':>8} {'Stop':>8} {'Type':<8} {'R:R':>6} {'Exit':>8} {'Reason':<15} {'P&L':>8}")
print("=" * 95)

total_trades = 0
total_skips = 0
total_pnl = 0

for date in pd.date_range('2025-12-01', '2025-12-31', freq='B'):
    day_df = df[df.index.date == date.date()]
    if len(day_df) == 0:
        continue
    
    # Build ORB with breakout tracking
    orb = ORBState()
    for ts, bar in day_df.iterrows():
        if ts.hour == 9 and 30 <= ts.minute < 35:
            if orb.session_high is None or bar['high'] > orb.session_high:
                orb.session_high = bar['high']
            if orb.session_low is None or bar['low'] < orb.session_low:
                orb.session_low = bar['low']
        elif ts.hour == 9 and ts.minute >= 35:
            orb.orb_complete = True
            break
    
    if not orb.orb_complete:
        continue
    
    vol = VolState()
    position = None
    last_exit_bar = -10
    exited_long = False
    exited_short = False
    
    for bar_num, (ts, bar) in enumerate(day_df.iterrows()):
        mins = ts.hour * 60 + ts.minute
        if mins < 9*60+30:
            continue
        
        vol_state = calc_vol_state(bar, ts, vol)
        
        atr = bar['atr_rth']
        if pd.isna(atr):
            continue
        ema = bar['ema9']
        
        # Reset breakout flags if price comes back inside ORB
        orb.reset_breakout_if_back_inside(bar['close'], exited_long, exited_short)
        exited_long = False
        exited_short = False
        
        if position is None and mins >= 9*60+35 and mins <= 10*60+30:
            if bar_num - last_exit_bar < 5:
                continue
            
            is_long_breakout = check_long_breakout(bar['open'], bar['high'], bar['low'], bar['close'],
                orb.session_high, orb.session_low, atr, orb.orb_complete, orb.has_broken_out_high)
            is_short_breakout = check_short_breakout(bar['open'], bar['high'], bar['low'], bar['close'],
                orb.session_high, orb.session_low, atr, orb.orb_complete, orb.has_broken_out_low)
            
            if is_long_breakout or is_short_breakout:
                is_long = is_long_breakout
                entry_price = bar['close']
                vwap = bar['vwap']
                bar_idx = day_df.index.get_loc(ts)
                best_stop, stop_type, achievable_rr = select_best_stop(
                    entry_price, atr, vwap, is_long, day_df, bar_idx, vol_state=vol_state)
                
                is_feasible, should_skip = check_trade_feasibility(achievable_rr, MIN_ACCEPTABLE_RR)
                
                # Consume the breakout (whether we take or skip)
                if is_long:
                    orb.consume_long_breakout()
                else:
                    orb.consume_short_breakout()
                
                if should_skip:
                    direction = "LONG" if is_long else "SHORT"
                    print(f"{date.strftime('%Y-%m-%d'):<12} {direction:<6} {entry_price:>8.2f} {best_stop:>8.2f} {stop_type:<8} {achievable_rr:>6.2f} {'---':>8} {'SKIP':^15} {'---':>8}")
                    total_skips += 1
                    last_exit_bar = bar_num
                    continue
                
                position = PositionState(entry_price=entry_price, initial_stop=best_stop, stop_type=stop_type,
                    is_long=is_long, risk=abs(entry_price - best_stop), target_rr=achievable_rr)
                direction = "LONG" if is_long else "SHORT"
        
        elif position is not None:
            is_closed, exit_price, exit_reason = manage_position_exit(position, bar, atr,
                break_even_rr=0.5, profit_target_rr=2.0, trailing_stop_distance=1.2,
                use_break_even=True, use_trailing=True, use_ema_exit=True, ema=ema, ema_confirmation_bars=1,
                use_aggressive_trailing=True, ema_tighten_zone=0.3, tightened_trail_distance=0.3)
            
            if is_closed:
                pnl = (exit_price - position.entry_price) if position.is_long else (position.entry_price - exit_price)
                total_pnl += pnl
                print(f"{date.strftime('%Y-%m-%d'):<12} {direction:<6} {position.entry_price:>8.2f} {position.initial_stop:>8.2f} {position.stop_type:<8} {position.target_rr:>6.2f} {exit_price:>8.2f} {exit_reason:<15} {pnl:>+8.2f}")
                
                # Mark exit for breakout reset logic
                if position.is_long:
                    exited_long = True
                else:
                    exited_short = True
                    
                position = None
                last_exit_bar = bar_num
                total_trades += 1
        
        if mins >= 15*60+55 and position is not None:
            exit_price = bar['close']
            pnl = (exit_price - position.entry_price) if position.is_long else (position.entry_price - exit_price)
            total_pnl += pnl
            print(f"{date.strftime('%Y-%m-%d'):<12} {direction:<6} {position.entry_price:>8.2f} {position.initial_stop:>8.2f} {position.stop_type:<8} {position.target_rr:>6.2f} {exit_price:>8.2f} {'EOD Close':<15} {pnl:>+8.2f}")
            
            if position.is_long:
                exited_long = True
            else:
                exited_short = True
                
            position = None
            total_trades += 1

print("=" * 95)
print(f"Total trades: {total_trades}, Total skips: {total_skips}, Total P&L: ${total_pnl:+.2f}")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (0.5 hours old)
═══ DECEMBER 2025 AMD - WITH BREAKOUT TRACKING ═══
Min Acceptable R:R: 1.5
Date         Dir       Entry     Stop Type        R:R     Exit Reason               P&L
2025-12-01   LONG     216.22   214.90 ATR        1.50   214.90 Initial Stop       -1.33
2025-12-02   LONG     224.33   222.89 ATR        1.50   224.71 EMA Exit           +0.38
2025-12-03   SHORT    213.28   214.68 ATR        1.20      ---      SKIP            ---
2025-12-03   LONG     216.95   215.90 SWING      1.78   215.90 Initial Stop       -1.05
2025-12-04   SHORT    215.28   216.71 ATR        1.50   216.71 Initial Stop       -1.43
2025-12-05   SHORT    216.74   218.25 ATR        1.50   218.25 Initial Stop       -1.51
2025-12-05   LONG     222.21   221.64 SWING      2.50   221.64 Initial Stop       -0.57
2025-12-08   LONG     221.94   220.82 ATR        1.50   221.94 Break

In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
from orb_backtester import ORBBacktester
from orb_core import (VolState, calc_vol_state, prepare_vol_columns,
                      select_best_stop, check_trade_feasibility, build_orb,
                      check_long_breakout, check_short_breakout, 
                      PositionState, manage_position_exit, ORBState)

bt = ORBBacktester(symbol='AMD', start_date='2025-12-01', end_date='2025-12-31', verbose=False)
bt.load_data()
df = bt.df
df = prepare_vol_columns(df)
df['ema9'] = df['close'].ewm(span=9, adjust=False).mean()

MIN_ACCEPTABLE_RR = 1.5

print("═══ DECEMBER 2025 AMD - WITH BREAKOUT TRACKING ═══")
print(f"Min Acceptable R:R: {MIN_ACCEPTABLE_RR}")
print(f"{'Date':<12} {'Time':<6} {'Dir':<6} {'Entry':>8} {'Stop':>8} {'Type':<8} {'R:R':>6} {'Exit':>8} {'Reason':<15} {'P&L':>8}")
print("=" * 105)

total_trades = 0
total_skips = 0
total_pnl = 0

for date in pd.date_range('2025-12-01', '2025-12-31', freq='B'):
    day_df = df[df.index.date == date.date()]
    if len(day_df) == 0:
        continue
    
    # Build ORB with breakout tracking
    orb = ORBState()s():
        if ts.hour == 9 and 30 <= ts.minute < 35:
            if orb.session_high is None or bar['
    for ts, bar in day_df.iterrowhigh'] > orb.session_high:
                orb.session_high = bar['high']
            if orb.session_low is None or bar['low'] < orb.session_low:
                orb.session_low = bar['low']
        elif ts.hour == 9 and ts.minute >= 35:
            orb.orb_complete = True
            break
    
    if not orb.orb_complete:
        continue
    
    vol = VolState()
    position = None
    last_exit_bar = -10
    exited_long = False
    exited_short = False
    entry_time = None
    
    for bar_num, (ts, bar) in enumerate(day_df.iterrows()):
        mins = ts.hour * 60 + ts.minute
        if mins < 9*60+30:
            continue
        
        vol_state = calc_vol_state(bar, ts, vol)
        
        atr = bar['atr_rth']
        if pd.isna(atr):
            continue
        ema = bar['ema9']
        
        # Reset breakout flags if price comes back inside ORB
        orb.reset_breakout_if_back_inside(bar['close'], exited_long, exited_short)
        exited_long = False
        exited_short = False
        
        if position is None and mins >= 9*60+35 and mins <= 10*60+30:
            if bar_num - last_exit_bar < 5:
                continue
            
            is_long_breakout = check_long_breakout(bar['open'], bar['high'], bar['low'], bar['close'],
                orb.session_high, orb.session_low, atr, orb.orb_complete,
                has_broken_out_high=orb.has_broken_out_high)
            is_short_breakout = check_short_breakout(bar['open'], bar['high'], bar['low'], bar['close'],
                orb.session_high, orb.session_low, atr, orb.orb_complete,
                has_broken_out_low=orb.has_broken_out_low)
            
            if is_long_breakout or is_short_breakout:
                is_long = is_long_breakout
                entry_price = bar['close']
                entry_time = ts.strftime('%H:%M')
                vwap = bar['vwap']
                bar_idx = day_df.index.get_loc(ts)
                best_stop, stop_type, achievable_rr = select_best_stop(
                    entry_price, atr, vwap, is_long, day_df, bar_idx, vol_state=vol_state)
                
                is_feasible, should_skip = check_trade_feasibility(achievable_rr, MIN_ACCEPTABLE_RR)
                
                # Consume the breakout (whether we take or skip)
                if is_long:
                    orb.consume_long_breakout()
                else:
                    orb.consume_short_breakout()
                
                if should_skip:
                    direction = "LONG" if is_long else "SHORT"
                    print(f"{date.strftime('%Y-%m-%d'):<12} {entry_time:<6} {direction:<6} {entry_price:>8.2f} {best_stop:>8.2f} {stop_type:<8} {achievable_rr:>6.2f} {'---':>8} {'SKIP':^15} {'---':>8}")
                    total_skips += 1
                    last_exit_bar = bar_num
                    continue
                
                position = PositionState(entry_price=entry_price, initial_stop=best_stop, stop_type=stop_type,
                    is_long=is_long, risk=abs(entry_price - best_stop), target_rr=achievable_rr)
                direction = "LONG" if is_long else "SHORT"
        
        elif position is not None:
            is_closed, exit_price, exit_reason = manage_position_exit(position, bar, atr,
                break_even_rr=0.5, profit_target_rr=2.0, trailing_stop_distance=1.2,
                use_break_even=True, use_trailing=True, use_ema_exit=True, ema=ema, ema_confirmation_bars=1,
                use_aggressive_trailing=True, ema_tighten_zone=0.3, tightened_trail_distance=0.3)
            
            if is_closed:
                pnl = (exit_price - position.entry_price) if position.is_long else (position.entry_price - exit_price)
                total_pnl += pnl
                print(f"{date.strftime('%Y-%m-%d'):<12} {entry_time:<6} {direction:<6} {position.entry_price:>8.2f} {position.initial_stop:>8.2f} {position.stop_type:<8} {position.target_rr:>6.2f} {exit_price:>8.2f} {exit_reason:<15} {pnl:>+8.2f}")
                
                # Mark exit for breakout reset logic
                if position.is_long:
                    exited_long = True
                else:
                    exited_short = True
                    
                position = None
                last_exit_bar = bar_num
                total_trades += 1
        
        if mins >= 15*60+55 and position is not None:
            exit_price = bar['close']
            pnl = (exit_price - position.entry_price) if position.is_long else (position.entry_price - exit_price)
            total_pnl += pnl
            print(f"{date.strftime('%Y-%m-%d'):<12} {entry_time:<6} {direction:<6} {position.entry_price:>8.2f} {position.initial_stop:>8.2f} {position.stop_type:<8} {position.target_rr:>6.2f} {exit_price:>8.2f} {'EOD Close':<15} {pnl:>+8.2f}")
            
            if position.is_long:
                exited_long = True
            else:
                exited_short = True
                
            position = None
            total_trades += 1

print("=" * 105)
print(f"Total trades: {total_trades}, Total skips: {total_skips}, Total P&L: ${total_pnl:+.2f}")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (0.6 hours old)
═══ DECEMBER 2025 AMD - WITH BREAKOUT TRACKING ═══
Min Acceptable R:R: 1.5
Date         Time   Dir       Entry     Stop Type        R:R     Exit Reason               P&L
2025-12-01   09:36  LONG     216.22   214.90 ATR        1.50   214.90 Initial Stop       -1.33
2025-12-02   09:36  LONG     224.33   222.89 ATR        1.50   224.71 EMA Exit           +0.38
2025-12-03   09:35  SHORT    213.28   214.68 ATR        1.20      ---      SKIP            ---
2025-12-03   10:17  LONG     216.95   215.90 SWING      1.78   215.90 Initial Stop       -1.05
2025-12-04   09:51  SHORT    215.28   216.71 ATR        1.50   216.71 Initial Stop       -1.43
2025-12-05   09:41  SHORT    216.74   218.25 ATR        1.50   218.25 Initial Stop       -1.51
2025-12-05   10:25  LONG     222.21   221.64 SWING      2.50   221.64 Initial Stop       -0.57
2025-12-08  

In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
from orb_backtester import ORBBacktester
from orb_core import prepare_vol_columns

bt = ORBBacktester(symbol='AMD', start_date='2025-12-01', end_date='2025-12-01', verbose=False)
bt.load_data()
df = bt.df
df = prepare_vol_columns(df)

day_df = df[df.index.date == pd.Timestamp('2025-12-01').date()]

for ts, bar in day_df.iterrows():
    if ts.hour == 9 and ts.minute == 36:
        entry = 216.22
        atr = bar['atr_rth']
        atr_stop = entry - (atr * 2.0)
        
        print(f"═══ Dec 1 @ 09:36 ═══")
        print(f"Entry: {entry}")
        print(f"ATR (Python): {atr:.4f}")
        print(f"ATR Stop (Python): {atr_stop:.4f}")
        print(f"ATR Stop (Pine): 214.83")
        print(f"\nTo get Pine stop, ATR should be: {(entry - 214.83) / 2.0:.4f}")
        break

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (0.8 hours old)
═══ Dec 1 @ 09:36 ═══
Entry: 216.22
ATR (Python): 0.6646
ATR Stop (Python): 214.8907
ATR Stop (Pine): 214.83

To get Pine stop, ATR should be: 0.6950


In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
from orb_backtester import ORBBacktester
from orb_core import prepare_vol_columns

bt = ORBBacktester(symbol='AMD', start_date='2025-12-01', end_date='2025-12-01', verbose=False)
bt.load_data()
df = bt.df
df = prepare_vol_columns(df)

day_df = df[df.index.date == pd.Timestamp('2025-12-01').date()]

print("═══ Dec 1 ATR values around entry ═══")
for ts, bar in day_df.iterrows():
    if ts.hour == 9 and 33 <= ts.minute <= 38:
        print(f"{ts.strftime('%H:%M')} | atr_rth={bar['atr_rth']:.4f}")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (0.9 hours old)
═══ Dec 1 ATR values around entry ═══
09:33 | atr_rth=0.6120
09:34 | atr_rth=0.6218
09:35 | atr_rth=0.6588
09:36 | atr_rth=0.6646
09:37 | atr_rth=0.6841
09:38 | atr_rth=0.6866


In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
from orb_backtester import ORBBacktester
from orb_core import prepare_vol_columns

bt = ORBBacktester(symbol='AMD', start_date='2025-12-01', end_date='2025-12-01', verbose=False)
bt.load_data()
df = bt.df
df = prepare_vol_columns(df)

day_df = df[df.index.date == pd.Timestamp('2025-12-01').date()]

print("═══ Dec 1 ATR values from 09:30 ═══")
for ts, bar in day_df.iterrows():
    if ts.hour == 9 and 30 <= ts.minute <= 36:
        print(f"{ts.strftime('%H:%M')} | atr_rth={bar['atr_rth']:.4f}")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (1.0 hours old)
═══ Dec 1 ATR values from 09:30 ═══
09:30 | atr_rth=0.2957
09:31 | atr_rth=0.5553
09:32 | atr_rth=0.5914
09:33 | atr_rth=0.6120
09:34 | atr_rth=0.6218
09:35 | atr_rth=0.6588
09:36 | atr_rth=0.6646


In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
from orb_backtester import ORBBacktester

bt = ORBBacktester(symbol='AMD', start_date='2025-11-29', end_date='2025-12-01', verbose=False)
bt.load_data()
df = bt.df

# Mark RTH
df['is_rth'] = df.index.map(lambda x: 9*60+30 <= x.hour*60+x.minute < 16*60)
rth_df = df[df['is_rth']].copy()

# Show last bars of Nov 29 and first bars of Dec 1
print("═══ RTH bars around Nov 29 → Dec 1 boundary ═══")
nov29 = rth_df[rth_df.index.date == pd.Timestamp('2025-11-29').date()].tail(3)
dec01 = rth_df[rth_df.index.date == pd.Timestamp('2025-12-01').date()].head(3)

for ts, bar in pd.concat([nov29, dec01]).iterrows():
    print(f"{ts} | close={bar['close']:.2f}")

print("\n═══ prev_close at Dec 1 09:30 ═══")
rth_df['prev_close'] = rth_df['close'].shift(1)
dec01_0930 = rth_df[rth_df.index.date == pd.Timestamp('2025-12-01').date()].iloc[0]
print(f"prev_close: {dec01_0930['prev_close']:.2f}")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (1.1 hours old)
═══ RTH bars around Nov 29 → Dec 1 boundary ═══
2025-12-01 09:30:00-05:00 | close=214.60
2025-12-01 09:31:00-05:00 | close=214.56
2025-12-01 09:32:00-05:00 | close=214.81

═══ prev_close at Dec 1 09:30 ═══
prev_close: 217.43


In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
import numpy as np
from orb_backtester import ORBBacktester

bt = ORBBacktester(symbol='AMD', start_date='2025-11-25', end_date='2025-12-01', verbose=False)
bt.load_data()
df = bt.df

# Mark RTH
df['is_rth'] = df.index.map(lambda x: 9*60+30 <= x.hour*60+x.minute < 16*60)
rth_df = df[df['is_rth']].copy()

# Calculate TR
rth_df['prev_close'] = rth_df['close'].shift(1)
tr1 = rth_df['high'] - rth_df['low']
tr2 = (rth_df['high'] - rth_df['prev_close']).abs()
tr3 = (rth_df['low'] - rth_df['prev_close']).abs()
rth_df['tr'] = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)

# Wilder RMA
def wilder_rma(series, n):
    rma = series.copy().astype(float)
    rma.iloc[:n] = series.iloc[:n].expanding().mean()
    for i in range(n, len(series)):
        rma.iloc[i] = (rma.iloc[i-1] * (n - 1) + series.iloc[i]) / n
    return rma

rth_df['atr'] = wilder_rma(rth_df['tr'], 14)
rth_df['atr_lagged'] = rth_df['atr'].shift(1)

# Show last 5 bars of Nov 28 and first 5 bars of Dec 1
print("═══ Last 5 RTH bars of Nov 28 ═══")
nov28 = rth_df[rth_df.index.date == pd.Timestamp('2025-11-28').date()].tail(5)
for ts, bar in nov28.iterrows():
    print(f"{ts.strftime('%H:%M')} | TR={bar['tr']:.4f} | ATR={bar['atr']:.4f}")

print("\n═══ First 5 RTH bars of Dec 1 ═══")
dec01 = rth_df[rth_df.index.date == pd.Timestamp('2025-12-01').date()].head(5)
for ts, bar in dec01.iterrows():
    print(f"{ts.strftime('%H:%M')} | prev_close={bar['prev_close']:.2f} | TR={bar['tr']:.4f} | ATR={bar['atr']:.4f} | ATR_lagged={bar['atr_lagged']:.4f}")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (1.2 hours old)
═══ Last 5 RTH bars of Nov 28 ═══
12:56 | TR=0.1953 | ATR=0.3071
12:57 | TR=0.1900 | ATR=0.2987
12:58 | TR=0.2301 | ATR=0.2938
12:59 | TR=0.5100 | ATR=0.3092
13:00 | TR=0.1200 | ATR=0.2957

═══ First 5 RTH bars of Dec 1 ═══
09:30 | prev_close=217.43 | TR=3.9300 | ATR=0.5553 | ATR_lagged=0.2957
09:31 | prev_close=214.60 | TR=1.0600 | ATR=0.5914 | ATR_lagged=0.5553
09:32 | prev_close=214.56 | TR=0.8800 | ATR=0.6120 | ATR_lagged=0.5914
09:33 | prev_close=214.81 | TR=0.7500 | ATR=0.6218 | ATR_lagged=0.6120
09:34 | prev_close=215.28 | TR=1.1400 | ATR=0.6588 | ATR_lagged=0.6218


In [2]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
from orb_backtester import ORBBacktester

bt = ORBBacktester(symbol='AMD', start_date='2025-11-28', end_date='2025-11-28', verbose=False)
bt.load_data()
df = bt.df

# Show bars around 13:00
for ts, bar in df.iterrows():
    if ts.hour == 12 and ts.minute >= 55 or ts.hour == 13 and ts.minute <= 5:
        print(f"{ts.strftime('%H:%M')} | O:{bar['open']:.2f} H:{bar['high']:.2f} L:{bar['low']:.2f} C:{bar['close']:.2f} V:{bar['volume']}")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (1.3 hours old)
12:55 | O:158.35 H:158.44 L:158.30 C:158.39 V:72129.0
12:56 | O:158.40 H:158.44 L:158.36 C:158.39 V:59889.0
12:57 | O:158.38 H:158.39 L:158.29 C:158.33 V:43020.0
12:58 | O:158.32 H:158.32 L:158.24 C:158.29 V:36685.0
12:59 | O:158.29 H:158.33 L:158.27 C:158.28 V:21115.0
13:00 | O:158.28 H:158.37 L:158.28 C:158.34 V:31930.0
13:01 | O:158.36 H:158.37 L:158.31 C:158.36 V:33488.0
13:02 | O:158.37 H:158.47 L:158.37 C:158.44 V:51995.0
13:03 | O:158.43 H:158.61 L:158.41 C:158.53 V:83290.0
13:04 | O:158.54 H:158.60 L:158.48 C:158.52 V:84566.0
13:05 | O:158.54 H:158.63 L:158.50 C:158.62 V:46419.0
12:55 | O:153.86 H:153.94 L:153.71 C:153.84 V:95920.0
12:56 | O:153.82 H:153.86 L:153.77 C:153.80 V:46736.0
12:57 | O:153.80 H:153.83 L:153.72 C:153.74 V:72274.0
12:58 | O:153.74 H:153.86 L:153.73 C:153.82 V:47009.0
12:59 | O:153.82 H:153.85 L:153.73 C:

In [ ]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
from orb_backtester import ORBBacktester
from orb_core import calc_atr_rth, detect_early_close_days

bt = ORBBacktester(symbol='AMD', start_date='2024-11-25', end_date='2024-12-05', verbose=False)
bt.load_data()
df = bt.df

# Check early close detection
early_closes = detect_early_close_days(df)
print(f"Early close days detected: {early_closes}")

# Check ATR on Nov 28 (should end at 12:59, not 13:00)
atr = calc_atr_rth(df, period=14)
nov28 = df[df.index.date == pd.Timestamp('2024-11-28').date()]
print(f"\nNov 28 last RTH bars and ATR:")
for ts in nov28.index[-5:]:
    print(f"  {ts.strftime('%H:%M')} ATR={atr.loc[ts]:.4f}")

In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
from orb_backtester import ORBBacktester
from orb_core import calc_atr_rth, detect_early_close_days

bt = ORBBacktester(symbol='AMD', start_date='2024-11-25', end_date='2024-12-05', verbose=False)
bt.load_data()
df = bt.df

# Check early close detection
early_closes = detect_early_close_days(df)
print(f"Early close days detected: {early_closes}")

# Check ATR on Nov 28 (should end at 12:59, not 13:00)
atr = calc_atr_rth(df, period=14)
nov28 = df[df.index.date == pd.Timestamp('2024-11-28').date()]
print(f"\nNov 28 last RTH bars and ATR:")
for ts in nov28.index[-5:]:
    print(f"  {ts.strftime('%H:%M')} ATR={atr.loc[ts]:.4f}")

ImportError: cannot import name 'detect_early_close_days' from 'orb_core' (C:\Users\phemm\orb_lab\src\orb_core.py)

In [1]:
"""
Test Early Close Fix - Validates ATR calculation handles early close days correctly.

Run from: C:\Users\phemm\orb_lab\src\
"""
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')

import pandas as pd
from orb_backtester import ORBBacktester
from orb_core import calc_atr_rth, detect_early_close_days

print("=" * 60)
print("EARLY CLOSE FIX VALIDATION")
print("=" * 60)

# Load AMD data spanning Nov 28 (early close day)
bt = ORBBacktester(symbol='AMD', start_date='2024-11-01', end_date='2024-12-10', verbose=False)
bt.load_data()
df = bt.df

# Detect early close days
early_closes = detect_early_close_days(df)
print(f"\n✓ Early close days detected: {sorted(early_closes)}")

# Calculate ATR with early close handling
atr = calc_atr_rth(df, period=14)

# Show Nov 28 data (early close day)
print("\n" + "=" * 60)
print("NOV 28, 2024 (Day Before Thanksgiving - 1PM Close)")
print("=" * 60)

nov28 = df[df.index.date == pd.Timestamp('2024-11-28').date()]
print(f"Total bars on Nov 28: {len(nov28)}")
print(f"First bar: {nov28.index[0].strftime('%H:%M')}")
print(f"Last bar:  {nov28.index[-1].strftime('%H:%M')}")

# Show last few bars with ATR
print("\nLast RTH bars on Nov 28:")
print("-" * 40)
for ts in nov28.index[-8:]:
    mins = ts.hour * 60 + ts.minute
    is_rth = 9*60+30 <= mins < 13*60  # Early close RTH
    marker = "✓ RTH" if is_rth else "✗ EXCLUDED"
    atr_val = atr.loc[ts]
    print(f"  {ts.strftime('%H:%M')} | ATR={atr_val:.4f} | {marker}")

# Show Dec 2 ATR (Dec 1 is Sunday)
print("\n" + "=" * 60)
print("DEC 2, 2024 (First Trading Day After Thanksgiving)")
print("=" * 60)

dec2 = df[df.index.date == pd.Timestamp('2024-12-02').date()]
print("\nFirst few RTH bars on Dec 2:")
print("-" * 40)
for ts in dec2.index[:10]:
    mins = ts.hour * 60 + ts.minute
    if 9*60+30 <= mins < 16*60:
        atr_val = atr.loc[ts]
        print(f"  {ts.strftime('%H:%M')} | ATR={atr_val:.4f}")

print("\n" + "=" * 60)
print("Compare Nov 28 12:59 ATR against Pine Script.")
print("Pine should show last bar at 12:59, ATR should MATCH.")
print("=" * 60)

SyntaxError: (unicode error) 'unicodeescape' codec can't decode bytes in position 99-100: truncated \UXXXXXXXX escape (3465364868.py, line 1)

In [2]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')

import pandas as pd
from orb_backtester import ORBBacktester
from orb_core import calc_atr_rth, detect_early_close_days

print("=" * 60)
print("EARLY CLOSE FIX VALIDATION")
print("=" * 60)

bt = ORBBacktester(symbol='AMD', start_date='2024-11-01', end_date='2024-12-10', verbose=False)
bt.load_data()
df = bt.df

early_closes = detect_early_close_days(df)
print(f"\nEarly close days detected: {sorted(early_closes)}")

atr = calc_atr_rth(df, period=14)

print("\n" + "=" * 60)
print("NOV 28, 2024 (Day Before Thanksgiving - 1PM Close)")
print("=" * 60)

nov28 = df[df.index.date == pd.Timestamp('2024-11-28').date()]
print(f"Total bars on Nov 28: {len(nov28)}")
print(f"Last bar: {nov28.index[-1].strftime('%H:%M')}")

print("\nLast RTH bars on Nov 28:")
for ts in nov28.index[-8:]:
    mins = ts.hour * 60 + ts.minute
    is_rth = 9*60+30 <= mins < 13*60
    marker = "RTH" if is_rth else "EXCLUDED"
    atr_val = atr.loc[ts]
    print(f"  {ts.strftime('%H:%M')} | ATR={atr_val:.4f} | {marker}")

print("\nCompare Nov 28 12:59 ATR against Pine Script.")

EARLY CLOSE FIX VALIDATION
[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (1.6 hours old)

Early close days detected: [datetime.date(2025, 11, 28), datetime.date(2025, 12, 24)]

NOV 28, 2024 (Day Before Thanksgiving - 1PM Close)
Total bars on Nov 28: 0


IndexError: index -1 is out of bounds for axis 0 with size 0

In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')

import pandas as pd
from orb_backtester import ORBBacktester
from orb_core import calc_atr_rth, detect_early_close_days

print("=" * 60)
print("EARLY CLOSE FIX VALIDATION")
print("=" * 60)

bt = ORBBacktester(symbol='AMD', start_date='2024-07-01', end_date='2025-01-15', verbose=False)
bt.load_data()
df = bt.df

print(f"\nData range: {df.index[0].date()} to {df.index[-1].date()}")

early_closes = detect_early_close_days(df)
print(f"Early close days detected: {sorted(early_closes)}")

atr = calc_atr_rth(df, period=14)

# Check each early close day found
for ec_date in sorted(early_closes):
    print("\n" + "=" * 60)
    print(f"{ec_date} (Early Close Day)")
    print("=" * 60)
    
    day_df = df[df.index.date == ec_date]
    if len(day_df) > 0:
        print(f"Total bars: {len(day_df)}")
        print(f"Last bar: {day_df.index[-1].strftime('%H:%M')}")
        
        print("\nLast few bars:")
        for ts in day_df.index[-6:]:
            mins = ts.hour * 60 + ts.minute
            is_rth = 9*60+30 <= mins < 13*60
            marker = "RTH" if is_rth else "EXCLUDED"
            atr_val = atr.loc[ts]
            print(f"  {ts.strftime('%H:%M')} | ATR={atr_val:.4f} | {marker}")

EARLY CLOSE FIX VALIDATION
[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (1.7 hours old)

Data range: 2025-07-21 to 2026-01-16
Early close days detected: [datetime.date(2025, 11, 28), datetime.date(2025, 12, 24)]

2025-11-28 (Early Close Day)
Total bars: 473
Last bar: 16:59

Last few bars:
  16:53 | ATR=0.3092 | EXCLUDED
  16:54 | ATR=0.3092 | EXCLUDED
  16:55 | ATR=0.3092 | EXCLUDED
  16:56 | ATR=0.3092 | EXCLUDED
  16:57 | ATR=0.3092 | EXCLUDED
  16:59 | ATR=0.3092 | EXCLUDED

2025-12-24 (Early Close Day)
Total bars: 345
Last bar: 16:59

Last few bars:
  16:46 | ATR=0.1787 | EXCLUDED
  16:52 | ATR=0.1787 | EXCLUDED
  16:55 | ATR=0.1787 | EXCLUDED
  16:56 | ATR=0.1787 | EXCLUDED
  16:57 | ATR=0.1787 | EXCLUDED
  16:59 | ATR=0.1787 | EXCLUDED


In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
from orb_backtester import ORBBacktester
from orb_core import (VolState, calc_vol_state, prepare_vol_columns,
                      select_best_stop, check_trade_feasibility, build_orb,
                      check_long_breakout, check_short_breakout, 
                      PositionState, manage_position_exit, ORBState)

bt = ORBBacktester(symbol='AMD', start_date='2025-12-01', end_date='2025-12-31', verbose=False)
bt.load_data()
df = bt.df
df = prepare_vol_columns(df)
df['ema9'] = df['close'].ewm(span=9, adjust=False).mean()

MIN_ACCEPTABLE_RR = 1.5

print("═══ DECEMBER 2025 AMD - WITH BREAKOUT TRACKING ═══")
print(f"Min Acceptable R:R: {MIN_ACCEPTABLE_RR}")
print(f"{'Date':<12} {'Time':<6} {'Dir':<6} {'Entry':>8} {'Stop':>8} {'Type':<8} {'R:R':>6} {'Exit':>8} {'Reason':<15} {'P&L':>8}")
print("=" * 105)

total_trades = 0
total_skips = 0
total_pnl = 0

for date in pd.date_range('2025-12-01', '2025-12-31', freq='B'):
    day_df = df[df.index.date == date.date()]
    if len(day_df) == 0:
        continue
    
    # Build ORB with breakout tracking
    orb = ORBState()
    for ts, bar in day_df.iterrows():
        if ts.hour == 9 and 30 <= ts.minute < 35:
            if orb.session_high is None or bar['high'] > orb.session_high:
                orb.session_high = bar['high']
            if orb.session_low is None or bar['low'] < orb.session_low:
                orb.session_low = bar['low']
        elif ts.hour == 9 and ts.minute >= 35:
            orb.orb_complete = True
            break
    
    if not orb.orb_complete:
        continue
    
    vol = VolState()
    position = None
    last_exit_bar = -10
    exited_long = False
    exited_short = False
    entry_time = None
    
    for bar_num, (ts, bar) in enumerate(day_df.iterrows()):
        mins = ts.hour * 60 + ts.minute
        if mins < 9*60+30:
            continue
        
        vol_state = calc_vol_state(bar, ts, vol)
        
        atr = bar['atr_rth']
        if pd.isna(atr):
            continue
        ema = bar['ema9']
        
        # Reset breakout flags if price comes back inside ORB
        orb.reset_breakout_if_back_inside(bar['close'], exited_long, exited_short)
        exited_long = False
        exited_short = False
        
        if position is None and mins >= 9*60+35 and mins <= 10*60+30:
            if bar_num - last_exit_bar < 5:
                continue
            
            is_long_breakout = check_long_breakout(bar['open'], bar['high'], bar['low'], bar['close'],
                orb.session_high, orb.session_low, atr, orb.orb_complete,
                has_broken_out_high=orb.has_broken_out_high)
            is_short_breakout = check_short_breakout(bar['open'], bar['high'], bar['low'], bar['close'],
                orb.session_high, orb.session_low, atr, orb.orb_complete,
                has_broken_out_low=orb.has_broken_out_low)
            
            if is_long_breakout or is_short_breakout:
                is_long = is_long_breakout
                entry_price = bar['close']
                entry_time = ts.strftime('%H:%M')
                vwap = bar['vwap']
                bar_idx = day_df.index.get_loc(ts)
                best_stop, stop_type, achievable_rr = select_best_stop(
                    entry_price, atr, vwap, is_long, day_df, bar_idx, vol_state=vol_state)
                
                is_feasible, should_skip = check_trade_feasibility(achievable_rr, MIN_ACCEPTABLE_RR)
                
                # Consume the breakout (whether we take or skip)
                if is_long:
                    orb.consume_long_breakout()
                else:
                    orb.consume_short_breakout()
                
                if should_skip:
                    direction = "LONG" if is_long else "SHORT"
                    print(f"{date.strftime('%Y-%m-%d'):<12} {entry_time:<6} {direction:<6} {entry_price:>8.2f} {best_stop:>8.2f} {stop_type:<8} {achievable_rr:>6.2f} {'---':>8} {'SKIP':^15} {'---':>8}")
                    total_skips += 1
                    last_exit_bar = bar_num
                    continue
                
                position = PositionState(entry_price=entry_price, initial_stop=best_stop, stop_type=stop_type,
                    is_long=is_long, risk=abs(entry_price - best_stop), target_rr=achievable_rr)
                direction = "LONG" if is_long else "SHORT"
        
        elif position is not None:
            is_closed, exit_price, exit_reason = manage_position_exit(position, bar, atr,
                break_even_rr=0.5, profit_target_rr=2.0, trailing_stop_distance=1.2,
                use_break_even=True, use_trailing=True, use_ema_exit=True, ema=ema, ema_confirmation_bars=1,
                use_aggressive_trailing=True, ema_tighten_zone=0.3, tightened_trail_distance=0.3)
            
            if is_closed:
                pnl = (exit_price - position.entry_price) if position.is_long else (position.entry_price - exit_price)
                total_pnl += pnl
                print(f"{date.strftime('%Y-%m-%d'):<12} {entry_time:<6} {direction:<6} {position.entry_price:>8.2f} {position.initial_stop:>8.2f} {position.stop_type:<8} {position.target_rr:>6.2f} {exit_price:>8.2f} {exit_reason:<15} {pnl:>+8.2f}")
                
                # Mark exit for breakout reset logic
                if position.is_long:
                    exited_long = True
                else:
                    exited_short = True
                    
                position = None
                last_exit_bar = bar_num
                total_trades += 1
        
        if mins >= 15*60+55 and position is not None:
            exit_price = bar['close']
            pnl = (exit_price - position.entry_price) if position.is_long else (position.entry_price - exit_price)
            total_pnl += pnl
            print(f"{date.strftime('%Y-%m-%d'):<12} {entry_time:<6} {direction:<6} {position.entry_price:>8.2f} {position.initial_stop:>8.2f} {position.stop_type:<8} {position.target_rr:>6.2f} {exit_price:>8.2f} {'EOD Close':<15} {pnl:>+8.2f}")
            
            if position.is_long:
                exited_long = True
            else:
                exited_short = True
                
            position = None
            total_trades += 1

print("=" * 105)
print(f"Total trades: {total_trades}, Total skips: {total_skips}, Total P&L: ${total_pnl:+.2f}")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (2.0 hours old)
═══ DECEMBER 2025 AMD - WITH BREAKOUT TRACKING ═══
Min Acceptable R:R: 1.5
Date         Time   Dir       Entry     Stop Type        R:R     Exit Reason               P&L
2025-12-01   09:36  LONG     216.22   214.90 ATR        1.50   214.90 Initial Stop       -1.33
2025-12-02   09:36  LONG     224.33   222.89 ATR        1.50   224.71 EMA Exit           +0.38
2025-12-03   09:35  SHORT    213.28   214.68 ATR        1.20      ---      SKIP            ---
2025-12-03   10:17  LONG     216.95   215.90 SWING      1.78   215.90 Initial Stop       -1.05
2025-12-04   09:51  SHORT    215.28   216.71 ATR        1.50   216.71 Initial Stop       -1.43
2025-12-05   09:41  SHORT    216.74   218.25 ATR        1.50   218.25 Initial Stop       -1.51
2025-12-05   10:25  LONG     222.21   221.64 SWING      2.50   221.64 Initial Stop       -0.57
2025-12-08  

In [1]:
from orb_core import ORBState
orb = ORBState()
print(hasattr(orb, 'has_broken_out_high'))
print(hasattr(orb, 'consume_long_breakout'))

ModuleNotFoundError: No module named 'orb_core'

In [2]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')

from orb_core import ORBState
orb = ORBState()
print(hasattr(orb, 'has_broken_out_high'))
print(hasattr(orb, 'consume_long_breakout'))

True
True


In [3]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')

import pandas as pd
from orb_backtester import ORBBacktester
from confluence_indicators import ConfluenceCalculator

# Load AMD data
bt = ORBBacktester(symbol='AMD', start_date='2025-12-01', end_date='2025-12-05', verbose=False)
bt.load_data()
df = bt.df

# AMD-specific presets
amd_params = {
    'ssl_baseline_length': 20,
    'ssl_length': 10,
    'wae_fast_ema': 14,
    'wae_slow_ema': 29,
    'wae_bb_length': 20,
    'wae_sensitivity': 300,
    'qqe_rsi1_length': 8,
    'qqe_rsi1_smoothing': 5,
    'qqe_rsi2_length': 4,
    'qqe_rsi2_smoothing': 4,
    'qqe_bb_length': 25,
    'vol_lookback': 12,
}

calc = ConfluenceCalculator(amd_params)
df = calc.compute_indicators(df)

print("=" * 70)
print("AMD DEC 2, 2025 - CONFLUENCE SCORES (AMD PRESETS)")
print("=" * 70)

dec2 = df[df.index.date == pd.Timestamp('2025-12-02').date()]

print(f"\n{'Time':<8} {'SSL_B':>6} {'SSL_S':>6} {'WAE_B':>6} {'WAE_S':>6} {'QQE_B':>6} {'QQE_S':>6} {'VOL':>6}")
print("-" * 70)

for ts, row in dec2.iterrows():
    if 9*60+34 <= ts.hour*60+ts.minute <= 9*60+45:
        print(f"{ts.strftime('%H:%M'):<8} "
              f"{row['ssl_score_bull']:>6.0f} {row['ssl_score_bear']:>6.0f} "
              f"{row['wae_score_bull']:>6.0f} {row['wae_score_bear']:>6.0f} "
              f"{row['qqe_score_bull']:>6.0f} {row['qqe_score_bear']:>6.0f} "
              f"{row['vol_score']:>6.2f}")

print("\n" + "=" * 70)
print("DETAILED VALUES AT 09:37 (likely breakout bar)")
print("=" * 70)

bar = dec2[dec2.index.map(lambda x: x.hour==9 and x.minute==37)]
if len(bar) > 0:
    row = bar.iloc[0]
    print(f"\nSSL (baseline={amd_params['ssl_baseline_length']}, len={amd_params['ssl_length']}):")
    print(f"  Baseline: {row['ssl_baseline']:.4f}")
    print(f"  Close: {row['close']:.4f}")
    print(f"  Bull Confirm: {row['ssl_bull_confirm']}, Accel: {row['ssl_accel_bull']}")
    print(f"  SCORE BULL: {row['ssl_score_bull']:.0f}")
    
    print(f"\nWAE (fast={amd_params['wae_fast_ema']}, slow={amd_params['wae_slow_ema']}, sens={amd_params['wae_sensitivity']}):")
    print(f"  Trend Up: {row['wae_trend_up']:.4f}")
    print(f"  Explosion: {row['wae_explosion_line']:.4f}")
    print(f"  Bull Confirm: {row['wae_bull_confirm']}, Accel: {row['wae_accel_bull']}")
    print(f"  SCORE BULL: {row['wae_score_bull']:.0f}")
    
    print(f"\nQQE (rsi1={amd_params['qqe_rsi1_length']}, rsi2={amd_params['qqe_rsi2_length']}, bb={amd_params['qqe_bb_length']}):")
    print(f"  Primary RSI: {row['qqe_primary_rsi']:.4f}")
    print(f"  Bull Signal: {row['qqe_bull_signal']}, Momentum Rising: {row['qqe_momentum_rising']}")
    print(f"  SCORE BULL: {row['qqe_score_bull']:.0f}")
    
    print(f"\nVOLUME (lookback={amd_params['vol_lookback']}):")
    print(f"  Vol Ratio: {row['vol_ratio']:.4f}")
    print(f"  SCORE: {row['vol_score']:.2f}")

print("\n" + "=" * 70)
print("Compare against Pine Data Window on AMD Dec 2")
print("=" * 70)

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ⚠ Cache stale (6 hours), refreshing...
  ✓ Fetched 103601 bars, cached for next time
AMD DEC 2, 2025 - CONFLUENCE SCORES (AMD PRESETS)

Time      SSL_B  SSL_S  WAE_B  WAE_S  QQE_B  QQE_S    VOL
----------------------------------------------------------------------
09:34         0      0      0      0      1      0   0.50
09:35         0      0      0      0      1      0   0.50
09:36         1      0      1      0      1      0   0.50
09:37         0      0      0      0      1      0   0.50
09:38         0      0      0      0      1      0   0.50
09:39         0      0      0      0      1      0   0.25
09:40         0      1      0      0      0      0   0.25
09:41         0      0      1      0      0      0   0.25
09:42         1      0      1      0      1      0   0.50
09:43         0      1      0      0      1      0   0.25
09:44         1      0      1      0   

In [4]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')

import pandas as pd
from orb_backtester import ORBBacktester
from confluence_indicators import ConfluenceCalculator

# Load AMD data
bt = ORBBacktester(symbol='AMD', start_date='2025-12-01', end_date='2025-12-05', verbose=False)
bt.load_data()
df = bt.df

# EXACT settings from Pine (presets OFF)
params = {
    'wae_fast_ema': 10,
    'wae_slow_ema': 20,
    'wae_bb_length': 12,
    'wae_bb_mult': 2.0,
    'wae_sensitivity': 225,
    'use_wae_acceleration': True,
}

calc = ConfluenceCalculator(params)
df = calc.compute_indicators(df)

print("=" * 70)
print("AMD DEC 2, 2025 - WAE VALUES (Pine settings: Fast=10, Slow=20, BB=12, Sens=225)")
print("=" * 70)

dec2 = df[df.index.date == pd.Timestamp('2025-12-02').date()]

print(f"\n{'Time':<8} {'TrendUp':>10} {'TrendPrev':>10} {'Explosion':>10} {'Confirm':>8} {'Accel':>6} {'Score':>6}")
print("-" * 70)

for ts, row in dec2.iterrows():
    if 9*60+34 <= ts.hour*60+ts.minute <= 9*60+40:
        trend_prev = df['wae_trend_up'].shift(1).loc[ts] if ts in df.index else 0
        print(f"{ts.strftime('%H:%M'):<8} "
              f"{row['wae_trend_up']:>10.4f} "
              f"{trend_prev:>10.4f} "
              f"{row['wae_explosion_line']:>10.4f} "
              f"{str(row['wae_bull_confirm']):>8} "
              f"{str(row['wae_accel_bull']):>6} "
              f"{row['wae_score_bull']:>6.0f}")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (1.0 hours old)
AMD DEC 2, 2025 - WAE VALUES (Pine settings: Fast=10, Slow=20, BB=12, Sens=225)

Time        TrendUp  TrendPrev  Explosion  Confirm  Accel  Score
----------------------------------------------------------------------
09:34       25.6463    33.2637     0.6035     True  False      0
09:35       22.0958    25.6463     0.6035     True  False      0
09:36       29.0141    22.0958     0.6035     True   True      1
09:37       18.0604    29.0141     0.6035     True  False      0
09:38       13.1981    18.0604     0.6035     True  False      0
09:39        8.8637    13.1981     0.6035     True  False      0
09:40        2.7354     8.8637     0.6035     True  False      0


In [5]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')

import pandas as pd
import numpy as np
from orb_backtester import ORBBacktester

bt = ORBBacktester(symbol='AMD', start_date='2025-12-01', end_date='2025-12-05', verbose=False)
bt.load_data()
df = bt.df

# === SSL CALCULATION (matching Pine exactly) ===

# HMA for baseline (Pine: ta.hma)
def hma(series, length):
    half_length = int(length / 2)
    sqrt_length = int(np.sqrt(length))
    wma1 = series.rolling(window=half_length, min_periods=half_length).apply(
        lambda x: np.sum(x * np.arange(1, len(x)+1)) / np.sum(np.arange(1, len(x)+1)), raw=True)
    wma2 = series.rolling(window=length, min_periods=length).apply(
        lambda x: np.sum(x * np.arange(1, len(x)+1)) / np.sum(np.arange(1, len(x)+1)), raw=True)
    raw_hma = 2 * wma1 - wma2
    return raw_hma.rolling(window=sqrt_length, min_periods=sqrt_length).apply(
        lambda x: np.sum(x * np.arange(1, len(x)+1)) / np.sum(np.arange(1, len(x)+1)), raw=True)

# JMA approximation (Pine: jma function)
def jma(series, length):
    momentum = series.diff()  # ta.change(src)
    # ta.atr(length) - need TR first
    tr = pd.concat([
        df['high'] - df['low'],
        (df['high'] - df['close'].shift(1)).abs(),
        (df['low'] - df['close'].shift(1)).abs()
    ], axis=1).max(axis=1)
    volatility = tr.ewm(span=length, adjust=False).mean()  # RMA approximation
    ratio = np.where(volatility != 0, momentum / volatility, 0)
    return (series + ratio * volatility).ewm(span=length, adjust=False).mean()

# Settings
ssl_baseline_length = 45
ssl_length = 5

# Calculate
ssl_baseline = hma(df['close'], ssl_baseline_length)
ssl_up = jma(df['high'], ssl_length)
ssl_down = jma(df['low'], ssl_length)

ssl_bull_confirm = df['close'] > ssl_baseline
ssl_bear_confirm = df['close'] < ssl_baseline

ssl_gap_bull = df['close'] - ssl_baseline
ssl_gap_bull_prev = df['close'].shift(1) - ssl_baseline.shift(1)
ssl_gap_bear = ssl_baseline - df['close']
ssl_gap_bear_prev = ssl_baseline.shift(1) - df['close'].shift(1)

ssl_accel_bull = ssl_gap_bull.fillna(0) > ssl_gap_bull_prev.fillna(0)
ssl_accel_bear = ssl_gap_bear.fillna(0) > ssl_gap_bear_prev.fillna(0)

ssl_score_bull = (ssl_bull_confirm & ssl_accel_bull).astype(float)
ssl_score_bear = (ssl_bear_confirm & ssl_accel_bear).astype(float)

# Output
print("=" * 70)
print("AMD DEC 2, 2025 - SSL VALUES (Baseline=45, Type=JMA, Length=5)")
print("=" * 70)

dec2 = df[df.index.date == pd.Timestamp('2025-12-02').date()]

print(f"\n{'Time':<8} {'Close':>10} {'Baseline':>10} {'Gap':>10} {'GapPrev':>10} {'Confirm':>8} {'Accel':>6} {'Score':>6}")
print("-" * 80)

for ts in dec2.index:
    if 9*60+34 <= ts.hour*60+ts.minute <= 9*60+40:
        idx = df.index.get_loc(ts)
        print(f"{ts.strftime('%H:%M'):<8} "
              f"{df['close'].iloc[idx]:>10.4f} "
              f"{ssl_baseline.iloc[idx]:>10.4f} "
              f"{ssl_gap_bull.iloc[idx]:>10.4f} "
              f"{ssl_gap_bull_prev.iloc[idx]:>10.4f} "
              f"{str(ssl_bull_confirm.iloc[idx]):>8} "
              f"{str(ssl_accel_bull.iloc[idx]):>6} "
              f"{ssl_score_bull.iloc[idx]:>6.0f}")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (1.1 hours old)
AMD DEC 2, 2025 - SSL VALUES (Baseline=45, Type=JMA, Length=5)

Time          Close   Baseline        Gap    GapPrev  Confirm  Accel  Score
--------------------------------------------------------------------------------
09:34      223.4800   221.9890     1.4910     1.5057     True  False      0
09:35      223.6500   222.0957     1.5543     1.4910     True   True      1
09:36      224.3297   222.2648     2.0649     1.5543     True   True      1
09:37      224.2100   222.4640     1.7460     2.0649     True  False      0
09:38      224.2730   222.6762     1.5968     1.7460     True  False      0
09:39      224.3100   222.8937     1.4163     1.5968     True  False      0
09:40      224.2050   223.1063     1.0987     1.4163     True  False      0


In [6]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')

import pandas as pd
import numpy as np
from orb_backtester import ORBBacktester

bt = ORBBacktester(symbol='AMD', start_date='2025-12-01', end_date='2025-12-05', verbose=False)
bt.load_data()
df = bt.df

# === WMA (Weighted Moving Average) ===
def wma(series, length):
    weights = np.arange(1, length + 1)
    return series.rolling(window=length, min_periods=length).apply(
        lambda x: np.dot(x, weights) / weights.sum(), raw=True)

# === HMA (Hull Moving Average) ===
def hma(series, length):
    half_length = length // 2
    sqrt_length = round(np.sqrt(length))
    wma_half = wma(series, half_length)
    wma_full = wma(series, length)
    raw = 2 * wma_half - wma_full
    return wma(raw, sqrt_length)

# === RTH FILTER ===
def is_rth(ts):
    return 9*60+30 <= ts.hour*60+ts.minute < 16*60

rth_mask = df.index.map(is_rth)
rth_close = df['close'].where(rth_mask)

# Calculate HMA on RTH-only, then forward-fill
ssl_baseline_rth = hma(rth_close, 45).ffill()

print("=" * 70)
print("AMD DEC 2, 2025 - SSL BASELINE (HMA 45, RTH-ONLY)")
print("=" * 70)

dec2 = df[df.index.date == pd.Timestamp('2025-12-02').date()]

print(f"\n{'Time':<8} {'Close':>10} {'Baseline':>10} {'Gap':>10}")
print("-" * 50)

for ts in dec2.index:
    if 9*60+34 <= ts.hour*60+ts.minute <= 9*60+40:
        idx = df.index.get_loc(ts)
        baseline = ssl_baseline_rth.iloc[idx]
        close_val = df['close'].iloc[idx]
        gap = close_val - baseline
        print(f"{ts.strftime('%H:%M'):<8} "
              f"{close_val:>10.4f} "
              f"{baseline:>10.4f} "
              f"{gap:>10.4f}")
        

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (1.3 hours old)
AMD DEC 2, 2025 - SSL BASELINE (HMA 45, RTH-ONLY)

Time          Close   Baseline        Gap
--------------------------------------------------
09:34      223.4800   220.0987     3.3813
09:35      223.6500   220.0987     3.5513
09:36      224.3297   220.0987     4.2310
09:37      224.2100   220.0987     4.1113
09:38      224.2730   220.0987     4.1743
09:39      224.3100   220.0987     4.2113
09:40      224.2050   220.0987     4.1063


In [7]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')

import pandas as pd
import numpy as np
from orb_backtester import ORBBacktester

bt = ORBBacktester(symbol='AMD', start_date='2025-12-01', end_date='2025-12-05', verbose=False)
bt.load_data()
df = bt.df

# === WMA (Weighted Moving Average) ===
def wma(series, length):
    weights = np.arange(1, length + 1)
    return series.rolling(window=length, min_periods=length).apply(
        lambda x: np.dot(x, weights) / weights.sum(), raw=True)

# === HMA (Hull Moving Average) ===
def hma(series, length):
    half_length = length // 2
    sqrt_length = round(np.sqrt(length))
    wma_half = wma(series, half_length)
    wma_full = wma(series, length)
    raw = 2 * wma_half - wma_full
    return wma(raw, sqrt_length)

# === RTH FILTER ===
def is_rth(ts):
    return 9*60+30 <= ts.hour*60+ts.minute < 16*60

# Create RTH-only dataframe (no gaps)
rth_df = df[df.index.map(is_rth)].copy()

# Calculate HMA on continuous RTH data
rth_df['ssl_baseline'] = hma(rth_df['close'], 45)

# Map back to full dataframe
df['ssl_baseline'] = rth_df['ssl_baseline'].reindex(df.index).ffill()

print("=" * 70)
print("AMD DEC 2, 2025 - SSL BASELINE (HMA 45, RTH-ONLY)")
print("=" * 70)

dec2 = df[df.index.date == pd.Timestamp('2025-12-02').date()]

print(f"\n{'Time':<8} {'Close':>10} {'Baseline':>10} {'Gap':>10}")
print("-" * 50)

for ts in dec2.index:
    if 9*60+34 <= ts.hour*60+ts.minute <= 9*60+40:
        baseline = df.loc[ts, 'ssl_baseline']
        close_val = df.loc[ts, 'close']
        gap = close_val - baseline
        print(f"{ts.strftime('%H:%M'):<8} "
              f"{close_val:>10.4f} "
              f"{baseline:>10.4f} "
              f"{gap:>10.4f}")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (1.4 hours old)
AMD DEC 2, 2025 - SSL BASELINE (HMA 45, RTH-ONLY)

Time          Close   Baseline        Gap
--------------------------------------------------
09:34      223.4800   220.7325     2.7475
09:35      223.6500   221.0556     2.5944
09:36      224.3297   221.4310     2.8987
09:37      224.2100   221.8302     2.3798
09:38      224.2730   222.2358     2.0372
09:39      224.3100   222.6332     1.6768
09:40      224.2050   223.0115     1.1935


In [8]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')

import numpy as np
import pandas as pd
from orb_backtester import ORBBacktester

bt = ORBBacktester(symbol='AMD', start_date='2025-12-01', end_date='2025-12-05', verbose=False)
bt.load_data()
df = bt.df

def wma(series: pd.Series, length: int) -> pd.Series:
    length = int(length)
    weights = np.arange(1, length + 1, dtype=float)
    denom = weights.sum()
    return series.rolling(length, min_periods=length).apply(
        lambda x: np.dot(x, weights) / denom, raw=True
    )

def hma_variant(series: pd.Series, length: int, half_mode: str, sqrt_mode: str) -> pd.Series:
    L = int(length)

    if half_mode == "floor":
        half = L // 2
    elif half_mode == "ceil":
        half = int(np.ceil(L / 2))
    elif half_mode == "round":
        half = int(np.round(L / 2))
    else:
        raise ValueError("half_mode must be floor|ceil|round")

    r = np.sqrt(L)
    if sqrt_mode == "floor":
        sq = int(np.floor(r))
    elif sqrt_mode == "ceil":
        sq = int(np.ceil(r))
    elif sqrt_mode == "round":
        sq = int(np.round(r))
    else:
        raise ValueError("sqrt_mode must be floor|ceil|round")

    raw = 2 * wma(series, half) - wma(series, L)
    return wma(raw, sq)

def is_rth(ts):
    m = ts.hour * 60 + ts.minute
    return (9*60+30) <= m < (16*60)

rth_df = df[df.index.map(is_rth)].copy()

targets = [
    pd.Timestamp("2025-12-02 09:36:00"),
    pd.Timestamp("2025-12-02 09:37:00"),
]

# Pine RTH baselines from your debug box
pine_baseline = {
    targets[0]: 221.5540,   # 09:36 RTH
    targets[1]: 221.9643,   # 09:37 RTH
}

results = []

for half_mode in ["floor", "ceil", "round"]:
    for sqrt_mode in ["floor", "ceil", "round"]:
        h = hma_variant(rth_df["close"], 45, half_mode, sqrt_mode)
        err = 0.0
        ok = True
        for t in targets:
            if t not in rth_df.index:
                ok = False
                break
            val = float(h.loc[t])
            err += abs(val - pine_baseline[t])
        if ok and np.isfinite(err):
            results.append((err, half_mode, sqrt_mode))

results.sort(key=lambda x: x[0])

print("Top HMA rounding matches (lowest total abs error):")
for err, half_mode, sqrt_mode in results[:5]:
    print(f"  err={err:.6f}  half={half_mode}  sqrt={sqrt_mode}")

# Print winning values
if results:
    _, half_mode, sqrt_mode = results[0]
    h = hma_variant(rth_df["close"], 45, half_mode, sqrt_mode)
    print("\nWinning variant details:")
    for t in targets:
        print(f"  {t.strftime('%H:%M')} HMA={float(h.loc[t]):.4f}  Pine={pine_baseline[t]:.4f}  diff={float(h.loc[t]) - pine_baseline[t]:.4f}")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (1.6 hours old)
Top HMA rounding matches (lowest total abs error):


In [9]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')

import numpy as np
import pandas as pd
from orb_backtester import ORBBacktester

bt = ORBBacktester(symbol='AMD', start_date='2025-12-01', end_date='2025-12-05', verbose=False)
bt.load_data()
df = bt.df

def wma(series: pd.Series, length: int) -> pd.Series:
    length = int(length)
    weights = np.arange(1, length + 1, dtype=float)
    denom = weights.sum()
    return series.rolling(length, min_periods=length).apply(
        lambda x: np.dot(x, weights) / denom, raw=True
    )

def hma_variant(series: pd.Series, length: int, half_mode: str, sqrt_mode: str) -> pd.Series:
    L = int(length)

    if half_mode == "floor":
        half = L // 2
    elif half_mode == "ceil":
        half = int(np.ceil(L / 2))
    elif half_mode == "round":
        half = int(np.round(L / 2))
    else:
        raise ValueError("half_mode must be floor|ceil|round")

    r = np.sqrt(L)
    if sqrt_mode == "floor":
        sq = int(np.floor(r))
    elif sqrt_mode == "ceil":
        sq = int(np.ceil(r))
    elif sqrt_mode == "round":
        sq = int(np.round(r))
    else:
        raise ValueError("sqrt_mode must be floor|ceil|round")

    raw = 2 * wma(series, half) - wma(series, L)
    return wma(raw, sq)

def is_rth(ts):
    m = ts.hour * 60 + ts.minute
    return (9*60+30) <= m < (16*60)

rth_df = df[df.index.map(is_rth)].copy()

# Debug: show what's in the index
print("RTH df shape:", rth_df.shape)
print("Sample RTH index:")
dec2_rth = rth_df[rth_df.index.date == pd.Timestamp('2025-12-02').date()]
print(dec2_rth.index[:10])

# Find actual 09:36 and 09:37 timestamps
targets = []
for ts in dec2_rth.index:
    if ts.hour == 9 and ts.minute in [36, 37]:
        targets.append(ts)
        print(f"Found: {ts}")

if len(targets) < 2:
    print("ERROR: Could not find target timestamps")
else:
    # Pine RTH baselines from your debug box
    pine_baseline = {
        targets[0]: 221.5540,   # 09:36 RTH
        targets[1]: 221.9643,   # 09:37 RTH
    }

    results = []

    for half_mode in ["floor", "ceil", "round"]:
        for sqrt_mode in ["floor", "ceil", "round"]:
            h = hma_variant(rth_df["close"], 45, half_mode, sqrt_mode)
            err = 0.0
            ok = True
            for t in targets:
                if t not in h.index:
                    ok = False
                    break
                val = h.loc[t]
                if pd.isna(val):
                    ok = False
                    break
                err += abs(val - pine_baseline[t])
            if ok:
                results.append((err, half_mode, sqrt_mode))

    results.sort(key=lambda x: x[0])

    print("\nTop HMA rounding matches (lowest total abs error):")
    for err, half_mode, sqrt_mode in results[:5]:
        print(f"  err={err:.6f}  half={half_mode}  sqrt={sqrt_mode}")

    # Print winning values
    if results:
        _, half_mode, sqrt_mode = results[0]
        h = hma_variant(rth_df["close"], 45, half_mode, sqrt_mode)
        print("\nWinning variant details:")
        for t in targets:
            print(f"  {t.strftime('%H:%M')} HMA={float(h.loc[t]):.4f}  Pine={pine_baseline[t]:.4f}  diff={float(h.loc[t]) - pine_baseline[t]:.4f}")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (1.6 hours old)
RTH df shape: (48782, 65)
Sample RTH index:
DatetimeIndex(['2025-12-02 09:30:00-05:00', '2025-12-02 09:31:00-05:00',
               '2025-12-02 09:32:00-05:00', '2025-12-02 09:33:00-05:00',
               '2025-12-02 09:34:00-05:00', '2025-12-02 09:35:00-05:00',
               '2025-12-02 09:36:00-05:00', '2025-12-02 09:37:00-05:00',
               '2025-12-02 09:38:00-05:00', '2025-12-02 09:39:00-05:00'],
              dtype='datetime64[ns, US/Eastern]', name='timestamp', freq=None)
Found: 2025-12-02 09:36:00-05:00
Found: 2025-12-02 09:37:00-05:00

Top HMA rounding matches (lowest total abs error):
  err=0.000047  half=floor  sqrt=floor
  err=0.000047  half=round  sqrt=floor
  err=0.168178  half=ceil  sqrt=floor
  err=0.257140  half=floor  sqrt=ceil
  err=0.257140  half=floor  sqrt=round

Winning variant details:
  09:36 HMA=221.5540 

In [10]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')

import numpy as np
import pandas as pd
from orb_backtester import ORBBacktester

bt = ORBBacktester(symbol='AMD', start_date='2025-12-01', end_date='2025-12-05', verbose=False)
bt.load_data()
df = bt.df

# === RTH FILTER ===
def is_rth(ts):
    m = ts.hour * 60 + ts.minute
    return (9*60+30) <= m < (16*60)

rth_df = df[df.index.map(is_rth)].copy()

# === WMA (Weighted Moving Average) ===
def wma(series, length):
    length = int(length)
    weights = np.arange(1, length + 1, dtype=float)
    denom = weights.sum()
    return series.rolling(length, min_periods=length).apply(
        lambda x: np.dot(x, weights) / denom, raw=True)

# === HMA (Hull Moving Average) - Pine parity confirmed ===
def hma(series, length):
    half = length // 2              # floor
    sq = int(np.sqrt(length))       # floor
    raw = 2 * wma(series, half) - wma(series, length)
    return wma(raw, sq)

# === ATR (Wilder's RMA) - RTH only ===
def atr_rth(high, low, close, length):
    tr = pd.concat([
        high - low,
        (high - close.shift(1)).abs(),
        (low - close.shift(1)).abs()
    ], axis=1).max(axis=1)
    # Wilder's RMA: alpha = 1/length
    return tr.ewm(alpha=1/length, adjust=False).mean()

# === JMA (matching Pine exactly) ===
# jma(src, length) =>
#     momentum = ta.change(src)
#     volatility = ta.atr(length)
#     ratio = volatility != 0 ? momentum / volatility : 0
#     ta.ema(src + ratio * volatility, length)
def jma(src, length, high, low, close):
    momentum = src.diff()  # ta.change(src)
    volatility = atr_rth(high, low, close, length)
    ratio = np.where(volatility != 0, momentum / volatility, 0)
    return (src + ratio * volatility).ewm(span=length, adjust=False).mean()

# === SSL SETTINGS ===
ssl_baseline_length = 45
ssl_length = 5

# === SSL CALCULATIONS (all on RTH-only data) ===
rth_df['ssl_baseline'] = hma(rth_df['close'], ssl_baseline_length)
rth_df['ssl_up'] = jma(rth_df['high'], ssl_length, rth_df['high'], rth_df['low'], rth_df['close'])
rth_df['ssl_down'] = jma(rth_df['low'], ssl_length, rth_df['high'], rth_df['low'], rth_df['close'])

# Confirmation
rth_df['ssl_bull_confirm'] = rth_df['close'] > rth_df['ssl_baseline']
rth_df['ssl_bear_confirm'] = rth_df['close'] < rth_df['ssl_baseline']

# Gap calculation
rth_df['ssl_gap_bull'] = rth_df['close'] - rth_df['ssl_baseline']
rth_df['ssl_gap_bull_prev'] = rth_df['close'].shift(1) - rth_df['ssl_baseline'].shift(1)
rth_df['ssl_gap_bear'] = rth_df['ssl_baseline'] - rth_df['close']
rth_df['ssl_gap_bear_prev'] = rth_df['ssl_baseline'].shift(1) - rth_df['close'].shift(1)

# Acceleration (nz = fillna(0))
rth_df['ssl_accel_bull'] = rth_df['ssl_gap_bull'].fillna(0) > rth_df['ssl_gap_bull_prev'].fillna(0)
rth_df['ssl_accel_bear'] = rth_df['ssl_gap_bear'].fillna(0) > rth_df['ssl_gap_bear_prev'].fillna(0)

# Score (with useSSLMomentumFilter = true)
rth_df['ssl_score_bull'] = (rth_df['ssl_bull_confirm'] & rth_df['ssl_accel_bull']).astype(float)
rth_df['ssl_score_bear'] = (rth_df['ssl_bear_confirm'] & rth_df['ssl_accel_bear']).astype(float)

# === OUTPUT ===
print("=" * 90)
print("AMD DEC 2, 2025 - FULL SSL (RTH-ONLY)")
print("=" * 90)

dec2 = rth_df[rth_df.index.date == pd.Timestamp('2025-12-02').date()]

print(f"\n{'Time':<8} {'Close':>9} {'Baseline':>9} {'Gap':>8} {'GapPrev':>8} {'Confirm':>8} {'Accel':>6} {'Score':>6}")
print("-" * 90)

for ts in dec2.index:
    if 9*60+34 <= ts.hour*60+ts.minute <= 9*60+40:
        r = rth_df.loc[ts]
        print(f"{ts.strftime('%H:%M'):<8} "
              f"{r['close']:>9.4f} "
              f"{r['ssl_baseline']:>9.4f} "
              f"{r['ssl_gap_bull']:>8.4f} "
              f"{r['ssl_gap_bull_prev']:>8.4f} "
              f"{str(r['ssl_bull_confirm']):>8} "
              f"{str(r['ssl_accel_bull']):>6} "
              f"{r['ssl_score_bull']:>6.0f}")

print("\n" + "=" * 90)
print("Compare to Pine SSL Debug at 09:36 and 09:37")
print("=" * 90)

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (1.8 hours old)
AMD DEC 2, 2025 - FULL SSL (RTH-ONLY)

Time         Close  Baseline      Gap  GapPrev  Confirm  Accel  Score
------------------------------------------------------------------------------------------
09:34     223.4800  220.8089   2.6711   2.9326     True  False      0
09:35     223.6500  221.1583   2.4917   2.6711     True  False      0
09:36     224.3297  221.5540   2.7757   2.4917     True   True      1
09:37     224.2100  221.9643   2.2457   2.7757     True  False      0
09:38     224.2730  222.3720   1.9010   2.2457     True  False      0
09:39     224.3100  222.7682   1.5418   1.9010     True  False      0
09:40     224.2050  223.1430   1.0620   1.5418     True  False      0

Compare to Pine SSL Debug at 09:36 and 09:37


In [11]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
from orb_backtester import ORBBacktester
from confluence_indicators import ConfluenceCalculator

# Load data
bt = ORBBacktester(symbol='AMD', start_date='2025-12-01', end_date='2025-12-05', verbose=False)
bt.load_data()
df = bt.df

# Compute SSL using the new module
calc = ConfluenceCalculator({'ssl_baseline_length': 45, 'ssl_length': 5, 'use_ssl_momentum': True})
calc.compute_indicators(df)

# Output
print("=" * 90)
print("AMD DEC 2, 2025 - SSL FROM confluence_indicators.py")
print("=" * 90)

dec2 = df[df.index.date == pd.Timestamp('2025-12-02').date()]

print(f"\n{'Time':<8} {'Close':>9} {'Baseline':>9} {'Score Bull':>11} {'Score Bear':>11}")
print("-" * 90)

for ts in dec2.index:
    if 9*60+34 <= ts.hour*60+ts.minute <= 9*60+40:
        r = df.loc[ts]
        baseline = r['ssl_baseline'] if pd.notna(r['ssl_baseline']) else 0
        score_bull = r['ssl_score_bull'] if pd.notna(r['ssl_score_bull']) else 0
        score_bear = r['ssl_score_bear'] if pd.notna(r['ssl_score_bear']) else 0
        print(f"{ts.strftime('%H:%M'):<8} "
              f"{r['close']:>9.4f} "
              f"{baseline:>9.4f} "
              f"{score_bull:>11.0f} "
              f"{score_bear:>11.0f}")

print("\n" + "=" * 90)
print("Expected: 09:36 Score Bull=1, 09:37 Score Bull=0")
print("=" * 90)

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ⚠ Cache stale (18 hours), refreshing...
  ✓ Fetched 102820 bars, cached for next time
AMD DEC 2, 2025 - SSL FROM confluence_indicators.py

Time         Close  Baseline  Score Bull  Score Bear
------------------------------------------------------------------------------------------
09:34     223.4800  221.9716           0           0
09:35     223.6500  222.3910           0           0
09:36     224.3297  222.8745           1           0
09:37     224.2100  223.3526           0           0
09:38     224.2730  223.8024           0           0
09:39     224.3100  224.1989           0           0
09:40     224.2050  224.5073           0           1

Expected: 09:36 Score Bull=1, 09:37 Score Bull=0


In [12]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
from orb_backtester import ORBBacktester
from confluence_indicators import ConfluenceCalculator

bt = ORBBacktester(symbol='AMD', start_date='2025-12-01', end_date='2025-12-05', verbose=False)
bt.load_data()
df = bt.df

calc = ConfluenceCalculator({'ssl_baseline_length': 45, 'ssl_length': 5})
calc.compute_indicators(df)

# Check what rth_df looks like
print(f"Full df rows: {len(df)}")
print(f"RTH df rows: {len(calc.rth_df)}")
print(f"\nRTH df Dec 2 09:34-09:37:")
dec2_rth = calc.rth_df[calc.rth_df.index.date == pd.Timestamp('2025-12-02').date()]
print(dec2_rth.loc['2025-12-02 09:34:00':'2025-12-02 09:37:00'][['close', 'ssl_baseline']])

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (0.0 hours old)
Full df rows: 102820


AttributeError: 'ConfluenceCalculator' object has no attribute 'rth_df'

In [13]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
from orb_backtester import ORBBacktester
from confluence_indicators import ConfluenceCalculator

# Load data
bt = ORBBacktester(symbol='AMD', start_date='2025-12-01', end_date='2025-12-05', verbose=False)
bt.load_data()
df = bt.df

# Compute SSL using the new module
calc = ConfluenceCalculator({'ssl_baseline_length': 45, 'ssl_length': 5, 'use_ssl_momentum': True})
calc.compute_indicators(df)

# Output
print("=" * 90)
print("AMD DEC 2, 2025 - SSL FROM confluence_indicators.py")
print("=" * 90)

dec2 = df[df.index.date == pd.Timestamp('2025-12-02').date()]

print(f"\n{'Time':<8} {'Close':>9} {'Baseline':>9} {'Score Bull':>11} {'Score Bear':>11}")
print("-" * 90)

for ts in dec2.index:
    if 9*60+34 <= ts.hour*60+ts.minute <= 9*60+40:
        r = df.loc[ts]
        baseline = r['ssl_baseline'] if pd.notna(r['ssl_baseline']) else 0
        score_bull = r['ssl_score_bull'] if pd.notna(r['ssl_score_bull']) else 0
        score_bear = r['ssl_score_bear'] if pd.notna(r['ssl_score_bear']) else 0
        print(f"{ts.strftime('%H:%M'):<8} "
              f"{r['close']:>9.4f} "
              f"{baseline:>9.4f} "
              f"{score_bull:>11.0f} "
              f"{score_bear:>11.0f}")

print("\n" + "=" * 90)
print("Expected: 09:36 Score Bull=1, 09:37 Score Bull=0")
print("=" * 90)

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (0.1 hours old)
AMD DEC 2, 2025 - SSL FROM confluence_indicators.py

Time         Close  Baseline  Score Bull  Score Bear
------------------------------------------------------------------------------------------
09:34     223.4800  221.9716           0           0
09:35     223.6500  222.3910           0           0
09:36     224.3297  222.8745           1           0
09:37     224.2100  223.3526           0           0
09:38     224.2730  223.8024           0           0
09:39     224.3100  224.1989           0           0
09:40     224.2050  224.5073           0           1

Expected: 09:36 Score Bull=1, 09:37 Score Bull=0


In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
from orb_backtester import ORBBacktester
from confluence_indicators import ConfluenceCalculator

bt = ORBBacktester(symbol='AMD', start_date='2025-12-01', end_date='2025-12-05', verbose=False)
bt.load_data()
df = bt.df

calc = ConfluenceCalculator({'ssl_baseline_length': 45, 'ssl_length': 5})
calc.compute_indicators(df)

print(f"Full df rows: {len(df)}")
print(f"RTH df rows: {len(calc.rth_df)}")
print(f"\nRTH df Dec 2 09:34-09:37:")
dec2_rth = calc.rth_df[calc.rth_df.index.date == pd.Timestamp('2025-12-02').date()]
print(dec2_rth.loc['2025-12-02 09:34:00':'2025-12-02 09:37:00'][['close', 'ssl_baseline']])

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (0.2 hours old)


AttributeError: 'NoneType' object has no attribute 'index'

In [2]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
from orb_backtester import ORBBacktester

bt = ORBBacktester(symbol='AMD', start_date='2025-12-01', end_date='2025-12-05', verbose=False)
bt.load_data()
print(f"bt.df is None: {bt.df is None}")
print(f"bt.df type: {type(bt.df)}")
if bt.df is not None:
    print(f"bt.df rows: {len(bt.df)}")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (0.2 hours old)
bt.df is None: True
bt.df type: <class 'NoneType'>


In [3]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
from orb_backtester import ORBBacktester

bt = ORBBacktester(symbol='AMD', start_date='2025-12-01', end_date='2025-12-05', verbose=False)
result = bt.load_data()

print(f"bt.df: {bt.df is None}")
print(f"result: {result is None}")
print(f"result type: {type(result)}")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (0.2 hours old)
bt.df: True
result: True
result type: <class 'NoneType'>


In [4]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
from data_collector import PolygonDataCollector

collector = PolygonDataCollector()
df = collector.fetch_bars('AMD', days_back=180, bar_size=1, extended_hours=True)

print(f"df is None: {df is None}")
print(f"df type: {type(df)}")
if df is not None:
    print(f"df rows: {len(df)}")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (0.2 hours old)
df is None: False
df type: <class 'pandas.core.frame.DataFrame'>
df rows: 102820


In [5]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
from data_collector import PolygonDataCollector
from confluence_indicators import ConfluenceCalculator

# Load data directly
collector = PolygonDataCollector()
df = collector.fetch_bars('AMD', days_back=180, bar_size=1, extended_hours=True)

# Compute SSL
calc = ConfluenceCalculator({'ssl_baseline_length': 45, 'ssl_length': 5, 'use_ssl_momentum': True})
calc.compute_indicators(df)

# Output
print("=" * 90)
print("AMD DEC 2, 2025 - SSL FROM confluence_indicators.py")
print("=" * 90)

dec2 = df[df.index.date == pd.Timestamp('2025-12-02').date()]

print(f"\n{'Time':<8} {'Close':>9} {'Baseline':>9} {'Score Bull':>11} {'Score Bear':>11}")
print("-" * 90)

for ts in dec2.index:
    if 9*60+34 <= ts.hour*60+ts.minute <= 9*60+40:
        r = df.loc[ts]
        baseline = r['ssl_baseline'] if pd.notna(r['ssl_baseline']) else 0
        score_bull = r['ssl_score_bull'] if pd.notna(r['ssl_score_bull']) else 0
        score_bear = r['ssl_score_bear'] if pd.notna(r['ssl_score_bear']) else 0
        print(f"{ts.strftime('%H:%M'):<8} "
              f"{r['close']:>9.4f} "
              f"{baseline:>9.4f} "
              f"{score_bull:>11.0f} "
              f"{score_bear:>11.0f}")

print("\n" + "=" * 90)
print("Expected: Baseline 221.554 @ 09:36, Score Bull=1 @ 09:36, Score Bull=0 @ 09:37")
print("=" * 90)

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (0.3 hours old)
AMD DEC 2, 2025 - SSL FROM confluence_indicators.py

Time         Close  Baseline  Score Bull  Score Bear
------------------------------------------------------------------------------------------
09:34     223.4800  220.8089           0           0
09:35     223.6500  221.1583           0           0
09:36     224.3297  221.5540           1           0
09:37     224.2100  221.9643           0           0
09:38     224.2730  222.3720           0           0
09:39     224.3100  222.7682           0           0
09:40     224.2050  223.1430           0           0

Expected: Baseline 221.554 @ 09:36, Score Bull=1 @ 09:36, Score Bull=0 @ 09:37


In [6]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
import numpy as np
from data_collector import PolygonDataCollector

# Load data
collector = PolygonDataCollector()
df = collector.fetch_bars('AMD', days_back=180, bar_size=1, extended_hours=True)

# === QQE SETTINGS (matching your Pine) ===
qqe_rsi1_length = 6
qqe_rsi1_smoothing = 5
qqe_factor_primary = 3.0
qqe_rsi2_length = 6
qqe_rsi2_smoothing = 5
qqe_factor_secondary = 1.61
threshold_secondary = 3.0
qqe_bb_length = 50
qqe_bb_mult = 0.35

# === RSI CALCULATION ===
def calc_rsi(series, length):
    delta = series.diff()
    gain = delta.where(delta > 0, 0.0)
    loss = (-delta).where(delta < 0, 0.0)
    avg_gain = gain.ewm(alpha=1/length, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/length, adjust=False).mean()
    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))

# === SMOOTHED RSI (EMA of RSI) ===
def calc_smoothed_rsi(close, rsi_length, smoothing):
    rsi = calc_rsi(close, rsi_length)
    return rsi.ewm(span=smoothing, adjust=False).mean()

# === TEST: Primary RSI values ===
primary_rsi = calc_smoothed_rsi(df['close'], qqe_rsi1_length, qqe_rsi1_smoothing)

# Output Dec 2 values
print("=" * 60)
print("AMD DEC 2, 2025 - PRIMARY RSI (smoothed)")
print("=" * 60)
print(f"\n{'Time':<8} {'Close':>9} {'PriRSI':>8} {'PriRSI-50':>10}")
print("-" * 60)

dec2 = df[df.index.date == pd.Timestamp('2025-12-02').date()]
for ts in dec2.index:
    if 9*60+34 <= ts.hour*60+ts.minute <= 9*60+38:
        pri_rsi = primary_rsi.loc[ts]
        print(f"{ts.strftime('%H:%M'):<8} "
              f"{df.loc[ts, 'close']:>9.4f} "
              f"{pri_rsi:>8.2f} "
              f"{pri_rsi - 50:>10.2f}")

print("\n" + "=" * 60)
print("Expected from Pine QQE MOD:")
print("09:34 PriRSI=83.38, 09:35=86.56, 09:36=89.27, 09:37=89.50")
print("=" * 60)

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ⚠ Cache stale (0 hours), refreshing...
  ✓ Fetched 101956 bars, cached for next time
AMD DEC 2, 2025 - PRIMARY RSI (smoothed)

Time         Close   PriRSI  PriRSI-50
------------------------------------------------------------
09:34     223.4800    56.21       6.21
09:35     223.6500    61.59      11.59
09:36     224.3297    67.31      17.31
09:37     224.2100    69.90      19.90
09:38     224.2730    71.86      21.86

Expected from Pine QQE MOD:
09:34 PriRSI=83.38, 09:35=86.56, 09:36=89.27, 09:37=89.50


In [7]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
import numpy as np
from data_collector import PolygonDataCollector

# Load data
collector = PolygonDataCollector()
df = collector.fetch_bars('AMD', days_back=180, bar_size=1, extended_hours=True)

# === RTH FILTER ===
def is_rth(ts):
    m = ts.hour * 60 + ts.minute
    return (9*60+30) <= m < (16*60)

rth_df = df[df.index.map(is_rth)].copy()

# === QQE SETTINGS ===
qqe_rsi1_length = 6
qqe_rsi1_smoothing = 5

# === RSI CALCULATION ===
def calc_rsi(series, length):
    delta = series.diff()
    gain = delta.where(delta > 0, 0.0)
    loss = (-delta).where(delta < 0, 0.0)
    avg_gain = gain.ewm(alpha=1/length, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/length, adjust=False).mean()
    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))

# === SMOOTHED RSI (EMA of RSI) ===
def calc_smoothed_rsi(close, rsi_length, smoothing):
    rsi = calc_rsi(close, rsi_length)
    return rsi.ewm(span=smoothing, adjust=False).mean()

# === TEST: Primary RSI on RTH data ===
primary_rsi = calc_smoothed_rsi(rth_df['close'], qqe_rsi1_length, qqe_rsi1_smoothing)

# Output Dec 2 values
print("=" * 60)
print("AMD DEC 2, 2025 - PRIMARY RSI (RTH-only)")
print("=" * 60)
print(f"\n{'Time':<8} {'Close':>9} {'PriRSI':>8} {'PriRSI-50':>10}")
print("-" * 60)

dec2 = rth_df[rth_df.index.date == pd.Timestamp('2025-12-02').date()]
for ts in dec2.index:
    if 9*60+34 <= ts.hour*60+ts.minute <= 9*60+38:
        pri_rsi = primary_rsi.loc[ts]
        print(f"{ts.strftime('%H:%M'):<8} "
              f"{rth_df.loc[ts, 'close']:>9.4f} "
              f"{pri_rsi:>8.2f} "
              f"{pri_rsi - 50:>10.2f}")

print("\n" + "=" * 60)
print("Expected from Pine QQE MOD:")
print("09:34 PriRSI=83.38, 09:35=86.56, 09:36=89.27, 09:37=89.50")
print("=" * 60)

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (0.0 hours old)
AMD DEC 2, 2025 - PRIMARY RSI (RTH-only)

Time         Close   PriRSI  PriRSI-50
------------------------------------------------------------
09:34     223.4800    83.38      33.38
09:35     223.6500    86.56      36.56
09:36     224.3297    89.27      39.27
09:37     224.2100    89.50      39.50
09:38     224.2730    89.76      39.76

Expected from Pine QQE MOD:
09:34 PriRSI=83.38, 09:35=86.56, 09:36=89.27, 09:37=89.50


In [8]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
import numpy as np
from data_collector import PolygonDataCollector

# Load data
collector = PolygonDataCollector()
df = collector.fetch_bars('AMD', days_back=180, bar_size=1, extended_hours=True)

# === RTH FILTER ===
def is_rth(ts):
    m = ts.hour * 60 + ts.minute
    return (9*60+30) <= m < (16*60)

rth_df = df[df.index.map(is_rth)].copy()

# === QQE SETTINGS ===
qqe_rsi1_length = 6
qqe_rsi1_smoothing = 5
qqe_factor_primary = 3.0
qqe_rsi2_length = 6
qqe_rsi2_smoothing = 5
qqe_factor_secondary = 1.61
threshold_secondary = 3.0
qqe_bb_length = 50
qqe_bb_mult = 0.35

# === RSI CALCULATION ===
def calc_rsi(series, length):
    delta = series.diff()
    gain = delta.where(delta > 0, 0.0)
    loss = (-delta).where(delta < 0, 0.0)
    avg_gain = gain.ewm(alpha=1/length, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/length, adjust=False).mean()
    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))

# === SMOOTHED RSI ===
def calc_smoothed_rsi(close, rsi_length, smoothing):
    rsi = calc_rsi(close, rsi_length)
    return rsi.ewm(span=smoothing, adjust=False).mean()

# === CALCULATE QQE BANDS ===
def calc_qqe(close, rsi_length, smoothing, qqe_factor):
    wilders_length = rsi_length * 2 - 1
    rsi = calc_rsi(close, rsi_length)
    smoothed_rsi = rsi.ewm(span=smoothing, adjust=False).mean()
    
    # ATR of RSI
    atr_rsi = (smoothed_rsi - smoothed_rsi.shift(1)).abs()
    smoothed_atr_rsi = atr_rsi.ewm(span=wilders_length, adjust=False).mean()
    dynamic_atr = smoothed_atr_rsi * qqe_factor
    
    return smoothed_rsi, dynamic_atr

# === CALCULATIONS ===
primary_rsi, primary_atr = calc_qqe(rth_df['close'], qqe_rsi1_length, qqe_rsi1_smoothing, qqe_factor_primary)
secondary_rsi, secondary_atr = calc_qqe(rth_df['close'], qqe_rsi2_length, qqe_rsi2_smoothing, qqe_factor_secondary)

# Bollinger Bands on (primaryRSI - 50)
pri_centered = primary_rsi - 50
bb_basis = pri_centered.rolling(window=qqe_bb_length, min_periods=qqe_bb_length).mean()
bb_std = pri_centered.rolling(window=qqe_bb_length, min_periods=qqe_bb_length).std()
bb_upper = bb_basis + qqe_bb_mult * bb_std
bb_lower = bb_basis - qqe_bb_mult * bb_std

# Raw conditions
sec_centered = secondary_rsi - 50
raw_bull = (sec_centered > threshold_secondary) & (pri_centered > bb_upper)
raw_bear = (sec_centered < -threshold_secondary) & (pri_centered < bb_lower)

# Momentum
mom_rising = primary_rsi > primary_rsi.shift(1)
mom_falling = primary_rsi < primary_rsi.shift(1)

# Store in dataframe
rth_df['pri_rsi'] = primary_rsi
rth_df['sec_rsi'] = secondary_rsi
rth_df['bb_upper'] = bb_upper
rth_df['bb_lower'] = bb_lower
rth_df['raw_bull'] = raw_bull
rth_df['raw_bear'] = raw_bear
rth_df['mom_rising'] = mom_rising

# Output Dec 2 values
print("=" * 80)
print("AMD DEC 2, 2025 - QQE COMPONENTS (RTH-only)")
print("=" * 80)
print(f"\n{'Time':<7} {'PriRSI':>7} {'SecRSI':>7} {'BBupr':>8} {'BBlwr':>8} {'RawBull':>8} {'MomRise':>8}")
print("-" * 80)

dec2 = rth_df[rth_df.index.date == pd.Timestamp('2025-12-02').date()]
for ts in dec2.index:
    if 9*60+34 <= ts.hour*60+ts.minute <= 9*60+38:
        r = rth_df.loc[ts]
        print(f"{ts.strftime('%H:%M'):<7} "
              f"{r['pri_rsi']:>7.2f} "
              f"{r['sec_rsi']:>7.2f} "
              f"{r['bb_upper']:>8.4f} "
              f"{r['bb_lower']:>8.4f} "
              f"{str(r['raw_bull']):>8} "
              f"{str(r['mom_rising']):>8}")

print("\n" + "=" * 80)
print("Pine Debug @ 09:36: BBupper=11.274, BBlower=5.7857, RawBull=T, MomRise=T")
print("=" * 80)

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (0.0 hours old)
AMD DEC 2, 2025 - QQE COMPONENTS (RTH-only)

Time     PriRSI  SecRSI    BBupr    BBlwr  RawBull  MomRise
--------------------------------------------------------------------------------
09:34     83.38   83.38  10.1433   2.0330     True     True
09:35     86.56   86.56  10.7548   2.1739     True     True
09:36     89.27   89.27  11.4995   2.3668     True     True
09:37     89.50   89.50  12.4430   2.7589     True     True
09:38     89.76   89.76  13.5188   3.3859     True     True

Pine Debug @ 09:36: BBupper=11.274, BBlower=5.7857, RawBull=T, MomRise=T


In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
import numpy as np
from data_collector import PolygonDataCollector

# Load data
collector = PolygonDataCollector()
df = collector.fetch_bars('AMD', days_back=180, bar_size=1, extended_hours=True)

# === RTH FILTER ===
def is_rth(ts):
    m = ts.hour * 60 + ts.minute
    return (9*60+30) <= m < (16*60)

rth_df = df[df.index.map(is_rth)].copy()

# === QQE SETTINGS ===
qqe_rsi1_length = 6
qqe_rsi1_smoothing = 5
qqe_factor_primary = 3.0
qqe_rsi2_length = 6
qqe_rsi2_smoothing = 5
qqe_factor_secondary = 1.61
threshold_secondary = 3.0
qqe_bb_length = 50
qqe_bb_mult = 0.35

# === RSI CALCULATION ===
def calc_rsi(series, length):
    delta = series.diff()
    gain = delta.where(delta > 0, 0.0)
    loss = (-delta).where(delta < 0, 0.0)
    avg_gain = gain.ewm(alpha=1/length, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/length, adjust=False).mean()
    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))

# === FULL QQE CALCULATION WITH TREND LINE ===
def calc_qqe_full(close, rsi_length, smoothing, qqe_factor):
    wilders_length = rsi_length * 2 - 1
    rsi = calc_rsi(close, rsi_length)
    smoothed_rsi = rsi.ewm(span=smoothing, adjust=False).mean()
    
    # ATR of RSI
    atr_rsi = (smoothed_rsi - smoothed_rsi.shift(1)).abs()
    smoothed_atr_rsi = atr_rsi.ewm(span=wilders_length, adjust=False).mean()
    dynamic_atr = smoothed_atr_rsi * qqe_factor
    
    # Initialize bands and trend
    n = len(smoothed_rsi)
    long_band = np.zeros(n)
    short_band = np.zeros(n)
    trend = np.zeros(n, dtype=int)
    qqe_line = np.zeros(n)
    
    smoothed_arr = smoothed_rsi.values
    atr_arr = dynamic_atr.values
    
    for i in range(1, n):
        if np.isnan(smoothed_arr[i]) or np.isnan(atr_arr[i]):
            long_band[i] = long_band[i-1]
            short_band[i] = short_band[i-1]
            trend[i] = trend[i-1]
            qqe_line[i] = qqe_line[i-1]
            continue
            
        new_short = smoothed_arr[i] + atr_arr[i]
        new_long = smoothed_arr[i] - atr_arr[i]
        
        # Long band logic
        if smoothed_arr[i-1] > long_band[i-1] and smoothed_arr[i] > long_band[i-1]:
            long_band[i] = max(long_band[i-1], new_long)
        else:
            long_band[i] = new_long
            
        # Short band logic
        if smoothed_arr[i-1] < short_band[i-1] and smoothed_arr[i] < short_band[i-1]:
            short_band[i] = min(short_band[i-1], new_short)
        else:
            short_band[i] = new_short
        
        # Trend direction
        # Cross above short band -> bullish (1)
        # Cross below long band -> bearish (-1)
        if smoothed_arr[i] > short_band[i-1] and smoothed_arr[i-1] <= short_band[i-1]:
            trend[i] = 1
        elif smoothed_arr[i] < long_band[i-1] and smoothed_arr[i-1] >= long_band[i-1]:
            trend[i] = -1
        else:
            trend[i] = trend[i-1]
        
        # QQE line follows trend
        qqe_line[i] = long_band[i] if trend[i] == 1 else short_band[i]
    
    return pd.Series(smoothed_rsi.values, index=close.index), pd.Series(qqe_line, index=close.index)

# === CALCULATIONS ===
primary_rsi, primary_qqe_line = calc_qqe_full(rth_df['close'], qqe_rsi1_length, qqe_rsi1_smoothing, qqe_factor_primary)
secondary_rsi, secondary_qqe_line = calc_qqe_full(rth_df['close'], qqe_rsi2_length, qqe_rsi2_smoothing, qqe_factor_secondary)

# Bollinger Bands on (primaryQQETrendLine - 50)
qqe_centered = primary_qqe_line - 50
bb_basis = qqe_centered.rolling(window=qqe_bb_length, min_periods=qqe_bb_length).mean()
bb_std = qqe_centered.rolling(window=qqe_bb_length, min_periods=qqe_bb_length).std()
bb_upper = bb_basis + qqe_bb_mult * bb_std
bb_lower = bb_basis - qqe_bb_mult * bb_std

# Raw conditions (use RSI for conditions, not QQ

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (0.0 hours old)


In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
import numpy as np
from data_collector import PolygonDataCollector

# Load data
collector = PolygonDataCollector()
df = collector.fetch_bars('AMD', days_back=180, bar_size=1, extended_hours=True)

# === RTH FILTER ===
def is_rth(ts):
    m = ts.hour * 60 + ts.minute
    return (9*60+30) <= m < (16*60)

rth_df = df[df.index.map(is_rth)].copy()

# === LIMIT TO NOVEMBER + DECEMBER FOR SPEED ===
rth_df = rth_df[rth_df.index >= '2025-11-01'].copy()
print(f"Testing on {len(rth_df)} RTH bars (Nov-Dec)")

# === QQE SETTINGS ===
qqe_rsi1_length = 6
qqe_rsi1_smoothing = 5
qqe_factor_primary = 3.0
qqe_bb_length = 50
qqe_bb_mult = 0.35
threshold_secondary = 3.0

# === RSI CALCULATION ===
def calc_rsi(series, length):
    delta = series.diff()
    gain = delta.where(delta > 0, 0.0)
    loss = (-delta).where(delta < 0, 0.0)
    avg_gain = gain.ewm(alpha=1/length, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/length, adjust=False).mean()
    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))

# === FULL QQE WITH TREND LINE ===
def calc_qqe_full(close, rsi_length, smoothing, qqe_factor):
    wilders_length = rsi_length * 2 - 1
    rsi = calc_rsi(close, rsi_length)
    smoothed_rsi = rsi.ewm(span=smoothing, adjust=False).mean()
    
    atr_rsi = (smoothed_rsi - smoothed_rsi.shift(1)).abs()
    smoothed_atr_rsi = atr_rsi.ewm(span=wilders_length, adjust=False).mean()
    dynamic_atr = smoothed_atr_rsi * qqe_factor
    
    n = len(smoothed_rsi)
    long_band = np.zeros(n)
    short_band = np.zeros(n)
    trend = np.zeros(n, dtype=int)
    qqe_line = np.zeros(n)
    
    smoothed_arr = smoothed_rsi.values
    atr_arr = dynamic_atr.values
    
    for i in range(1, n):
        if np.isnan(smoothed_arr[i]) or np.isnan(atr_arr[i]):
            long_band[i] = long_band[i-1]
            short_band[i] = short_band[i-1]
            trend[i] = trend[i-1]
            qqe_line[i] = qqe_line[i-1]
            continue
            
        new_short = smoothed_arr[i] + atr_arr[i]
        new_long = smoothed_arr[i] - atr_arr[i]
        
        if smoothed_arr[i-1] > long_band[i-1] and smoothed_arr[i] > long_band[i-1]:
            long_band[i] = max(long_band[i-1], new_long)
        else:
            long_band[i] = new_long
            
        if smoothed_arr[i-1] < short_band[i-1] and smoothed_arr[i] < short_band[i-1]:
            short_band[i] = min(short_band[i-1], new_short)
        else:
            short_band[i] = new_short
        
        if smoothed_arr[i] > short_band[i-1] and smoothed_arr[i-1] <= short_band[i-1]:
            trend[i] = 1
        elif smoothed_arr[i] < long_band[i-1] and smoothed_arr[i-1] >= long_band[i-1]:
            trend[i] = -1
        else:
            trend[i] = trend[i-1]
        
        qqe_line[i] = long_band[i] if trend[i] == 1 else short_band[i]
    
    return pd.Series(smoothed_rsi.values, index=close.index), pd.Series(qqe_line, index=close.index)

# === CALCULATE ===
primary_rsi, primary_qqe_line = calc_qqe_full(rth_df['close'], qqe_rsi1_length, qqe_rsi1_smoothing, qqe_factor_primary)
secondary_rsi, _ = calc_qqe_full(rth_df['close'], 6, 5, 1.61)

# BB on QQE line
qqe_centered = primary_qqe_line - 50
bb_basis = qqe_centered.rolling(window=qqe_bb_length, min_periods=qqe_bb_length).mean()
bb_std = qqe_centered.rolling(window=qqe_bb_length, min_periods=qqe_bb_length).std()
bb_upper = bb_basis + qqe_bb_mult * bb_std
bb_lower = bb_basis - qqe_bb_mult * bb_std

# Output
print("=" * 90)
print("AMD DEC 2, 2025 - QQE WITH TREND LINE")
print("=" * 90)
print(f"\n{'Time':<7} {'PriRSI':>7} {'QQEline':>8} {'BBupr':>8} {'BBlwr':>8}")
print("-" * 90)

dec2 = rth_df[rth_df.index.date == pd.Timestamp('2025-12-02').date()]
for ts in dec2.index:
    if 9*60+34 <= ts.hour*60+ts.minute <= 9*60+38:
        print(f"{ts.strftime('%H:%M'):<7} "
              f"{primary_rsi.loc[ts]:>7.2f} "
              f"{primary_qqe_line.loc[ts]:>8.2f} "
              f"{bb_upper.loc[ts]:>8.4f} "
              f"{bb_lower.loc[ts]:>8.4f}")

print("\n" + "=" * 90)
print("Pine Debug @ 09:36: BBupper=11.274, BBlower=5.7857")
print("=" * 90)

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (0.1 hours old)
Testing on 19922 RTH bars (Nov-Dec)
AMD DEC 2, 2025 - QQE WITH TREND LINE

Time     PriRSI  QQEline    BBupr    BBlwr
------------------------------------------------------------------------------------------
09:34     83.38    66.95  10.7010   5.5386
09:35     86.56    71.28  10.9378   5.6338
09:36     89.27    75.18  11.3018   5.7578
09:37     89.50    77.64  11.6258   5.8344
09:38     89.76    79.75  12.1753   6.0250

Pine Debug @ 09:36: BBupper=11.274, BBlower=5.7857


In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
import numpy as np
from data_collector import PolygonDataCollector

# Load data
collector = PolygonDataCollector()
df = collector.fetch_bars('AMD', days_back=180, bar_size=1, extended_hours=True)

# === RTH FILTER ===
def is_rth(ts):
    m = ts.hour * 60 + ts.minute
    return (9*60+30) <= m < (16*60)

rth_df = df[df.index.map(is_rth)].copy()
rth_df = rth_df[rth_df.index >= '2025-11-01'].copy()
print(f"Testing on {len(rth_df)} RTH bars (Nov-Dec)")

# === QQE SETTINGS ===
qqe_rsi1_length = 6
qqe_rsi1_smoothing = 5
qqe_factor_primary = 3.0
qqe_rsi2_length = 6
qqe_rsi2_smoothing = 5
qqe_factor_secondary = 1.61
qqe_bb_length = 50
qqe_bb_mult = 0.35
threshold_secondary = 3.0

# === RSI ===
def calc_rsi(series, length):
    delta = series.diff()
    gain = delta.where(delta > 0, 0.0)
    loss = (-delta).where(delta < 0, 0.0)
    avg_gain = gain.ewm(alpha=1/length, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/length, adjust=False).mean()
    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))

# === QQE FULL ===
def calc_qqe_full(close, rsi_length, smoothing, qqe_factor):
    wilders_length = rsi_length * 2 - 1
    rsi = calc_rsi(close, rsi_length)
    smoothed_rsi = rsi.ewm(span=smoothing, adjust=False).mean()
    
    atr_rsi = (smoothed_rsi - smoothed_rsi.shift(1)).abs()
    smoothed_atr_rsi = atr_rsi.ewm(span=wilders_length, adjust=False).mean()
    dynamic_atr = smoothed_atr_rsi * qqe_factor
    
    n = len(smoothed_rsi)
    long_band = np.zeros(n)
    short_band = np.zeros(n)
    trend = np.zeros(n, dtype=int)
    qqe_line = np.zeros(n)
    
    smoothed_arr = smoothed_rsi.values
    atr_arr = dynamic_atr.values
    
    for i in range(1, n):
        if np.isnan(smoothed_arr[i]) or np.isnan(atr_arr[i]):
            long_band[i] = long_band[i-1]
            short_band[i] = short_band[i-1]
            trend[i] = trend[i-1]
            qqe_line[i] = qqe_line[i-1]
            continue
            
        new_short = smoothed_arr[i] + atr_arr[i]
        new_long = smoothed_arr[i] - atr_arr[i]
        
        if smoothed_arr[i-1] > long_band[i-1] and smoothed_arr[i] > long_band[i-1]:
            long_band[i] = max(long_band[i-1], new_long)
        else:
            long_band[i] = new_long
            
        if smoothed_arr[i-1] < short_band[i-1] and smoothed_arr[i] < short_band[i-1]:
            short_band[i] = min(short_band[i-1], new_short)
        else:
            short_band[i] = new_short
        
        if smoothed_arr[i] > short_band[i-1] and smoothed_arr[i-1] <= short_band[i-1]:
            trend[i] = 1
        elif smoothed_arr[i] < long_band[i-1] and smoothed_arr[i-1] >= long_band[i-1]:
            trend[i] = -1
        else:
            trend[i] = trend[i-1]
        
        qqe_line[i] = long_band[i] if trend[i] == 1 else short_band[i]
    
    return pd.Series(smoothed_rsi.values, index=close.index), pd.Series(qqe_line, index=close.index)

# === CALCULATE ===
primary_rsi, primary_qqe_line = calc_qqe_full(rth_df['close'], qqe_rsi1_length, qqe_rsi1_smoothing, qqe_factor_primary)
secondary_rsi, _ = calc_qqe_full(rth_df['close'], qqe_rsi2_length, qqe_rsi2_smoothing, qqe_factor_secondary)

# BB on QQE line
qqe_centered = primary_qqe_line - 50
bb_basis = qqe_centered.rolling(window=qqe_bb_length, min_periods=qqe_bb_length).mean()
bb_std = qqe_centered.rolling(window=qqe_bb_length, min_periods=qqe_bb_length).std()
bb_upper = bb_basis + qqe_bb_mult * bb_std
bb_lower = bb_basis - qqe_bb_mult * bb_std

# Raw conditions
pri_centered = primary_rsi - 50
sec_centered = secondary_rsi - 50
raw_bull = (sec_centered > threshold_secondary) & (pri_centered > bb_upper)
raw_bear = (sec_centered < -threshold_secondary) & (pri_centered < bb_lower)

# Momentum
mom_rising = primary_rsi > primary_rsi.shift(1)
mom_falling = primary_rsi < primary_rsi.shift(1)

# === QQE STATE MACHINE ===
n = len(rth_df)
qqe_bull_state = np.zeros(n, dtype=bool)
qqe_bear_state = np.zeros(n, dtype=bool)

raw_bull_arr = raw_bull.values
raw_bear_arr = raw_bear.values
mom_rise_arr = mom_rising.values

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (0.1 hours old)
Testing on 19922 RTH bars (Nov-Dec)


In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
import numpy as np
from data_collector import PolygonDataCollector

# Load data
collector = PolygonDataCollector()
df = collector.fetch_bars('AMD', days_back=180, bar_size=1, extended_hours=True)

# === RTH FILTER - DECEMBER ONLY ===
def is_rth(ts):
    m = ts.hour * 60 + ts.minute
    return (9*60+30) <= m < (16*60)

rth_df = df[df.index.map(is_rth)].copy()
rth_df = rth_df[rth_df.index >= '2025-12-01'].copy()
print(f"Testing on {len(rth_df)} RTH bars (Dec only)")

# === QQE SETTINGS ===
qqe_rsi1_length = 6
qqe_rsi1_smoothing = 5
qqe_bb_length = 50
qqe_bb_mult = 0.35
threshold_secondary = 3.0

# === RSI ===
def calc_rsi(series, length):
    delta = series.diff()
    gain = delta.where(delta > 0, 0.0)
    loss = (-delta).where(delta < 0, 0.0)
    avg_gain = gain.ewm(alpha=1/length, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/length, adjust=False).mean()
    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))

# === SMOOTHED RSI ONLY (skip QQE bands for now) ===
rsi = calc_rsi(rth_df['close'], qqe_rsi1_length)
primary_rsi = rsi.ewm(span=qqe_rsi1_smoothing, adjust=False).mean()
secondary_rsi = primary_rsi.copy()  # Same settings

# Simple BB approximation on RSI itself for now
pri_centered = primary_rsi - 50
bb_basis = pri_centered.rolling(window=qqe_bb_length, min_periods=qqe_bb_length).mean()
bb_std = pri_centered.rolling(window=qqe_bb_length, min_periods=qqe_bb_length).std()
bb_upper = bb_basis + qqe_bb_mult * bb_std
bb_lower = bb_basis - qqe_bb_mult * bb_std

# Raw conditions
sec_centered = secondary_rsi - 50
raw_bull = (sec_centered > threshold_secondary) & (pri_centered > bb_upper)

# Momentum
mom_rising = primary_rsi > primary_rsi.shift(1)

# Output
print("=" * 70)
print("AMD DEC 2 - QUICK QQE CHECK")
print("=" * 70)
print(f"\n{'Time':<7} {'PriRSI':>7} {'Pri-50':>7} {'BBupr':>8} {'RawBull':>8} {'MomRise':>8}")
print("-" * 70)

dec2 = rth_df[rth_df.index.date == pd.Timestamp('2025-12-02').date()]
for ts in dec2.index:
    if 9*60+34 <= ts.hour*60+ts.minute <= 9*60+40:
        print(f"{ts.strftime('%H:%M'):<7} "
              f"{primary_rsi.loc[ts]:>7.2f} "
              f"{pri_centered.loc[ts]:>7.2f} "
              f"{bb_upper.loc[ts]:>8.4f} "
              f"{str(raw_bull.loc[ts]):>8} "
              f"{str(mom_rising.loc[ts]):>8}")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (0.1 hours old)
Testing on 12691 RTH bars (Dec only)
AMD DEC 2 - QUICK QQE CHECK

Time     PriRSI  Pri-50    BBupr  RawBull  MomRise
----------------------------------------------------------------------
09:34     83.38   33.38  10.1433     True     True
09:35     86.56   36.56  10.7548     True     True
09:36     89.27   39.27  11.4995     True     True
09:37     89.50   39.50  12.4430     True     True
09:38     89.76   39.76  13.5188     True     True
09:39     90.00   40.00  14.6210     True     True
09:40     88.14   38.14  15.6182     True    False


In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
import numpy as np
from numba import jit
from data_collector import PolygonDataCollector

# Load data
collector = PolygonDataCollector()
df = collector.fetch_bars('AMD', days_back=180, bar_size=1, extended_hours=True)

# === RTH FILTER ===
def is_rth(ts):
    m = ts.hour * 60 + ts.minute
    return (9*60+30) <= m < (16*60)

rth_df = df[df.index.map(is_rth)].copy()
print(f"Testing on {len(rth_df)} RTH bars (full history)")

# === QQE SETTINGS ===
qqe_rsi1_length = 6
qqe_rsi1_smoothing = 5
qqe_factor_primary = 3.0
qqe_rsi2_length = 6
qqe_rsi2_smoothing = 5
qqe_factor_secondary = 1.61
qqe_bb_length = 50
qqe_bb_mult = 0.35
threshold_secondary = 3.0

# === RSI ===
def calc_rsi(series, length):
    delta = series.diff()
    gain = delta.where(delta > 0, 0.0)
    loss = (-delta).where(delta < 0, 0.0)
    avg_gain = gain.ewm(alpha=1/length, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/length, adjust=False).mean()
    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))

# === NUMBA-OPTIMIZED QQE BAND CALCULATION ===
@jit(nopython=True)
def calc_qqe_bands(smoothed_arr, atr_arr):
    n = len(smoothed_arr)
    long_band = np.zeros(n)
    short_band = np.zeros(n)
    trend = np.zeros(n, dtype=np.int32)
    qqe_line = np.zeros(n)
    
    for i in range(1, n):
        if np.isnan(smoothed_arr[i]) or np.isnan(atr_arr[i]):
            long_band[i] = long_band[i-1]
            short_band[i] = short_band[i-1]
            trend[i] = trend[i-1]
            qqe_line[i] = qqe_line[i-1]
            continue
            
        new_short = smoothed_arr[i] + atr_arr[i]
        new_long = smoothed_arr[i] - atr_arr[i]
        
        if smoothed_arr[i-1] > long_band[i-1] and smoothed_arr[i] > long_band[i-1]:
            long_band[i] = max(long_band[i-1], new_long)
        else:
            long_band[i] = new_long
            
        if smoothed_arr[i-1] < short_band[i-1] and smoothed_arr[i] < short_band[i-1]:
            short_band[i] = min(short_band[i-1], new_short)
        else:
            short_band[i] = new_short
        
        # Trend crosses
        if smoothed_arr[i] > short_band[i-1] and smoothed_arr[i-1] <= short_band[i-1]:
            trend[i] = 1
        elif smoothed_arr[i] < long_band[i-1] and smoothed_arr[i-1] >= long_band[i-1]:
            trend[i] = -1
        else:
            trend[i] = trend[i-1]
        
        qqe_line[i] = long_band[i] if trend[i] == 1 else short_band[i]
    
    return qqe_line

def calc_qqe_full(close, rsi_length, smoothing, qqe_factor):
    wilders_length = rsi_length * 2 - 1
    rsi = calc_rsi(close, rsi_length)
    smoothed_rsi = rsi.ewm(span=smoothing, adjust=False).mean()
    
    atr_rsi = (smoothed_rsi - smoothed_rsi.shift(1)).abs()
    smoothed_atr_rsi = atr_rsi.ewm(span=wilders_length, adjust=False).mean()
    dynamic_atr = smoothed_atr_rsi * qqe_factor
    
    qqe_line = calc_qqe_bands(smoothed_rsi.values, dynamic_atr.values)
    
    return smoothed_rsi, pd.Series(qqe_line, index=close.index)

# === CALCULATE ===
print("Calculating QQE (numba optimized)...")
primary_rsi, primary_qqe_line = calc_qqe_full(rth_df['close'], qqe_rsi1_length, qqe_rsi1_smoothing, qqe_factor_primary)
secondary_rsi, _ = calc_qqe_full(rth_df['close'], qqe_rsi2_length, qqe_rsi2_smoothing, qqe_factor_secondary)
print("Done.")

# BB on QQE TrendLine
qqe_centered = primary_qqe_line - 50
bb_basis = qqe_centered.rolling(window=qqe_bb_length, min_periods=qqe_bb_length).mean()
bb_std = qqe_centered.rolling(window=qqe_bb_length, min_periods=qqe_bb_length).std()
bb_upper = bb_basis + qqe_bb_mult * bb_std
bb_lower = bb_basis - qqe_bb_mult * bb_std

# Raw conditions
pri_centered = primary_rsi - 50
sec_centered = secondary_rsi - 50
raw_bull = (sec_centered > threshold_secondary) & (pri_centered > bb_upper)
raw_bear = (sec_centered < -threshold_secondary) & (pri_centered < bb_lower)

# Momentum
mom_rising = primary_rsi > primary_rsi.shift(1)

# Output
print("=" * 90)
print("AMD DEC 2, 2025 - QQE WITH NUMBA-OPTIMIZED BANDS")
print("=" * 90)
print(f"\n{'Time':<7} {'PriRSI':>7} {'QQEline':>8} {'BBupr':>8} {'BBlwr':>8} {'RawBull':>8} {'MomRise':>8}")
print("-" * 90)

dec2 = rth_df[rth_df.index.date == pd.Timestamp('2025-12-02').date()]
for ts in dec2.index:
    if 9*60+34 <= ts.hour*60+ts.minute <= 9*60+40:
        print(f"{ts.strftime('%H:%M'):<7} "
              f"{primary_rsi.loc[ts]:>7.2f} "
              f"{primary_qqe_line.loc[ts]:>8.2f} "
              f"{bb_upper.loc[ts]:>8.4f} "
              f"{bb_lower.loc[ts]:>8.4f} "
              f"{str(raw_bull.loc[ts]):>8} "
              f"{str(mom_rising.loc[ts]):>8}")

print("\n" + "=" * 90)
print("Pine Debug @ 09:36: BBupper=11.274, BBlower=5.7857")
print("=" * 90)

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (0.2 hours old)
Testing on 48002 RTH bars (full history)
Calculating QQE (numba optimized)...
Done.
AMD DEC 2, 2025 - QQE WITH NUMBA-OPTIMIZED BANDS

Time     PriRSI  QQEline    BBupr    BBlwr  RawBull  MomRise
------------------------------------------------------------------------------------------
09:34     83.38    66.95  10.7010   5.5386     True     True
09:35     86.56    71.28  10.9378   5.6338     True     True
09:36     89.27    75.18  11.3018   5.7578     True     True
09:37     89.50    77.64  11.6258   5.8344     True     True
09:38     89.76    79.75  12.1753   6.0250     True     True
09:39     90.00    81.54  12.8537   6.3187     True     True
09:40     88.14    81.54  13.5156   6.6351     True    False

Pine Debug @ 09:36: BBupper=11.274, BBlower=5.7857


In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
import numpy as np
from numba import jit
from data_collector import PolygonDataCollector

collector = PolygonDataCollector()
df = collector.fetch_bars('AMD', days_back=180, bar_size=1, extended_hours=True)

def is_rth(ts):
    m = ts.hour * 60 + ts.minute
    return (9*60+30) <= m < (16*60)

rth_df = df[df.index.map(is_rth)].copy()
print(f"Testing on {len(rth_df)} RTH bars")

# === QQE SETTINGS ===
qqe_rsi1_length = 6
qqe_rsi1_smoothing = 5
qqe_factor_primary = 3.0
qqe_rsi2_length = 6
qqe_rsi2_smoothing = 5
qqe_factor_secondary = 1.61
qqe_bb_length = 50
qqe_bb_mult = 0.35
threshold_secondary = 3.0

def calc_rsi(series, length):
    delta = series.diff()
    gain = delta.where(delta > 0, 0.0)
    loss = (-delta).where(delta < 0, 0.0)
    avg_gain = gain.ewm(alpha=1/length, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/length, adjust=False).mean()
    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))

@jit(nopython=True)
def calc_qqe_bands(smoothed_arr, atr_arr):
    n = len(smoothed_arr)
    long_band = np.zeros(n)
    short_band = np.zeros(n)
    trend = np.zeros(n, dtype=np.int32)
    qqe_line = np.zeros(n)
    
    for i in range(1, n):
        if np.isnan(smoothed_arr[i]) or np.isnan(atr_arr[i]):
            long_band[i] = long_band[i-1]
            short_band[i] = short_band[i-1]
            trend[i] = trend[i-1]
            qqe_line[i] = qqe_line[i-1]
            continue
            
        new_short = smoothed_arr[i] + atr_arr[i]
        new_long = smoothed_arr[i] - atr_arr[i]
        
        if smoothed_arr[i-1] > long_band[i-1] and smoothed_arr[i] > long_band[i-1]:
            long_band[i] = max(long_band[i-1], new_long)
        else:
            long_band[i] = new_long
            
        if smoothed_arr[i-1] < short_band[i-1] and smoothed_arr[i] < short_band[i-1]:
            short_band[i] = min(short_band[i-1], new_short)
        else:
            short_band[i] = new_short
        
        if smoothed_arr[i] > short_band[i-1] and smoothed_arr[i-1] <= short_band[i-1]:
            trend[i] = 1
        elif smoothed_arr[i] < long_band[i-1] and smoothed_arr[i-1] >= long_band[i-1]:
            trend[i] = -1
        else:
            trend[i] = trend[i-1]
        
        qqe_line[i] = long_band[i] if trend[i] == 1 else short_band[i]
    
    return qqe_line

def calc_qqe_full(close, rsi_length, smoothing, qqe_factor):
    wilders_length = rsi_length * 2 - 1
    rsi = calc_rsi(close, rsi_length)
    smoothed_rsi = rsi.ewm(span=smoothing, adjust=False).mean()
    atr_rsi = (smoothed_rsi - smoothed_rsi.shift(1)).abs()
    smoothed_atr_rsi = atr_rsi.ewm(span=wilders_length, adjust=False).mean()
    dynamic_atr = smoothed_atr_rsi * qqe_factor
    qqe_line = calc_qqe_bands(smoothed_rsi.values, dynamic_atr.values)
    return smoothed_rsi, pd.Series(qqe_line, index=close.index)

# === QQE STATE MACHINE (numba) ===
@jit(nopython=True)
def calc_qqe_state(raw_bull, raw_bear, mom_rising, mom_falling):
    n = len(raw_bull)
    bull_state = np.zeros(n, dtype=np.bool_)
    bear_state = np.zeros(n, dtype=np.bool_)
    
    for i in range(1, n):
        # Bull: ON when raw_bull AND mom_rising, OFF when mom_falling
        if raw_bull[i] and mom_rising[i]:
            bull_state[i] = True
        elif mom_falling[i]:
            bull_state[i] = False
        else:
            bull_state[i] = bull_state[i-1]
        
        # Bear: ON when raw_bear AND mom_falling, OFF when mom_rising
        if raw_bear[i] and mom_falling[i]:
            bear_state[i] = True
        elif mom_rising[i]:
            bear_state[i] = False
        else:
            bear_state[i] = bear_state[i-1]
    
    return bull_state, bear_state

# === CALCULATE ===
print("Calculating QQE...")
primary_rsi, primary_qqe_line = calc_qqe_full(rth_df['close'], qqe_rsi1_length, qqe_rsi1_smoothing, qqe_factor_primary)
secondary_rsi, _ = calc_qqe_full(rth_df['close'], qqe_rsi2_length, qqe_rsi2_smoothing, qqe_factor_secondary)

qqe_centered = primary_qqe_line - 50
bb_basis = qqe_centered.rolling(window=qqe_bb_length, min_periods=qqe_bb_length).mean()
bb_std = qqe_centered.rolling(window=qqe_bb_length, min_periods=qqe_bb_length).std()
bb_upper = bb_basis + qqe_bb_mult * bb_std
bb_lower = bb_basis - qqe_bb_mult * bb_std

pri_centered = primary_rsi - 50
sec_centered = secondary_rsi - 50
raw_bull = (sec_centered > threshold_secondary) & (pri_centered > bb_upper)
raw_bear = (sec_centered < -threshold_secondary) & (pri_centered < bb_lower)

mom_rising = (primary_rsi > primary_rsi.shift(1)).fillna(False)
mom_falling = (primary_rsi < primary_rsi.shift(1)).fillna(False)

bull_state, bear_state = calc_qqe_state(
    raw_bull.values, raw_bear.values, 
    mom_rising.values, mom_falling.values
)
print("Done.")

# Output
print("=" * 90)
print("AMD DEC 2, 2025 - QQE STATE MACHINE")
print("=" * 90)
print(f"\n{'Time':<7} {'PriRSI':>7} {'RawBull':>8} {'MomRise':>8} {'BullState':>10} {'QQE_Score':>10}")
print("-" * 90)

dec2_idx = rth_df[rth_df.index.date == pd.Timestamp('2025-12-02').date()].index
for ts in dec2_idx:
    if 9*60+34 <= ts.hour*60+ts.minute <= 9*60+42:
        i = rth_df.index.get_loc(ts)
        print(f"{ts.strftime('%H:%M'):<7} "
              f"{primary_rsi.iloc[i]:>7.2f} "
              f"{str(raw_bull.iloc[i]):>8} "
              f"{str(mom_rising.iloc[i]):>8} "
              f"{str(bull_state[i]):>10} "
              f"{int(bull_state[i]):>10}")

print("\n" + "=" * 90)
print("Expected: QQE=1 from 09:35-09:39, QQE=0 at 09:40+ (when MomRise=Fal

SyntaxError: unterminated string literal (detected at line 158) (3392431080.py, line 158)

In [2]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
import numpy as np
from numba import jit
from data_collector import PolygonDataCollector

collector = PolygonDataCollector()
df = collector.fetch_bars('AMD', days_back=180, bar_size=1, extended_hours=True)

def is_rth(ts):
    m = ts.hour * 60 + ts.minute
    return (9*60+30) <= m < (16*60)

rth_df = df[df.index.map(is_rth)].copy()
print(f"Testing on {len(rth_df)} RTH bars")

# === QQE SETTINGS ===
qqe_rsi1_length = 6
qqe_rsi1_smoothing = 5
qqe_factor_primary = 3.0
qqe_rsi2_length = 6
qqe_rsi2_smoothing = 5
qqe_factor_secondary = 1.61
qqe_bb_length = 50
qqe_bb_mult = 0.35
threshold_secondary = 3.0

def calc_rsi(series, length):
    delta = series.diff()
    gain = delta.where(delta > 0, 0.0)
    loss = (-delta).where(delta < 0, 0.0)
    avg_gain = gain.ewm(alpha=1/length, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/length, adjust=False).mean()
    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))

@jit(nopython=True)
def calc_qqe_bands(smoothed_arr, atr_arr):
    n = len(smoothed_arr)
    long_band = np.zeros(n)
    short_band = np.zeros(n)
    trend = np.zeros(n, dtype=np.int32)
    qqe_line = np.zeros(n)
    
    for i in range(1, n):
        if np.isnan(smoothed_arr[i]) or np.isnan(atr_arr[i]):
            long_band[i] = long_band[i-1]
            short_band[i] = short_band[i-1]
            trend[i] = trend[i-1]
            qqe_line[i] = qqe_line[i-1]
            continue
            
        new_short = smoothed_arr[i] + atr_arr[i]
        new_long = smoothed_arr[i] - atr_arr[i]
        
        if smoothed_arr[i-1] > long_band[i-1] and smoothed_arr[i] > long_band[i-1]:
            long_band[i] = max(long_band[i-1], new_long)
        else:
            long_band[i] = new_long
            
        if smoothed_arr[i-1] < short_band[i-1] and smoothed_arr[i] < short_band[i-1]:
            short_band[i] = min(short_band[i-1], new_short)
        else:
            short_band[i] = new_short
        
        if smoothed_arr[i] > short_band[i-1] and smoothed_arr[i-1] <= short_band[i-1]:
            trend[i] = 1
        elif smoothed_arr[i] < long_band[i-1] and smoothed_arr[i-1] >= long_band[i-1]:
            trend[i] = -1
        else:
            trend[i] = trend[i-1]
        
        qqe_line[i] = long_band[i] if trend[i] == 1 else short_band[i]
    
    return qqe_line

def calc_qqe_full(close, rsi_length, smoothing, qqe_factor):
    wilders_length = rsi_length * 2 - 1
    rsi = calc_rsi(close, rsi_length)
    smoothed_rsi = rsi.ewm(span=smoothing, adjust=False).mean()
    atr_rsi = (smoothed_rsi - smoothed_rsi.shift(1)).abs()
    smoothed_atr_rsi = atr_rsi.ewm(span=wilders_length, adjust=False).mean()
    dynamic_atr = smoothed_atr_rsi * qqe_factor
    qqe_line = calc_qqe_bands(smoothed_rsi.values, dynamic_atr.values)
    return smoothed_rsi, pd.Series(qqe_line, index=close.index)

# === QQE STATE MACHINE (numba) ===
@jit(nopython=True)
def calc_qqe_state(raw_bull, raw_bear, mom_rising, mom_falling):
    n = len(raw_bull)
    bull_state = np.zeros(n, dtype=np.bool_)
    bear_state = np.zeros(n, dtype=np.bool_)
    
    for i in range(1, n):
        # Bull: ON when raw_bull AND mom_rising, OFF when mom_falling
        if raw_bull[i] and mom_rising[i]:
            bull_state[i] = True
        elif mom_falling[i]:
            bull_state[i] = False
        else:
            bull_state[i] = bull_state[i-1]
        
        # Bear: ON when raw_bear AND mom_falling, OFF when mom_rising
        if raw_bear[i] and mom_falling[i]:
            bear_state[i] = True
        elif mom_rising[i]:
            bear_state[i] = False
        else:
            bear_state[i] = bear_state[i-1]
    
    return bull_state, bear_state

# === CALCULATE ===
print("Calculating QQE...")
primary_rsi, primary_qqe_line = calc_qqe_full(rth_df['close'], qqe_rsi1_length, qqe_rsi1_smoothing, qqe_factor_primary)
secondary_rsi, _ = calc_qqe_full(rth_df['close'], qqe_rsi2_length, qqe_rsi2_smoothing, qqe_factor_secondary)

qqe_centered = primary_qqe_line - 50
bb_basis = qqe_centered.rolling(window=qqe_bb_length, min_periods=qqe_bb_length).mean()
bb_std = qqe_centered.rolling(window=qqe_bb_length, min_periods=qqe_bb_length).std()
bb_upper = bb_basis + qqe_bb_mult * bb_std
bb_lower = bb_basis - qqe_bb_mult * bb_std

pri_centered = primary_rsi - 50
sec_centered = secondary_rsi - 50
raw_bull = (sec_centered > threshold_secondary) & (pri_centered > bb_upper)
raw_bear = (sec_centered < -threshold_secondary) & (pri_centered < bb_lower)

mom_rising = (primary_rsi > primary_rsi.shift(1)).fillna(False)
mom_falling = (primary_rsi < primary_rsi.shift(1)).fillna(False)

bull_state, bear_state = calc_qqe_state(
    raw_bull.values, raw_bear.values, 
    mom_rising.values, mom_falling.values
)
print("Done.")

# Output
print("=" * 90)
print("AMD DEC 2, 2025 - QQE STATE MACHINE")
print("=" * 90)
print(f"\n{'Time':<7} {'PriRSI':>7} {'RawBull':>8} {'MomRise':>8} {'BullState':>10} {'QQE_Score':>10}")
print("-" * 90)

dec2_idx = rth_df[rth_df.index.date == pd.Timestamp('2025-12-02').date()].index
for ts in dec2_idx:
    if 9*60+34 <= ts.hour*60+ts.minute <= 9*60+42:
        i = rth_df.index.get_loc(ts)
        print(f"{ts.strftime('%H:%M'):<7} "
              f"{primary_rsi.iloc[i]:>7.2f} "
              f"{str(raw_bull.iloc[i]):>8} "
              f"{str(mom_rising.iloc[i]):>8} "
              f"{str(bull_state[i]):>10} "
              f"{int(bull_state[i]):>10}")

print("\n" + "=" * 90)
print("Expected: QQE=1 from 09:35-09:39, QQE=0 at 09:40+ (when MomRise=False)")
print("=" * 90)

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (0.2 hours old)
Testing on 48002 RTH bars
Calculating QQE...
Done.
AMD DEC 2, 2025 - QQE STATE MACHINE

Time     PriRSI  RawBull  MomRise  BullState  QQE_Score
------------------------------------------------------------------------------------------
09:34     83.38     True     True       True          1
09:35     86.56     True     True       True          1
09:36     89.27     True     True       True          1
09:37     89.50     True     True       True          1
09:38     89.76     True     True       True          1
09:39     90.00     True     True       True          1
09:40     88.14     True    False      False          0
09:41     87.88     True    False      False          0
09:42     88.90     True     True       True          1

Expected: QQE=1 from 09:35-09:39, QQE=0 at 09:40+ (when MomRise=False)


In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
import numpy as np
from numba import jit
from data_collector import PolygonDataCollector

collector = PolygonDataCollector()
df = collector.fetch_bars('AMD', days_back=180, bar_size=1, extended_hours=True)

def is_rth(ts):
    m = ts.hour * 60 + ts.minute
    return (9*60+30) <= m < (16*60)

rth_df = df[df.index.map(is_rth)].copy()
print(f"Testing on {len(rth_df)} RTH bars")

# === QQE SETTINGS ===
qqe_rsi1_length = 6
qqe_rsi1_smoothing = 5
qqe_factor_primary = 3.0
qqe_rsi2_length = 6
qqe_rsi2_smoothing = 5
qqe_factor_secondary = 1.61
qqe_bb_length = 50
qqe_bb_mult = 0.35
threshold_secondary = 3.0
consecutive_bars = 3  # consecutiveIncreasingBars

def calc_rsi(series, length):
    delta = series.diff()
    gain = delta.where(delta > 0, 0.0)
    loss = (-delta).where(delta < 0, 0.0)
    avg_gain = gain.ewm(alpha=1/length, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/length, adjust=False).mean()
    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))

@jit(nopython=True)
def calc_qqe_bands(smoothed_arr, atr_arr):
    n = len(smoothed_arr)
    long_band = np.zeros(n)
    short_band = np.zeros(n)
    trend = np.zeros(n, dtype=np.int32)
    qqe_line = np.zeros(n)
    
    for i in range(1, n):
        if np.isnan(smoothed_arr[i]) or np.isnan(atr_arr[i]):
            long_band[i] = long_band[i-1]
            short_band[i] = short_band[i-1]
            trend[i] = trend[i-1]
            qqe_line[i] = qqe_line[i-1]
            continue
            
        new_short = smoothed_arr[i] + atr_arr[i]
        new_long = smoothed_arr[i] - atr_arr[i]
        
        if smoothed_arr[i-1] > long_band[i-1] and smoothed_arr[i] > long_band[i-1]:
            long_band[i] = max(long_band[i-1], new_long)
        else:
            long_band[i] = new_long
            
        if smoothed_arr[i-1] < short_band[i-1] and smoothed_arr[i] < short_band[i-1]:
            short_band[i] = min(short_band[i-1], new_short)
        else:
            short_band[i] = new_short
        
        if smoothed_arr[i] > short_band[i-1] and smoothed_arr[i-1] <= short_band[i-1]:
            trend[i] = 1
        elif smoothed_arr[i] < long_band[i-1] and smoothed_arr[i-1] >= long_band[i-1]:
            trend[i] = -1
        else:
            trend[i] = trend[i-1]
        
        qqe_line[i] = long_band[i] if trend[i] == 1 else short_band[i]
    
    return qqe_line

def calc_qqe_full(close, rsi_length, smoothing, qqe_factor):
    wilders_length = rsi_length * 2 - 1
    rsi = calc_rsi(close, rsi_length)
    smoothed_rsi = rsi.ewm(span=smoothing, adjust=False).mean()
    atr_rsi = (smoothed_rsi - smoothed_rsi.shift(1)).abs()
    smoothed_atr_rsi = atr_rsi.ewm(span=wilders_length, adjust=False).mean()
    dynamic_atr = smoothed_atr_rsi * qqe_factor
    qqe_line = calc_qqe_bands(smoothed_rsi.values, dynamic_atr.values)
    return smoothed_rsi, pd.Series(qqe_line, index=close.index)

# === QQE SCORE WITH CONSECUTIVE BARS (exact Pine logic) ===
@jit(nopython=True)
def calc_qqe_score(raw_bull, raw_bear, bull_strength, bear_strength, 
                   mom_rising, mom_falling, consecutive_bars):
    n = len(raw_bull)
    bull_cnt = np.zeros(n, dtype=np.int32)
    bear_cnt = np.zeros(n, dtype=np.int32)
    prev_bull_str = np.zeros(n)
    prev_bear_str = np.zeros(n)
    score_bull = np.zeros(n)
    score_bear = np.zeros(n)
    
    for i in range(1, n):
        # Bull side
        if raw_bull[i]:
            if bull_strength[i] > prev_bull_str[i-1]:
                bull_cnt[i] = bull_cnt[i-1] + 1
            else:
                bull_cnt[i] = 0
        else:
            bull_cnt[i] = 0
        
        # Update prev_bull_strength (after count logic, like Pine)
        prev_bull_str[i] = bull_strength[i] if raw_bull[i] else 0.0
        
        # Bear side
        if raw_bear[i]:
            if bear_strength[i] > prev_bear_str[i-1]:
                bear_cnt[i] = bear_cnt[i-1] + 1
            else:
                bear_cnt[i] = 0
        else:
            bear_cnt[i] = 0
        
        prev_bear_str[i] = bear_strength[i] if raw_bear[i] else 0.0
        
        # Final signals
        qqe_bull_signal = (bull_cnt[i] >= consecutive_bars - 1) and raw_bull[i]
        qqe_bear_signal = (bear_cnt[i] >= consecutive_bars - 1) and raw_bear[i]
        
        # Score with momentum filter
        score_bull[i] = 1.0 if (qqe_bull_signal and mom_rising[i]) else 0.0
        score_bear[i] = 1.0 if (qqe_bear_signal and mom_falling[i]) else 0.0
    
    return score_bull, score_bear, bull_cnt, prev_bull_str

# === CALCULATE ===
print("Calculating QQE...")
primary_rsi, primary_qqe_line = calc_qqe_full(rth_df['close'], qqe_rsi1_length, qqe_rsi1_smoothing, qqe_factor_primary)
secondary_rsi, _ = calc_qqe_full(rth_df['close'], qqe_rsi2_length, qqe_rsi2_smoothing, qqe_factor_secondary)

qqe_centered = primary_qqe_line - 50
bb_basis = qqe_centered.rolling(window=qqe_bb_length, min_periods=qqe_bb_length).mean()
bb_std = qqe_centered.rolling(window=qqe_bb_length, min_periods=qqe_bb_length).std()
bb_upper = bb_basis + qqe_bb_mult * bb_std
bb_lower = bb_basis - qqe_bb_mult * bb_std

pri_centered = primary_rsi - 50
sec_centered = secondary_rsi - 50

# Raw conditions
raw_bull = ((sec_centered > threshold_secondary) & (pri_centered > bb_upper)).values
raw_bear = ((sec_centered < -threshold_secondary) & (pri_centered < bb_lower)).values

# Signal strength (Pine: min of two margins)
bull_strength = np.minimum(
    (sec_centered - threshold_secondary).values,
    (pri_centered - bb_upper).values
)
bear_strength = np.minimum(
    (-threshold_secondary - sec_centered).values,
    (bb_lower - pri_centered).values
)

# Momentum
mom_rising = (primary_rsi > primary_rsi.shift(1)).fillna(False).values
mom_falling = (primary_rsi < primary_rsi.shift(1)).fillna(False).values

# Calculate scores
score_bull, score_bear, bull_cnt, prev_bull_str = calc_qqe_score(
    raw_bull, raw_bear, bull_strength, bear_strength,
    mom_rising, mom_falling, consecutive_bars
)
print("Done.")

# Output
print("=" * 100)
print("AMD DEC 2, 2025 - QQE WITH CONSECUTIVE BARS LOGIC")
print("=" * 100)
print(f"\n{'Time':<7} {'PriRSI':>7} {'BullStr':>8} {'PrevStr':>8} {'BullCnt':>8} {'RawBull':>8} {'MomRise':>8} {'Score':>6}")
print("-" * 100)

dec2_idx = rth_df[rth_df.index.date == pd.Timestamp('2025-12-02').date()].index
for ts in dec2_idx:
    if 9*60+34 <= ts.hour*60+ts.minute <= 9*60+42:
        i = rth_df.index.get_loc(ts)
        print(f"{ts.strftime('%H:%M'):<7} "
              f"{primary_rsi.iloc[i]:>7.2f} "
              f"{bull_strength[i]:>8.4f} "
              f"{prev_bull_str[i]:>8.4f} "
              f"{bull_cnt[i]:>8} "
              f"{str(raw_bull[i]):>8} "
              f"{str(mom_rising[i]):>8} "
              f"{int(score_bull[i]):>6}")

print("\n" + "=" * 100)
print("Pine @ 09:39: BullCnt=0, Score=0 | Pine @ 09:40: BullCnt=0, Score=0")
print("=" * 100)

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (0.3 hours old)
Testing on 48002 RTH bars
Calculating QQE...
Done.
AMD DEC 2, 2025 - QQE WITH CONSECUTIVE BARS LOGIC

Time     PriRSI  BullStr  PrevStr  BullCnt  RawBull  MomRise  Score
----------------------------------------------------------------------------------------------------
09:34     83.38  22.6823  22.6823        4     True     True      1
09:35     86.56  25.6242  25.6242        5     True     True      1
09:36     89.27  27.9633  27.9633        6     True     True      1
09:37     89.50  27.8734  27.8734        0     True     True      0
09:38     89.76  27.5818  27.5818        0     True     True      0
09:39     90.00  27.1433  27.1433        0     True     True      0
09:40     88.14  24.6292  24.6292        0     True    False      0
09:41     87.88  23.7147  23.7147        0     True    False      0
09:42     88.90  24.0643  24.064

In [2]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
from data_collector import PolygonDataCollector
from confluence_indicators import ConfluenceCalculator

collector = PolygonDataCollector()
df = collector.fetch_bars('AMD', days_back=180, bar_size=1, extended_hours=True)

# Test with default QQE settings
calc = ConfluenceCalculator({
    'ssl_baseline_length': 45, 
    'ssl_length': 5,
    'qqe_rsi1_length': 6,
    'qqe_rsi1_smoothing': 5,
    'qqe_factor_primary': 3.0,
    'qqe_rsi2_length': 6,
    'qqe_rsi2_smoothing': 5,
    'qqe_factor_secondary': 1.61,
    'qqe_threshold': 3.0,
    'qqe_bb_length': 50,
    'qqe_bb_mult': 0.35,
    'qqe_consecutive_bars': 3
})
calc.compute_indicators(df)

print("=" * 80)
print("AMD DEC 2, 2025 - SSL + QQE FROM confluence_indicators.py")
print("=" * 80)
print(f"\n{'Time':<7} {'Close':>9} {'SSL_Bull':>9} {'QQE_Bull':>9}")
print("-" * 80)

dec2 = df[df.index.date == pd.Timestamp('2025-12-02').date()]
for ts in dec2.index:
    if 9*60+34 <= ts.hour*60+ts.minute <= 9*60+42:
        r = df.loc[ts]
        ssl = r['ssl_score_bull'] if pd.notna(r['ssl_score_bull']) else 0
        qqe = r['qqe_score_bull'] if pd.notna(r['qqe_score_bull']) else 0
        print(f"{ts.strftime('%H:%M'):<7} {r['close']:>9.4f} {int(ssl):>9} {int(qqe):>9}")

print("\n" + "=" * 80)
print("Expected QQE: 1 at 09:35-09:36, 0 at 09:37+")
print("=" * 80)

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (5.6 hours old)
AMD DEC 2, 2025 - SSL + QQE FROM confluence_indicators.py

Time        Close  SSL_Bull  QQE_Bull
--------------------------------------------------------------------------------
09:34    223.4800         0         1
09:35    223.6500         0         1
09:36    224.3297         1         1
09:37    224.2100         0         0
09:38    224.2730         0         0
09:39    224.3100         0         0
09:40    224.2050         0         0
09:41    224.5050         0         0
09:42    225.0400         1         0

Expected QQE: 1 at 09:35-09:36, 0 at 09:37+


In [1]:
# Quick test - add to your local environment
from confluence_indicators import ConfluenceCalculator

params = {
    'wae_fast_ema': 10,
    'wae_slow_ema': 20,
    'wae_sensitivity': 225,
    'wae_bb_length': 12,
    'wae_bb_mult': 2.0,
    'use_wae_acceleration': True,
}

calc = ConfluenceCalculator(params)
calc.compute_indicators(df)  # your AMD dataframe

# At 09:37
ts = '2024-12-02 09:37:00'
print(df.loc[ts][['wae_trend_up', 'wae_trend_down', 'wae_explosion_line', 'wae_score_bull']])

ModuleNotFoundError: No module named 'confluence_indicators'

In [2]:
# wae_test.py - Run from your project directory
import pandas as pd
from confluence_indicators import ConfluenceCalculator
from data_collector import PolygonDataCollector

collector = PolygonDataCollector()
df = collector.get_intraday_bars('AMD', '2024-12-01', '2024-12-03')
df = df.set_index('timestamp')

params = {
    'wae_fast_ema': 10,
    'wae_slow_ema': 20,
    'wae_sensitivity': 225,
    'wae_bb_length': 12,
    'wae_bb_mult': 2.0,
    'use_wae_acceleration': True,
}

calc = ConfluenceCalculator(params)
calc.compute_indicators(df)

# Show 09:35-09:38
for t in ['09:35', '09:36', '09:37', '09:38']:
    ts = [x for x in df.index if f'2024-12-02 {t}' in str(x)][0]
    r = df.loc[ts]
    print(f"{t} | TrendUp: {r['wae_trend_up']:.4f} | Explosion: {r['wae_explosion_line']:.4f} | Score: {r['wae_score_bull']:.0f}")

ModuleNotFoundError: No module named 'confluence_indicators'

In [3]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')  # from the session summary

import pandas as pd
from confluence_indicators import ConfluenceCalculator
from data_collector import PolygonDataCollector

collector = PolygonDataCollector()
df = collector.fetch_bars('AMD', days_back=60, bar_size=1, extended_hours=True)

params = {
    'wae_fast_ema': 10,
    'wae_slow_ema': 20,
    'wae_sensitivity': 225,
    'wae_bb_length': 12,
    'wae_bb_mult': 2.0,
    'use_wae_acceleration': True,
}

calc = ConfluenceCalculator(params)
calc.compute_indicators(df)

# Show 09:35-09:38 on Dec 2
dec2 = df[df.index.date == pd.Timestamp('2024-12-02').date()]
for ts in dec2.index:
    if 9*60+35 <= ts.hour*60+ts.minute <= 9*60+38:
        r = df.loc[ts]
        print(f"{ts.strftime('%H:%M')} | TrendUp: {r['wae_trend_up']:.4f} | Explosion: {r['wae_explosion_line']:.4f} | Score: {r['wae_score_bull']:.0f}")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 60 days, extended)...
  ✓ Fetched 29264 bars, cached for next time


In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
import numpy as np
from data_collector import PolygonDataCollector

collector = PolygonDataCollector()
df = collector.fetch_bars('AMD', days_back=60, bar_size=1, extended_hours=True)

# RTH filter (learned from QQE)
def is_rth(ts):
    m = ts.hour * 60 + ts.minute
    return (9*60+30) <= m < (16*60)

rth_df = df[df.index.map(is_rth)].copy()
print(f"Testing on {len(rth_df)} RTH bars")

# WAE settings (your presets OFF)
wae_fast = 10
wae_slow = 20
wae_sens = 225
wae_bb_len = 12
wae_bb_mult = 2.0

# Calculations
close = rth_df['close']
fast_ma = close.ewm(span=wae_fast, adjust=False).mean()
slow_ma = close.ewm(span=wae_slow, adjust=False).mean()
macd = fast_ma - slow_ma
t1 = (macd - macd.shift(1)) * wae_sens

# Explosion line
basis = close.rolling(wae_bb_len).mean()
std = close.rolling(wae_bb_len).std()
dev = wae_bb_mult * std
explosion = (basis + dev) - (basis - dev)  # = 2 * dev

# Trend
trend_up = t1.where(t1 >= 0, 0.0)
trend_dn = (-t1).where(t1 < 0, 0.0)

# Confirmations
bull_conf = trend_up > explosion
prev_up = trend_up.shift(1).fillna(0.0)
bull_accel = (trend_up > prev_up) & (trend_up > explosion)
score_bull = (bull_conf & bull_accel).astype(float)

# Output Dec 2
print("=" * 90)
print("AMD DEC 2 - WAE DEBUG (Python vs Pine)")
print("=" * 90)
print(f"\n{'Time':<7} {'t1':>10} {'TrendUp':>10} {'Explosion':>10} {'BullConf':>9} {'Accel':>6} {'Score':>6}")
print("-" * 90)

dec2 = rth_df[rth_df.index.date == pd.Timestamp('2024-12-02').date()]
for ts in dec2.index:
    if 9*60+35 <= ts.hour*60+ts.minute <= 9*60+38:
        print(f"{ts.strftime('%H:%M'):<7} "
              f"{t1.loc[ts]:>10.4f} "
              f"{trend_up.loc[ts]:>10.4f} "
              f"{explosion.loc[ts]:>10.4f} "
              f"{str(bull_conf.loc[ts]):>9} "
              f"{str(bull_accel.loc[ts]):>6} "
              f"{int(score_bull.loc[ts]):>6}")

print("\n" + "=" * 90)
print("Pine reference:")
print("09:35 t1=24.1253 Explosion=6.3054 Score=0")
print("09:36 t1=27.9153 Explosion=6.98   Score=1")
print("09:37 t1=14.6648 Explosion=7.2592 Score=0")
print("=" * 90)

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 60 days, extended)...
  ✓ Loading from cache (0.1 hours old)
Testing on 14852 RTH bars
AMD DEC 2 - WAE DEBUG (Python vs Pine)

Time            t1    TrendUp  Explosion  BullConf  Accel  Score
------------------------------------------------------------------------------------------

Pine reference:
09:35 t1=24.1253 Explosion=6.3054 Score=0
09:36 t1=27.9153 Explosion=6.98   Score=1
09:37 t1=14.6648 Explosion=7.2592 Score=0


In [2]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
import numpy as np
from data_collector import PolygonDataCollector

collector = PolygonDataCollector()
df = collector.fetch_bars('AMD', days_back=60, bar_size=1, extended_hours=True)

# RTH filter
def is_rth(ts):
    m = ts.hour * 60 + ts.minute
    return (9*60+30) <= m < (16*60)

rth_df = df[df.index.map(is_rth)].copy()
print(f"Testing on {len(rth_df)} RTH bars")

# WAE settings (your presets OFF)
wae_fast = 10
wae_slow = 20
wae_sens = 225
wae_bb_len = 12
wae_bb_mult = 2.0

# Calculations
close = rth_df['close']
fast_ma = close.ewm(span=wae_fast, adjust=False).mean()
slow_ma = close.ewm(span=wae_slow, adjust=False).mean()
macd = fast_ma - slow_ma
t1 = (macd - macd.shift(1)) * wae_sens

# Explosion line
basis = close.rolling(wae_bb_len).mean()
std = close.rolling(wae_bb_len).std()
dev = wae_bb_mult * std
explosion = (basis + dev) - (basis - dev)  # = 2 * dev

# Trend
trend_up = t1.where(t1 >= 0, 0.0)
trend_dn = (-t1).where(t1 < 0, 0.0)

# Confirmations
bull_conf = trend_up > explosion
prev_up = trend_up.shift(1).fillna(0.0)
bull_accel = (trend_up > prev_up) & (trend_up > explosion)
score_bull = (bull_conf & bull_accel).astype(float)

# Output Dec 2 - FIX: 2025 not 2024
print("=" * 90)
print("AMD DEC 2, 2025 - WAE DEBUG (Python vs Pine)")
print("=" * 90)
print(f"\n{'Time':<7} {'t1':>10} {'TrendUp':>10} {'Explosion':>10} {'BullConf':>9} {'Accel':>6} {'Score':>6}")
print("-" * 90)

dec2 = rth_df[rth_df.index.date == pd.Timestamp('2025-12-02').date()]
for ts in dec2.index:
    if 9*60+35 <= ts.hour*60+ts.minute <= 9*60+38:
        print(f"{ts.strftime('%H:%M'):<7} "
              f"{t1.loc[ts]:>10.4f} "
              f"{trend_up.loc[ts]:>10.4f} "
              f"{explosion.loc[ts]:>10.4f} "
              f"{str(bull_conf.loc[ts]):>9} "
              f"{str(bull_accel.loc[ts]):>6} "
              f"{int(score_bull.loc[ts]):>6}")

print("\n" + "=" * 90)
print("Pine reference:")
print("09:35 t1=24.1253 Explosion=6.3054 Score=0")
print("09:36 t1=27.9153 Explosion=6.98   Score=1")
print("09:37 t1=14.6648 Explosion=7.2592 Score=0")
print("=" * 90)

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 60 days, extended)...
  ✓ Loading from cache (0.1 hours old)
Testing on 14852 RTH bars
AMD DEC 2, 2025 - WAE DEBUG (Python vs Pine)

Time            t1    TrendUp  Explosion  BullConf  Accel  Score
------------------------------------------------------------------------------------------
09:35      24.1253    24.1253     6.5858      True  False      0
09:36      27.9153    27.9153     7.2904      True   True      1
09:37      14.6648    14.6648     7.5820      True  False      0
09:38       8.1611     8.1611     7.4935      True  False      0

Pine reference:
09:35 t1=24.1253 Explosion=6.3054 Score=0
09:36 t1=27.9153 Explosion=6.98   Score=1
09:37 t1=14.6648 Explosion=7.2592 Score=0


In [4]:
# Change this line:
std = close.rolling(wae_bb_len).std()

# To use population stdev (ddof=0) like Pine:
std = close.rolling(wae_bb_len).std(ddof=0)

In [5]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
import numpy as np
from data_collector import PolygonDataCollector

collector = PolygonDataCollector()
df = collector.fetch_bars('AMD', days_back=60, bar_size=1, extended_hours=True)

# RTH filter
def is_rth(ts):
    m = ts.hour * 60 + ts.minute
    return (9*60+30) <= m < (16*60)

rth_df = df[df.index.map(is_rth)].copy()
print(f"Testing on {len(rth_df)} RTH bars")

# WAE settings (your presets OFF)
wae_fast = 10
wae_slow = 20
wae_sens = 225
wae_bb_len = 12
wae_bb_mult = 2.0

# Calculations
close = rth_df['close']
fast_ma = close.ewm(span=wae_fast, adjust=False).mean()
slow_ma = close.ewm(span=wae_slow, adjust=False).mean()
macd = fast_ma - slow_ma
t1 = (macd - macd.shift(1)) * wae_sens

# Explosion line - use ddof=0 for population stdev (Pine parity)
basis = close.rolling(wae_bb_len).mean()
std = close.rolling(wae_bb_len).std(ddof=0)
dev = wae_bb_mult * std
explosion = (basis + dev) - (basis - dev)

# Trend
trend_up = t1.where(t1 >= 0, 0.0)
trend_dn = (-t1).where(t1 < 0, 0.0)

# Confirmations
bull_conf = trend_up > explosion
prev_up = trend_up.shift(1).fillna(0.0)
bull_accel = (trend_up > prev_up) & (trend_up > explosion)
score_bull = (bull_conf & bull_accel).astype(float)

# Output Dec 2, 2025
print("=" * 90)
print("AMD DEC 2, 2025 - WAE DEBUG (Python vs Pine) - with ddof=0")
print("=" * 90)
print(f"\n{'Time':<7} {'t1':>10} {'TrendUp':>10} {'Explosion':>10} {'BullConf':>9} {'Accel':>6} {'Score':>6}")
print("-" * 90)

dec2 = rth_df[rth_df.index.date == pd.Timestamp('2025-12-02').date()]
for ts in dec2.index:
    if 9*60+35 <= ts.hour*60+ts.minute <= 9*60+38:
        print(f"{ts.strftime('%H:%M'):<7} "
              f"{t1.loc[ts]:>10.4f} "
              f"{trend_up.loc[ts]:>10.4f} "
              f"{explosion.loc[ts]:>10.4f} "
              f"{str(bull_conf.loc[ts]):>9} "
              f"{str(bull_accel.loc[ts]):>6} "
              f"{int(score_bull.loc[ts]):>6}")

print("\n" + "=" * 90)
print("Pine reference:")
print("09:35 t1=24.1253 Explosion=6.3054 Score=0")
print("09:36 t1=27.9153 Explosion=6.98   Score=1")
print("09:37 t1=14.6648 Explosion=7.2592 Score=0")
print("=" * 90)

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 60 days, extended)...
  ✓ Loading from cache (0.2 hours old)
Testing on 14852 RTH bars
AMD DEC 2, 2025 - WAE DEBUG (Python vs Pine) - with ddof=0

Time            t1    TrendUp  Explosion  BullConf  Accel  Score
------------------------------------------------------------------------------------------
09:35      24.1253    24.1253     6.3054      True  False      0
09:36      27.9153    27.9153     6.9800      True   True      1
09:37      14.6648    14.6648     7.2592      True  False      0
09:38       8.1611     8.1611     7.1745      True  False      0

Pine reference:
09:35 t1=24.1253 Explosion=6.3054 Score=0
09:36 t1=27.9153 Explosion=6.98   Score=1
09:37 t1=14.6648 Explosion=7.2592 Score=0


In [6]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
import numpy as np
from data_collector import PolygonDataCollector

collector = PolygonDataCollector()
df = collector.fetch_bars('NVDA', days_back=5, bar_size=1, extended_hours=True)

# RTH filter
def is_rth(ts):
    m = ts.hour * 60 + ts.minute
    return (9*60+30) <= m < (16*60)

rth_df = df[df.index.map(is_rth)].copy()

# New Z-score volume logic
vol_lookback = 20
vol_mean = rth_df['volume'].rolling(vol_lookback).mean()
vol_std = rth_df['volume'].rolling(vol_lookback).std(ddof=0)  # population stdev
vol_z = np.where(vol_std > 0, (rth_df['volume'] - vol_mean) / vol_std, 0.0)

# Score
vol_score = np.where(vol_z >= 1.0, 1.0, np.where(vol_z >= 0.2, 0.5, 0.0))

rth_df['vol_mean'] = vol_mean
rth_df['vol_std'] = vol_std
rth_df['vol_z'] = vol_z
rth_df['vol_score'] = vol_score

print("=" * 80)
print("NVDA Jan 21, 2025 - VOLUME Z-SCORE DEBUG")
print("=" * 80)
print(f"\n{'Time':<7} {'Volume':>10} {'Mean':>12} {'Std':>12} {'Z-Score':>8} {'Score':>6}")
print("-" * 80)

today = rth_df[rth_df.index.date == pd.Timestamp('2025-01-21').date()]
for ts in today.index:
    if 9*60+50 <= ts.hour*60+ts.minute <= 9*60+53:
        r = rth_df.loc[ts]
        print(f"{ts.strftime('%H:%M'):<7} "
              f"{r['volume']:>10.0f} "
              f"{r['vol_mean']:>12.2f} "
              f"{r['vol_std']:>12.2f} "
              f"{r['vol_z']:>8.2f}σ "
              f"{r['vol_score']:>6.1f}")

print("\n" + "=" * 80)
print("Pine @ 09:52: Volume=663384, Mean=611165.9, Std=130685.66, Z=0.4σ, Score=0.5")
print("=" * 80)

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching NVDA (1-min bars, 5 days, extended)...
  ✓ Fetched 2851 bars, cached for next time
NVDA Jan 21, 2025 - VOLUME Z-SCORE DEBUG

Time        Volume         Mean          Std  Z-Score  Score
--------------------------------------------------------------------------------

Pine @ 09:52: Volume=663384, Mean=611165.9, Std=130685.66, Z=0.4σ, Score=0.5


In [7]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
import numpy as np
from data_collector import PolygonDataCollector

collector = PolygonDataCollector()
df = collector.fetch_bars('NVDA', days_back=5, bar_size=1, extended_hours=True)

# RTH filter
def is_rth(ts):
    m = ts.hour * 60 + ts.minute
    return (9*60+30) <= m < (16*60)

rth_df = df[df.index.map(is_rth)].copy()

# New Z-score volume logic
vol_lookback = 20
vol_mean = rth_df['volume'].rolling(vol_lookback).mean()
vol_std = rth_df['volume'].rolling(vol_lookback).std(ddof=0)
vol_z = np.where(vol_std > 0, (rth_df['volume'] - vol_mean) / vol_std, 0.0)

vol_score = np.where(vol_z >= 1.0, 1.0, np.where(vol_z >= 0.2, 0.5, 0.0))

rth_df['vol_mean'] = vol_mean
rth_df['vol_std'] = vol_std
rth_df['vol_z'] = vol_z
rth_df['vol_score'] = vol_score

print("=" * 80)
print("NVDA Jan 21, 2026 - VOLUME Z-SCORE DEBUG")
print("=" * 80)
print(f"\n{'Time':<7} {'Volume':>10} {'Mean':>12} {'Std':>12} {'Z-Score':>8} {'Score':>6}")
print("-" * 80)

today = rth_df[rth_df.index.date == pd.Timestamp('2026-01-21').date()]
for ts in today.index:
    if 9*60+50 <= ts.hour*60+ts.minute <= 9*60+53:
        r = rth_df.loc[ts]
        print(f"{ts.strftime('%H:%M'):<7} "
              f"{r['volume']:>10.0f} "
              f"{r['vol_mean']:>12.2f} "
              f"{r['vol_std']:>12.2f} "
              f"{r['vol_z']:>8.2f}σ "
              f"{r['vol_score']:>6.1f}")

print("\n" + "=" * 80)
print("Pine @ 09:52: Volume=663384, Mean=611165.9, Std=130685.66, Z=0.4σ, Score=0.5")
print("=" * 80)

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching NVDA (1-min bars, 5 days, extended)...
  ✓ Loading from cache (0.0 hours old)
NVDA Jan 21, 2026 - VOLUME Z-SCORE DEBUG

Time        Volume         Mean          Std  Z-Score  Score
--------------------------------------------------------------------------------
09:50      1049383    796691.35    160583.01     1.57σ    1.0
09:51       648711    780137.95    157897.89    -0.83σ    0.0
09:52       800017    775521.10    155886.38     0.16σ    0.0
09:53       899172    768147.35    146065.76     0.90σ    0.5

Pine @ 09:52: Volume=663384, Mean=611165.9, Std=130685.66, Z=0.4σ, Score=0.5


In [8]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import pandas as pd
import numpy as np
from data_collector import PolygonDataCollector

collector = PolygonDataCollector()
df = collector.fetch_bars('AMD', days_back=60, bar_size=1, extended_hours=True)

# RTH filter
def is_rth(ts):
    m = ts.hour * 60 + ts.minute
    return (9*60+30) <= m < (16*60)

rth_df = df[df.index.map(is_rth)].copy()

# New Z-score volume logic
vol_lookback = 20
vol_mean = rth_df['volume'].rolling(vol_lookback).mean()
vol_std = rth_df['volume'].rolling(vol_lookback).std(ddof=0)
vol_z = np.where(vol_std > 0, (rth_df['volume'] - vol_mean) / vol_std, 0.0)

vol_score = np.where(vol_z >= 1.0, 1.0, np.where(vol_z >= 0.2, 0.5, 0.0))

rth_df['vol_mean'] = vol_mean
rth_df['vol_std'] = vol_std
rth_df['vol_z'] = vol_z
rth_df['vol_score'] = vol_score

print("=" * 80)
print("AMD Dec 2, 2025 - VOLUME Z-SCORE DEBUG")
print("=" * 80)
print(f"\n{'Time':<7} {'Volume':>10} {'Mean':>12} {'Std':>12} {'Z-Score':>8} {'Score':>6}")
print("-" * 80)

dec2 = rth_df[rth_df.index.date == pd.Timestamp('2025-12-02').date()]
for ts in dec2.index:
    if 9*60+35 <= ts.hour*60+ts.minute <= 9*60+38:
        r = rth_df.loc[ts]
        print(f"{ts.strftime('%H:%M'):<7} "
              f"{r['volume']:>10.0f} "
              f"{r['vol_mean']:>12.2f} "
              f"{r['vol_std']:>12.2f} "
              f"{r['vol_z']:>8.2f}σ "
              f"{r['vol_score']:>6.1f}")

print("\n" + "=" * 80)
print("Pine reference: (paste values here)")
print("=" * 80)

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 60 days, extended)...
  ✓ Loading from cache (1.1 hours old)
AMD Dec 2, 2025 - VOLUME Z-SCORE DEBUG

Time        Volume         Mean          Std  Z-Score  Score
--------------------------------------------------------------------------------
09:35       219729    175778.55    150075.40     0.29σ    0.5
09:36       226909    184774.25    147455.62     0.29σ    0.5
09:37       230459    193839.00    144380.88     0.25σ    0.5
09:38       239642    203827.35    140234.52     0.26σ    0.5

Pine reference: (paste values here)


In [1]:
python -c "
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
from orb_backtester import ORBBacktester

bt = ORBBacktester('AMD', '2025-10-01', '2025-10-15', enable_confluence=True, min_confluence=2, verbose=True)
bt.load_data()

# Check a few bars around 9:40 (after ORB window)
import pandas as pd
for i in range(100, 110):
    ts = bt.df.index[i]
    if ts.hour == 9 and ts.minute >= 35:
        scores = bt.confluence_calc.get_scores(i)
        print(f'{ts}: SSL={scores.ssl_score_bull:.0f} WAE={scores.wae_score_bull:.0f} QQE={scores.qqe_score_bull:.0f} Vol={scores.vol_score:.1f}')
"

SyntaxError: unterminated string literal (detected at line 1) (2658145612.py, line 1)

In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
from orb_backtester import ORBBacktester

bt = ORBBacktester('AMD', '2025-10-01', '2025-10-15', enable_confluence=True, min_confluence=2, verbose=True)
bt.load_data()

# Check confluence scores
for i in range(len(bt.df)):
    ts = bt.df.index[i]
    if ts.hour == 9 and 35 <= ts.minute <= 45:
        scores = bt.confluence_calc.get_scores(i)
        total = 1 + scores.ssl_score_bull + scores.wae_score_bull + scores.qqe_score_bull + scores.vol_score
        print(f'{ts}: SSL={scores.ssl_score_bull:.0f} WAE={scores.wae_score_bull:.0f} QQE={scores.qqe_score_bull:.0f} Vol={scores.vol_score:.1f} | Total={total:.1f}')

[Presets] Loaded AMD: SSL_base=20, WAE_sens=300, QQE_rsi1=8
[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (3.2 hours old)
âœ“ Loaded 102851 bars for AMD
  [*] Calculating confluence indicators (SSL, WAE, QQE, Volume)...
  [âœ“] Confluence indicators calculated
2025-07-28 09:35:00-04:00: SSL=0 WAE=0 QQE=1 Vol=0.5 | Total=2.5
2025-07-28 09:36:00-04:00: SSL=0 WAE=0 QQE=1 Vol=0.5 | Total=2.5
2025-07-28 09:37:00-04:00: SSL=0 WAE=0 QQE=0 Vol=0.0 | Total=1.0
2025-07-28 09:38:00-04:00: SSL=0 WAE=0 QQE=0 Vol=0.0 | Total=1.0
2025-07-28 09:39:00-04:00: SSL=0 WAE=0 QQE=0 Vol=0.0 | Total=1.0
2025-07-28 09:40:00-04:00: SSL=0 WAE=0 QQE=0 Vol=0.0 | Total=1.0
2025-07-28 09:41:00-04:00: SSL=1 WAE=0 QQE=0 Vol=0.0 | Total=2.0
2025-07-28 09:42:00-04:00: SSL=0 WAE=0 QQE=0 Vol=0.0 | Total=1.0
2025-07-28 09:43:00-04:00: SSL=0 WAE=0 QQE=0 Vol=0.0 | Total=1.0
2025-07-28 09:44:00-04:00: SSL=0 WAE=0 QQE=0 Vol=0.0 | Total=1.

In [2]:
bt = ORBBacktester('AMD', '2025-10-01', '2025-10-15', enable_confluence=True, min_confluence=2, verbose=True)
bt.load_data()
results = bt.run()
print(f"Total trades: {results['total_trades']}")


[Presets] Loaded AMD: SSL_base=20, WAE_sens=300, QQE_rsi1=8
[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (3.3 hours old)
âœ“ Loaded 102851 bars for AMD
  [*] Calculating confluence indicators (SSL, WAE, QQE, Volume)...
  [âœ“] Confluence indicators calculated

  ORB BACKTESTER: AMD
  2025-10-01 to 2025-10-15

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (3.3 hours old)
âœ“ Loaded 102851 bars for AMD
  [*] Calculating confluence indicators (SSL, WAE, QQE, Volume)...
  [âœ“] Confluence indicators calculated
Processing 11 trading days...


  BACKTEST SUMMARY: AMD
  2025-10-01 to 2025-10-15

  Total Trades:    0
  Wins/Losses:     0W / 0L
  Win Rate:        0.0%

  Total P/L:       $+0.00
  Total R:         +0.00R
  Avg R/Trade:     +0.000R

  Avg Win:         $0.00 (0.00R)
  Avg Loss:        $0.00 (0.00R)
  

In [5]:
import numpy as np
import pandas as pd

# Diagnostic: Trace entry decisions
bt = ORBBacktester('AMD', '2025-10-01', '2025-10-15', enable_confluence=True, min_confluence=2, verbose=True)
bt.load_data()

# Pick a specific day to trace
date_str = '2025-10-01'
day_df = bt.df[bt.df.index.date.astype(str) == date_str].copy()

bt.orb.reset_session()
bt.vol.orb_session_baseline = np.nan

for i in range(len(day_df)):
    ts = day_df.index[i]
    bar = day_df.iloc[i]
    
    if not bt._is_rth(ts):
        continue
    
    bt.orb.bars_since_last_trigger += 1
    atr = bar['atr_rth']
    
    # ORB building
    if bt._is_orb_window(ts):
        if ts.hour == 9 and ts.minute == 30:
            bt.orb.reset_session()
            bt.orb.session_high = bar['high']
            bt.orb.session_low = bar['low']
        else:
            bt.orb.session_high = max(bt.orb.session_high, bar['high'])
            bt.orb.session_low = min(bt.orb.session_low, bar['low'])
        continue
    
    # ORB complete check
    if not bt.orb.orb_complete and not pd.isna(bt.orb.session_high):
        bt.orb.orb_complete = True
        print(f"{ts} ORB COMPLETE: High={bt.orb.session_high:.2f}, Low={bt.orb.session_low:.2f}")
    
    bt._calc_vol_state(bar, ts)
    
    # Breakout detection
    new_long, new_short = bt._check_breakout(bar)
    
    if new_long and not bt.orb.has_broken_out_high:
        bt.orb.long_breakout_pending = True
        bt.orb.has_broken_out_high = True
        print(f"{ts} LONG BREAKOUT DETECTED! Close={bar['close']:.2f}")
    
    if new_short and not bt.orb.has_broken_out_low:
        bt.orb.short_breakout_pending = True
        bt.orb.has_broken_out_low = True
        print(f"{ts} SHORT BREAKOUT DETECTED! Close={bar['close']:.2f}")
    
    # Entry evaluation
    in_pos = bt._in_position()
    in_window = bt._is_trading_window(ts)
    can_trig = bt._can_trigger()
    
    if not in_pos and in_window and can_trig:
        if bt.orb.long_breakout_pending:
            print(f"{ts} CHECKING LONG ENTRY...")
            
            full_idx = bt.df.index.get_loc(ts)
            passes_conf, conf_score, conf_details = bt.confluence_calc.check_confluence(
                full_idx, is_long=True, orb_breakout=True
            )
            print(f"  Confluence: passes={passes_conf}, score={conf_score}")
            print(f"  SSL={conf_details.ssl_score_bull}, WAE={conf_details.wae_score_bull}, QQE={conf_details.qqe_score_bull}, Vol={conf_details.vol_score}")
            
            if passes_conf:
                entry = bar['close']
                vwap = bar['vwap']
                stops = bt._calc_stops(entry, atr, vwap, True, day_df, i)
                stop_price, stop_type, achievable_rr = bt._select_best_stop(stops, entry, atr, bt.profit_target_rr, True)
                print(f"  Stop: {stop_type}@{stop_price:.2f}, R:R={achievable_rr:.2f}")
                
                if achievable_rr >= bt.min_acceptable_rr - 0.0001:
                    print(f"  ✓ TRADE WOULD BE TAKEN!")
                else:
                    print(f"  ✗ R:R too low (need {bt.min_acceptable_rr}, got {achievable_rr:.2f})")
            else:
                print(f"  ✗ Confluence not met (need {bt.min_confluence})")
        
        if bt.orb.short_breakout_pending:
            print(f"{ts} CHECKING SHORT ENTRY...")
            # Similar logic for shorts...
    else:
        if bt.orb.long_breakout_pending or bt.orb.short_breakout_pending:
            if in_pos:
                print(f"{ts} Pending breakout but IN POSITION")
            if not in_window:
                print(f"{ts} Pending breakout but NOT IN TRADING WINDOW")
            if not can_trig:
                print(f"{ts} Pending breakout but CANNOT TRIGGER (cooldown)")

[Presets] Loaded AMD: SSL_base=20, WAE_sens=300, QQE_rsi1=8
[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (3.5 hours old)
âœ“ Loaded 102851 bars for AMD
  [*] Calculating confluence indicators (SSL, WAE, QQE, Volume)...
  [âœ“] Confluence indicators calculated
2025-10-01 09:35:00-04:00 ORB COMPLETE: High=162.38, Low=160.49
2025-10-01 09:44:00-04:00 LONG BREAKOUT DETECTED! Close=162.55
2025-10-01 09:44:00-04:00 CHECKING LONG ENTRY...
  Confluence: passes=True, score=5
  SSL=1.0, WAE=1.0, QQE=1.0, Vol=1.0
  Stop: ATR@162.21, R:R=1.50
  ✓ TRADE WOULD BE TAKEN!
2025-10-01 09:45:00-04:00 CHECKING LONG ENTRY...
  Confluence: passes=True, score=3
  SSL=0.0, WAE=0.0, QQE=1.0, Vol=0.5
  Stop: ATR@161.96, R:R=1.50
  ✓ TRADE WOULD BE TAKEN!
2025-10-01 09:46:00-04:00 CHECKING LONG ENTRY...
  Confluence: passes=True, score=2
  SSL=0.0, WAE=0.0, QQE=1.0, Vol=0.0
  Stop: ATR@161.92, R:R=1.50
  ✓ TRADE WOULD BE 

In [6]:
# Fresh backtester - don't reuse the traced one
bt2 = ORBBacktester('AMD', '2025-10-01', '2025-10-15', enable_confluence=True, min_confluence=2, verbose=True)
results = bt2.run()
print(f"Total trades: {results['total_trades']}")
for t in results['trades']:
    print(f"  {t['date']} {t['direction']} @ {t['entry_price']:.2f} -> {t['exit_reason']}")

[Presets] Loaded AMD: SSL_base=20, WAE_sens=300, QQE_rsi1=8

  ORB BACKTESTER: AMD
  2025-10-01 to 2025-10-15

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (3.5 hours old)
âœ“ Loaded 102851 bars for AMD
  [*] Calculating confluence indicators (SSL, WAE, QQE, Volume)...
  [âœ“] Confluence indicators calculated
Processing 11 trading days...


  BACKTEST SUMMARY: AMD
  2025-10-01 to 2025-10-15

  Total Trades:    0
  Wins/Losses:     0W / 0L
  Win Rate:        0.0%

  Total P/L:       $+0.00
  Total R:         +0.00R
  Avg R/Trade:     +0.000R

  Avg Win:         $0.00 (0.00R)
  Avg Loss:        $0.00 (0.00R)
  Profit Factor:   0.00
  Max Drawdown:    0.00R


Total trades: 0


In [7]:
# Save original method
original_process_day = bt2._process_day

def debug_process_day(date_str):
    day_df = bt2.df[bt2.df.index.date.astype(str) == date_str].copy()
    if len(day_df) == 0:
        return
    
    bt2.orb.reset_session()
    bt2.vol.orb_session_baseline = np.nan
    
    for i in range(len(day_df)):
        ts = day_df.index[i]
        bar = day_df.iloc[i]
        
        if not bt2._is_rth(ts):
            continue
        
        bt2.orb.bars_since_last_trigger += 1
        
        if bt2._is_orb_window(ts):
            if ts.hour == 9 and ts.minute == 30:
                bt2.orb.reset_session()
                bt2.orb.session_high = bar['high']
                bt2.orb.session_low = bar['low']
            else:
                bt2.orb.session_high = max(bt2.orb.session_high, bar['high'])
                bt2.orb.session_low = min(bt2.orb.session_low, bar['low'])
            continue
        
        if not bt2.orb.orb_complete and not pd.isna(bt2.orb.session_high):
            bt2.orb.orb_complete = True
        
        bt2._calc_vol_state(bar, ts)
        
        new_long, new_short = bt2._check_breakout(bar)
        
        if new_long and not bt2.orb.has_broken_out_high:
            bt2.orb.long_breakout_pending = True
            bt2.orb.has_broken_out_high = True
            print(f"DEBUG {ts}: BREAKOUT DETECTED")
        
        # Check conditions for entry
        in_pos = bt2._in_position()
        in_window = bt2._is_trading_window(ts)
        can_trig = bt2._can_trigger()
        has_pending = bt2.orb.long_breakout_pending or bt2.orb.short_breakout_pending
        
        if has_pending and ts.minute <= 50:  # limit output
            print(f"DEBUG {ts}: in_pos={in_pos}, in_window={in_window}, can_trig={can_trig}, pending_L={bt2.orb.long_breakout_pending}, pending_S={bt2.orb.short_breakout_pending}")
        
        # Call original entry logic... actually let's just check one thing
        if not in_pos and in_window and can_trig and bt2.orb.long_breakout_pending:
            print(f"DEBUG {ts}: SHOULD BE CHECKING CONFLUENCE NOW!")

# Don't actually replace - just run this on one day manually:
bt3 = ORBBacktester('AMD', '2025-10-01', '2025-10-15', enable_confluence=True, min_confluence=2, verbose=False)
bt3.load_data()

# Run just one day with debug
debug_process_day('2025-10-01')
print(f"\nTrades taken: {len(bt3.trades)}")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (3.6 hours old)
DEBUG 2025-10-01 09:44:00-04:00: BREAKOUT DETECTED
DEBUG 2025-10-01 09:44:00-04:00: in_pos=False, in_window=True, can_trig=True, pending_L=True, pending_S=False
DEBUG 2025-10-01 09:44:00-04:00: SHOULD BE CHECKING CONFLUENCE NOW!
DEBUG 2025-10-01 09:45:00-04:00: in_pos=False, in_window=True, can_trig=True, pending_L=True, pending_S=False
DEBUG 2025-10-01 09:45:00-04:00: SHOULD BE CHECKING CONFLUENCE NOW!
DEBUG 2025-10-01 09:46:00-04:00: in_pos=False, in_window=True, can_trig=True, pending_L=True, pending_S=False
DEBUG 2025-10-01 09:46:00-04:00: SHOULD BE CHECKING CONFLUENCE NOW!
DEBUG 2025-10-01 09:47:00-04:00: in_pos=False, in_window=True, can_trig=True, pending_L=True, pending_S=False
DEBUG 2025-10-01 09:47:00-04:00: SHOULD BE CHECKING CONFLUENCE NOW!
DEBUG 2025-10-01 09:48:00-04:00: in_pos=False, in_window=True, can_trig=True, pendin

In [8]:
bt4 = ORBBacktester('AMD', '2025-10-01', '2025-10-15', enable_confluence=True, min_confluence=2)
bt4.load_data()

# Check what min_confluence each component thinks it has
print(f"Backtester min_confluence: {bt4.min_confluence}")
print(f"Calculator min_confluence: {bt4.confluence_calc.min_confluence}")
print(f"Confluence mode: {bt4.confluence_mode}")

# Check a specific bar where we KNOW confluence should pass
ts = pd.Timestamp('2025-10-01 09:44:00-04:00')
idx = bt4.df.index.get_loc(ts)
passes, score, details = bt4.confluence_calc.check_confluence(idx, is_long=True, orb_breakout=True)
print(f"\nAt {ts}:")
print(f"  check_confluence returns: passes={passes}, score={score}")
print(f"  SSL={details.ssl_score_bull}, WAE={details.wae_score_bull}, QQE={details.qqe_score_bull}, Vol={details.vol_score}")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (3.6 hours old)
Backtester min_confluence: 2
Calculator min_confluence: 2
Confluence mode: all

At 2025-10-01 09:44:00-04:00:
  check_confluence returns: passes=True, score=5
  SSL=1.0, WAE=1.0, QQE=1.0, Vol=1.0


In [1]:
# Let's trace what happens inside the ACTUAL entry block
# Temporarily patch the backtester to add debug output

import types

def patched_process_day(self, date_str):
    """Process a single trading day - WITH DEBUG"""
    day_df = self.df[self.df.index.date.astype(str) == date_str].copy()
    
    if len(day_df) == 0:
        return
    
    # Reset session state
    self.orb.reset_session()
    self.vol.orb_session_baseline = np.nan
    
    for i in range(len(day_df)):
        ts = day_df.index[i]
        bar = day_df.iloc[i]
        
        if not self._is_rth(ts):
            continue
        
        time_str = ts.strftime('%H:%M')
        
        # Cooldown counter
        self.orb.bars_since_last_trigger += 1
        
        atr = bar['atr_rth']
        ema = bar['ema9']
        
        # Skip exit checks for brevity...
        
        # ORB BUILDING
        if self._is_orb_window(ts):
            if ts.hour == 9 and ts.minute == 30:
                self.orb.reset_session()
                self.orb.session_high = bar['high']
                self.orb.session_low = bar['low']
            else:
                self.orb.session_high = max(self.orb.session_high, bar['high'])
                self.orb.session_low = min(self.orb.session_low, bar['low'])
            continue
        
        # ORB complete
        if not self.orb.orb_complete and not pd.isna(self.orb.session_high):
            self.orb.orb_complete = True
        
        self._calc_vol_state(bar, ts)
        
        # BREAKOUT DETECTION
        new_long, new_short = self._check_breakout(bar)
        
        if new_long and not self.orb.has_broken_out_high:
            self.orb.long_breakout_pending = True
            self.orb.has_broken_out_high = True
        
        if new_short and not self.orb.has_broken_out_low:
            self.orb.short_breakout_pending = True
            self.orb.has_broken_out_low = True
        
        # ENTRY EVALUATION - ONLY DEBUG FIRST FEW
        if not self._in_position() and self._is_trading_window(ts) and self._can_trigger():

                if self.orb.long_breakout_pending:
                    print(f"[DEBUG] {ts} LONG ENTRY BLOCK REACHED")
                    entry = bar['close']
                    vwap = bar['vwap']
                
                if self.enable_confluence:
                    full_idx = self.df.index.get_loc(ts)
                    passes_conf, conf_score, conf_details = self.confluence_calc.check_confluence(
                        full_idx, is_long=True, orb_breakout=True
                    )
                    print(f"DEBUG {ts}: passes_conf={passes_conf}, score={conf_score}, mode={self.confluence_mode}")
                    
                    # EXACT COPY OF MODE LOGIC
                    if self.confluence_mode == 'ssl':
                        mode_score = 1 + conf_details.ssl_score_bull
                        passes_conf = mode_score >= self.min_confluence
                    elif self.confluence_mode == 'wae':
                        mode_score = 1 + conf_details.wae_score_bull
                        passes_conf = mode_score >= self.min_confluence
                    elif self.confluence_mode == 'qqe':
                        mode_score = 1 + conf_details.qqe_score_bull
                        passes_conf = mode_score >= self.min_confluence
                    elif self.confluence_mode == 'vol':
                        from confluence_indicators import pine_round
                        mode_score = 1 + pine_round(conf_details.vol_score)
                        passes_conf = mode_score >= self.min_confluence
                    # else 'all' - use passes_conf from check_confluence
                    
                    print(f"  After mode logic: passes_conf={passes_conf}")
                    
                    if passes_conf:
                        rr_desired = self.profit_target_rr
                        stops = self._calc_stops(entry, atr, vwap, True, day_df, i)
                        stop_price, stop_type, achievable_rr = self._select_best_stop(stops, entry, atr, rr_desired, True)
                        print(f"  R:R check: achievable={achievable_rr:.2f}, min={self.min_acceptable_rr}")
                        
                        if achievable_rr >= self.min_acceptable_rr - 0.0001:
                            print(f"  >>> ENTERING TRADE!")
                            # Actually enter...
                            risk = abs(entry - stop_price)
                            self.position.direction = "LONG"
                            self.position.entry_price = entry
                            self.position.entry_time = time_str
                            self.position.entry_date = date_str
                            self.position.initial_stop = stop_price
                            self.position.current_stop = stop_price
                            self.position.stop_type = stop_type
                            self.position.risk = risk
                            self.position.target_rr = achievable_rr
                            self.orb.bars_since_last_trigger = 0
                            self.orb.long_breakout_pending = False

# Apply patch and test
bt5 = ORBBacktester('AMD', '2025-10-01', '2025-10-01', enable_confluence=True, min_confluence=2, verbose=False)
bt5.load_data()
bt5._process_day = types.MethodType(patched_process_day, bt5)
bt5._process_day('2025-10-01')
print(f"\nTrades: {len(bt5.trades)}, Position: {bt5.position.direction}")

NameError: name 'ORBBacktester' is not defined

In [10]:
# Check if real _process_day enters a position
bt6 = ORBBacktester('AMD', '2025-10-01', '2025-10-01', enable_confluence=True, min_confluence=2, verbose=False)
bt6.load_data()

# Run the REAL _process_day
bt6._process_day('2025-10-01')

# Check position state BEFORE run() calculates summary
print(f"After _process_day: Position={bt6.position.direction}, Trades={len(bt6.trades)}")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (3.7 hours old)
After _process_day: Position=, Trades=0


In [11]:
bt9 = ORBBacktester('AMD', '2025-10-01', '2025-10-01', enable_confluence=True, min_confluence=2, verbose=False)
bt9.load_data()

# Add a tracer to see what happens at EOD
original_close = bt9._close_position

def traced_close(exit_price, reason, time_str, date_str):
    print(f"[_close_position] reason={reason}, price={exit_price}, pos={bt9.position.direction}")
    return original_close(exit_price, reason, time_str, date_str)

bt9._close_position = traced_close

# Also check Position.reset if it exists
if hasattr(bt9.position, 'reset'):
    original_reset = bt9.position.reset
    def traced_reset():
        print(f"[Position.reset] CALLED! Was: {bt9.position.direction}")
        return original_reset()
    bt9.position.reset = traced_reset

bt9._process_day('2025-10-01')
print(f"\nFinal state: Position={bt9.position.direction}, Trades={len(bt9.trades)}")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (3.9 hours old)

Final state: Position=, Trades=0


In [12]:
import orb_backtester
print(f"File location: {orb_backtester.__file__}")

# Also check if there's something different about the loaded module
import inspect
source_lines = inspect.getsourcelines(ORBBacktester._process_day)
print(f"Line number of _process_day: {source_lines[1]}")
print(f"First 5 lines of _process_day:")
for line in source_lines[0][:5]:
    print(f"  {line.rstrip()}")

File location: C:\Users\phemm\orb_lab\src\orb_backtester.py
Line number of _process_day: 1037
First 5 lines of _process_day:
      def _process_day(self, date_str: str):
          """Process a single trading day"""
          day_df = self.df[self.df.index.date.astype(str) == date_str].copy()
  
          if len(day_df) == 0:


In [13]:
import inspect
source_lines = inspect.getsourcelines(ORBBacktester._process_day)

# Find the entry evaluation section
for i, line in enumerate(source_lines[0]):
    if 'ENTRY EVALUATION' in line or 'long_breakout_pending' in line:
        # Print 20 lines starting from here
        start = max(0, i-2)
        end = min(len(source_lines[0]), i+25)
        print(f"--- Lines {start}-{end} ---")
        for j in range(start, end):
            print(f"{source_lines[1]+j}: {source_lines[0][j].rstrip()}")
        print()
        break

--- Lines 81-108 ---
1118: 
1119:             if new_long and not self.orb.has_broken_out_high:
1120:                 self.orb.long_breakout_pending = True
1121:                 self.orb.has_broken_out_high = True
1122: 
1123:             if new_short and not self.orb.has_broken_out_low:
1124:                 self.orb.short_breakout_pending = True
1125:                 self.orb.has_broken_out_low = True
1126: 
1127:             # â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
1128:             # ENTRY EVALUATION (with confluence check - v70)
1129:             # â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
1130: 
1131:             if not self._in_position() and self._is_trading_window(ts) and self._can_trigger():
1132: 
1133:  

In [14]:
bt11 = ORBBacktester('AMD', '2025-10-01', '2025-10-01', enable_confluence=True, min_confluence=2, verbose=True)
bt11.load_data()

# Check if skips attribute exists
print(f"Has 'skips' attribute: {hasattr(bt11, 'skips')}")
if hasattr(bt11, 'skips'):
    print(f"skips type: {type(bt11.skips)}")

# Now run with verbose=True to see the DIAG output
bt11._process_day('2025-10-01')

print(f"\nPosition: {bt11.position.direction}")
print(f"Trades: {len(bt11.trades)}")
if hasattr(bt11, 'skips'):
    print(f"Skips: {len(bt11.skips)}")

[Presets] Loaded AMD: SSL_base=20, WAE_sens=300, QQE_rsi1=8
[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (4.1 hours old)
âœ“ Loaded 102851 bars for AMD
  [*] Calculating confluence indicators (SSL, WAE, QQE, Volume)...
  [âœ“] Confluence indicators calculated
Has 'skips' attribute: True
skips type: <class 'list'>

Position: 
Trades: 0
Skips: 0


In [15]:
# Create fresh backtester
bt12 = ORBBacktester('AMD', '2025-10-01', '2025-10-01', enable_confluence=True, min_confluence=2, verbose=False)
bt12.load_data()

# Manually walk through just the 9:44 bar to see what happens
date_str = '2025-10-01'
day_df = bt12.df[bt12.df.index.date.astype(str) == date_str].copy()

bt12.orb.reset_session()
bt12.vol.orb_session_baseline = np.nan

for i, (ts, bar) in enumerate(day_df.iterrows()):
    if not bt12._is_rth(ts):
        continue
    
    bt12.orb.bars_since_last_trigger += 1
    
    # ORB window
    if bt12._is_orb_window(ts):
        if ts.hour == 9 and ts.minute == 30:
            bt12.orb.reset_session()
            bt12.orb.session_high = bar['high']
            bt12.orb.session_low = bar['low']
        else:
            bt12.orb.session_high = max(bt12.orb.session_high, bar['high'])
            bt12.orb.session_low = min(bt12.orb.session_low, bar['low'])
        continue
    
    # ORB complete
    if not bt12.orb.orb_complete and not pd.isna(bt12.orb.session_high):
        bt12.orb.orb_complete = True
        print(f"ORB COMPLETE at {ts}")
    
    bt12._calc_vol_state(bar, ts)
    
    # Only process 9:44 bar
    if ts.hour == 9 and ts.minute == 44:
        print(f"\n=== Processing {ts} ===")
        print(f"bar['close'] = {bar['close']}")
        print(f"bar['atr_rth'] = {bar['atr_rth']}")
        
        new_long, new_short = bt12._check_breakout(bar)
        print(f"_check_breakout returned: long={new_long}, short={new_short}")
        print(f"has_broken_out_high BEFORE: {bt12.orb.has_broken_out_high}")
        
        if new_long and not bt12.orb.has_broken_out_high:
            bt12.orb.long_breakout_pending = True
            bt12.orb.has_broken_out_high = True
            print(">>> Set long_breakout_pending = True")
        
        print(f"long_breakout_pending: {bt12.orb.long_breakout_pending}")
        print(f"_in_position: {bt12._in_position()}")
        print(f"_is_trading_window: {bt12._is_trading_window(ts)}")
        print(f"_can_trigger: {bt12._can_trigger()}")
        
        break

print(f"\nFinal state: orb_complete={bt12.orb.orb_complete}, pending_L={bt12.orb.long_breakout_pending}")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (4.2 hours old)
ORB COMPLETE at 2025-10-01 09:35:00-04:00

=== Processing 2025-10-01 09:44:00-04:00 ===
bar['close'] = 162.55
bar['atr_rth'] = 0.17232939206762019
_check_breakout returned: long=True, short=False
has_broken_out_high BEFORE: False
>>> Set long_breakout_pending = True
long_breakout_pending: True
_in_position: False
_is_trading_window: True
_can_trigger: True

Final state: orb_complete=True, pending_L=True


In [16]:
bt13 = ORBBacktester('AMD', '2025-10-01', '2025-10-01', enable_confluence=True, min_confluence=2, verbose=False)
bt13.load_data()

date_str = '2025-10-01'
day_df = bt13.df[bt13.df.index.date.astype(str) == date_str].copy()

bt13.orb.reset_session()
bt13.vol.orb_session_baseline = np.nan

for i in range(len(day_df)):
    ts = day_df.index[i]
    bar = day_df.iloc[i]
    
    if not bt13._is_rth(ts):
        continue
    
    bt13.orb.bars_since_last_trigger += 1
    atr = bar['atr_rth']
    
    if bt13._is_orb_window(ts):
        if ts.hour == 9 and ts.minute == 30:
            bt13.orb.reset_session()
            bt13.orb.session_high = bar['high']
            bt13.orb.session_low = bar['low']
        else:
            bt13.orb.session_high = max(bt13.orb.session_high, bar['high'])
            bt13.orb.session_low = min(bt13.orb.session_low, bar['low'])
        continue
    
    if not bt13.orb.orb_complete and not pd.isna(bt13.orb.session_high):
        bt13.orb.orb_complete = True
    
    bt13._calc_vol_state(bar, ts)
    
    new_long, new_short = bt13._check_breakout(bar)
    
    if new_long and not bt13.orb.has_broken_out_high:
        bt13.orb.long_breakout_pending = True
        bt13.orb.has_broken_out_high = True
    
    # Only check 9:44
    if ts.hour == 9 and ts.minute == 44:
        print(f"At {ts}: pending_L={bt13.orb.long_breakout_pending}")
        
        if bt13.orb.long_breakout_pending:
            try:
                entry = bar['close']
                vwap = bar['vwap']
                print(f"  entry={entry}, vwap={vwap}")
                
                full_idx = bt13.df.index.get_loc(ts)
                print(f"  full_idx={full_idx}")
                
                passes_conf, conf_score, conf_details = bt13.confluence_calc.check_confluence(
                    full_idx, is_long=True, orb_breakout=True
                )
                print(f"  passes_conf={passes_conf}, score={conf_score}")
                
                if passes_conf:
                    stops = bt13._calc_stops(entry, atr, vwap, True, day_df, i)
                    print(f"  stops={stops}")
                    
                    stop_price, stop_type, achievable_rr = bt13._select_best_stop(stops, entry, atr, bt13.profit_target_rr, True)
                    print(f"  stop_price={stop_price}, achievable_rr={achievable_rr}")
                    
                    if achievable_rr >= bt13.min_acceptable_rr - 0.0001:
                        print(f"  >>> WOULD ENTER TRADE!")
                        bt13.position.direction = "LONG"
                        bt13.position.entry_price = entry
                    else:
                        print(f"  >>> RR too low")
                        
            except Exception as e:
                print(f"  !!! EXCEPTION: {type(e).__name__}: {e}")
                import traceback
                traceback.print_exc()
        
        break

print(f"\nFinal: Position={bt13.position.direction}")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (4.2 hours old)
At 2025-10-01 09:44:00-04:00: pending_L=True
  entry=162.55, vwap=161.56986252436712
  full_idx=38791
  passes_conf=True, score=5
  stops={'atr_stop': np.float64(162.20534121586476), 'swing_stop': np.float64(160.87665341215865), 'vwap_stop': np.float64(161.51816370674683), 'swing_valid': np.True_, 'vwap_valid': np.True_}
  stop_price=162.20534121586476, achievable_rr=1.4999999999999658
  >>> WOULD ENTER TRADE!

Final: Position=LONG


In [2]:
from orb_backtester import ORBBacktester
bt = ORBBacktester('AMD', '2025-10-01', '2025-10-01', enable_confluence=True, min_confluence=2)
bt.load_data()
bt._process_day('2025-10-01')
print(f"Position: {bt.position.direction}, Trades: {len(bt.trades)}")

ModuleNotFoundError: No module named 'orb_backtester'

In [3]:
# Complete diagnostic - no file editing needed
import numpy as np
import pandas as pd
from orb_backtester import ORBBacktester

# Create backtester
bt = ORBBacktester('AMD', '2025-10-01', '2025-10-01', enable_confluence=True, min_confluence=2, verbose=False)
bt.load_data()

# Manually run through the day with debug output
date_str = '2025-10-01'
day_df = bt.df[bt.df.index.date.astype(str) == date_str].copy()

bt.orb.reset_session()
bt.vol.orb_session_baseline = np.nan

for i in range(len(day_df)):
    ts = day_df.index[i]
    bar = day_df.iloc[i]
    
    if not bt._is_rth(ts):
        continue
    
    time_str = ts.strftime('%H:%M')
    bt.orb.bars_since_last_trigger += 1
    atr = bar['atr_rth']
    ema = bar['ema9']
    
    # ORB Building
    if bt._is_orb_window(ts):
        if ts.hour == 9 and ts.minute == 30:
            bt.orb.reset_session()
            bt.orb.session_high = bar['high']
            bt.orb.session_low = bar['low']
        else:
            bt.orb.session_high = max(bt.orb.session_high, bar['high'])
            bt.orb.session_low = min(bt.orb.session_low, bar['low'])
        continue
    
    if not bt.orb.orb_complete and not pd.isna(bt.orb.session_high):
        bt.orb.orb_complete = True
    
    bt._calc_vol_state(bar, ts)
    
    # Breakout detection
    new_long, new_short = bt._check_breakout(bar)
    
    if new_long and not bt.orb.has_broken_out_high:
        bt.orb.long_breakout_pending = True
        bt.orb.has_broken_out_high = True
        print(f"{ts} BREAKOUT LONG DETECTED")
    
    # Entry evaluation - FULL LOGIC
    if not bt._in_position() and bt._is_trading_window(ts) and bt._can_trigger():
        if bt.orb.long_breakout_pending:
            entry = bar['close']
            vwap = bar['vwap']
            
            full_idx = bt.df.index.get_loc(ts)
            passes_conf, conf_score, conf_details = bt.confluence_calc.check_confluence(
                full_idx, is_long=True, orb_breakout=True
            )
            
            if passes_conf:
                stops = bt._calc_stops(entry, atr, vwap, True, day_df, i)
                stop_price, stop_type, achievable_rr = bt._select_best_stop(stops, entry, atr, bt.profit_target_rr, True)
                
                if achievable_rr >= bt.min_acceptable_rr - 0.0001:
                    # ENTER THE TRADE
                    risk = abs(entry - stop_price)
                    bt.position.direction = "LONG"
                    bt.position.entry_price = entry
                    bt.position.entry_time = time_str
                    bt.position.entry_date = date_str
                    bt.position.initial_stop = stop_price
                    bt.position.current_stop = stop_price
                    bt.position.stop_type = stop_type
                    bt.position.risk = risk
                    bt.orb.long_breakout_pending = False
                    print(f"{ts} ENTERED LONG @ {entry:.2f}, stop={stop_price:.2f}")
    
    # Exit check
    if bt._in_position():
        if bt._check_stop_hit(bar):
            print(f"{ts} STOP HIT")
            bt._close_position(bt.position.current_stop, "STOP", time_str, date_str)

# End of day - close position
if bt._in_position():
    last_bar = day_df[day_df.index.map(bt._is_rth)].iloc[-1]
    bt._close_position(last_bar['close'], "END OF DAY", "16:00", date_str)
    print(f"END OF DAY: Closed position")

print(f"\n=== RESULTS ===")
print(f"Position: {bt.position.direction}")
print(f"Trades: {len(bt.trades)}")
for t in bt.trades:
    print(f"  {t.direction} {t.entry_price:.2f} -> {t.exit_price:.2f} ({t.exit_reason}) R={t.r_multiple:.2f}")"

SyntaxError: unterminated string literal (detected at line 98) (2224298628.py, line 98)

In [4]:
# Complete diagnostic - no file editing needed
import numpy as np
import pandas as pd
from orb_backtester import ORBBacktester

# Create backtester
bt = ORBBacktester('AMD', '2025-10-01', '2025-10-01', enable_confluence=True, min_confluence=2, verbose=False)
bt.load_data()

# Manually run through the day with debug output
date_str = '2025-10-01'
day_df = bt.df[bt.df.index.date.astype(str) == date_str].copy()

bt.orb.reset_session()
bt.vol.orb_session_baseline = np.nan

for i in range(len(day_df)):
    ts = day_df.index[i]
    bar = day_df.iloc[i]
    
    if not bt._is_rth(ts):
        continue
    
    time_str = ts.strftime('%H:%M')
    bt.orb.bars_since_last_trigger += 1
    atr = bar['atr_rth']
    ema = bar['ema9']
    
    # ORB Building
    if bt._is_orb_window(ts):
        if ts.hour == 9 and ts.minute == 30:
            bt.orb.reset_session()
            bt.orb.session_high = bar['high']
            bt.orb.session_low = bar['low']
        else:
            bt.orb.session_high = max(bt.orb.session_high, bar['high'])
            bt.orb.session_low = min(bt.orb.session_low, bar['low'])
        continue
    
    if not bt.orb.orb_complete and not pd.isna(bt.orb.session_high):
        bt.orb.orb_complete = True
    
    bt._calc_vol_state(bar, ts)
    
    # Breakout detection
    new_long, new_short = bt._check_breakout(bar)
    
    if new_long and not bt.orb.has_broken_out_high:
        bt.orb.long_breakout_pending = True
        bt.orb.has_broken_out_high = True
        print(f"{ts} BREAKOUT LONG DETECTED")
    
    # Entry evaluation - FULL LOGIC
    if not bt._in_position() and bt._is_trading_window(ts) and bt._can_trigger():
        if bt.orb.long_breakout_pending:
            entry = bar['close']
            vwap = bar['vwap']
            
            full_idx = bt.df.index.get_loc(ts)
            passes_conf, conf_score, conf_details = bt.confluence_calc.check_confluence(
                full_idx, is_long=True, orb_breakout=True
            )
            
            if passes_conf:
                stops = bt._calc_stops(entry, atr, vwap, True, day_df, i)
                stop_price, stop_type, achievable_rr = bt._select_best_stop(stops, entry, atr, bt.profit_target_rr, True)
                
                if achievable_rr >= bt.min_acceptable_rr - 0.0001:
                    # ENTER THE TRADE
                    risk = abs(entry - stop_price)
                    bt.position.direction = "LONG"
                    bt.position.entry_price = entry
                    bt.position.entry_time = time_str
                    bt.position.entry_date = date_str
                    bt.position.initial_stop = stop_price
                    bt.position.current_stop = stop_price
                    bt.position.stop_type = stop_type
                    bt.position.risk = risk
                    bt.orb.long_breakout_pending = False
                    print(f"{ts} ENTERED LONG @ {entry:.2f}, stop={stop_price:.2f}")
    
    # Exit check
    if bt._in_position():
        if bt._check_stop_hit(bar):
            print(f"{ts} STOP HIT")
            bt._close_position(bt.position.current_stop, "STOP", time_str, date_str)

# End of day - close position
if bt._in_position():
    last_bar = day_df[day_df.index.map(bt._is_rth)].iloc[-1]
    bt._close_position(last_bar['close'], "END OF DAY", "16:00", date_str)
    print(f"END OF DAY: Closed position")

print(f"\n=== RESULTS ===")
print(f"Position: {bt.position.direction}")
print(f"Trades: {len(bt.trades)}")
for t in bt.trades:
    print(f"  {t.direction} {t.entry_price:.2f} -> {t.exit_price:.2f} ({t.exit_reason}) R={t.r_multiple:.2f}")

ModuleNotFoundError: No module named 'orb_backtester'

In [5]:
# Complete diagnostic - no file editing needed
import numpy as np
import pandas as pd
from orb_backtester import ORBBacktester

# Create backtester
bt = ORBBacktester('AMD', '2025-10-01', '2025-10-01', enable_confluence=True, min_confluence=2, verbose=False)
bt.load_data()

# Manually run through the day with debug output
date_str = '2025-10-01'
day_df = bt.df[bt.df.index.date.astype(str) == date_str].copy()

bt.orb.reset_session()
bt.vol.orb_session_baseline = np.nan

for i in range(len(day_df)):
    ts = day_df.index[i]
    bar = day_df.iloc[i]
    
    if not bt._is_rth(ts):
        continue
    
    time_str = ts.strftime('%H:%M')
    bt.orb.bars_since_last_trigger += 1
    atr = bar['atr_rth']
    ema = bar['ema9']
    
    # ORB Building
    if bt._is_orb_window(ts):
        if ts.hour == 9 and ts.minute == 30:
            bt.orb.reset_session()
            bt.orb.session_high = bar['high']
            bt.orb.session_low = bar['low']
        else:
            bt.orb.session_high = max(bt.orb.session_high, bar['high'])
            bt.orb.session_low = min(bt.orb.session_low, bar['low'])
        continue
    
    if not bt.orb.orb_complete and not pd.isna(bt.orb.session_high):
        bt.orb.orb_complete = True
    
    bt._calc_vol_state(bar, ts)
    
    # Breakout detection
    new_long, new_short = bt._check_breakout(bar)
    
    if new_long and not bt.orb.has_broken_out_high:
        bt.orb.long_breakout_pending = True
        bt.orb.has_broken_out_high = True
        print(f"{ts} BREAKOUT LONG DETECTED")
    
    # Entry evaluation - FULL LOGIC
    if not bt._in_position() and bt._is_trading_window(ts) and bt._can_trigger():
        if bt.orb.long_breakout_pending:
            entry = bar['close']
            vwap = bar['vwap']
            
            full_idx = bt.df.index.get_loc(ts)
            passes_conf, conf_score, conf_details = bt.confluence_calc.check_confluence(
                full_idx, is_long=True, orb_breakout=True
            )
            
            if passes_conf:
                stops = bt._calc_stops(entry, atr, vwap, True, day_df, i)
                stop_price, stop_type, achievable_rr = bt._select_best_stop(stops, entry, atr, bt.profit_target_rr, True)
                
                if achievable_rr >= bt.min_acceptable_rr - 0.0001:
                    # ENTER THE TRADE
                    risk = abs(entry - stop_price)
                    bt.position.direction = "LONG"
                    bt.position.entry_price = entry
                    bt.position.entry_time = time_str
                    bt.position.entry_date = date_str
                    bt.position.initial_stop = stop_price
                    bt.position.current_stop = stop_price
                    bt.position.stop_type = stop_type
                    bt.position.risk = risk
                    bt.orb.long_breakout_pending = False
                    print(f"{ts} ENTERED LONG @ {entry:.2f}, stop={stop_price:.2f}")
    
    # Exit check
    if bt._in_position():
        if bt._check_stop_hit(bar):
            print(f"{ts} STOP HIT")
            bt._close_position(bt.position.current_stop, "STOP", time_str, date_str)

# End of day - close position
if bt._in_position():
    last_bar = day_df[day_df.index.map(bt._is_rth)].iloc[-1]
    bt._close_position(last_bar['close'], "END OF DAY", "16:00", date_str)
    print(f"END OF DAY: Closed position")

print(f"\n=== RESULTS ===")
print(f"Position: {bt.position.direction}")
print(f"Trades: {len(bt.trades)}")
for t in bt.trades:
    print(f"  {t.direction} {t.entry_price:.2f} -> {t.exit_price:.2f} ({t.exit_reason}) R={t.r_multiple:.2f}")

ModuleNotFoundError: No module named 'orb_backtester'

In [6]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')

import numpy as np
import pandas as pd
from orb_backtester import ORBBacktester

# Create backtester
bt = ORBBacktester('AMD', '2025-10-01', '2025-10-01', enable_confluence=True, min_confluence=2, verbose=False)
bt.load_data()

# Manually run through the day with debug output
date_str = '2025-10-01'
day_df = bt.df[bt.df.index.date.astype(str) == date_str].copy()

bt.orb.reset_session()
bt.vol.orb_session_baseline = np.nan

for i in range(len(day_df)):
    ts = day_df.index[i]
    bar = day_df.iloc[i]
    
    if not bt._is_rth(ts):
        continue
    
    time_str = ts.strftime('%H:%M')
    bt.orb.bars_since_last_trigger += 1
    atr = bar['atr_rth']
    ema = bar['ema9']
    
    # ORB Building
    if bt._is_orb_window(ts):
        if ts.hour == 9 and ts.minute == 30:
            bt.orb.reset_session()
            bt.orb.session_high = bar['high']
            bt.orb.session_low = bar['low']
        else:
            bt.orb.session_high = max(bt.orb.session_high, bar['high'])
            bt.orb.session_low = min(bt.orb.session_low, bar['low'])
        continue
    
    if not bt.orb.orb_complete and not pd.isna(bt.orb.session_high):
        bt.orb.orb_complete = True
    
    bt._calc_vol_state(bar, ts)
    
    # Breakout detection
    new_long, new_short = bt._check_breakout(bar)
    
    if new_long and not bt.orb.has_broken_out_high:
        bt.orb.long_breakout_pending = True
        bt.orb.has_broken_out_high = True
        print(f"{ts} BREAKOUT LONG DETECTED")
    
    # Entry evaluation - FULL LOGIC
    if not bt._in_position() and bt._is_trading_window(ts) and bt._can_trigger():
        if bt.orb.long_breakout_pending:
            entry = bar['close']
            vwap = bar['vwap']
            
            full_idx = bt.df.index.get_loc(ts)
            passes_conf, conf_score, conf_details = bt.confluence_calc.check_confluence(
                full_idx, is_long=True, orb_breakout=True
            )
            
            if passes_conf:
                stops = bt._calc_stops(entry, atr, vwap, True, day_df, i)
                stop_price, stop_type, achievable_rr = bt._select_best_stop(stops, entry, atr, bt.profit_target_rr, True)
                
                if achievable_rr >= bt.min_acceptable_rr - 0.0001:
                    # ENTER THE TRADE
                    risk = abs(entry - stop_price)
                    bt.position.direction = "LONG"
                    bt.position.entry_price = entry
                    bt.position.entry_time = time_str
                    bt.position.entry_date = date_str
                    bt.position.initial_stop = stop_price
                    bt.position.current_stop = stop_price
                    bt.position.stop_type = stop_type
                    bt.position.risk = risk
                    bt.orb.long_breakout_pending = False
                    print(f"{ts} ENTERED LONG @ {entry:.2f}, stop={stop_price:.2f}")
    
    # Exit check
    if bt._in_position():
        if bt._check_stop_hit(bar):
            print(f"{ts} STOP HIT")
            bt._close_position(bt.position.current_stop, "STOP", time_str, date_str)

# End of day - close position
if bt._in_position():
    last_bar = day_df[day_df.index.map(bt._is_rth)].iloc[-1]
    bt._close_position(last_bar['close'], "END OF DAY", "16:00", date_str)
    print(f"END OF DAY: Closed position")

print(f"\n=== RESULTS ===")
print(f"Position: {bt.position.direction}")
print(f"Trades: {len(bt.trades)}")
for t in bt.trades:
    print(f"  {t.direction} {t.entry_price:.2f} -> {t.exit_price:.2f} ({t.exit_reason}) R={t.r_multiple:.2f}")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (4.5 hours old)
2025-10-01 09:44:00-04:00 BREAKOUT LONG DETECTED
2025-10-01 09:44:00-04:00 ENTERED LONG @ 162.55, stop=162.21
2025-10-01 09:44:00-04:00 STOP HIT

=== RESULTS ===
Position: 
Trades: 1
  LONG 162.55 -> 162.21 (STOP) R=-1.00


In [7]:
import sys
import importlib

# Remove any cached modules
if 'orb_backtester' in sys.modules:
    del sys.modules['orb_backtester']
if 'confluence_indicators' in sys.modules:
    del sys.modules['confluence_indicators']

# Clear bytecode cache
import orb_backtester
print(f"File loaded: {orb_backtester.__file__}")

# Check line 1133
import inspect
lines = inspect.getsource(orb_backtester.ORBBacktester._process_day).split('\n')
for i, line in enumerate(lines):
    if 'long_breakout_pending' in line:
        print(f"Line {i}: {line}")

File loaded: C:\Users\phemm\orb_lab\src\orb_backtester.py
Line 83:                 self.orb.long_breakout_pending = True
Line 96:                 if self.orb.long_breakout_pending:
Line 138:                             self.orb.long_breakout_pending = False
Line 179:                             self.orb.long_breakout_pending = False  # Clear pending on entry
Line 190:                             self.orb.long_breakout_pending = False  # Pine consumes pending on SKIP too!
Line 298:                     self.orb.long_breakout_pending = False


In [8]:
import sys
if 'orb_backtester' in sys.modules:
    del sys.modules['orb_backtester']

sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
from orb_backtester import ORBBacktester

bt = ORBBacktester('AMD', '2025-10-01', '2025-10-15', enable_confluence=True, min_confluence=2, verbose=False)
results = bt.run()
print(f"Total trades: {results['total_trades']}")
for t in results['trades']:
    print(f"  {t['date']} {t['direction']} @ {t['entry_price']:.2f} -> {t['exit_reason']} R={t['r_multiple']:.2f}")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ⚠ Cache stale (8 hours), refreshing...
  ✓ Fetched 102949 bars, cached for next time
Total trades: 0


In [9]:
import os
import shutil

# Delete pycache
cache_path = r'C:\Users\phemm\orb_lab\src\__pycache__'
if os.path.exists(cache_path):
    shutil.rmtree(cache_path)
    print("Deleted __pycache__")
else:
    print("No __pycache__ found")

# Now force reimport
import sys
for mod in list(sys.modules.keys()):
    if 'orb' in mod or 'confluence' in mod:
        del sys.modules[mod]
        print(f"Removed {mod} from cache")

# Fresh import
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
from orb_backtester import ORBBacktester

bt = ORBBacktester('AMD', '2025-10-01', '2025-10-15', enable_confluence=True, min_confluence=2, verbose=False)
results = bt.run()
print(f"\nTotal trades: {results['total_trades']}")

Deleted __pycache__
Removed confluence_indicators from cache
Removed orb_backtester from cache
[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (0.0 hours old)

Total trades: 0


In [10]:
import os
import shutil
import sys

# Delete pycache
cache_path = r'C:\Users\phemm\orb_lab\src\__pycache__'
if os.path.exists(cache_path):
    shutil.rmtree(cache_path)
    print("Deleted __pycache__")

# Clear module cache
for mod in list(sys.modules.keys()):
    if 'orb' in mod or 'confluence' in mod:
        del sys.modules[mod]

# Fresh import and test
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
from orb_backtester import ORBBacktester

bt = ORBBacktester('AMD', '2025-10-01', '2025-10-15', enable_confluence=True, min_confluence=2, verbose=False)
results = bt.run()
print(f"\nTotal trades: {results['total_trades']}")
for t in results['trades']:
    print(f"  {t['date']} {t['direction']} @ {t['entry_price']:.2f} -> {t['exit_reason']} R={t['r_multiple']:.2f}")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (0.3 hours old)

Total trades: 17
  2025-10-01 LONG @ 162.55 -> INITIAL STOP R=-1.00
  2025-10-01 LONG @ 162.60 -> BREAK-EVEN STOP R=0.00
  2025-10-02 LONG @ 169.90 -> INITIAL STOP R=-1.00
  2025-10-02 SHORT @ 166.67 -> INITIAL STOP R=-1.00
  2025-10-03 SHORT @ 169.49 -> BREAK-EVEN STOP R=0.00
  2025-10-03 SHORT @ 169.10 -> BREAK-EVEN STOP R=0.00
  2025-10-06 SHORT @ 213.90 -> INITIAL STOP R=-1.00
  2025-10-06 SHORT @ 213.90 -> EMA EXIT R=1.32
  2025-10-07 SHORT @ 213.98 -> INITIAL STOP R=-1.00
  2025-10-08 LONG @ 213.82 -> BREAK-EVEN STOP R=0.00
  2025-10-10 SHORT @ 230.37 -> EMA EXIT R=1.11
  2025-10-13 SHORT @ 218.94 -> INITIAL STOP R=-1.00
  2025-10-13 LONG @ 222.23 -> INITIAL STOP R=-1.00
  2025-10-13 SHORT @ 219.19 -> BREAK-EVEN STOP R=0.00
  2025-10-14 SHORT @ 216.43 -> INITIAL STOP R=-1.00
  2025-10-14 LONG @ 221.88 -> INITIAL STOP R=-1.00
  2

In [11]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
from orb_backtester import ORBBacktester

bt = ORBBacktester(
    symbol='AMD',
    start_date='2025-10-01',
    end_date='2025-12-01',
    enable_confluence=True,
    min_confluence=2,
    verbose=False
)
results = bt.run()

print(f"=== AMD 60-Day Backtest (min_confluence=2) ===")
print(f"Total Trades: {results['total_trades']}")
print(f"Win Rate: {results['win_rate']:.1f}%")
print(f"Total R: {results['total_r']:.2f}")
print(f"Avg R/Trade: {results['avg_r_per_trade']:.3f}")
print(f"Profit Factor: {results['profit_factor']:.2f}")
print(f"Max Drawdown: {results['max_drawdown_r']:.2f}R")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ⚠ Cache stale (11 hours), refreshing...
  ✓ Fetched 102949 bars, cached for next time
=== AMD 60-Day Backtest (min_confluence=2) ===
Total Trades: 64
Win Rate: 47.5%
Total R: -4.05
Avg R/Trade: -0.063
Profit Factor: 0.90
Max Drawdown: 9.48R


In [12]:
# Compare different confluence thresholds
for min_conf in [2, 3, 4]:
    bt = ORBBacktester(
        symbol='AMD',
        start_date='2025-10-01',
        end_date='2025-12-01',
        enable_confluence=True,
        min_confluence=min_conf,
        verbose=False
    )
    results = bt.run()
    print(f"min_confluence={min_conf}: {results['total_trades']} trades, {results['win_rate']:.1f}% WR, {results['total_r']:+.2f}R, PF={results['profit_factor']:.2f}")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (0.1 hours old)
min_confluence=2: 64 trades, 47.5% WR, -4.05R, PF=0.90
[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (0.1 hours old)
min_confluence=3: 62 trades, 47.5% WR, -4.05R, PF=0.90
[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (0.1 hours old)
min_confluence=4: 45 trades, 56.7% WR, +2.18R, PF=1.11


In [13]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
sys.path.insert(0, r'C:\hmm_trading\scripts')

from orb_optimizer import ORBOptimizer

opt = ORBOptimizer(
    symbol='AMD',
    days=120,
    verbose=True
)

results = opt.run_optuna(n_trials=10)
print(f"\nBest score: {results['best_score']:.3f}")
print(f"Best params: {results['best_params']}")

TypeError: ORBOptimizer.__init__() got an unexpected keyword argument 'days'

In [14]:
import sys
sys.path.insert(0, r'C:\hmm_trading\scripts')
from orb_optimizer import ORBOptimizer
import inspect

print(inspect.signature(ORBOptimizer.__init__))

(self, symbol: str, start_date: str, end_date: str, param_grid: Dict = None, train_ratio: float = 0.7, min_trades_train: int = 5, min_trades_test: int = 3, verbose: bool = True)


In [15]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
sys.path.insert(0, r'C:\hmm_trading\scripts')

from orb_optimizer import ORBOptimizer

opt = ORBOptimizer(
    symbol='AMD',
    start_date='2025-08-01',
    end_date='2025-12-01',
    verbose=True
)

results = opt.run_optuna(n_trials=10)
print(f"\nBest score: {results['best_score']:.3f}")
print(f"Best params: {results['best_params']}")

AttributeError: 'ORBOptimizer' object has no attribute 'run_optuna'

In [16]:
print([m for m in dir(opt) if not m.startswith('_')])

['best_result', 'end_date', 'get_top_n', 'min_trades_test', 'min_trades_train', 'optimize', 'optimize_optuna', 'param_grid', 'results', 'start_date', 'symbol', 'test_end', 'test_start', 'train_end', 'train_ratio', 'train_start', 'verbose']


In [17]:
results = opt.optimize_optuna(n_trials=10)
print(f"\nBest score: {results['best_score']:.3f}")
print(f"Best params: {results['best_params']}")


  OPTUNA OPTIMIZING: AMD
  Train: 2025-08-01 to 2025-10-25
  Test:  2025-10-26 to 2025-12-01
  Trials: 10

  Study stored at: C:\hmm_trading\orb_optuna.db
  Run 'optuna-dashboard sqlite:///C:\hmm_trading\orb_optuna.db' for web UI



  0%|          | 0/10 [00:00<?, ?it/s]

  [Trial 25] Testing: SSL=25, WAE_sens=225, min_conf=2
[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (0.6 hours old)
    Train trades: 77
[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (0.6 hours old)
    Test trades: 33
  ★ Trial 25: New best! Score=0.267, WR=61.1%, PF=1.35, Trades=33
[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (0.6 hours old)
[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (0.7 hours old)
[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (0.7 hours old)
[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching

TypeError: 'OptimizationResult' object is not subscriptable

In [18]:
optuna-dashboard sqlite:///C:\hmm_trading\orb_optuna.db

SyntaxError: invalid syntax (1048147449.py, line 1)

In [1]:
import optuna

# Create fresh study with new file
storage = "sqlite:///C:/hmm_trading/orb_optuna_fresh.db"
study = optuna.create_study(
    study_name="ORB_AMD_v2",
    storage=storage,
    direction="maximize"
)
print("Fresh database created!")

[I 2026-01-24 13:20:14,668] A new study created in RDB with name: ORB_AMD_v2


Fresh database created!


In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
sys.path.insert(0, r'C:\hmm_trading\scripts')

from orb_optimizer import ORBOptimizer

opt = ORBOptimizer(
    symbol='AMD',
    start_date='2025-08-01',
    end_date='2025-12-01',
    verbose=True
)

results = opt.optimize_optuna(n_trials=50)


  OPTUNA OPTIMIZING: AMD
  Train: 2025-08-01 to 2025-10-25
  Test:  2025-10-26 to 2025-12-01
  Trials: 50

  Study stored at: C:\hmm_trading\orb_optuna.db
  Run 'optuna-dashboard sqlite:///C:\hmm_trading\orb_optuna.db' for web UI



  0%|          | 0/50 [00:00<?, ?it/s]

  [Trial 0] Testing: SSL=15, WAE_sens=325, min_conf=2
[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (1.5 hours old)
    Train trades: 77
[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (1.5 hours old)
    Test trades: 35
  ★ Trial 0: New best! Score=0.181, WR=55.6%, PF=0.93, Trades=35
[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (1.5 hours old)
[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (1.5 hours old)
[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (1.6 hours old)
[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching A

In [2]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
sys.path.insert(0, r'C:\hmm_trading\scripts')

from orb_optimizer_v3 import ORBOptimizer

opt = ORBOptimizer(
    symbol='AMD',
    start_date='2025-08-01',
    end_date='2025-12-01',
    mode='preset',
    verbose=True
)

results = opt.optimize_optuna(n_trials=50)

[W 2026-01-24 15:53:09,101] Trial 0 failed with parameters: {'min_confluence': 3, 'min_acceptable_rr': 1.8, 'profit_target_rr': 2.3, 'atr_stop_mult': 2.1, 'break_even_rr': 0.3, 'trailing_activation_rr': 0.9, 'trailing_stop_distance': 0.8, 'use_ema_exit': True} because of the following error: TypeError("ORBBacktester.__init__() got an unexpected keyword argument 'trailing_activation_rr'").
Traceback (most recent call last):
  File "C:\Users\phemm\anaconda3\Lib\site-packages\optuna\study\_optimize.py", line 205, in _run_trial
    value_or_values = func(trial)
  File "C:\hmm_trading\scripts\orb_optimizer_v3.py", line 283, in objective
    train_results = self._run_backtest(params, self.train_start, self.train_end)
  File "C:\hmm_trading\scripts\orb_optimizer_v3.py", line 219, in _run_backtest
    bt = ORBBacktester(
        symbol=self.symbol,
    ...<5 lines>...
        **params
    )
TypeError: ORBBacktester.__init__() got an unexpected keyword argument 'trailing_activation_rr'
[W 2026-


  ORB Optimizer v3 - AMD
  Mode: PRESET
  Train: 2025-08-01 to 2025-10-25
  Test:  2025-10-26 to 2025-12-01
  Storage: sqlite:///C:/hmm_trading/orb_optuna.db

  Study: ORB_AMD_preset
  Dashboard: optuna-dashboard sqlite:///C:/hmm_trading/orb_optuna.db
  Running 50 trials...

  [Trial 0] min_conf=3, RR=1.8-2.3


TypeError: ORBBacktester.__init__() got an unexpected keyword argument 'trailing_activation_rr'

In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
sys.path.insert(0, r'C:\hmm_trading\scripts')

from orb_optimizer_v3 import ORBOptimizer

opt = ORBOptimizer(
    symbol='AMD',
    start_date='2025-08-01',
    end_date='2025-12-01',
    mode='preset',
    verbose=True
)

results = opt.optimize_optuna(n_trials=50)


  ORB Optimizer v3 - AMD
  Mode: PRESET
  Train: 2025-08-01 to 2025-10-25
  Test:  2025-10-26 to 2025-12-01
  Storage: sqlite:///C:/hmm_trading/orb_optuna.db



[I 2026-01-24 15:53:45,914] Using an existing study with name 'ORB_AMD_preset' instead of creating a new one.
[W 2026-01-24 15:53:46,003] Trial 1 failed with parameters: {'min_confluence': 3, 'min_acceptable_rr': 1.8, 'profit_target_rr': 2.3, 'atr_stop_mult': 2.1, 'break_even_rr': 0.3, 'trailing_activation_rr': 0.9, 'trailing_stop_distance': 0.8, 'use_ema_exit': True} because of the following error: TypeError("ORBBacktester.__init__() got an unexpected keyword argument 'trailing_activation_rr'").
Traceback (most recent call last):
  File "C:\Users\phemm\anaconda3\Lib\site-packages\optuna\study\_optimize.py", line 205, in _run_trial
    value_or_values = func(trial)
  File "C:\hmm_trading\scripts\orb_optimizer_v3.py", line 283, in objective
    train_results = self._run_backtest(params, self.train_start, self.train_end)
  File "C:\hmm_trading\scripts\orb_optimizer_v3.py", line 219, in _run_backtest
    bt = ORBBacktester(
        symbol=self.symbol,
    ...<5 lines>...
        **params


  Study: ORB_AMD_preset
  Dashboard: optuna-dashboard sqlite:///C:/hmm_trading/orb_optuna.db
  Running 50 trials...

  [Trial 1] min_conf=3, RR=1.8-2.3


TypeError: ORBBacktester.__init__() got an unexpected keyword argument 'trailing_activation_rr'

In [2]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
from orb_backtester import ORBBacktester
import inspect

sig = inspect.signature(ORBBacktester.__init__)
print("Valid parameters:")
for name, param in sig.parameters.items():
    if name != 'self':
        print(f"  {name}: {param.default}")

Valid parameters:
  symbol: AMD
  start_date: 2025-10-01
  end_date: 2025-12-31
  orb_minutes: 5
  breakout_threshold_mult: 0.1
  min_body_strength: 0.5
  require_open_in_range: True
  atr_period: 14
  atr_stop_mult: 2.0
  swing_lookback: 5
  swing_buffer: 0.02
  vwap_stop_distance: 0.3
  min_stop_atr: 0.15
  max_target_atr: 3.0
  min_acceptable_rr: 1.5
  profit_target_rr: 2.0
  enable_confluence: True
  min_confluence: 5
  confluence_mode: all
  ssl_baseline_length: 45
  ssl_length: 5
  ssl_type: JMA
  use_ssl_momentum: True
  wae_fast_ema: 20
  wae_slow_ema: 40
  wae_sensitivity: 190
  wae_bb_length: 20
  wae_bb_mult: 2.0
  use_wae_acceleration: True
  qqe_rsi1_length: 6
  qqe_rsi1_smoothing: 5
  qqe_factor_primary: 3.0
  qqe_rsi2_length: 6
  qqe_rsi2_smoothing: 5
  qqe_factor_secondary: 1.61
  qqe_bb_length: 50
  qqe_bb_mult: 0.35
  qqe_threshold: 3
  qqe_consecutive_bars: 3
  use_qqe_momentum: True
  vol_lookback: 5
  use_break_even: True
  break_even_rr: 0.5
  use_trailing_stop: T

In [2]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
sys.path.insert(0, r'C:\hmm_trading\scripts')

from orb_optimizer_v3 import ORBOptimizer

opt = ORBOptimizer(
    symbol='AMD',
    start_date='2025-08-01',
    end_date='2025-12-01',
    mode='preset',
    verbose=True
)

results = opt.optimize_optuna(n_trials=50)


  ORB Optimizer v3 - AMD
  Mode: PRESET
  Train: 2025-08-01 to 2025-10-25
  Test:  2025-10-26 to 2025-12-01
  Storage: sqlite:///C:/hmm_trading/orb_optuna.db



[I 2026-01-24 16:04:25,168] Using an existing study with name 'ORB_AMD_preset' instead of creating a new one.


  Study: ORB_AMD_preset
  Dashboard: optuna-dashboard sqlite:///C:/hmm_trading/orb_optuna.db
  Running 50 trials...

  [Trial 3] min_conf=3, RR=1.8-2.3
[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (4.1 hours old)


[I 2026-01-24 16:05:14,497] Trial 3 pruned. 


    ✗ Train trades 8 < 10
  [Trial 4] min_conf=4, RR=1.2-2.5
[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (4.1 hours old)
    Train: 58 trades, WR=55.0%, PF=3.66
[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (4.1 hours old)


...

    Test: 14 trades, WR=66.7%, PF=2.12, Score=0.990
  [Trial 51] min_conf=4, RR=1.5-1.5
[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (5.1 hours old)
    Train: 11 trades, WR=85.7%, PF=7.35
[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (5.2 hours old)


[I 2026-01-24 17:10:25,467] Trial 51 finished with value: 5.35757465971326 and parameters: {'min_confluence': 4, 'min_acceptable_rr': 1.5, 'profit_target_rr': 1.5, 'atr_stop_mult': 2.5, 'break_even_rr': 0.7, 'trailing_stop_distance': 1.5, 'use_ema_exit': False, 'ema_tighten_zone': 0.4}. Best is trial 51 with value: 5.35757465971326.


    Test: 8 trades, WR=100.0%, PF=inf, Score=5.358
  ★ New best! Score=5.358
  [Trial 52] min_conf=4, RR=1.6-1.5
[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (5.2 hours old)


[I 2026-01-24 17:11:10,629] Trial 52 pruned. 


    ✗ Train trades 2 < 10

  OPTIMIZATION COMPLETE - AMD (PRESET mode)
  Completed 53 trials, 40 valid

  BEST PARAMETERS:
    ssl_baseline_length: 20
    wae_sensitivity: 300
    wae_fast_ema: 14
    wae_slow_ema: 29
    qqe_rsi1_length: 8
    vol_lookback: 12
    min_confluence: 4
    min_acceptable_rr: 1.50
    profit_target_rr: 1.50
    atr_stop_mult: 2.50
    break_even_rr: 0.70
    trailing_stop_distance: 1.50
    use_ema_exit: False
    ema_tighten_zone: 0.40

  TRAIN (2025-08-01 to 2025-10-25):
    Trades: 11
    Win Rate: 85.7%
    Profit Factor: 7.35
    Total R: +6.61
    Max Drawdown: 1.00R

  TEST (2025-10-26 to 2025-12-01):
    Trades: 8
    Win Rate: 100.0%
    Profit Factor: inf
    Total R: +11.51
    Sortino: 0.000
    Max Drawdown: 0.00R

  COMPOSITE SCORE: 5.358



In [3]:
opt_nvda = ORBOptimizer(
    symbol='NVDA',
    start_date='2025-08-01',
    end_date='2025-12-01',
    mode='preset',
    verbose=True
)

results_nvda = opt_nvda.optimize_optuna(n_trials=50)

[I 2026-01-24 17:42:39,635] A new study created in RDB with name: ORB_NVDA_preset



  ORB Optimizer v3 - NVDA
  Mode: PRESET
  Train: 2025-08-01 to 2025-10-25
  Test:  2025-10-26 to 2025-12-01
  Storage: sqlite:///C:/hmm_trading/orb_optuna.db

  Study: ORB_NVDA_preset
  Dashboard: optuna-dashboard sqlite:///C:/hmm_trading/orb_optuna.db
  Running 50 trials...

  [Trial 0] min_conf=3, RR=1.8-2.3
[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching NVDA (1-min bars, 180 days, extended)...
  ⚠ Cache stale (18 hours), refreshing...
  ✓ Fetched 118281 bars, cached for next time
    Train: 17 trades, WR=12.5%, PF=0.32
[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching NVDA (1-min bars, 180 days, extended)...
  ✓ Loading from cache (0.0 hours old)


[I 2026-01-24 17:44:25,008] Trial 0 finished with value: -2.033768322766746 and parameters: {'min_confluence': 3, 'min_acceptable_rr': 1.8, 'profit_target_rr': 2.3, 'atr_stop_mult': 2.1, 'break_even_rr': 0.3, 'trailing_stop_distance': 0.9, 'use_ema_exit': False, 'ema_tighten_zone': 0.4}. Best is trial 0 with value: -2.033768322766746.


    Test: 7 trades, WR=20.0%, PF=1.07, Score=-2.034
  ★ New best! Score=-2.034
  [Trial 1] min_conf=4, RR=1.2-2.5
[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching NVDA (1-min bars, 180 days, extended)...
  ✓ Loading from cache (0.0 hours old)
    Train: 53 trades, WR=7.1%, PF=0.22
[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching NVDA (1-min bars, 180 days, extended)...
  ✓ Loading from cache (0.0 hours old)


[I 2026-01-24 17:46:02,518] Trial 1 pruned. 


...

    ✗ Train trades 5 < 10
  [Trial 48] min_conf=3, RR=1.6-2.0
[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching NVDA (1-min bars, 180 days, extended)...
  ✓ Loading from cache (1.1 hours old)
    Train: 23 trades, WR=23.5%, PF=0.37
[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching NVDA (1-min bars, 180 days, extended)...
  ✓ Loading from cache (1.1 hours old)


[I 2026-01-24 18:51:08,178] Trial 48 pruned. 


    ✗ Test PF 0.56 < 0.8
  [Trial 49] min_conf=3, RR=1.7-1.8
[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching NVDA (1-min bars, 180 days, extended)...
  ✓ Loading from cache (1.1 hours old)
    Train: 19 trades, WR=33.3%, PF=0.66
[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching NVDA (1-min bars, 180 days, extended)...
  ✓ Loading from cache (1.1 hours old)


[I 2026-01-24 18:52:46,269] Trial 49 finished with value: -1.590204992859232 and parameters: {'min_confluence': 3, 'min_acceptable_rr': 1.7, 'profit_target_rr': 1.8, 'atr_stop_mult': 2.0, 'break_even_rr': 0.8, 'trailing_stop_distance': 1.5, 'use_ema_exit': True, 'ema_tighten_zone': 0.30000000000000004}. Best is trial 23 with value: -1.590204992859232.


    Test: 8 trades, WR=50.0%, PF=1.21, Score=-1.590

  OPTIMIZATION COMPLETE - NVDA (PRESET mode)
  Completed 50 trials, 13 valid

  BEST PARAMETERS:
    ssl_baseline_length: 20
    wae_sensitivity: 325
    wae_fast_ema: 17
    wae_slow_ema: 35
    qqe_rsi1_length: 9
    vol_lookback: 13
    min_confluence: 3
    min_acceptable_rr: 1.70
    profit_target_rr: 2.10
    atr_stop_mult: 2.20
    break_even_rr: 0.70
    trailing_stop_distance: 1.30
    use_ema_exit: True
    ema_tighten_zone: 0.30

  TRAIN (2025-08-01 to 2025-10-25):
    Trades: 19
    Win Rate: 23.1%
    Profit Factor: 0.29
    Total R: -7.03
    Max Drawdown: 7.03R

  TEST (2025-10-26 to 2025-12-01):
    Trades: 8
    Win Rate: 50.0%
    Profit Factor: 1.21
    Total R: +0.27
    Sortino: 0.000
    Max Drawdown: 2.00R

  COMPOSITE SCORE: -1.590



In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
sys.path.insert(0, r'C:\hmm_trading\scripts')

from orb_optimizer_v3 import ORBOptimizer

opt_pltr = ORBOptimizer(
    symbol='PLTR',
    start_date='2025-08-01',
    end_date='2025-12-01',
    mode='preset',
    verbose=True
)

results_pltr = opt_pltr.optimize_optuna(n_trials=50)

SyntaxError: '{' was never closed (orb_backtester.py, line 62)

In [2]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
sys.path.insert(0, r'C:\hmm_trading\scripts')

from orb_optimizer_v3 import ORBOptimizer

opt_pltr = ORBOptimizer(
    symbol='PLTR',
    start_date='2025-08-01',
    end_date='2025-12-01',
    mode='preset',
    verbose=True
)

results_pltr = opt_pltr.optimize_optuna(n_trials=50)


  ORB Optimizer v3 - PLTR
  Mode: PRESET
  Train: 2025-08-01 to 2025-10-25
  Test:  2025-10-26 to 2025-12-01
  Storage: sqlite:///C:/hmm_trading/orb_optuna.db



[I 2026-01-25 00:14:14,032] A new study created in RDB with name: ORB_PLTR_preset


  Study: ORB_PLTR_preset
  Dashboard: optuna-dashboard sqlite:///C:/hmm_trading/orb_optuna.db
  Running 50 trials...

  [Trial 0] min_conf=3, RR=1.8-2.3
[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching PLTR (1-min bars, 180 days, extended)...
  ⚠ Cache stale (13 hours), refreshing...
  ✓ Fetched 103903 bars, cached for next time


[I 2026-01-25 00:15:07,865] Trial 0 pruned. 


    ✗ Train trades 8 < 10
  [Trial 1] min_conf=4, RR=1.2-2.5
[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching PLTR (1-min bars, 180 days, extended)...
  ✓ Loading from cache (0.0 hours old)
    Train: 47 trades, WR=30.0%, PF=1.11
[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching PLTR (1-min bars, 180 days, extended)...
  ✓ Loading from cache (0.0 hours old)


...

    ✗ Test PF 0.00 < 0.8
  [Trial 48] min_conf=2, RR=1.3-2.3
[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching PLTR (1-min bars, 180 days, extended)...
  ✓ Loading from cache (0.9 hours old)
    Train: 66 trades, WR=52.9%, PF=0.78
[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching PLTR (1-min bars, 180 days, extended)...
  ✓ Loading from cache (0.9 hours old)


[I 2026-01-25 01:10:59,161] Trial 48 pruned. 


    ✗ Test PF 0.35 < 0.8
  [Trial 49] min_conf=3, RR=1.6-2.4
[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching PLTR (1-min bars, 180 days, extended)...
  ✓ Loading from cache (0.9 hours old)
    Train: 61 trades, WR=29.7%, PF=0.90
[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching PLTR (1-min bars, 180 days, extended)...
  ✓ Loading from cache (0.9 hours old)


[I 2026-01-25 01:12:29,243] Trial 49 pruned. 


    ✗ Test PF 0.00 < 0.8

  OPTIMIZATION COMPLETE - PLTR (PRESET mode)
  Completed 50 trials, 0 valid

  ✗ No valid trials found!



In [3]:
from orb_backtester import ORBBacktester

bt = ORBBacktester(
    symbol='PLTR',
    start_date='2025-08-01',
    end_date='2025-12-01',
    enable_confluence=True,
    min_confluence=2,  # Loosest setting
    verbose=True
)
r = bt.run()
print(f"\nTrades: {r['total_trades']}, WR: {r['win_rate']:.1f}%, PF: {r['profit_factor']:.2f}")

[Presets] Loaded PLTR: SSL_base=20, WAE_sens=200, QQE_rsi1=8

  ORB BACKTESTER: PLTR
  2025-08-01 to 2025-12-01

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching PLTR (1-min bars, 180 days, extended)...
  ⚠ Cache stale (10 hours), refreshing...
  ✓ Fetched 103903 bars, cached for next time
✓ Loaded 103903 bars for PLTR
  [*] Calculating confluence indicators (SSL, WAE, QQE, Volume)...
  [✓] Confluence indicators calculated
Processing 85 trading days...

  ✅ SHORT 2025-08-01 09:42→09:51: $+0.02 (+0.02R) - EMA EXIT [C:4]
  ❌ SHORT 2025-08-01 10:00→10:06: $-0.93 (-1.00R) - INITIAL STOP [C:2]
  ❌ LONG 2025-08-01 10:26→10:51: $-0.92 (-1.00R) - INITIAL STOP [C:3]
  ❌ LONG 2025-08-08 10:13→10:17: $+0.00 (+0.00R) - BREAK-EVEN STOP [C:2]
  ✅ SHORT 2025-08-13 09:55→10:19: $+0.38 (+0.43R) - EMA EXIT [C:5]
  ❌ LONG 2025-08-14 09:51→10:07: $-0.78 (-1.00R) - INITIAL STOP [C:4]
  ✅ LONG 2025-08-14 10:19→10:33: $+0.31 (+0.49R) - EMA EXIT [C:2]
  ❌ SHORT 2025-08-18 09:48→09:50:

In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')

from orb_backtester import SYMBOL_PRESETS

print("GOOGL presets in Python:")
if 'GOOGL' in SYMBOL_PRESETS:
    for k, v in SYMBOL_PRESETS['GOOGL'].items():
        print(f"  {k}: {v}")
else:
    print("  NOT FOUND!")

IndentationError: unexpected indent (orb_backtester.py, line 108)

In [2]:
%run C:\hmm_trading\scripts\enhanced_preset_generator.py

ENHANCED PRESET GENERATOR v4.3
WITH INTEGRATED CAM BEHAVIOR + WAE/QQE SYNC
Dual timeframes: ORB (1-min) + HMM/Cam (2-min)

[CACHE] Checking for stale cache files (>16 hours old)...
[CACHE] All cache files are fresh (<16 hours old)
[OK] Enhanced Preset Generator v4.2 initialized
   - Cache auto-expires after 16 hours
   - ORB analysis: 1-minute bars
   - HMM/Cam analysis: 2-minute bars
   - VF/BE/Volume (original)
   - WAE/SSL optimization (original)
   - WAE EMA lengths (Fast/Slow/BB)
   - QQE RSI lengths (RSI1/RSI2/Smoothing/BB)
   - Volume lookback bars
   - IB period minutes
   - Level respect rates
   - Confluence weights
   - ORB analysis
   - Cam pivots
   - Weekly/Monthly
   - Time patterns
   - CAM BEHAVIOR ANALYSIS (90-day lookback)
   - OUTPUT: Dual files (ORB + HMM)
[OK] Loaded 2 symbols from GUI config
[DATA] Processing symbols: PLTR, GOOGL

ANALYZING PLTR
  [*] Fetching 1-minute bars for ORB analysis...
📊 Fetching PLTR...
  ✓ Loading from cache
  [OK] Loaded 7429 bars (1-m

In [3]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')

# Force reload
import importlib
import orb_backtester
importlib.reload(orb_backtester)

from orb_backtester import SYMBOL_PRESETS

print("GOOGL:", SYMBOL_PRESETS.get('GOOGL', 'NOT FOUND'))
print("\nAll symbols:", list(SYMBOL_PRESETS.keys()))

GOOGL: {'ssl_baseline_length': 20, 'ssl_length': 10, 'wae_fast_ema': 17, 'wae_slow_ema': 35, 'wae_bb_length': 20, 'wae_sensitivity': 325, 'qqe_rsi1_length': 9, 'qqe_rsi1_smoothing': 5, 'qqe_rsi2_length': 4, 'qqe_rsi2_smoothing': 4, 'qqe_bb_length': 25, 'vol_lookback': 12, 'low_vol_threshold': 0.8, 'high_vol_threshold': 1.3, 'extreme_vol_threshold': 2.0}

All symbols: ['GOOGL', 'PLTR']


In [4]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
sys.path.insert(0, r'C:\hmm_trading\scripts')

import importlib
import orb_optimizer_v3
importlib.reload(orb_optimizer_v3)

from orb_optimizer_v3 import ORBOptimizer

opt_googl = ORBOptimizer(
    symbol='GOOGL',
    start_date='2025-08-01',
    end_date='2025-12-01',
    mode='preset',
    verbose=True
)

results_googl = opt_googl.optimize_optuna(n_trials=50)

KeyError: 'AMD'

In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
sys.path.insert(0, r'C:\hmm_trading\scripts')

from orb_optimizer_v3 import ORBOptimizer

opt_googl = ORBOptimizer(
    symbol='GOOGL',
    start_date='2025-08-01',
    end_date='2025-12-01',
    mode='preset',
    verbose=True
)

results_googl = opt_googl.optimize_optuna(n_trials=50)


  ORB Optimizer v3 - GOOGL
  Mode: PRESET
  Train: 2025-08-01 to 2025-10-25
  Test:  2025-10-26 to 2025-12-01
  Storage: sqlite:///C:/hmm_trading/orb_optuna.db



[I 2026-01-25 13:02:29,459] A new study created in RDB with name: ORB_GOOGL_preset


  Study: ORB_GOOGL_preset
  Dashboard: optuna-dashboard sqlite:///C:/hmm_trading/orb_optuna.db
  Running 50 trials...

  [Trial 0] min_conf=3, RR=1.8-2.3
[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching GOOGL (1-min bars, 180 days, extended)...
  ⚠ Cache stale (17 hours), refreshing...
  ✓ Fetched 88081 bars, cached for next time
    Train: 10 trades, WR=0.0%, PF=0.00
[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching GOOGL (1-min bars, 180 days, extended)...
  ✓ Loading from cache (0.0 hours old)


[I 2026-01-25 13:03:55,775] Trial 0 pruned. 


    ✗ Test PF 0.00 < 0.8
  [Trial 1] min_conf=4, RR=1.2-2.5
[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching GOOGL (1-min bars, 180 days, extended)...
  ✓ Loading from cache (0.0 hours old)


...

    Test: 6 trades, WR=50.0%, PF=1.50, Score=-1.832
  [Trial 48] min_conf=3, RR=1.7-1.6
[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching GOOGL (1-min bars, 180 days, extended)...
  ✓ Loading from cache (0.7 hours old)


[I 2026-01-25 13:49:36,289] Trial 48 pruned. 


    ✗ Train trades 2 < 10
  [Trial 49] min_conf=3, RR=1.8-1.8
[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching GOOGL (1-min bars, 180 days, extended)...
  ✓ Loading from cache (0.7 hours old)
    Train: 10 trades, WR=25.0%, PF=0.37
[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching GOOGL (1-min bars, 180 days, extended)...
  ✓ Loading from cache (0.7 hours old)


[I 2026-01-25 13:50:55,060] Trial 49 finished with value: -2.5810404869305956 and parameters: {'min_confluence': 3, 'min_acceptable_rr': 1.8, 'profit_target_rr': 1.8, 'atr_stop_mult': 1.9, 'break_even_rr': 0.7, 'trailing_stop_distance': 1.2000000000000002, 'use_ema_exit': True, 'ema_tighten_zone': 0.2}. Best is trial 28 with value: -1.831978599545606.


    Test: 6 trades, WR=50.0%, PF=0.91, Score=-2.581

  OPTIMIZATION COMPLETE - GOOGL (PRESET mode)
  Completed 50 trials, 18 valid

  BEST PARAMETERS:
    ssl_baseline_length: 20
    wae_sensitivity: 325
    wae_fast_ema: 17
    wae_slow_ema: 35
    qqe_rsi1_length: 9
    vol_lookback: 12
    min_confluence: 3
    min_acceptable_rr: 1.80
    profit_target_rr: 1.90
    atr_stop_mult: 1.80
    break_even_rr: 0.80
    trailing_stop_distance: 1.30
    use_ema_exit: True
    ema_tighten_zone: 0.20

  TRAIN (2025-08-01 to 2025-10-25):
    Trades: 10
    Win Rate: 25.0%
    Profit Factor: 0.33
    Total R: -4.62
    Max Drawdown: 4.62R

  TEST (2025-10-26 to 2025-12-01):
    Trades: 6
    Win Rate: 50.0%
    Profit Factor: 1.50
    Total R: +0.59
    Sortino: 0.000
    Max Drawdown: 3.00R

  COMPOSITE SCORE: -1.832



In [2]:
from orb_backtester import ORBBacktester

bt = ORBBacktester(
    symbol='GOOGL',
    start_date='2025-08-25',
    end_date='2025-12-01',
    enable_confluence=False,  # Disable confluence
    verbose=True
)
r = bt.run()
print(f"\nTrades: {r['total_trades']}, Skips: {r['total_skips']}")

[Presets] Loaded GOOGL: SSL_base=20, WAE_sens=325, QQE_rsi1=9

  ORB BACKTESTER: GOOGL
  2025-08-25 to 2025-12-01

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching GOOGL (1-min bars, 180 days, extended)...
  ✓ Loading from cache (1.1 hours old)
✓ Loaded 88081 bars for GOOGL
Processing 69 trading days...

  ❌ LONG 2025-08-25 09:38→09:45: $-0.85 (-1.00R) - INITIAL STOP
  ✅ LONG 2025-08-25 09:46→10:07: $+0.68 (+1.04R) - EMA EXIT
  ❌ SHORT 2025-08-26 09:46→09:52: $+0.00 (+0.00R) - BREAK-EVEN STOP
  ❌ SHORT 2025-08-26 09:54→09:57: $-0.31 (-1.00R) - INITIAL STOP
  ❌ SHORT 2025-08-26 10:06→10:18: $-0.54 (-1.00R) - INITIAL STOP
  ✅ LONG 2025-08-27 09:39→10:03: $+0.68 (+1.07R) - EMA EXIT
  ✅ LONG 2025-08-28 09:36→09:49: $+0.31 (+0.54R) - EMA EXIT
  ❌ LONG 2025-08-28 09:52→09:56: $+0.00 (+0.00R) - BREAK-EVEN STOP
  ❌ LONG 2025-08-29 09:38→09:50: $-0.57 (-1.00R) - INITIAL STOP
  ❌ LONG 2025-08-29 09:52→09:56: $-0.53 (-1.00R) - INITIAL STOP
  ❌ LONG 2025-08-29 10:23→10:26:

In [5]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
from orb_backtester import ORBBacktester

bt = ORBBacktester(
    symbol='GOOGL',
    start_date='2025-09-01',
    end_date='2025-11-30',
    enable_confluence=False,
    verbose=True
)
results = bt.run()

# Quick comparison
print(f"\nPython: {results['wins']}W / {results['losses']}L / {results['breakevens']}BE")
print(f"Win Rate: {results['win_rate']:.1f}%")
print(f"Total R: {results['total_r']:+.2f}R")

[Presets] Loaded GOOGL: SSL_base=20, WAE_sens=325, QQE_rsi1=9

  ORB BACKTESTER: GOOGL
  2025-09-01 to 2025-11-30

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching GOOGL (1-min bars, 180 days, extended)...
  ✓ Loading from cache (3.5 hours old)
✓ Loaded 88081 bars for GOOGL
Processing 63 trading days...

  ❌ LONG 2025-09-02 09:36→09:43: $-0.76 (-1.00R) - INITIAL STOP
  ❌ LONG 2025-09-02 09:56→10:14: $-0.63 (-1.00R) - INITIAL STOP
  ❌ LONG 2025-09-03 09:39→09:43: $+0.00 (+0.00R) - BREAK-EVEN STOP
  ✅ SHORT 2025-09-04 09:40→09:50: $+0.54 (+0.51R) - EMA EXIT
  ✅ LONG 2025-09-05 09:37→09:55: $+0.81 (+1.00R) - EMA EXIT
  ❌ LONG 2025-09-05 10:17→10:18: $-0.47 (-1.00R) - INITIAL STOP
  ✅ LONG 2025-09-08 09:37→09:52: $+0.65 (+0.75R) - EMA EXIT
  ❌ LONG 2025-09-09 09:43→09:51: $+0.00 (+0.00R) - BREAK-EVEN STOP
  ✅ LONG 2025-09-09 09:54→10:14: $+0.86 (+2.10R) - TRAILING STOP
  ❌ SHORT 2025-09-10 09:49→09:56: $-0.80 (-1.00R) - INITIAL STOP
  ✅ LONG 2025-09-10 10:12→10:22:

In [4]:
import random

# Filter by exit reason
initial_stops = [t for t in bt.trades if t.exit_reason == "INITIAL STOP"]
ema_exits = [t for t in bt.trades if t.exit_reason == "EMA EXIT"]
trailing_stops = [t for t in bt.trades if t.exit_reason == "TRAILING STOP"]
break_evens = [t for t in bt.trades if t.exit_reason == "BREAK-EVEN STOP"]

print(f"Exit Counts: {len(initial_stops)} Initial, {len(ema_exits)} EMA, {len(trailing_stops)} Trail, {len(break_evens)} BE")
print()

# Random sample of each (up to 3)
def show_sample(trades, label, n=3):
    print(f"=== {label} (sample of {min(n, len(trades))}) ===")
    sample = random.sample(trades, min(n, len(trades))) if trades else []
    for t in sample:
        print(f"  {t.date} {t.direction} {t.entry_time}→{t.exit_time} @ {t.entry_price:.2f}→{t.exit_price:.2f} {t.r_multiple:+.2f}R")
    print()

show_sample(initial_stops, "INITIAL STOP")
show_sample(ema_exits, "EMA EXIT")
show_sample(trailing_stops, "TRAILING STOP")
show_sample(break_evens, "BREAK-EVEN STOP")

Exit Counts: 40 Initial, 18 EMA, 8 Trail, 22 BE

=== INITIAL STOP (sample of 3) ===
  2025-10-14 SHORT 09:41→09:48 @ 241.07→241.65 -1.00R
  2025-11-17 LONG 10:28→10:31 @ 290.84→290.19 -1.00R
  2025-09-29 LONG 09:35→09:37 @ 248.39→247.81 -1.00R

=== EMA EXIT (sample of 3) ===
  2025-09-10 LONG 10:12→10:22 @ 240.40→240.63 +0.43R
  2025-11-07 SHORT 09:37→09:52 @ 280.39→279.31 +0.88R
  2025-10-17 LONG 10:06→10:17 @ 251.25→251.46 +0.21R

=== TRAILING STOP (sample of 3) ===
  2025-09-09 LONG 09:54→10:14 @ 235.27→236.13 +2.10R
  2025-09-29 LONG 09:40→09:50 @ 249.00→250.46 +2.13R
  2025-11-28 SHORT 09:45→10:08 @ 321.61→318.68 +1.83R

=== BREAK-EVEN STOP (sample of 3) ===
  2025-10-09 SHORT 09:42→09:45 @ 243.13→243.13 +0.00R
  2025-11-17 LONG 09:35→09:39 @ 291.25→291.25 +0.00R
  2025-09-30 SHORT 10:06→10:07 @ 240.66→240.66 +0.00R



In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')

from orb_backtester import ORBBacktester

bt = ORBBacktester(
    symbol='GOOGL',
    start_date='2025-09-01',
    end_date='2025-11-30',
    enable_confluence=False,
    verbose=True
)

results = bt.run()

# Exit breakdown
print("\n" + "="*80)
print("EXIT REASON BREAKDOWN")
print("="*80)
exit_reasons = {}
for t in bt.trades:
    reason = t.exit_reason
    if reason not in exit_reasons:
        exit_reasons[reason] = {'count': 0, 'r': 0}
    exit_reasons[reason]['count'] += 1
    exit_reasons[reason]['r'] += t.r_multiple

for reason, data in sorted(exit_reasons.items()):
    print(f"  {reason}: {data['count']} trades, {data['r']:+.2f}R")

print(f"\nTotal: {len(bt.trades)} trades, {results['total_r']:+.2f}R")
print(f"Win Rate: {results['win_rate']:.1f}%")
print(f"Profit Factor: {results['profit_factor']:.2f}")

[Presets] Loaded GOOGL: SSL_base=20, WAE_sens=325, QQE_rsi1=9

  ORB BACKTESTER: GOOGL
  2025-09-01 to 2025-11-30

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching GOOGL (1-min bars, 180 days, extended)...
  ✓ Loading from cache (4.6 hours old)
✓ Loaded 88081 bars for GOOGL
Processing 63 trading days...

  ❌ LONG 2025-09-02 09:36→09:43: $-0.76 (-1.00R) - INITIAL STOP
  ❌ LONG 2025-09-02 09:56→10:14: $-0.63 (-1.00R) - INITIAL STOP
  ✅ LONG 2025-09-03 09:39→09:48: $+0.14 (+0.10R) - EMA EXIT
  ✅ SHORT 2025-09-04 09:40→09:50: $+0.54 (+0.51R) - EMA EXIT
  ✅ LONG 2025-09-05 09:37→09:55: $+0.81 (+1.00R) - EMA EXIT
  ❌ LONG 2025-09-05 10:17→10:18: $-0.47 (-1.00R) - INITIAL STOP
  ✅ LONG 2025-09-08 09:37→09:52: $+0.65 (+0.75R) - EMA EXIT
  ❌ LONG 2025-09-09 09:43→09:51: $+0.00 (+0.00R) - BREAK-EVEN STOP
  ✅ LONG 2025-09-09 09:54→10:14: $+0.84 (+2.06R) - TRAILING STOP
  ❌ SHORT 2025-09-10 09:49→09:56: $-0.80 (-1.00R) - INITIAL STOP
  ✅ LONG 2025-09-10 10:12→10:22: $+0.24

In [1]:
# Debug Oct 14 breakout detection
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
from orb_backtester import ORBBacktester

bt = ORBBacktester(
    symbol='GOOGL',
    start_date='2025-10-14',
    end_date='2025-10-14',
    enable_confluence=False,
    verbose=False
)
bt.load_data()

# Get Oct 14 data
day_df = bt.df[bt.df.index.date.astype(str) == '2025-10-14'].copy()

# Build ORB manually
orb_bars = day_df[(day_df.index.hour == 9) & (day_df.index.minute >= 30) & (day_df.index.minute < 35)]
orb_high = orb_bars['high'].max()
orb_low = orb_bars['low'].min()
print(f"ORB High: {orb_high:.2f}")
print(f"ORB Low: {orb_low:.2f}")
print()

# Check each bar after ORB
bt.orb.session_high = orb_high
bt.orb.session_low = orb_low
bt.orb.orb_complete = True

print("Bar-by-bar breakout check:")
for i, (ts, bar) in enumerate(day_df.iterrows()):
    if ts.hour == 9 and ts.minute >= 36 and ts.minute <= 55:
        new_long, new_short = bt._check_breakout(bar)
        time_str = ts.strftime('%H:%M')
        if new_long or new_short:
            print(f"  {time_str} O={bar['open']:.2f} H={bar['high']:.2f} L={bar['low']:.2f} C={bar['close']:.2f} | LONG={new_long} SHORT={new_short}")
        else:
            print(f"  {time_str} O={bar['open']:.2f} C={bar['close']:.2f} | no breakout")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching GOOGL (1-min bars, 180 days, extended)...
  ✓ Loading from cache (5.1 hours old)
ORB High: 242.30
ORB Low: 241.12

Bar-by-bar breakout check:
  09:36 O=241.46 C=241.35 | no breakout
  09:37 O=241.32 C=241.24 | no breakout
  09:38 O=241.27 C=241.31 | no breakout
  09:39 O=241.29 C=241.29 | no breakout
  09:40 O=241.28 C=241.68 | no breakout
  09:41 O=241.66 H=241.68 L=241.06 C=241.07 | LONG=False SHORT=True
  09:42 O=241.09 C=241.28 | no breakout
  09:43 O=241.30 H=241.38 L=240.80 C=240.99 | LONG=False SHORT=True
  09:44 O=241.02 C=240.86 | no breakout
  09:45 O=240.85 C=240.82 | no breakout
  09:46 O=240.80 C=240.92 | no breakout
  09:47 O=240.93 C=241.41 | no breakout
  09:48 O=241.43 C=241.66 | no breakout
  09:49 O=241.61 C=241.21 | no breakout
  09:50 O=241.21 C=241.14 | no breakout
  09:51 O=241.12 H=241.21 L=240.80 C=240.88 | LONG=False SHORT=True
  09:52 O=240.85 C=241.16 | no breakout
  09:53 O=241.18 C=2

In [2]:
bar_0941 = day_df[day_df.index.strftime('%H:%M') == '09:41'].iloc[0]
atr = bar_0941['atr_rth']
threshold = atr * 0.1
required_close = orb_low - threshold

print(f"ATR at 09:41: {atr:.4f}")
print(f"Threshold (ATR * 0.1): {threshold:.4f}")
print(f"Required close for SHORT: < {required_close:.4f}")
print(f"Actual close: {bar_0941['close']:.4f}")
print(f"Passes threshold? {bar_0941['close'] < required_close}")

ATR at 09:41: 0.3943
Threshold (ATR * 0.1): 0.0394
Required close for SHORT: < 241.0806
Actual close: 241.0750
Passes threshold? True


In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
from orb_backtester import ORBBacktester

# Larger date range - Aug 1 to Nov 30
bt = ORBBacktester(
    symbol='GOOGL',
    start_date='2025-08-01',
    end_date='2025-11-30',
    enable_confluence=False,
    verbose=False
)

results = bt.run()

# Full trade list
print("="*100)
print("ALL TRADES - GOOGL Aug 1 - Nov 30, 2025")
print("="*100)
print(f"{'Date':<12} {'Dir':<6} {'Entry':<6} {'Exit':<6} {'Entry$':<10} {'Exit$':<10} {'Stop$':<10} {'StopType':<10} {'Exit Reason':<18} {'R':>8}")
print("-"*100)

for t in bt.trades:
    print(f"{t.date:<12} {t.direction:<6} {t.entry_time:<6} {t.exit_time:<6} {t.entry_price:<10.2f} {t.exit_price:<10.2f} {t.initial_stop:<10.2f} {t.stop_type:<10} {t.exit_reason:<18} {t.r_multiple:>+8.2f}")

print("-"*100)
print(f"Total: {len(bt.trades)} trades, {results['total_r']:+.2f}R")

# Summary stats
print("\n" + "="*100)
print("SUMMARY")
print("="*100)
print(f"Total Trades: {len(bt.trades)}")
print(f"Wins: {results['wins']} | Losses: {results['losses']} | Break-even: {results['breakevens']}")
print(f"Win Rate: {results['win_rate']:.1f}%")
print(f"Total R: {results['total_r']:+.2f}")
print(f"Profit Factor: {results['profit_factor']:.2f}")
print(f"Avg R/Trade: {results['avg_r_per_trade']:+.3f}")

# Exit breakdown
print("\n" + "="*100)
print("EXIT REASON BREAKDOWN")
print("="*100)
exit_reasons = {}
for t in bt.trades:
    reason = t.exit_reason
    if reason not in exit_reasons:
        exit_reasons[reason] = {'count': 0, 'wins': 0, 'losses': 0, 'r': 0}
    exit_reasons[reason]['count'] += 1
    exit_reasons[reason]['r'] += t.r_multiple
    if t.r_multiple > 0:
        exit_reasons[reason]['wins'] += 1
    elif t.r_multiple < 0:
        exit_reasons[reason]['losses'] += 1

for reason, data in sorted(exit_reasons.items()):
    print(f"  {reason:<18}: {data['count']:>3} trades ({data['wins']}W/{data['losses']}L), {data['r']:>+7.2f}R")

# Monthly breakdown
print("\n" + "="*100)
print("MONTHLY BREAKDOWN")
print("="*100)
monthly = {}
for t in bt.trades:
    month = t.date[:7]  # YYYY-MM
    if month not in monthly:
        monthly[month] = {'count': 0, 'wins': 0, 'losses': 0, 'r': 0}
    monthly[month]['count'] += 1
    monthly[month]['r'] += t.r_multiple
    if t.r_multiple > 0:
        monthly[month]['wins'] += 1
    elif t.r_multiple < 0:
        monthly[month]['losses'] += 1

for month, data in sorted(monthly.items()):
    wr = data['wins'] / (data['wins'] + data['losses']) * 100 if (data['wins'] + data['losses']) > 0 else 0
    print(f"  {month}: {data['count']:>3} trades, {data['wins']}W/{data['losses']}L ({wr:.0f}%), {data['r']:>+7.2f}R")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching GOOGL (1-min bars, 180 days, extended)...
  ✓ Loading from cache (5.1 hours old)
ALL TRADES - GOOGL Aug 1 - Nov 30, 2025
Date         Dir    Entry  Exit   Entry$     Exit$      Stop$      StopType   Exit Reason               R
----------------------------------------------------------------------------------------------------
2025-08-01   SHORT  09:41  09:56  188.71     189.49     189.49     ATR        INITIAL STOP          -1.00
2025-08-01   SHORT  10:00  10:04  188.55     188.55     189.30     ATR        BREAK-EVEN STOP       +0.00
2025-08-04   LONG   09:35  09:52  192.22     193.13     191.55     ATR        EMA EXIT              +1.37
2025-08-05   LONG   09:37  09:53  196.47     197.32     195.77     ATR        EMA EXIT              +1.22
2025-08-07   SHORT  09:40  09:48  196.62     195.94     197.10     VWAP       TRAILING STOP         +1.43
2025-08-07   SHORT  10:07  10:13  196.44     196.38     196.75     S

In [2]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
from orb_backtester import ORBBacktester

# Aug 25 to Nov 30 (matches TV 5-month lookback)
bt = ORBBacktester(
    symbol='GOOGL',
    start_date='2025-08-25',
    end_date='2025-11-30',
    enable_confluence=False,
    verbose=False
)

results = bt.run()

# Full trade list
print("="*100)
print("ALL TRADES - GOOGL Aug 25 - Nov 30, 2025")
print("="*100)
print(f"{'Date':<12} {'Dir':<6} {'Entry':<6} {'Exit':<6} {'Entry$':<10} {'Exit$':<10} {'Stop$':<10} {'StopType':<10} {'Exit Reason':<18} {'R':>8}")
print("-"*100)

for t in bt.trades:
    print(f"{t.date:<12} {t.direction:<6} {t.entry_time:<6} {t.exit_time:<6} {t.entry_price:<10.2f} {t.exit_price:<10.2f} {t.initial_stop:<10.2f} {t.stop_type:<10} {t.exit_reason:<18} {t.r_multiple:>+8.2f}")

print("-"*100)
print(f"Total: {len(bt.trades)} trades, {results['total_r']:+.2f}R")

# Summary stats
print("\n" + "="*100)
print("SUMMARY")
print("="*100)
print(f"Total Trades: {len(bt.trades)}")
print(f"Wins: {results['wins']} | Losses: {results['losses']} | Break-even: {results['breakevens']}")
print(f"Win Rate: {results['win_rate']:.1f}%")
print(f"Total R: {results['total_r']:+.2f}")
print(f"Profit Factor: {results['profit_factor']:.2f}")
print(f"Avg R/Trade: {results['avg_r_per_trade']:+.3f}")

# Exit breakdown
print("\n" + "="*100)
print("EXIT REASON BREAKDOWN")
print("="*100)
exit_reasons = {}
for t in bt.trades:
    reason = t.exit_reason
    if reason not in exit_reasons:
        exit_reasons[reason] = {'count': 0, 'wins': 0, 'losses': 0, 'r': 0}
    exit_reasons[reason]['count'] += 1
    exit_reasons[reason]['r'] += t.r_multiple
    if t.r_multiple > 0:
        exit_reasons[reason]['wins'] += 1
    elif t.r_multiple < 0:
        exit_reasons[reason]['losses'] += 1

for reason, data in sorted(exit_reasons.items()):
    print(f"  {reason:<18}: {data['count']:>3} trades ({data['wins']}W/{data['losses']}L), {data['r']:>+7.2f}R")

# Monthly breakdown
print("\n" + "="*100)
print("MONTHLY BREAKDOWN")
print("="*100)
monthly = {}
for t in bt.trades:
    month = t.date[:7]
    if month not in monthly:
        monthly[month] = {'count': 0, 'wins': 0, 'losses': 0, 'r': 0}
    monthly[month]['count'] += 1
    monthly[month]['r'] += t.r_multiple
    if t.r_multiple > 0:
        monthly[month]['wins'] += 1
    elif t.r_multiple < 0:
        monthly[month]['losses'] += 1

for month, data in sorted(monthly.items()):
    wr = data['wins'] / (data['wins'] + data['losses']) * 100 if (data['wins'] + data['losses']) > 0 else 0
    print(f"  {month}: {data['count']:>3} trades, {data['wins']}W/{data['losses']}L ({wr:.0f}%), {data['r']:>+7.2f}R")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching GOOGL (1-min bars, 180 days, extended)...
  ✓ Loading from cache (5.2 hours old)
ALL TRADES - GOOGL Aug 25 - Nov 30, 2025
Date         Dir    Entry  Exit   Entry$     Exit$      Stop$      StopType   Exit Reason               R
----------------------------------------------------------------------------------------------------
2025-08-25   LONG   09:38  09:45  207.40     206.55     206.55     ATR        INITIAL STOP          -1.00
2025-08-25   LONG   09:46  10:07  207.16     207.84     206.51     SWING      EMA EXIT              +1.04
2025-08-26   SHORT  09:46  09:52  206.27     206.27     206.95     ATR        BREAK-EVEN STOP       +0.00
2025-08-26   SHORT  09:54  09:57  206.29     206.60     206.60     SWING      INITIAL STOP          -1.00
2025-08-26   SHORT  10:06  10:18  206.25     206.79     206.79     VWAP       INITIAL STOP          -1.00
2025-08-27   LONG   09:39  10:03  207.12     207.80     206.48     

In [3]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
from orb_backtester import ORBBacktester

bt = ORBBacktester(
    symbol='GOOGL',
    start_date='2025-09-02',
    end_date='2025-09-02',
    enable_confluence=False,
    verbose=False
)
bt.load_data()

# Get Sep 2 data
day_df = bt.df[bt.df.index.date.astype(str) == '2025-09-02'].copy()

# Check ATR at 09:36
bar_0936 = day_df[day_df.index.strftime('%H:%M') == '09:36'].iloc[0]
print(f"Time: 09:36")
print(f"Close: {bar_0936['close']:.2f}")
print(f"ATR (atr_rth): {bar_0936['atr_rth']:.4f}")
print(f"ATR Stop (entry - ATR*2): {bar_0936['close'] - bar_0936['atr_rth'] * 2:.2f}")
print()
print(f"Pine shows ATR Stop: 208.15")
print(f"Pine ATR would be: {(209.28 - 208.15) / 2:.4f}")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching GOOGL (1-min bars, 180 days, extended)...
  ⚠ Cache stale (1 hours), refreshing...
  ✓ Fetched 87789 bars, cached for next time
Time: 09:36
Close: 209.28
ATR (atr_rth): 0.3792
ATR Stop (entry - ATR*2): 208.52

Pine shows ATR Stop: 208.15
Pine ATR would be: 0.5650


In [4]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
from orb_backtester import ORBBacktester
import pandas as pd

bt = ORBBacktester(
    symbol='GOOGL',
    start_date='2025-09-02',
    end_date='2025-09-02',
    enable_confluence=False,
    verbose=False
)
bt.load_data()
df = bt.df

# Filter to RTH only
rth_df = df[df['is_rth']].copy()

# Look at last few RTH bars before and at 09:36 on Sep 2
sep2_rth = rth_df[rth_df.index.date.astype(str) == '2025-09-02']

print("Sep 2 RTH bars with ATR components:")
print(f"{'Time':<8} {'Open':>8} {'High':>8} {'Low':>8} {'Close':>8} {'TR':>8} {'ATR':>8}")
print("-" * 70)

for ts, row in sep2_rth.head(10).iterrows():
    time_str = ts.strftime('%H:%M')
    print(f"{time_str:<8} {row['open']:>8.2f} {row['high']:>8.2f} {row['low']:>8.2f} {row['close']:>8.2f} {row.get('tr', 0):>8.4f} {row['atr_rth']:>8.4f}")

# Also check what the PREVIOUS session's last ATR was
sep1_rth = rth_df[rth_df.index.date.astype(str) == '2025-08-29']  # Likely Friday before Labor Day
if len(sep1_rth) > 0:
    print(f"\nPrevious session last bar ATR: {sep1_rth.iloc[-1]['atr_rth']:.4f}")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching GOOGL (1-min bars, 180 days, extended)...
  ✓ Loading from cache (0.0 hours old)
Sep 2 RTH bars with ATR components:
Time         Open     High      Low    Close       TR      ATR
----------------------------------------------------------------------
09:30      208.44   209.03   208.28   208.72   0.7800   0.1795
09:31      208.69   208.72   207.56   207.82   1.1625   0.2199
09:32      207.90   208.77   207.69   208.59   1.0800   0.2873
09:33      208.61   208.87   208.08   208.85   0.7900   0.3439
09:34      208.92   209.02   208.67   208.73   0.3475   0.3757
09:35      208.71   209.08   208.63   208.90   0.4500   0.3737
09:36      208.92   209.34   208.90   209.28   0.4400   0.3792
09:37      209.28   209.32   209.00   209.31   0.3200   0.3835
09:38      209.33   209.56   209.11   209.13   0.4480   0.3790
09:39      209.14   209.31   208.85   208.96   0.4600   0.3839

Previous session last bar ATR: 0.1707


In [5]:
# Check ATR continuity around Labor Day
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
from orb_backtester import ORBBacktester

bt = ORBBacktester(
    symbol='GOOGL',
    start_date='2025-08-29',  # Friday before Labor Day
    end_date='2025-09-02',
    enable_confluence=False,
    verbose=False
)
bt.load_data()

# Check ATR at end of Aug 29 and start of Sep 2
aug29 = bt.df[bt.df.index.date.astype(str) == '2025-08-29']
sep02 = bt.df[bt.df.index.date.astype(str) == '2025-09-02']

print("=== Aug 29 (Friday) - Last RTH bars ===")
aug29_rth = aug29[aug29['is_rth']].tail(5)
for ts, row in aug29_rth.iterrows():
    print(f"{ts.strftime('%H:%M')} ATR: {row['atr_rth']:.4f}")

print("\n=== Sep 2 (Tuesday after Labor Day) - First RTH bars ===")
sep02_rth = sep02[sep02['is_rth']].head(10)
for ts, row in sep02_rth.iterrows():
    print(f"{ts.strftime('%H:%M')} ATR: {row['atr_rth']:.4f}")

print(f"\nPine ATR at 09:36 on Sep 2: 0.5671")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching GOOGL (1-min bars, 180 days, extended)...
  ✓ Loading from cache (0.1 hours old)
=== Aug 29 (Friday) - Last RTH bars ===
15:55 ATR: 0.1668
15:56 ATR: 0.1774
15:57 ATR: 0.1754
15:58 ATR: 0.1715
15:59 ATR: 0.1707

=== Sep 2 (Tuesday after Labor Day) - First RTH bars ===
09:30 ATR: 0.1795
09:31 ATR: 0.2199
09:32 ATR: 0.2873
09:33 ATR: 0.3439
09:34 ATR: 0.3757
09:35 ATR: 0.3737
09:36 ATR: 0.3792
09:37 ATR: 0.3835
09:38 ATR: 0.3790
09:39 ATR: 0.3839

Pine ATR at 09:36 on Sep 2: 0.5671


In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
from orb_backtester import ORBBacktester

bt = ORBBacktester(
    symbol='GOOGL',
    start_date='2025-09-02',
    end_date='2025-09-02',
    enable_confluence=False,
    verbose=False
)
bt.load_data()

# Check ATR at 09:36
day_df = bt.df[bt.df.index.date.astype(str) == '2025-09-02']
bar_0936 = day_df[day_df.index.strftime('%H:%M') == '09:36'].iloc[0]
print(f"Python ATR at 09:36: {bar_0936['atr_rth']:.4f}")
print(f"Pine ATR at 09:36:   0.5671")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching GOOGL (1-min bars, 180 days, extended)...
  ✓ Loading from cache (0.5 hours old)
Python ATR at 09:36: 0.5671
Pine ATR at 09:36:   0.5671


In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
from orb_backtester import ORBBacktester

bt = ORBBacktester(
    symbol='GOOGL',
    start_date='2025-08-25',
    end_date='2025-11-30',
    enable_confluence=False,
    verbose=False
)

results = bt.run()

# Check Sep 2 trade specifically
for t in bt.trades:
    if t.date == '2025-09-02':
        print(f"Sep 2: {t.direction} @ {t.entry_price:.2f}, Stop: {t.initial_stop:.2f} ({t.stop_type})")
        print(f"  Pine showed: LONG @ 209.28, ATR Stop: 208.15")
        print(f"  Risk: ${t.entry_price - t.initial_stop:.2f}")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching GOOGL (1-min bars, 180 days, extended)...
  ✓ Loading from cache (0.5 hours old)
Sep 2: LONG @ 209.28, Stop: 208.46 (VWAP)
  Pine showed: LONG @ 209.28, ATR Stop: 208.15
  Risk: $0.82
Sep 2: LONG @ 209.22, Stop: 208.58 (VWAP)
  Pine showed: LONG @ 209.28, ATR Stop: 208.15
  Risk: $0.64


In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
from orb_backtester import ORBBacktester

bt = ORBBacktester(
    symbol='GOOGL',
    start_date='2025-09-02',
    end_date='2025-09-02',
    enable_confluence=False,
    verbose=False
)
results = bt.run()

for t in bt.trades:
    print(f"{t.entry_time}: {t.direction} @ {t.entry_price:.2f}, Stop: {t.initial_stop:.2f} ({t.stop_type})")

print(f"\nPine Trade 1: LONG @ 209.28, ATR Stop: 208.15")
print(f"Pine Trade 2: LONG @ 209.22, VWAP Stop")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching GOOGL (1-min bars, 180 days, extended)...
  ✓ Loading from cache (0.8 hours old)
09:36: LONG @ 209.28, Stop: 208.46 (VWAP)
09:56: LONG @ 209.22, Stop: 208.58 (VWAP)

Pine Trade 1: LONG @ 209.28, ATR Stop: 208.15
Pine Trade 2: LONG @ 209.22, VWAP Stop


In [2]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
from orb_backtester import ORBBacktester

bt = ORBBacktester(
    symbol='GOOGL',
    start_date='2025-09-02',
    end_date='2025-09-02',
    enable_confluence=False,
    verbose=False
)
bt.load_data()

# Manually run through Sep 2 to see vol_state at 09:36
day_df = bt.df[bt.df.index.date.astype(str) == '2025-09-02']

for ts, bar in day_df.iterrows():
    if ts.hour == 9 and ts.minute >= 30 and ts.minute <= 40:
        bt._calc_vol_state(bar, ts)
        time_str = ts.strftime('%H:%M')
        print(f"{time_str}: vol_state={bt.vol.vol_state}, vol_factor={bt.vol.vol_factor:.2f}")
        
        if ts.minute == 36:
            # Calculate what rr_desired would be
            rr_desired = 2.0  # base
            if bt.vol.vol_state == "LOW":
                rr_desired += 0.5
            elif bt.vol.vol_state == "HIGH":
                rr_desired = max(rr_desired - 0.5, 1.2)
            elif bt.vol.vol_state == "EXTREME":
                rr_desired = max(rr_desired - 0.8, 1.0)
            print(f"  -> Adjusted rr_desired: {rr_desired}")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching GOOGL (1-min bars, 180 days, extended)...
  ✓ Loading from cache (0.8 hours old)
09:30: vol_state=NORMAL, vol_factor=1.05
09:31: vol_state=NORMAL, vol_factor=1.06
09:32: vol_state=NORMAL, vol_factor=1.05
09:33: vol_state=NORMAL, vol_factor=0.99
09:34: vol_state=NORMAL, vol_factor=0.88
09:35: vol_state=NORMAL, vol_factor=0.81
09:36: vol_state=LOW, vol_factor=0.76
  -> Adjusted rr_desired: 2.5
09:37: vol_state=LOW, vol_factor=0.69
09:38: vol_state=LOW, vol_factor=0.68
09:39: vol_state=LOW, vol_factor=0.67
09:40: vol_state=LOW, vol_factor=0.65


In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
from orb_backtester import ORBBacktester

bt = ORBBacktester(
    symbol='GOOGL',
    start_date='2025-09-02',
    end_date='2025-09-02',
    enable_confluence=False,
    verbose=False
)
results = bt.run()

for t in bt.trades:
    print(f"{t.entry_time}: {t.direction} @ {t.entry_price:.2f}, Stop: {t.initial_stop:.2f} ({t.stop_type}), vol_state={t.vol_state}")

print(f"\nPine Trade 1: LONG @ 209.28, ATR Stop: 208.15, vol_state=HIGH")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching GOOGL (1-min bars, 180 days, extended)...
  ✓ Loading from cache (0.9 hours old)
09:36: LONG @ 209.28, Stop: 208.46 (VWAP), vol_state=NORMAL
09:56: LONG @ 209.22, Stop: 208.58 (VWAP), vol_state=LOW

Pine Trade 1: LONG @ 209.28, ATR Stop: 208.15, vol_state=HIGH


In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import importlib
import orb_backtester
importlib.reload(orb_backtester)
from orb_backtester import ORBBacktester

bt = ORBBacktester(
    symbol='GOOGL',
    start_date='2025-09-02',
    end_date='2025-09-02',
    enable_confluence=False,
    verbose=False
)
bt.load_data()

day_df = bt.df[bt.df.index.date.astype(str) == '2025-09-02']

print("═══ GOOGL Sep 2 - VF Debug (Python vs Pine) ═══")
print(f"{'Time':<8} {'atr_fast':>10} {'atr_sma':>10} {'baseline':>10} {'sessVF':>8} {'volFactor':>10} {'volState':>10}")
print("-" * 78)

for ts, bar in day_df.iterrows():
    if ts.hour == 9 and 30 <= ts.minute <= 36:
        bt._calc_vol_state(bar, ts)
        atr_fast = bar.get('atr_fast_all', 0)
        atr_sma = bar.get('atr_fast_sma_all', 0)
        time_str = ts.strftime('%H:%M')
        print(f"{time_str:<8} {atr_fast:>10.4f} {atr_sma:>10.4f} {bt.vol.orb_session_baseline:>10.4f} {bt.vol.session_vf:>8.2f} {bt.vol.vol_factor:>10.2f} {bt.vol.vol_state:>10}")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching GOOGL (1-min bars, 180 days, extended)...
  ✓ Loading from cache (1.4 hours old)
═══ GOOGL Sep 2 - VF Debug (Python vs Pine) ═══
Time       atr_fast    atr_sma   baseline   sessVF  volFactor   volState
------------------------------------------------------------------------------
09:30        0.5496     0.3054     0.3054     1.80       1.89       HIGH
09:31        0.6722     0.3603     0.3109     2.16       2.27    EXTREME
09:32        0.7537     0.4237     0.3221     2.34       2.45    EXTREME
09:33        0.7610     0.4882     0.3387     2.25       2.35    EXTREME
09:34        0.6783     0.5441     0.3593     1.89       1.98       HIGH
09:35        0.6326     0.5791     0.3813     1.66       1.74       HIGH
09:36        0.5941     0.6093     0.4041     1.47       1.54       HIGH


In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import importlib
import orb_backtester
importlib.reload(orb_backtester)
from orb_backtester import ORBBacktester

bt = ORBBacktester(
    symbol='GOOGL',
    start_date='2025-09-02',
    end_date='2025-09-02',
    enable_confluence=False,
    verbose=False
)
bt.load_data()

df = bt.df
sep2 = df[df.index.date.astype(str) == '2025-09-02']

print("═══ GOOGL Sep 2 - Last 10 bars before/at 09:30 ═══")
print(f"{'Time':<8} {'Open':>10} {'High':>10} {'Low':>10} {'Close':>10} {'TR':>10} {'atr_fast_all':>12}")
print("-" * 82)

# Get bars from 09:20 to 09:36
for ts, bar in sep2.iterrows():
    if (ts.hour == 9 and 20 <= ts.minute <= 36):
        print(f"{ts.strftime('%H:%M'):<8} {bar['open']:>10.2f} {bar['high']:>10.2f} {bar['low']:>10.2f} {bar['close']:>10.2f} {bar['tr']:>10.4f} {bar.get('atr_fast_all', 0):>12.4f}")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching GOOGL (1-min bars, 180 days, extended)...
  ✓ Loading from cache (1.5 hours old)
═══ GOOGL Sep 2 - Last 10 bars before/at 09:30 ═══
Time           Open       High        Low      Close         TR atr_fast_all
----------------------------------------------------------------------------------
09:20        209.03     209.06     208.96     209.03     0.0992       0.1359
09:21        209.03     209.03     208.96     209.00     0.0700       0.1227
09:22        208.89     209.00     208.89     209.00     0.1100       0.1202
09:23        208.90     208.90     208.90     208.90     0.0992       0.1160
09:24        208.99     209.03     208.96     209.03     0.1292       0.1186
09:25        208.80     209.00     208.09     208.41     0.9400       0.2829
09:26        208.44     208.62     208.29     208.59     0.3300       0.2923
09:27        208.66     209.35     208.56     209.29     0.7900       0.3919
09:28        208.8

In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import importlib
import orb_backtester
importlib.reload(orb_backtester)
from orb_backtester import ORBBacktester

bt = ORBBacktester(
    symbol='GOOGL',
    start_date='2025-09-02',
    end_date='2025-09-02',
    enable_confluence=False,
    verbose=False
)
bt.load_data()

df = bt.df

# Get Aug 29 late + Sep 2 early
print("═══ GOOGL - Aug 29 EOD + Sep 2 Start ═══")
print(f"{'Timestamp':<20} {'Open':>10} {'High':>10} {'Low':>10} {'Close':>10} {'TR':>10} {'atr_fast_all':>12}")
print("-" * 94)

# Last bars of Aug 29
aug29 = df[df.index.date.astype(str) == '2025-08-29']
for ts, bar in aug29.tail(5).iterrows():
    print(f"{ts.strftime('%Y-%m-%d %H:%M'):<20} {bar['open']:>10.2f} {bar['high']:>10.2f} {bar['low']:>10.2f} {bar['close']:>10.2f} {bar['tr']:>10.4f} {bar.get('atr_fast_all', 0):>12.4f}")

print("-" * 94)

# First bars of Sep 2
sep2 = df[df.index.date.astype(str) == '2025-09-02']
for ts, bar in sep2.head(10).iterrows():
    print(f"{ts.strftime('%Y-%m-%d %H:%M'):<20} {bar['open']:>10.2f} {bar['high']:>10.2f} {bar['low']:>10.2f} {bar['close']:>10.2f} {bar['tr']:>10.4f} {bar.get('atr_fast_all', 0):>12.4f}")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching GOOGL (1-min bars, 180 days, extended)...
  ✓ Loading from cache (1.7 hours old)
═══ GOOGL - Aug 29 EOD + Sep 2 Start ═══
Timestamp                  Open       High        Low      Close         TR atr_fast_all
----------------------------------------------------------------------------------------------
2025-08-29 19:35         212.68     212.68     212.68     212.68     0.1200       0.0991
2025-08-29 19:44         212.68     212.68     212.68     212.68     0.0000       0.0793
2025-08-29 19:52         212.84     212.85     212.84     212.85     0.1700       0.0974
2025-08-29 19:58         212.75     212.75     212.75     212.75     0.1000       0.0980
2025-08-29 19:59         212.74     212.74     212.74     212.74     0.0101       0.0804
----------------------------------------------------------------------------------------------
2025-09-02 04:00         212.16     212.50     211.47     211.47     1.2699     

In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import importlib
import orb_backtester
importlib.reload(orb_backtester)
from orb_backtester import ORBBacktester

bt = ORBBacktester(
    symbol='GOOGL',
    start_date='2025-09-02',
    end_date='2025-09-02',
    enable_confluence=False,
    verbose=False
)
bt.load_data()

day_df = bt.df[bt.df.index.date.astype(str) == '2025-09-02']

print("═══ GOOGL Sep 2 - VF Debug (with RTH gap fix) ═══")
print(f"{'Time':<8} {'atr_fast':>10} {'atr_sma':>10} {'baseline':>10} {'sessVF':>8} {'volFactor':>10} {'volState':>10}")
print("-" * 78)

for ts, bar in day_df.iterrows():
    if ts.hour == 9 and 30 <= ts.minute <= 36:
        bt._calc_vol_state(bar, ts)
        atr_fast = bar.get('atr_fast_all', 0)
        atr_sma = bar.get('atr_fast_sma_all', 0)
        time_str = ts.strftime('%H:%M')
        print(f"{time_str:<8} {atr_fast:>10.4f} {atr_sma:>10.4f} {bt.vol.orb_session_baseline:>10.4f} {bt.vol.session_vf:>8.2f} {bt.vol.vol_factor:>10.2f} {bt.vol.vol_state:>10}")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching GOOGL (1-min bars, 180 days, extended)...
  ✓ Loading from cache (1.7 hours old)
═══ GOOGL Sep 2 - VF Debug (with RTH gap fix) ═══
Time       atr_fast    atr_sma   baseline   sessVF  volFactor   volState
------------------------------------------------------------------------------
09:30        1.3446     0.3849     0.3849     3.49       3.66    EXTREME
09:31        1.3082     0.5034     0.3967     3.30       3.46    EXTREME
09:32        1.2625     0.6176     0.4188     3.01       3.16    EXTREME
09:33        1.1680     0.7228     0.4492     2.60       2.72    EXTREME
09:34        1.0039     0.8114     0.4854     2.07       2.17    EXTREME
09:35        0.8931     0.8724     0.5241     1.70       1.79       HIGH
09:36        0.8025     0.9234     0.5641     1.42       1.49       HIGH


In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import importlib
import orb_backtester
importlib.reload(orb_backtester)
from orb_backtester import ORBBacktester

# Test on the original problem day
bt = ORBBacktester(
    symbol='GOOGL',
    start_date='2025-09-02',
    end_date='2025-09-02',
    enable_confluence=False,
    verbose=True
)
r = bt.run()
print(f"\nTrades: {r['total_trades']}, Skips: {len(bt.skips)}")
for skip in bt.skips:
    print(f"  SKIP @ {skip.timestamp}: {skip.reason} | volState={skip.vol_state}")

[Presets] Loaded GOOGL: SSL_base=20, WAE_sens=325, QQE_rsi1=9

  ORB BACKTESTER: GOOGL
  2025-09-02 to 2025-09-02

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching GOOGL (1-min bars, 180 days, extended)...
  ✓ Loading from cache (1.7 hours old)
✓ Loaded 87789 bars for GOOGL
Processing 1 trading days...

  ❌ LONG 2025-09-02 09:36→09:44: $-0.82 (-1.00R) - INITIAL STOP
  ❌ LONG 2025-09-02 09:56→10:14: $-0.64 (-1.00R) - INITIAL STOP

  BACKTEST SUMMARY: GOOGL
  2025-09-02 to 2025-09-02

  Total Trades:    2
  Total Skips:     0 (R:R below 1.5)
  Wins/Losses/BE:  0W / 2L / 0BE
  Win Rate:        0.0% (of decided trades)

  Total P/L:       $-1.46
  Total R:         -2.00R
  Avg R/Trade:     -1.000R

  Avg Win:         $0.00 (0.00R)
  Avg Loss:        $-0.73 (-1.00R)
  Profit Factor:   0.00
  Max Drawdown:    2.00R



TRADE DETAILS
2025-09-02 LONG  09:36->09:44  $209.28->$208.46  INITIAL STOP       -1.00R
2025-09-02 LONG  09:56->10:14  $209.22->$208.58  INITIAL STOP 

In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import importlib
import orb_backtester
importlib.reload(orb_backtester)
from orb_backtester import ORBBacktester

bt = ORBBacktester(
    symbol='GOOGL',
    start_date='2025-09-02',
    end_date='2025-09-02',
    enable_confluence=False,
    verbose=False
)
bt.load_data()

day_df = bt.df[bt.df.index.date.astype(str) == '2025-09-02']

print("═══ GOOGL Sep 2 - volState at Entry Times ═══")
for ts, bar in day_df.iterrows():
    if (ts.hour == 9 and ts.minute in [36, 56]):
        bt._calc_vol_state(bar, ts)
        print(f"{ts.strftime('%H:%M')}: volFactor={bt.vol.vol_factor:.2f}, volState={bt.vol.vol_state}")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching GOOGL (1-min bars, 180 days, extended)...
  ✓ Loading from cache (1.8 hours old)
═══ GOOGL Sep 2 - volState at Entry Times ═══
09:36: volFactor=0.91, volState=NORMAL
09:56: volFactor=0.44, volState=LOW


In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import importlib
import orb_backtester
importlib.reload(orb_backtester)
from orb_backtester import ORBBacktester

bt = ORBBacktester(
    symbol='GOOGL',
    start_date='2025-09-02',
    end_date='2025-09-02',
    enable_confluence=False,
    verbose=False
)
bt.load_data()

day_df = bt.df[bt.df.index.date.astype(str) == '2025-09-02']

print("═══ GOOGL Sep 2 - volState (processing all bars in order) ═══")

# Process ALL bars in order to build state correctly
for ts, bar in day_df.iterrows():
    bt._calc_vol_state(bar, ts)
    
    # Only print the entry times
    if (ts.hour == 9 and ts.minute in [36, 56]):
        print(f"{ts.strftime('%H:%M')}: volFactor={bt.vol.vol_factor:.2f}, volState={bt.vol.vol_state}")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching GOOGL (1-min bars, 180 days, extended)...
  ✓ Loading from cache (1.8 hours old)
═══ GOOGL Sep 2 - volState (processing all bars in order) ═══
09:36: volFactor=1.49, volState=HIGH
09:56: volFactor=0.74, volState=LOW


In [ ]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import importlib
import orb_backtester
importlib.reload(orb_backtester)
from orb_backtester import ORBBacktester

bt = ORBBacktester(
    symbol='GOOGL',
    start_date='2025-09-02',
    end_date='2025-09-02',
    enable_confluence=False,
    verbose=False
)
bt.load_data()

day_df = bt.df[bt.df.index.date.astype(str) == '2025-09-02']

print("═══ GOOGL Sep 2 - STOP ATR Comparison ═══")
print(f"{'Time':<8} {'atr_rth':>10} {'Pine ATR':>10} {'Diff':>10}")
print("-" * 42)

# Pine values from your screenshots
pine_atr = {
    '09:30': 0.5064,
    '09:36': 0.5671,
}

for ts, bar in day_df.iterrows():
    if ts.hour == 9 and ts.minute in [30, 36]:
        time_str = ts.strftime('%H:%M')
        atr = bar.get('atr_rth', 0)
        pine = pine_atr.get(time_str, 0)
        diff = atr - pine
        print(f"{time_str:<8} {atr:>10.4f} {pine:>10.4f} {diff:>+10.4f}")

# Calculate what stop SHOULD be at 09:36
entry = 209.28
print(f"\n═══ Stop Calculation @ 09:36 ═══")
print(f"Entry: {entry}")
print(f"Pine ATR: 0.5671 → Stop = {entry - 0.5671 * 2:.2f}")
print(f"Python ATR: {day_df[day_df.index.hour == 9][day_df.index.minute == 36]['atr_rth'].values[0]:.4f}")

In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import importlib
import orb_backtester
importlib.reload(orb_backtester)
from orb_backtester import ORBBacktester

bt = ORBBacktester(
    symbol='GOOGL',
    start_date='2025-09-02',
    end_date='2025-09-02',
    enable_confluence=False,
    verbose=False
)
bt.load_data()

day_df = bt.df[bt.df.index.date.astype(str) == '2025-09-02']

print("═══ GOOGL Sep 2 - STOP ATR Comparison ═══")
print(f"{'Time':<8} {'atr_rth':>10} {'Pine ATR':>10} {'Diff':>10}")
print("-" * 42)

# Pine values from your screenshots
pine_atr = {
    '09:30': 0.5064,
    '09:36': 0.5671,
}

for ts, bar in day_df.iterrows():
    if ts.hour == 9 and ts.minute in [30, 36]:
        time_str = ts.strftime('%H:%M')
        atr = bar.get('atr_rth', 0)
        pine = pine_atr.get(time_str, 0)
        diff = atr - pine
        print(f"{time_str:<8} {atr:>10.4f} {pine:>10.4f} {diff:>+10.4f}")

# Calculate what stop SHOULD be at 09:36
entry = 209.28
print(f"\n═══ Stop Calculation @ 09:36 ═══")
print(f"Entry: {entry}")
print(f"Pine ATR: 0.5671 → Stop = {entry - 0.5671 * 2:.2f}")
print(f"Python ATR: {day_df[day_df.index.hour == 9][day_df.index.minute == 36]['atr_rth'].values[0]:.4f}")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching GOOGL (1-min bars, 180 days, extended)...
  ✓ Loading from cache (3.1 hours old)
═══ GOOGL Sep 2 - STOP ATR Comparison ═══
Time        atr_rth   Pine ATR       Diff
------------------------------------------
09:30        0.5064     0.5064    -0.0000
09:36        0.5671     0.5671    +0.0000

═══ Stop Calculation @ 09:36 ═══
Entry: 209.28
Pine ATR: 0.5671 → Stop = 208.15


ValueError: Item wrong length 826 instead of 60.

In [2]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import importlib
import orb_backtester
importlib.reload(orb_backtester)
from orb_backtester import ORBBacktester
import pandas as pd

bt = ORBBacktester(
    symbol='GOOGL',
    start_date='2025-09-02',
    end_date='2025-09-02',
    enable_confluence=False,
    verbose=False
)
bt.load_data()

df = bt.df

# Filter to RTH only
rth_mask = df.index.map(lambda x: 9*60+30 <= x.hour*60+x.minute < 16*60)
rth_df = df[rth_mask].copy()

# Check the shift
rth_df['prev_close_test'] = rth_df['close'].shift(1)

# Get Aug 29 last bar and Sep 2 first bar
print("═══ RTH Close Carryover Check ═══")
aug29_rth = rth_df[rth_df.index.date.astype(str) == '2025-08-29']
sep2_rth = rth_df[rth_df.index.date.astype(str) == '2025-09-02']

if len(aug29_rth) > 0:
    print(f"Aug 29 last RTH bar: {aug29_rth.index[-1]} close={aug29_rth['close'].iloc[-1]:.2f}")
if len(sep2_rth) > 0:
    print(f"Sep 2 first RTH bar: {sep2_rth.index[0]} close={sep2_rth['close'].iloc[0]:.2f}")
    print(f"Sep 2 first bar prev_close: {sep2_rth['prev_close_test'].iloc[0]:.2f}")
    
    # Calculate what TR should be
    h, l, c = sep2_rth['high'].iloc[0], sep2_rth['low'].iloc[0], sep2_rth['close'].iloc[0]
    prev_c = sep2_rth['prev_close_test'].iloc[0]
    tr = max(h - l, abs(h - prev_c), abs(l - prev_c))
    print(f"\nSep 2 09:30: H={h:.2f}, L={l:.2f}")
    print(f"TR with gap: {tr:.4f}")
    print(f"Expected TR ~4.75 if gap captured")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching GOOGL (1-min bars, 180 days, extended)...
  ✓ Loading from cache (3.2 hours old)
═══ RTH Close Carryover Check ═══
Aug 29 last RTH bar: 2025-08-29 15:59:00-04:00 close=213.04
Sep 2 first RTH bar: 2025-09-02 09:30:00-04:00 close=208.72
Sep 2 first bar prev_close: 213.04

Sep 2 09:30: H=209.03, L=208.28
TR with gap: 4.7550
Expected TR ~4.75 if gap captured


In [3]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import importlib
import orb_backtester
importlib.reload(orb_backtester)
from orb_backtester import ORBBacktester

bt = ORBBacktester(
    symbol='GOOGL',
    start_date='2025-09-02',
    end_date='2025-09-02',
    enable_confluence=False,
    verbose=False
)
bt.load_data()

day_df = bt.df[bt.df.index.date.astype(str) == '2025-09-02']

print("═══ GOOGL Sep 2 - STOP ATR (atr_rth) ═══")
print(f"{'Time':<8} {'atr_rth':>10} {'Pine ATR':>10}")
print("-" * 32)

# Pine values from your screenshots
pine_atr = {
    '09:30': 0.5064,
    '09:36': 0.5671,
}

for ts, bar in day_df.iterrows():
    if ts.hour == 9 and 30 <= ts.minute <= 36:
        time_str = ts.strftime('%H:%M')
        atr = bar.get('atr_rth', 0)
        pine = pine_atr.get(time_str, '?')
        print(f"{time_str:<8} {atr:>10.4f} {pine:>10}")

# Calculate stop comparison at 09:36
print("\n═══ Stop Price Comparison @ 09:36 ═══")
entry = 209.28
for ts, bar in day_df.iterrows():
    if ts.hour == 9 and ts.minute == 36:
        python_atr = bar.get('atr_rth', 0)
        python_stop = entry - (python_atr * 2.0)
        pine_stop = entry - (0.5671 * 2.0)  # Pine ATR at 09:36
        print(f"Entry: {entry}")
        print(f"Python ATR: {python_atr:.4f} → Stop: {python_stop:.2f}")
        print(f"Pine ATR:   0.5671 → Stop: {pine_stop:.2f}")
        print(f"Pine actual stop from screenshot: 208.15")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching GOOGL (1-min bars, 180 days, extended)...
  ✓ Loading from cache (3.2 hours old)
═══ GOOGL Sep 2 - STOP ATR (atr_rth) ═══
Time        atr_rth   Pine ATR
--------------------------------
09:30        0.5064     0.5064
09:31        0.5532          ?
09:32        0.5909          ?
09:33        0.6051          ?
09:34        0.5867          ?
09:35        0.5769          ?
09:36        0.5671     0.5671

═══ Stop Price Comparison @ 09:36 ═══
Entry: 209.28
Python ATR: 0.5671 → Stop: 208.15
Pine ATR:   0.5671 → Stop: 208.15
Pine actual stop from screenshot: 208.15


In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import importlib
import orb_backtester
importlib.reload(orb_backtester)
from orb_backtester import ORBBacktester

bt = ORBBacktester(
    symbol='GOOGL',
    start_date='2025-09-02',
    end_date='2025-09-02',
    enable_confluence=False,
    verbose=False
)
bt.load_data()

day_df = bt.df[bt.df.index.date.astype(str) == '2025-09-02']

print("═══ GOOGL Sep 2 - STOP ATR Comparison ═══")
print(f"{'Time':<8} {'Python atr_rth':>14}")
print("-" * 24)

for ts, bar in day_df.iterrows():
    if ts.hour == 9 and 30 <= ts.minute <= 40:
        time_str = ts.strftime('%H:%M')
        atr = bar.get('atr_rth', 0)
        print(f"{time_str:<8} {atr:>14.4f}")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching GOOGL (1-min bars, 180 days, extended)...
  ✓ Loading from cache (3.2 hours old)
═══ GOOGL Sep 2 - STOP ATR Comparison ═══
Time     Python atr_rth
------------------------
09:30            0.5064
09:31            0.5532
09:32            0.5909
09:33            0.6051
09:34            0.5867
09:35            0.5769
09:36            0.5671
09:37            0.5495
09:38            0.5422
09:39            0.5364
09:40            0.5271


In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import importlib
import orb_backtester
importlib.reload(orb_backtester)
from orb_backtester import ORBBacktester

bt = ORBBacktester(
    symbol='GOOGL',
    start_date='2025-09-02',
    end_date='2025-09-02',
    enable_confluence=False,
    verbose=False
)
bt.load_data()

day_df = bt.df[bt.df.index.date.astype(str) == '2025-09-02']

# Get the 09:36 bar
for ts, bar in day_df.iterrows():
    if ts.hour == 9 and ts.minute == 36:
        entry = 209.28
        atr = bar['atr_rth']
        atr_mult = 2.0  # default
        
        print(f"═══ Stop Calculation @ 09:36 ═══")
        print(f"Entry: {entry}")
        print(f"Python atr_rth: {atr:.4f}")
        print(f"ATR Multiplier: {atr_mult}")
        print(f"Python ATR Stop: {entry - (atr * atr_mult):.2f}")
        print(f"Pine ATR Stop: 208.15")
        print(f"\nTo match Pine, ATR should be: {(entry - 208.15) / atr_mult:.4f}")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching GOOGL (1-min bars, 180 days, extended)...
  ✓ Loading from cache (3.2 hours old)
═══ Stop Calculation @ 09:36 ═══
Entry: 209.28
Python atr_rth: 0.5671
ATR Multiplier: 2.0
Python ATR Stop: 208.15
Pine ATR Stop: 208.15

To match Pine, ATR should be: 0.5650


In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import importlib
import orb_backtester
importlib.reload(orb_backtester)
from orb_backtester import ORBBacktester

bt = ORBBacktester(
    symbol='GOOGL',
    start_date='2025-09-02',
    end_date='2025-09-02',
    enable_confluence=False,
    verbose=True
)
r = bt.run()

# Show trade details
for t in bt.trades:
    print(f"\n{t.entry_time.strftime('%H:%M')} → {t.exit_time.strftime('%H:%M')}")
    print(f"  Entry: {t.entry_price:.2f}, Stop: {t.stop_price:.2f}")
    print(f"  Exit: {t.exit_price:.2f}, Reason: {t.exit_reason}")
    print(f"  volState: {t.vol_state}")

[Presets] Loaded GOOGL: SSL_base=20, WAE_sens=325, QQE_rsi1=9

  ORB BACKTESTER: GOOGL
  2025-09-02 to 2025-09-02

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching GOOGL (1-min bars, 180 days, extended)...
  ✓ Loading from cache (3.3 hours old)
✓ Loaded 87789 bars for GOOGL
Processing 1 trading days...

  ❌ LONG 2025-09-02 09:36→09:44: $-0.82 (-1.00R) - INITIAL STOP
  ❌ LONG 2025-09-02 09:56→10:14: $-0.64 (-1.00R) - INITIAL STOP

  BACKTEST SUMMARY: GOOGL
  2025-09-02 to 2025-09-02

  Total Trades:    2
  Total Skips:     0 (R:R below 1.5)
  Wins/Losses/BE:  0W / 2L / 0BE
  Win Rate:        0.0% (of decided trades)

  Total P/L:       $-1.46
  Total R:         -2.00R
  Avg R/Trade:     -1.000R

  Avg Win:         $0.00 (0.00R)
  Avg Loss:        $-0.73 (-1.00R)
  Profit Factor:   0.00
  Max Drawdown:    2.00R



TRADE DETAILS
2025-09-02 LONG  09:36->09:44  $209.28->$208.46  INITIAL STOP       -1.00R
2025-09-02 LONG  09:56->10:14  $209.22->$208.58  INITIAL STOP 

AttributeError: 'str' object has no attribute 'strftime'

In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import importlib
import orb_backtester
importlib.reload(orb_backtester)
from orb_backtester import ORBBacktester

bt = ORBBacktester(
    symbol='GOOGL',
    start_date='2025-09-02',
    end_date='2025-09-02',
    enable_confluence=False,
    verbose=True
)
r = bt.run()

# Check what fields are available
if bt.trades:
    print("\nTrade fields:", bt.trades[0].__dict__ if hasattr(bt.trades[0], '__dict__') else bt.trades[0])

[Presets] Loaded GOOGL: SSL_base=20, WAE_sens=325, QQE_rsi1=9

  ORB BACKTESTER: GOOGL
  2025-09-02 to 2025-09-02

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching GOOGL (1-min bars, 180 days, extended)...
  ✓ Loading from cache (3.3 hours old)
✓ Loaded 87789 bars for GOOGL
Processing 1 trading days...

  ❌ LONG 2025-09-02 09:36→09:44: $-0.82 (-1.00R) - INITIAL STOP
  ❌ LONG 2025-09-02 09:56→10:14: $-0.64 (-1.00R) - INITIAL STOP

  BACKTEST SUMMARY: GOOGL
  2025-09-02 to 2025-09-02

  Total Trades:    2
  Total Skips:     0 (R:R below 1.5)
  Wins/Losses/BE:  0W / 2L / 0BE
  Win Rate:        0.0% (of decided trades)

  Total P/L:       $-1.46
  Total R:         -2.00R
  Avg R/Trade:     -1.000R

  Avg Win:         $0.00 (0.00R)
  Avg Loss:        $-0.73 (-1.00R)
  Profit Factor:   0.00
  Max Drawdown:    2.00R



TRADE DETAILS
2025-09-02 LONG  09:36->09:44  $209.28->$208.46  INITIAL STOP       -1.00R
2025-09-02 LONG  09:56->10:14  $209.22->$208.58  INITIAL STOP 

In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import importlib
import orb_backtester
importlib.reload(orb_backtester)
from orb_backtester import ORBBacktester

bt = ORBBacktester(
    symbol='GOOGL',
    start_date='2025-09-02',
    end_date='2025-09-02',
    enable_confluence=False,
    verbose=False
)
bt.load_data()

# Check if atr_fast_all exists in the dataframe
day_df = bt.df[bt.df.index.date.astype(str) == '2025-09-02']
bar_936 = day_df[day_df.index.hour == 9][day_df.index.minute == 36].iloc[0]

print("═══ Checking vol_state columns in df ═══")
print(f"atr_fast_all present: {'atr_fast_all' in bt.df.columns}")
print(f"atr_fast_sma_all present: {'atr_fast_sma_all' in bt.df.columns}")

if 'atr_fast_all' in bt.df.columns:
    print(f"\natr_fast_all @ 09:36: {bar_936.get('atr_fast_all', 'N/A')}")
    print(f"atr_fast_sma_all @ 09:36: {bar_936.get('atr_fast_sma_all', 'N/A')}")
else:
    print("\n⚠️ atr_fast_all NOT in dataframe - the fix didn't apply!")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching GOOGL (1-min bars, 180 days, extended)...
  ✓ Loading from cache (3.3 hours old)


ValueError: Item wrong length 826 instead of 60.

In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')

# Force reimport
import importlib
import orb_backtester
importlib.reload(orb_backtester)

# Check if the new columns exist
from orb_backtester import ORBBacktester
bt = ORBBacktester(symbol='GOOGL', start_date='2025-09-02', end_date='2025-09-02', verbose=False)
bt.load_data()

print("Columns with 'atr_fast':", [c for c in bt.df.columns if 'atr_fast' in c])

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching GOOGL (1-min bars, 180 days, extended)...
  ✓ Loading from cache (3.3 hours old)
Columns with 'atr_fast': ['atr_fast_all', 'atr_fast_sma_all', 'atr_fast_rth', 'atr_fast_sma_rth']


In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import importlib
import orb_backtester
importlib.reload(orb_backtester)
from orb_backtester import ORBBacktester

bt = ORBBacktester(
    symbol='GOOGL',
    start_date='2025-09-02',
    end_date='2025-09-02',
    enable_confluence=False,
    verbose=False
)
bt.load_data()

day_df = bt.df[bt.df.index.date.astype(str) == '2025-09-02']

# Simulate what run() does - process bars in order
print("═══ Tracing vol_state during bar processing ═══")
for ts, bar in day_df.iterrows():
    bt._calc_vol_state(bar, ts)
    
    if ts.hour == 9 and ts.minute == 36:
        print(f"09:36: vol_factor={bt.vol.vol_factor:.2f}, vol_state={bt.vol.vol_state}")
        print(f"  atr_fast_all in bar: {bar.get('atr_fast_all', 'MISSING'):.4f}")
        print(f"  atr_fast_sma_all in bar: {bar.get('atr_fast_sma_all', 'MISSING'):.4f}")
        print(f"  baseline: {bt.vol.orb_session_baseline:.4f}")
        break

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching GOOGL (1-min bars, 180 days, extended)...
  ✓ Loading from cache (3.3 hours old)
═══ Tracing vol_state during bar processing ═══
09:36: vol_factor=1.49, vol_state=HIGH
  atr_fast_all in bar: 0.8025
  atr_fast_sma_all in bar: 0.9234
  baseline: 0.5641


In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import importlib
import orb_backtester
importlib.reload(orb_backtester)
from orb_backtester import ORBBacktester

bt = ORBBacktester(
    symbol='GOOGL',
    start_date='2025-09-02',
    end_date='2025-09-02',
    enable_confluence=False,
    verbose=False
)
bt.load_data()

# Manually run the day loop like run() does
day_df = bt.df[bt.df.index.date.astype(str) == '2025-09-02']

# Reset state like run() does at start of each day
bt.orb = bt.orb.__class__()  # Reset ORB state
bt.vol = bt.vol.__class__()  # Reset vol state

print("═══ Tracing vol_state through bar loop ═══")
for ts, bar in day_df.iterrows():
    bt._calc_vol_state(bar, ts)
    
    if ts.hour == 9 and ts.minute in [30, 36]:
        print(f"{ts.strftime('%H:%M')}: baseline={bt.vol.orb_session_baseline:.4f}, vol_factor={bt.vol.vol_factor:.2f}, vol_state={bt.vol.vol_state}")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching GOOGL (1-min bars, 180 days, extended)...
  ✓ Loading from cache (3.3 hours old)
═══ Tracing vol_state through bar loop ═══
09:30: baseline=0.3849, vol_factor=3.66, vol_state=EXTREME
09:36: baseline=0.5641, vol_factor=1.49, vol_state=HIGH


In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import importlib
import orb_backtester
importlib.reload(orb_backtester)
from orb_backtester import ORBBacktester
import numpy as np

bt = ORBBacktester(
    symbol='GOOGL',
    start_date='2025-09-02',
    end_date='2025-09-02',
    enable_confluence=False,
    verbose=False
)

# Call load_data like run() does
bt.load_data()

# Get day_df exactly like _process_day does
day_df = bt.df[bt.df.index.date.astype(str) == '2025-09-02'].copy()

# Reset session state exactly like _process_day does
bt.orb.reset_session()
bt.vol.orb_session_baseline = np.nan

print("═══ Simulating _process_day exactly ═══")
for i in range(len(day_df)):
    ts = day_df.index[i]
    bar = day_df.iloc[i]
    
    # Skip non-RTH like _process_day does
    if not bt._is_rth(ts):
        continue
    
    bt._calc_vol_state(bar, ts)
    
    if ts.hour == 9 and ts.minute in [30, 36]:
        print(f"{ts.strftime('%H:%M')}: vol_factor={bt.vol.vol_factor:.2f}, vol_state={bt.vol.vol_state}")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching GOOGL (1-min bars, 180 days, extended)...
  ✓ Loading from cache (3.4 hours old)
═══ Simulating _process_day exactly ═══
09:30: vol_factor=3.66, vol_state=EXTREME
09:36: vol_factor=1.49, vol_state=HIGH


In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import importlib
import orb_backtester
importlib.reload(orb_backtester)
from orb_backtester import ORBBacktester

bt = ORBBacktester(
    symbol='GOOGL',
    start_date='2025-09-02',
    end_date='2025-09-02',
    enable_confluence=False,
    verbose=True
)
r = bt.run()
```

I added debug statements at:
1. After `_calc_vol_state()` is called (at 09:30 and 09:36)
2. At LONG entry evaluation

This will show us exactly what vol_state is at each step. We should see:
```
DEBUG _calc_vol_state 09:30: vol_state=EXTREME, vol_factor=3.66
DEBUG _calc_vol_state 09:36: vol_state=HIGH, vol_factor=1.49
DEBUG LONG 09:36: vol_state=HIGH, ...

SyntaxError: leading zeros in decimal integer literals are not permitted; use an 0o prefix for octal integers (1443494492.py, line 19)

In [2]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import importlib
import orb_backtester
importlib.reload(orb_backtester)
from orb_backtester import ORBBacktester

bt = ORBBacktester(
    symbol='GOOGL',
    start_date='2025-09-02',
    end_date='2025-09-02',
    enable_confluence=False,
    verbose=True
)
r = bt.run()

[Presets] Loaded GOOGL: SSL_base=20, WAE_sens=325, QQE_rsi1=9

  ORB BACKTESTER: GOOGL
  2025-09-02 to 2025-09-02

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching GOOGL (1-min bars, 180 days, extended)...
  ✓ Loading from cache (3.5 hours old)
✓ Loaded 87789 bars for GOOGL
Processing 1 trading days...

DEBUG _calc_vol_state 09:36: vol_state=NORMAL, vol_factor=0.96
DEBUG LONG 09:36: vol_state=NORMAL, vol_factor=0.96, baseline=0.8775
  ❌ LONG 2025-09-02 09:36→09:44: $-0.82 (-1.00R) - INITIAL STOP
DEBUG LONG 09:56: vol_state=LOW, vol_factor=0.69, baseline=0.5580
  ❌ LONG 2025-09-02 09:56→10:14: $-0.64 (-1.00R) - INITIAL STOP

  BACKTEST SUMMARY: GOOGL
  2025-09-02 to 2025-09-02

  Total Trades:    2
  Total Skips:     0 (R:R below 1.5)
  Wins/Losses/BE:  0W / 2L / 0BE
  Win Rate:        0.0% (of decided trades)

  Total P/L:       $-1.46
  Total R:         -2.00R
  Avg R/Trade:     -1.000R

  Avg Win:         $0.00 (0.00R)
  Avg Loss:        $-0.73 (-1.00R)
  Pro

In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import importlib
import orb_backtester
importlib.reload(orb_backtester)
from orb_backtester import ORBBacktester

bt = ORBBacktester(
    symbol='GOOGL',
    start_date='2025-09-02',
    end_date='2025-09-02',
    enable_confluence=False,
    verbose=True
)
r = bt.run()

[Presets] Loaded GOOGL: SSL_base=20, WAE_sens=325, QQE_rsi1=9

  ORB BACKTESTER: GOOGL
  2025-09-02 to 2025-09-02

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching GOOGL (1-min bars, 180 days, extended)...
  ✓ Loading from cache (3.5 hours old)
✓ Loaded 87789 bars for GOOGL
Processing 1 trading days...

DEBUG BAR[0] 2025-09-02 04:00:00-04:00: is_rth=False, atr_fast_all=0.3182891423848624
DEBUG BAR[1] 2025-09-02 04:01:00-04:00: is_rth=False, atr_fast_all=0.27463131390788875
DEBUG BAR[2] 2025-09-02 04:02:00-04:00: is_rth=False, atr_fast_all=0.24770505112630828
DEBUG BAR[197] 2025-09-02 09:30:00-04:00: is_rth=True, atr_fast_all=1.3445878032197283
DEBUG BASELINE SEEDED: atr_fast_sma=0.8724, baseline=0.8724
DEBUG _calc_vol_state 09:36: vol_state=NORMAL, vol_factor=0.96
DEBUG LONG 09:36: vol_state=NORMAL, vol_factor=0.96, baseline=0.8775
  ❌ LONG 2025-09-02 09:36→09:44: $-0.82 (-1.00R) - INITIAL STOP
DEBUG LONG 09:56: vol_state=LOW, vol_factor=0.69, baseline=0.5580
 

In [2]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import importlib
import orb_backtester
importlib.reload(orb_backtester)
from orb_backtester import ORBBacktester

bt = ORBBacktester(
    symbol='GOOGL',
    start_date='2025-09-02',
    end_date='2025-09-02',
    enable_confluence=False,
    verbose=True
)
r = bt.run()



[Presets] Loaded GOOGL: SSL_base=20, WAE_sens=325, QQE_rsi1=9

  ORB BACKTESTER: GOOGL
  2025-09-02 to 2025-09-02

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching GOOGL (1-min bars, 180 days, extended)...
  ✓ Loading from cache (3.6 hours old)
✓ Loaded 87789 bars for GOOGL
Processing 1 trading days...

DEBUG BAR[0] 2025-09-02 04:00:00-04:00: is_rth=False, atr_fast_all=0.3182891423848624
DEBUG BAR[1] 2025-09-02 04:01:00-04:00: is_rth=False, atr_fast_all=0.27463131390788875
DEBUG BAR[2] 2025-09-02 04:02:00-04:00: is_rth=False, atr_fast_all=0.24770505112630828
DEBUG BAR[197] 2025-09-02 09:30:00-04:00: is_rth=True, atr_fast_all=1.3445878032197283
DEBUG BASELINE SEEDED: atr_fast_sma=0.8114, baseline=0.8114
DEBUG _calc_vol_state 09:36: vol_state=NORMAL, vol_factor=1.03
DEBUG LONG 09:36: vol_state=NORMAL, vol_factor=1.03, baseline=0.8175
  ❌ LONG 2025-09-02 09:36→09:44: $-0.82 (-1.00R) - INITIAL STOP
DEBUG LONG 09:56: vol_state=LOW, vol_factor=0.68, baseline=0.5689
 

In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')
import importlib
import orb_backtester
importlib.reload(orb_backtester)
from orb_backtester import ORBBacktester

bt = ORBBacktester(
    symbol='GOOGL',
    start_date='2025-09-02',
    end_date='2025-09-02',
    enable_confluence=False,
    verbose=True
)
r = bt.run()

[Presets] Loaded GOOGL: SSL_base=20, WAE_sens=325, QQE_rsi1=9

  ORB BACKTESTER: GOOGL
  2025-09-02 to 2025-09-02

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching GOOGL (1-min bars, 180 days, extended)...
  ✓ Loading from cache (3.6 hours old)
✓ Loaded 87789 bars for GOOGL
Processing 1 trading days...

DEBUG BAR[0] 2025-09-02 04:00:00-04:00: is_rth=False, atr_fast_all=0.3182891423848624
DEBUG BAR[1] 2025-09-02 04:01:00-04:00: is_rth=False, atr_fast_all=0.27463131390788875
DEBUG BAR[2] 2025-09-02 04:02:00-04:00: is_rth=False, atr_fast_all=0.24770505112630828
DEBUG BAR[197] 2025-09-02 09:30:00-04:00: is_rth=True, atr_fast_all=1.3445878032197283
DEBUG BASELINE SEEDED: atr_fast_sma=0.8114, baseline=0.8114
DEBUG _calc_vol_state 09:36: vol_state=NORMAL, vol_factor=1.03
DEBUG LONG 09:36: vol_state=NORMAL, vol_factor=1.03, baseline=0.8175
  ❌ LONG 2025-09-02 09:36→09:44: $-0.82 (-1.00R) - INITIAL STOP
DEBUG LONG 09:56: vol_state=LOW, vol_factor=0.68, baseline=0.5689
 

In [2]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')

# Force a fresh import
if 'orb_backtester' in sys.modules:
    del sys.modules['orb_backtester']

from orb_backtester import ORBBacktester

bt = ORBBacktester(
    symbol='GOOGL',
    start_date='2025-09-02',
    end_date='2025-09-02',
    enable_confluence=False,
    verbose=True
)
r = bt.run()


[Presets] Loaded GOOGL: SSL_base=20, WAE_sens=325, QQE_rsi1=9

  ORB BACKTESTER: GOOGL
  2025-09-02 to 2025-09-02

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching GOOGL (1-min bars, 180 days, extended)...
  ✓ Loading from cache (3.7 hours old)
✓ Loaded 87789 bars for GOOGL
Processing 1 trading days...

DEBUG BAR[0] 2025-09-02 04:00:00-04:00: is_rth=False, atr_fast_all=0.3182891423848624
DEBUG BAR[1] 2025-09-02 04:01:00-04:00: is_rth=False, atr_fast_all=0.27463131390788875
DEBUG BAR[2] 2025-09-02 04:02:00-04:00: is_rth=False, atr_fast_all=0.24770505112630828
DEBUG BAR[197] 2025-09-02 09:30:00-04:00: is_rth=True, atr_fast_all=1.3445878032197283
DEBUG _calc_vol_state 09:36: vol_state=NORMAL, vol_factor=0.96
DEBUG LONG 09:36: vol_state=NORMAL, vol_factor=0.96, baseline=0.8775
  ❌ LONG 2025-09-02 09:36→09:44: $-0.82 (-1.00R) - INITIAL STOP
DEBUG LONG 09:56: vol_state=LOW, vol_factor=0.69, baseline=0.5580
  ❌ LONG 2025-09-02 09:56→10:14: $-0.64 (-1.00R) - INITIAL ST

In [2]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')

# Force fresh import
if 'orb_backtester' in sys.modules:
    del sys.modules['orb_backtester']

from orb_backtester import ORBBacktester

bt = ORBBacktester(
    symbol='GOOGL',
    start_date='2025-09-02',
    end_date='2025-09-02',
    enable_confluence=False,
    verbose=True
)
r = bt.run()


[Presets] Loaded GOOGL: SSL_base=20, WAE_sens=325, QQE_rsi1=9

  ORB BACKTESTER: GOOGL
  2025-09-02 to 2025-09-02

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching GOOGL (1-min bars, 180 days, extended)...
  ✓ Loading from cache (3.7 hours old)
✓ Loaded 87789 bars for GOOGL
Processing 1 trading days...

DEBUG BAR[0] 2025-09-02 04:00:00-04:00: is_rth=False, atr_fast_all=0.3182891423848624
DEBUG BAR[1] 2025-09-02 04:01:00-04:00: is_rth=False, atr_fast_all=0.27463131390788875
DEBUG BAR[2] 2025-09-02 04:02:00-04:00: is_rth=False, atr_fast_all=0.24770505112630828
DEBUG BAR[197] 2025-09-02 09:30:00-04:00: is_rth=True, atr_fast_all=1.3445878032197283
DEBUG: ts_idx=14911, looking up prev bar
DEBUG: prev_ts=2025-09-02 09:29:00-04:00
DEBUG: seed_value=0.26398504168928016, prev_atr_fast=0.4919847540246613
DEBUG BASELINE SEEDED from prev bar 2025-09-02 09:29:00-04:00: atr_fast=0.4920, sma=0.2640
DEBUG _calc_vol_state 09:30: vol_state=EXTREME, vol_factor=5.34
DEBUG _calc_vo

In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')

if 'orb_backtester' in sys.modules:
    del sys.modules['orb_backtester']

from orb_backtester import ORBBacktester

bt = ORBBacktester(
    symbol='GOOGL',
    start_date='2025-09-02',
    end_date='2025-09-02',
    enable_confluence=False,
    verbose=True
)
r = bt.run()

[Presets] Loaded GOOGL: SSL_base=20, WAE_sens=325, QQE_rsi1=9

  ORB BACKTESTER: GOOGL
  2025-09-02 to 2025-09-02

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching GOOGL (1-min bars, 180 days, extended)...
  ✓ Loading from cache (3.8 hours old)
✓ Loaded 87789 bars for GOOGL
Processing 1 trading days...

  ❌ LONG 2025-09-02 09:36→09:47: $-1.13 (-1.00R) - INITIAL STOP
  ❌ LONG 2025-09-02 09:56→10:14: $-0.64 (-1.00R) - INITIAL STOP

  BACKTEST SUMMARY: GOOGL
  2025-09-02 to 2025-09-02

  Total Trades:    2
  Total Skips:     0 (R:R below 1.5)
  Wins/Losses/BE:  0W / 2L / 0BE
  Win Rate:        0.0% (of decided trades)

  Total P/L:       $-1.78
  Total R:         -2.00R
  Avg R/Trade:     -1.000R

  Avg Win:         $0.00 (0.00R)
  Avg Loss:        $-0.89 (-1.00R)
  Profit Factor:   0.00
  Max Drawdown:    2.00R



TRADE DETAILS
2025-09-02 LONG  09:36->09:47  $209.28->$208.15  INITIAL STOP       -1.00R
2025-09-02 LONG  09:56->10:14  $209.22->$208.58  INITIAL STOP 

In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')

# Force fresh import
if 'orb_backtester' in sys.modules:
    del sys.modules['orb_backtester']

from orb_backtester import ORBBacktester

# Run backtest
bt = ORBBacktester(
    symbol='GOOGL',
    start_date='2025-08-27',
    end_date='2026-01-27',
    enable_confluence=False,
    verbose=False
)
r = bt.run()

# Generate audit report
print("=" * 130)
print("GOOGL TRADE AUDIT REPORT")
print("Period: August 27, 2025 - January 27, 2026")
print("=" * 130)
print()
print(f"{'Date':<12} {'Dir':<6} {'Entry':<6} {'Exit':<6} {'Entry$':>10} {'Exit$':>10} {'Stop$':>10} {'Stop Type':<10} {'Exit Reason':<20} {'R':>7}")
print("-" * 130)

for t in bt.trades:
    print(f"{t.date:<12} {t.direction:<6} {t.entry_time:<6} {t.exit_time:<6} {t.entry_price:>10.2f} {t.exit_price:>10.2f} {t.initial_stop:>10.2f} {t.stop_type:<10} {t.exit_reason:<20} {t.r_multiple:>+7.2f}")

print("-" * 130)
print(f"Total Trades: {len(bt.trades)}")
print(f"Total R: {sum(t.r_multiple for t in bt.trades):+.2f}")
wins = sum(1 for t in bt.trades if t.r_multiple > 0)
losses = sum(1 for t in bt.trades if t.r_multiple < 0)
be = sum(1 for t in bt.trades if t.r_multiple == 0)
print(f"Wins: {wins}, Losses: {losses}, BE: {be}")
print(f"Win Rate: {wins / len(bt.trades) * 100:.1f}%" if bt.trades else "N/A")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching GOOGL (1-min bars, 180 days, extended)...
  ✓ Loading from cache (4.0 hours old)
GOOGL TRADE AUDIT REPORT
Period: August 27, 2025 - January 27, 2026

Date         Dir    Entry  Exit       Entry$      Exit$      Stop$ Stop Type  Exit Reason                R
----------------------------------------------------------------------------------------------------------------------------------
2025-08-27   LONG   09:39  10:03      207.12     207.80     206.48 SWING      EMA EXIT               +1.07
2025-08-28   LONG   09:52  09:56      209.47     209.47     208.91 SWING      BREAK-EVEN STOP        +0.00
2025-08-29   LONG   09:38  09:56      212.06     211.43     211.43 ATR        INITIAL STOP           -1.00
2025-08-29   LONG   10:23  10:26      212.20     211.57     211.57 ATR        INITIAL STOP           -1.00
2025-09-02   LONG   09:36  09:47      209.28     208.15     208.15 ATR        INITIAL STOP           -1.00
202

In [1]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')

if 'orb_backtester' in sys.modules:
    del sys.modules['orb_backtester']

from orb_backtester import ORBBacktester

bt = ORBBacktester(
    symbol='AMD',
    start_date='2025-08-29',
    end_date='2026-01-27',
    enable_confluence=False,
    verbose=False
)
r = bt.run()

# Trades Report
print("=" * 130)
print("AMD TRADE AUDIT REPORT")
print("Period: August 29, 2025 - January 27, 2026")
print("=" * 130)
print()
print(f"{'Date':<12} {'Dir':<6} {'Entry':<6} {'Exit':<6} {'Entry$':>10} {'Exit$':>10} {'Stop$':>10} {'Stop Type':<10} {'Exit Reason':<20} {'R':>7}")
print("-" * 130)

for t in bt.trades:
    print(f"{t.date:<12} {t.direction:<6} {t.entry_time:<6} {t.exit_time:<6} {t.entry_price:>10.2f} {t.exit_price:>10.2f} {t.initial_stop:>10.2f} {t.stop_type:<10} {t.exit_reason:<20} {t.r_multiple:>+7.2f}")

print("-" * 130)
print(f"Total Trades: {len(bt.trades)}")
print(f"Total R: {sum(t.r_multiple for t in bt.trades):+.2f}")
wins = sum(1 for t in bt.trades if t.r_multiple > 0)
losses = sum(1 for t in bt.trades if t.r_multiple < 0)
be = sum(1 for t in bt.trades if t.r_multiple == 0)
print(f"Wins: {wins}, Losses: {losses}, BE: {be}")
print(f"Win Rate: {wins / len(bt.trades) * 100:.1f}%" if bt.trades else "N/A")

# Skips Report
print("\n" + "=" * 130)
print("AMD SKIPPED TRADES (R:R below minimum)")
print("=" * 130)
print()
print(f"{'Date':<12} {'Dir':<6} {'Time':<6} {'Entry$':>10} {'Stop$':>10} {'Stop Type':<10} {'Achievable RR':>14} {'Min RR':>8} {'Reason':<30}")
print("-" * 130)

for s in bt.skips:
    print(f"{s.date:<12} {s.direction:<6} {s.time:<6} {s.entry_price:>10.2f} {s.stop_price:>10.2f} {s.stop_type:<10} {s.achievable_rr:>14.2f} {s.min_rr:>8.1f} {s.reason:<30}")

print("-" * 130)
print(f"Total Skips: {len(bt.skips)}")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ⚠ Cache stale (6 hours), refreshing...
  ✓ Fetched 101722 bars, cached for next time
AMD TRADE AUDIT REPORT
Period: August 29, 2025 - January 27, 2026

Date         Dir    Entry  Exit       Entry$      Exit$      Stop$ Stop Type  Exit Reason                R
----------------------------------------------------------------------------------------------------------------------------------
2025-08-29   SHORT  09:38  09:51      166.07     165.03     167.10 ATR        EMA EXIT               +1.01
2025-09-03   LONG   09:41  09:55      162.09     162.63     161.31 VWAP       EMA EXIT               +0.70
2025-09-04   SHORT  09:38  09:44      158.54     158.54     159.72 ATR        BREAK-EVEN STOP        +0.00
2025-09-04   SHORT  09:48  09:51      158.24     159.10     159.10 VWAP       INITIAL STOP           -1.00
2025-09-04   SHORT  09:55  09:57      158.84     159.44     159.44